# Identifiability-Aware Parameter Estimation and Frequency-Band Assessment of Equivalent-Circuit Models: Application to In Vivo Tissue Bioimpedance
**Complete reproducibility notebook** (authors withheld for double-blind review)

The notebook keeps the item numbering of an earlier draft; `README.md` maps every result to the current manuscript. The additional analyses of the current version are in `src/revision/` (see its README).

This single notebook contains **all code** used in the manuscript and Online Resources 1–2:

| Part | Content | Manuscript items | Time (1 CPU core) |
|---|---|---|---|
| **A** | Original protocol: data QC, 11-model ECM screening, 8-algorithm benchmark (M10, M11), extended budget, statistics, profile likelihood, $f_c$ band coverage, synthetic recovery, M12 stress test, Kramers–Kronig check, repeatability (ICC), figures | Tables 2, 5–6; Figs. 2, 4–7; Figs. S1–S7, S10; Tables S1–S3 | 60–75 min |
| **B** | Variable projection (VarPro): benchmark, merged statistics, global VarPro profiles, same-machine timing | Tables 3–4; Fig. 3; Fig. S8 | 75–90 min |
| **C** | New analyses: band-design simulation, sensitivity (leave-one-subset-out), nested M13 model, feature stability, workflow and decision figures | Figs. 1, 8–9; Tables S4–S7 | 40 min |
| **D** | `eisvarpro` open-source package (fit, KK check, profile likelihood, reporting decision) and demo | Online Resource 2 | 1 min |

**Run modes.** `RUN_PART_A = False` (default) uses the original-protocol results shipped in `results/tables` (identical to the run reported in the manuscript) and starts directly with Parts B–D. Set `RUN_PART_A = True` to recompute everything from the raw data `PreparedData.zip`.

**Data.** Falagario C, De Momi E, Mattos LS. In vivo tissue bioimpedance dataset. *Sci Data* (2026). doi:10.1038/s41597-026-07652-y (`PreparedData.zip`, SHA-256 `3d255d1e…`).

**Colab.** Upload `eis-identifiability-code.zip`, run cell 0 (it unzips and changes into the folder). Locally: open the notebook inside the unzipped folder. Long steps cache finished spectra in `results/cache_*`, so a cell can be re-run after a disconnect.

In [ ]:
# 0. Environment (Colab: unzips the package; local Jupyter: just uses the current folder)
import os, sys, zipfile, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('eis-identifiability-code/PreparedData.zip') and not os.path.exists('PreparedData.zip'):
    USE_DRIVE = True                                      # keep work on Google Drive (survives disconnects)
    if USE_DRIVE:
        from google.colab import drive; drive.mount('/content/drive')
        os.makedirs('/content/drive/MyDrive/EIS_VarPro_run', exist_ok=True); os.chdir('/content/drive/MyDrive/EIS_VarPro_run')
    if not glob.glob('eis-identifiability-code.zip'):
        from google.colab import files; files.upload()    # choose eis-identifiability-code.zip
    zipfile.ZipFile('eis-identifiability-code.zip').extractall('.')
if os.path.exists('eis-identifiability-code/PreparedData.zip'): os.chdir('eis-identifiability-code')
print('Working folder:', os.getcwd()); assert os.path.exists('PreparedData.zip')

In [ ]:
# Run control
RUN_PART_A = False        # True: recompute the original protocol from raw data (60-75 min)
RUNS_VP    = 25           # runs per spectrum and algorithm in Part B (manuscript: 25)
os.makedirs('src', exist_ok=True); os.makedirs('results/tables', exist_ok=True); os.makedirs('results/figures_vp', exist_ok=True)

from IPython.core.magic import register_cell_magic
@register_cell_magic
def run_if(line, cell):
    """%%run_if FLAG : execute the cell only when FLAG is true (used for Part A)."""
    if eval(line, get_ipython().user_ns): get_ipython().run_cell(cell)
    else: print(f'skipped ({line} is False): using the shipped results in results/tables')

---
# Part A — Original protocol (data, ECM screening, 8-algorithm benchmark, identifiability)
Code identical to the original analysis. Skipped unless `RUN_PART_A = True`.

## 1. Setup and configuration

In [ ]:
%%run_if RUN_PART_A
# Optional (Colab): !pip install -q numpy scipy pandas matplotlib
import os, sys, re, json, math, time, zipfile, hashlib, platform, warnings, pickle, random
from pathlib import Path
import numpy as np, pandas as pd
from scipy.optimize import least_squares
from scipy.stats import friedmanchisquare, wilcoxon, rankdata, spearmanr
warnings.filterwarnings('ignore')

QUICK = False                         # True = fast smoke test
HERE = Path('.').resolve()
RAW_ZIP = Path(os.environ.get('EIS_RAW_ZIP', HERE/'PreparedData.zip'))
assert RAW_ZIP.exists(), f'PreparedData.zip not found at {RAW_ZIP}'
print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__)

## 2. Core library
Pure Python/NumPy implementation (no numba). Hyperparameters are frozen and documented in the docstring below and in Table 2 of the manuscript.

In [ ]:
%%run_if RUN_PART_A
"""
eis_core.py - Core library for the revised ASHO-LSHADE tissue-EIS study.

Pure Python/NumPy (no numba required). Algorithms are line-by-line ports of the
frozen numba implementation in the original notebook, generalised to any
dimension so that they can be applied to both the 5-parameter Cole+L model
(M10) and the identifiable 4-parameter R-CPE-L model (M11).

Hyperparameters (identical for every model and every spectrum):
  L-SHADE : N_init=20, N_min=4, H=6, p=0.11, |A|=NP, M_F=M_CR=0.5 initially,
            current-to-pbest/1/bin, linear population-size reduction.
  Trigger : stagnation >= max(4, round(12-7*progress)) generations AND
            normalised diversity < 0.20+0.20*progress; only the poorer half
            of the population is eligible.
  SAHO    : fixed rescue probability 0.25.
  ASHO    : p_HO(0)=0.25, p_min=0.05, p_max=0.45, lambda=0.25.
  Restart : same trigger, uniform re-sampling with probability 0.25.
"""

POP0, POPMIN, H, PBEST = 20, 4, 6, 0.11

# ------------------------------------------------------------------ models
MODELS = {
    # name: (parameter names, lower bounds, upper bounds)
    'M10_ColeL': (['logRinf', 'logdR', 'logtau', 'alpha', 'logL'],
                  np.array([-3., -3., -7., 0.30, -9.]), np.array([7., 8., 1., 1.0, 0.])),
    'M11_RCPEL': (['logRinf', 'logQ', 'alpha', 'logL'],
                  np.array([-3., -12., 0.30, -9.]), np.array([7., 0., 1.0, 0.])),
    # higher-dimensional stress test: double Cole + series inductance (8 parameters)
    'M12_DoubleColeL': (['logRinf', 'logdR1', 'logtau1', 'logdR2', 'logtau2', 'alpha1', 'alpha2', 'logL'],
                  np.array([-3., -3., -7., -3., -7., 0.30, 0.30, -9.]), np.array([7., 8., 1., 8., 1., 1.0, 1.0, 0.])),
}

def predict(model, x, w):
    x = np.asarray(x, float)
    if model == 'M10_ColeL':
        Rinf, dR, tau, L = 10**x[0], 10**x[1], 10**x[2], 10**x[4]; a = x[3]
        return Rinf + dR/(1 + (1j*w*tau)**a) + 1j*w*L
    if model == 'M11_RCPEL':
        Rinf, Q, L = 10**x[0], 10**x[1], 10**x[3]; a = x[2]
        return Rinf + 1/(Q*(1j*w)**a) + 1j*w*L
    if model == 'M12_DoubleColeL':
        Rinf, d1, t1, d2, t2, L = 10**x[0], 10**x[1], 10**x[2], 10**x[3], 10**x[4], 10**x[7]
        return Rinf + d1/(1 + (1j*w*t1)**x[5]) + d2/(1 + (1j*w*t2)**x[6]) + 1j*w*L
    raise KeyError(model)

def residual(model, x, w, z):
    e = (predict(model, x, w) - z)/np.maximum(np.abs(z), 1.0)
    return np.r_[e.real, e.imag]

def make_objective(model, w, z):
    """J(theta) = (1/N) sum_k |Zhat_k - Z_k|^2/|Z_k|^2 ; NRMSE = sqrt(J)."""
    sc = np.maximum(np.abs(z), 1.0); N = len(w)
    if model == 'M10_ColeL':
        jw = 1j*w
        def f(x):
            zh = 10**x[0] + 10**x[1]/(1 + (jw*10**x[2])**x[3]) + jw*10**x[4]
            e = np.abs(zh - z)/sc
            return float(np.dot(e, e)/N)
    elif model == 'M11_RCPEL':
        jw = 1j*w
        def f(x):
            zh = 10**x[0] + 1/(10**x[1]*jw**x[2]) + jw*10**x[3]
            e = np.abs(zh - z)/sc
            return float(np.dot(e, e)/N)
    else:
        def f(x):
            e = np.abs(predict(model, x, w) - z)/sc
            return float(np.dot(e, e)/N)
    return f

# ------------------------------------------------------------ utilities
class Tracker:
    """Counts NFE and records best-so-far objective at checkpoints."""
    def __init__(self, f, budget, checkpoints):
        self.f, self.budget, self.nfe = f, budget, 0
        self.best = math.inf; self.xbest = None
        self.cps = sorted(c for c in checkpoints if c <= budget); self.ci = 0
        self.trace = {}
    def __call__(self, x):
        v = self.f(x); self.nfe += 1
        if v < self.best: self.best = v; self.xbest = list(x)
        while self.ci < len(self.cps) and self.nfe >= self.cps[self.ci]:
            self.trace[self.cps[self.ci]] = self.best; self.ci += 1
        return v

def _clip(u, lo, hi):
    for d in range(len(u)):
        if u[d] < lo[d]: u[d] = lo[d]
        elif u[d] > hi[d]: u[d] = hi[d]
    return u

def _levy(rnd, beta=1.5):
    sigma = (math.gamma(1+beta)*math.sin(math.pi*beta/2) /
             (math.gamma((1+beta)/2)*beta*2**((beta-1)/2)))**(1/beta)
    return rnd.gauss(0, 1)*sigma/(abs(rnd.gauss(0, 1))**(1/beta) + 1e-12)

def _ho_rescue(rnd, x, best, progress, lo, hi):
    D = len(x); u = list(x)
    if rnd.random() < 0.5:   # HO river/pond position update (dominant-hippo guidance)
        I1 = 1 + rnd.randint(0, 1); y = rnd.random()
        for d in range(D): u[d] = x[d] + y*(best[d] - I1*x[d])
    else:                    # HO defence/escape-inspired heavy-tailed move
        lev = _levy(rnd); shrink = max(0.03, 0.35*(1.0 - progress))
        for d in range(D):
            u[d] = x[d] + rnd.random()*lev*shrink*(hi[d]-lo[d]) + 0.25*rnd.random()*(best[d]-x[d])
    return _clip(u, lo, hi)

def _init(rnd, lo, hi, n):
    D = len(lo)
    return [[lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)] for _ in range(n)]

# ------------------------------------------------------- L-SHADE family
def lshade_family(f, lo, hi, budget, seed, mode, checkpoints=()):
    """mode: 'lshade' | 'restart' | 'saho' | 'asho'"""
    rnd = random.Random(seed); D = len(lo); lo = list(lo); hi = list(hi)
    T = Tracker(f, budget, checkpoints)
    NP = POP0; X = _init(rnd, lo, hi, NP); fx = [T(x) for x in X]
    MF = [0.5]*H; MCR = [0.5]*H; mem = 0; archive = []
    best_prev = min(fx); stagn = 0; maxdiv = 1e-12
    rt = rs = 0; p_ho = 0.25; pmin, pmax, lam = 0.05, 0.45, 0.25; asum = 0.0; an = 0
    while T.nfe < budget and NP >= POPMIN:
        cen = [sum(X[i][d] for i in range(NP))/NP for d in range(D)]
        div = sum(math.sqrt(sum((X[i][d]-cen[d])**2 for d in range(D))) for i in range(NP))/NP
        maxdiv = max(maxdiv, div); dnorm = div/maxdiv; progress = T.nfe/budget
        stag_thr = max(4, int(round(12 - 7*progress)))
        active = mode != 'lshade' and stagn >= stag_thr and dnorm < (0.20 + 0.20*progress)
        prob = 0.25 if mode in ('saho', 'restart') else p_ho
        order = sorted(range(NP), key=fx.__getitem__); pnum = max(2, int(math.ceil(PBEST*NP)))
        Xold = [list(x) for x in X]; fold = list(fx); ib = order[0]
        sF = []; sCR = []; simp = []; arch_add = []; grt = grs = 0
        for i in range(NP):
            if T.nfe >= budget: break
            k = rnd.randrange(H)
            Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            while Fi <= 0: Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            Fi = min(1.0, Fi); CRi = min(1.0, max(0.0, rnd.gauss(MCR[k], 0.1)))
            special = active and i >= max(1, NP//2) and rnd.random() < prob
            if special and mode in ('saho', 'asho'):
                rt += 1; grt += 1; u = _ho_rescue(rnd, Xold[i], Xold[ib], progress, lo, hi)
            elif special and mode == 'restart':
                rt += 1; u = [lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)]
            else:
                special = False
                pb = order[rnd.randrange(pnum)]
                while True:
                    r1 = rnd.randrange(NP)
                    if r1 != i and r1 != pb: break
                tot = NP + len(archive)
                while True:
                    r2 = rnd.randrange(tot)
                    if r2 >= NP or (r2 != i and r2 != pb and r2 != r1): break
                xr2 = Xold[r2] if r2 < NP else archive[r2-NP]
                xi = Xold[i]; xp = Xold[pb]; x1 = Xold[r1]
                v = _clip([xi[d] + Fi*(xp[d]-xi[d]) + Fi*(x1[d]-xr2[d]) for d in range(D)], lo, hi)
                jr = rnd.randrange(D)
                u = [v[d] if (rnd.random() < CRi or d == jr) else xi[d] for d in range(D)]
            fu = T(u)
            if fu < fold[i]:
                X[i] = u; fx[i] = fu
                if special:
                    rs += 1; grs += 1
                else:
                    sF.append(Fi); sCR.append(CRi); simp.append(fold[i]-fu); arch_add.append(Xold[i])
        if mode == 'asho' and grt > 0:
            p_ho = (1-lam)*p_ho + lam*(pmin + (pmax-pmin)*grs/grt)
            p_ho = min(pmax, max(pmin, p_ho)); asum += p_ho; an += 1
        if arch_add:
            archive.extend(arch_add)
            if len(archive) > NP: archive = rnd.sample(archive, NP)
        if sF:
            tot = sum(simp); wts = [s/tot for s in simp]
            den = sum(wt*F for wt, F in zip(wts, sF))
            if den > 0: MF[mem] = sum(wt*F*F for wt, F in zip(wts, sF))/den
            MCR[mem] = sum(wt*c for wt, c in zip(wts, sCR)); mem = (mem+1) % H
        cur = min(fx)
        if cur < best_prev - 1e-14: best_prev = cur; stagn = 0
        else: stagn += 1
        if T.nfe < budget:
            tgt = int(round(POPMIN + (POP0-POPMIN)*(budget-T.nfe)/budget)); tgt = max(POPMIN, min(NP, tgt))
            if tgt < NP:
                ids = sorted(range(NP), key=fx.__getitem__)[:tgt]
                X = [X[j] for j in ids]; fx = [fx[j] for j in ids]; NP = tgt
                if len(archive) > NP: archive = rnd.sample(archive, NP)
    order = sorted(range(NP), key=fx.__getitem__)
    extra = {'final_pop': [X[j] for j in order], 'rescue_trials': rt, 'rescue_success': rs,
             'mean_pHO': (asum/an if an else p_ho) if mode == 'asho' else np.nan,
             'final_pHO': p_ho if mode == 'asho' else np.nan}
    return np.array(T.xbest), T.best, T.trace, extra

# ------------------------------------------------------------------- HO
def ho(f, lo, hi, budget, seed, checkpoints=()):
    """Compact NFE-fair HO (Amiri et al., 2024): position update, defence, escape."""
    rnd = random.Random(seed); D = len(lo); lo = list(lo); hi = list(hi)
    T = Tracker(f, budget, checkpoints)
    N = POP0; X = _init(rnd, lo, hi, N); fx = [T(x) for x in X]; it = 1
    maxit = max(1, (budget-N)//N)
    while T.nfe < budget:
        best = list(X[min(range(N), key=fx.__getitem__)]); half = max(1, N//2); Tt = math.exp(-it/maxit)
        for i in range(half):
            if T.nfe >= budget: break
            I1 = 1 + rnd.randint(0, 1); y = rnd.random()
            u = _clip([X[i][d] + y*(best[d]-I1*X[i][d]) for d in range(D)], lo, hi)
            fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        for i in range(half, N):
            if T.nfe >= budget: break
            pred = [lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)]; fp = T(pred)
            if T.nfe >= budget: break
            lev = _levy(rnd); u = list(X[i])
            for d in range(D):
                dist = abs(pred[d]-X[i][d]) + 1e-9; step = 0.05*lev*(hi[d]-lo[d])/dist
                u[d] = pred[d] + step if fp < fx[i] else X[i][d] + rnd.random()*(best[d]-X[i][d]) + step*rnd.random()
            u = _clip(u, lo, hi); fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        for _ in range(max(1, N//4)):
            if T.nfe >= budget: break
            i = rnd.randrange(N); shrink = max(0.02, 0.25*Tt)
            u = _clip([X[i][d] + (2*rnd.random()-1)*shrink*(hi[d]-lo[d]) + 0.2*rnd.random()*(best[d]-X[i][d])
                       for d in range(D)], lo, hi)
            fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        it += 1
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': 0, 'rescue_success': 0, 'mean_pHO': np.nan, 'final_pHO': np.nan}

# ----------------------------------------- multistart trust-region baseline
class _Budget(Exception): pass

def ms_trf(model, w, z, lo, hi, budget, seed, checkpoints=()):
    """Multistart bounded nonlinear least squares (trust-region reflective,
    2-point Jacobian). Every residual evaluation, including those used for the
    finite-difference Jacobian, is counted against the same NFE budget."""
    rng = np.random.default_rng(seed); f = make_objective(model, w, z)
    T = Tracker(f, budget, checkpoints)
    def res(x):
        if T.nfe >= budget: raise _Budget
        T(x); return residual(model, x, w, z)
    while T.nfe < budget:
        try:
            least_squares(res, rng.uniform(lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget:
            break
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': 0, 'rescue_success': 0, 'mean_pHO': np.nan, 'final_pHO': np.nan}

def hybrid_trf(model, w, z, lo, hi, budget, seed, mode, checkpoints=(), frac_global=0.5):
    """Memetic variant: L-SHADE-family global stage for frac_global*budget NFE,
    then bounded TRF refinement started from the best and the remaining final
    population members (then uniform random points) until the SAME total NFE
    budget is exhausted."""
    f = make_objective(model, w, z); b1 = int(frac_global*budget)
    x1, f1, tr1, ex = lshade_family(f, lo, hi, b1, seed, mode, checkpoints)
    T = Tracker(f, budget, checkpoints); T.nfe = b1; T.best = f1; T.xbest = list(x1)
    while T.ci < len(T.cps) and T.cps[T.ci] <= b1: T.ci += 1
    rng = np.random.default_rng(seed)
    starts = [np.asarray(p, float) for p in ex.pop('final_pop')]
    def res(x):
        if T.nfe >= budget: raise _Budget
        T(x); return residual(model, x, w, z)
    k = 0
    while T.nfe < budget:
        x0 = starts[k] if k < len(starts) else rng.uniform(lo, hi); k += 1
        try:
            least_squares(res, np.clip(x0, lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget:
            break
    trace = dict(tr1); trace.update(T.trace)
    return np.array(T.xbest), T.best, trace, ex

ALGORITHMS = ['HO', 'MS-TRF', 'L-SHADE', 'L-SHADE-Restart', 'SAHO-LSHADE', 'ASHO-LSHADE', 'L-SHADE+TRF', 'ASHO-LSHADE+TRF']

def run_algorithm(name, model, w, z, budget, seed, checkpoints=()):
    _, lo, hi = MODELS[model]; f = make_objective(model, w, z)
    t0 = time.perf_counter()
    if name == 'HO': out = ho(f, lo, hi, budget, seed, checkpoints)
    elif name == 'MS-TRF': out = ms_trf(model, w, z, lo, hi, budget, seed, checkpoints)
    elif name.endswith('+TRF'):
        out = hybrid_trf(model, w, z, lo, hi, budget, seed, 'asho' if name.startswith('ASHO') else 'lshade', checkpoints)
    else:
        mode = {'L-SHADE': 'lshade', 'L-SHADE-Restart': 'restart', 'SAHO-LSHADE': 'saho', 'ASHO-LSHADE': 'asho'}[name]
        out = lshade_family(f, lo, hi, budget, seed, mode, checkpoints)
    out[3].pop('final_pop', None)
    return (*out, time.perf_counter()-t0)

# --------------------------------------------------------- identifiability
def jacobian_svd(model, x, w, z):
    _, lo, hi = MODELS[model]; x = np.asarray(x, float); r0 = residual(model, x, w, z)
    J = np.empty((len(r0), len(x)))
    for d in range(len(x)):
        h = 1e-5*max(1, abs(x[d])); xp = x.copy(); xm = x.copy()
        xp[d] = min(hi[d], xp[d]+h); xm[d] = max(lo[d], xm[d]-h)
        J[:, d] = (residual(model, xp, w, z) - residual(model, xm, w, z))/max(xp[d]-xm[d], 1e-15)
    s = np.linalg.svd(J, compute_uv=False)
    return s[0]/max(s[-1], 1e-300), s[-1]

def best_fit(model, w, z, x0s, rng, extra_starts=20):
    _, lo, hi = MODELS[model]; best = None
    starts = [np.clip(np.asarray(x, float), lo, hi) for x in x0s] + [rng.uniform(lo, hi) for _ in range(extra_starts)]
    for x0 in starts:
        r = least_squares(lambda x: residual(model, x, w, z), x0, bounds=(lo, hi), max_nfev=5000,
                          xtol=1e-12, ftol=1e-12, gtol=1e-12)
        if best is None or r.cost < best.cost: best = r
    return best.x, 2*best.cost

def profile_likelihood(model, w, z, xhat, pidx, ngrid=41, rng=None):
    """Profile of parameter pidx (others re-optimised). Gaussian likelihood with
    unknown variance: LR(theta) = n*log(RSS(theta)/RSS_min); 95% CI where LR<=3.84."""
    names, lo, hi = MODELS[model]; n = 2*len(w)
    free = [d for d in range(len(lo)) if d != pidx]
    rss_min = float(np.sum(residual(model, xhat, w, z)**2))
    span = 0.05*(hi[pidx]-lo[pidx])
    grid = np.unique(np.clip(np.r_[np.linspace(lo[pidx], hi[pidx], ngrid),
                                   xhat[pidx] + np.linspace(-span, span, 21), xhat[pidx]], lo[pidx], hi[pidx]))
    prof = []
    for g in grid:
        best = None
        starts = [xhat[free]] + [rng.uniform(lo[free], hi[free]) for _ in range(3)]
        for x0 in starts:
            def rf(xf, g=g):
                x = np.array(xhat, float); x[free] = xf; x[pidx] = g; return residual(model, x, w, z)
            r = least_squares(rf, np.clip(x0, lo[free], hi[free]), bounds=(lo[free], hi[free]), max_nfev=3000)
            if best is None or r.cost < best: best = r.cost
        prof.append(2*best)
    prof = np.minimum(np.array(prof), np.inf); rss_min = min(rss_min, prof.min())
    lr = n*np.log(prof/rss_min); inside = grid[lr <= 3.84]
    i0 = int(np.argmin(lr)); thr = 3.84
    def cross(j_in, j_out):   # linear interpolation of the LR=3.84 crossing
        a, b = lr[j_in], lr[j_out]
        return grid[j_in] + (thr-a)*(grid[j_out]-grid[j_in])/(b-a) if b != a else grid[j_in]
    j = i0
    while j > 0 and lr[j-1] <= thr: j -= 1
    lo_ci = cross(j, j-1) if j > 0 else grid[0]
    j = i0
    while j < len(grid)-1 and lr[j+1] <= thr: j += 1
    hi_ci = cross(j, j+1) if j < len(grid)-1 else grid[-1]
    # identifiable (practically) if the 95% profile CI closes before both search bounds
    bounded = (lr[0] > 3.84) and (lr[-1] > 3.84)
    return grid, lr, lo_ci, hi_ci, bounded

# ------------------------------------------------------------------------------
# ASHO-LR: Adaptive Stagnation-triggered, HO-seeded Local Refinement (revision 3)
# ------------------------------------------------------------------------------
class _LocalCap(Exception): pass

def asho_lr(model, w, z, lo, hi, budget, seed, checkpoints=(), cap_per_dim=40, warmup_gens=2, cap_max_per_dim=160, adapt_cap=True):
    """L-SHADE backbone. When the best objective stagnates (same progress-dependent
    threshold as ASHO, no diversity condition), a bounded TRF refinement is launched.
    The first refinement starts from the population best; subsequent ones start from
    an HO-rescue move applied to a poorer-half individual, so HO proposes a new basin
    and TRF polishes it. Each refinement is capped at cap_per_dim*D evaluations
    (doubled after a refinement improves the global best, halved otherwise, within
    [10D, 160D]). The refined point replaces the worst individual if it is better.
    All evaluations (population, HO, TRF incl. Jacobians) share one NFE budget."""
    rnd = random.Random(seed); D = len(lo); lo_l = list(lo); hi_l = list(hi)
    f = make_objective(model, w, z); T = Tracker(f, budget, checkpoints)
    NP = POP0; X = _init(rnd, lo_l, hi_l, NP); fx = [T(x) for x in X]
    MF = [0.5]*H; MCR = [0.5]*H; mem = 0; archive = []
    best_prev = min(fx); stagn = 0; gen = 0; nref = 0; cap = cap_per_dim*D
    ref_trials = ref_success = 0; cap_max = cap_max_per_dim*D
    def refine(x0, cap_):
        start = T.nfe; best_seen = [math.inf, None]
        def res(x):
            if T.nfe >= budget: raise _Budget
            if T.nfe - start >= cap_: raise _LocalCap
            v = T(x)
            if v < best_seen[0]: best_seen[0] = v; best_seen[1] = list(x)
            return residual(model, x, w, z)
        try:
            least_squares(res, np.clip(np.asarray(x0, float), lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=100000)
        except (_Budget, _LocalCap):
            pass
        return best_seen
    while T.nfe < budget and NP >= POPMIN:
        progress = T.nfe/budget; stag_thr = max(4, int(round(12 - 7*progress)))
        if gen >= warmup_gens and stagn >= stag_thr:
            ib = min(range(NP), key=fx.__getitem__)
            if nref == 0: x0 = X[ib]
            else:
                i = rnd.randrange(max(1, NP//2), NP) if NP > 1 else 0
                order = sorted(range(NP), key=fx.__getitem__)
                x0 = _ho_rescue(rnd, X[order[i]], X[ib], progress, lo_l, hi_l)
            gb = T.best; v, xr = refine(x0, cap); nref += 1; ref_trials += 1
            if xr is not None:
                iw = max(range(NP), key=fx.__getitem__)
                if v < fx[iw]: X[iw] = xr; fx[iw] = v
            if T.best < gb*(1 - 1e-9): ref_success += 1; cap = min(cap_max, cap*2)
            else: cap = max(10*D, cap//2) if adapt_cap else cap
            stagn = 0; best_prev = min(fx)
            if T.nfe >= budget: break
        order = sorted(range(NP), key=fx.__getitem__); pnum = max(2, int(math.ceil(PBEST*NP)))
        Xold = [list(x) for x in X]; fold = list(fx)
        sF = []; sCR = []; simp = []; arch_add = []
        for i in range(NP):
            if T.nfe >= budget: break
            k = rnd.randrange(H)
            Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            while Fi <= 0: Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            Fi = min(1.0, Fi); CRi = min(1.0, max(0.0, rnd.gauss(MCR[k], 0.1)))
            pb = order[rnd.randrange(pnum)]
            while True:
                r1 = rnd.randrange(NP)
                if r1 != i and r1 != pb: break
            tot = NP + len(archive)
            while True:
                r2 = rnd.randrange(tot)
                if r2 >= NP or (r2 != i and r2 != pb and r2 != r1): break
            xr2 = Xold[r2] if r2 < NP else archive[r2-NP]; xi = Xold[i]
            v = _clip([xi[d] + Fi*(Xold[pb][d]-xi[d]) + Fi*(Xold[r1][d]-xr2[d]) for d in range(D)], lo_l, hi_l)
            jr = rnd.randrange(D); u = [v[d] if (rnd.random() < CRi or d == jr) else xi[d] for d in range(D)]
            fu = T(u)
            if fu < fold[i]:
                X[i] = u; fx[i] = fu; sF.append(Fi); sCR.append(CRi); simp.append(fold[i]-fu); arch_add.append(Xold[i])
        if arch_add:
            archive.extend(arch_add)
            if len(archive) > NP: archive = rnd.sample(archive, NP)
        if sF:
            tot = sum(simp); wts = [s/tot for s in simp]; den = sum(a*b for a, b in zip(wts, sF))
            if den > 0: MF[mem] = sum(a*b*b for a, b in zip(wts, sF))/den
            MCR[mem] = sum(a*b for a, b in zip(wts, sCR)); mem = (mem+1) % H
        cur = min(fx)
        if cur < best_prev - 1e-14: best_prev = cur; stagn = 0
        else: stagn += 1
        gen += 1
        if T.nfe < budget:
            tgt = int(round(POPMIN + (POP0-POPMIN)*(budget-T.nfe)/budget)); tgt = max(POPMIN, min(NP, tgt))
            if tgt < NP:
                ids = sorted(range(NP), key=fx.__getitem__)[:tgt]; X = [X[j] for j in ids]; fx = [fx[j] for j in ids]; NP = tgt
                if len(archive) > NP: archive = rnd.sample(archive, NP)
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': ref_trials, 'rescue_success': ref_success, 'mean_pHO': np.nan, 'final_pHO': np.nan}

_old_run_algorithm = run_algorithm
def run_algorithm(name, model, w, z, budget, seed, checkpoints=()):
    if name == 'ASHO-LR':
        _, lo, hi = MODELS[model]; t0 = time.perf_counter()
        out = asho_lr(model, w, z, lo, hi, budget, seed, checkpoints)
        return (*out, time.perf_counter()-t0)
    return _old_run_algorithm(name, model, w, z, budget, seed, checkpoints)

### Experiment configuration and cache helper

In [ ]:
%%run_if RUN_PART_A
OUT = HERE/('results_quick' if QUICK else 'results'); TAB = OUT/'tables'; OUT.mkdir(exist_ok=True); TAB.mkdir(exist_ok=True)
CACHE = OUT/'cache'; CACHE.mkdir(exist_ok=True)
import pickle
def cached(key, fn):
    p = CACHE/(re.sub(r'[^A-Za-z0-9_.-]', '_', key)+'.pkl')
    if p.exists(): return pickle.load(open(p, 'rb'))
    v = fn(); pickle.dump(v, open(p, 'wb')); return v
SEED0 = 20261101
RUNS, BUDGET = (3, 600) if QUICK else (25, 2000)
EXT_RUNS, EXT_BUDGET = (2, 2000) if QUICK else (10, 10000)
CPS = (100, 200, 500, 1000, 1500, 2000)
SYN_N, SYN_RUNS = (3, 2) if QUICK else (10, 10)
ECM_STARTS = 5 if QUICK else 20
def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)

## 3. Data loading and quality control

In [ ]:
%%run_if RUN_PART_A
COLS = ['device_state','frequency_khz','magnitude_ohm','phase_deg','resistance_ohm','reactance_ohm','timestamp_scaled','tissue']
TARGET_FREQ = np.arange(10, 101, 10, dtype=float)   # kHz
def preprocess(zip_path):
    frames, qc = [], []
    with zipfile.ZipFile(zip_path) as zf:
        for name in sorted(n for n in zf.namelist() if n.lower().endswith('.txt') and '/byExperimentalTask/' in n):
            with zf.open(name) as fh:
                d = pd.read_csv(fh, sep=';', comment='%', header=None, names=COLS, engine='python')
            d = d.dropna(subset=['frequency_khz','resistance_ohm','reactance_ohm'])
            if len(d) == 0:
                qc.append({'source_file': Path(name).name, 'subset': Path(name).parent.name, 'tissue': '', 'raw_rows': 0,
                           'rows_after_exact_dedup': 0, 'exact_duplicates_removed': 0, 'n_frequencies': 0, 'frequencies_khz': ''})
                continue
            before = len(d); d = d.drop_duplicates(); after = len(d)
            d['source_file'] = Path(name).name; d['subset'] = Path(name).parent.name
            frames.append(d); fr = np.sort(d.frequency_khz.unique())
            qc.append({'source_file': Path(name).name, 'subset': Path(name).parent.name, 'tissue': d.tissue.iloc[0],
                       'raw_rows': before, 'rows_after_exact_dedup': after, 'exact_duplicates_removed': before-after,
                       'n_frequencies': len(fr), 'frequencies_khz': ','.join(f'{x:g}' for x in fr)})
    raw = pd.concat(frames, ignore_index=True)
    reps = (raw.groupby(['subset','source_file','tissue','frequency_khz'], as_index=False)
            .agg(resistance_ohm=('resistance_ohm','median'), reactance_ohm=('reactance_ohm','median'),
                 resistance_iqr_ohm=('resistance_ohm', lambda x: np.subtract(*np.percentile(x,[75,25]))),
                 reactance_iqr_ohm=('reactance_ohm', lambda x: np.subtract(*np.percentile(x,[75,25]))),
                 n_obs=('resistance_ohm','size')))
    reps['frequency_hz'] = 1000*reps.frequency_khz
    comp = [g for _, g in reps.groupby('source_file') if np.array_equal(np.sort(g.frequency_khz.unique()), TARGET_FREQ)]
    return pd.DataFrame(qc), pd.concat(comp, ignore_index=True)

qc, complete = preprocess(RAW_ZIP)
qc.to_csv(TAB/'file_qc_summary.csv', index=False); complete.to_csv(TAB/'ecm_ready_complete_scans.csv', index=False)
SPECTRA = []
for si, (sf, g) in enumerate(complete.groupby('source_file')):
    g = g.sort_values('frequency_hz')
    SPECTRA.append(dict(si=si, sf=sf, tissue=g.tissue.iloc[0], w=2*np.pi*g.frequency_hz.to_numpy(float),
                        z=g.resistance_ohm.to_numpy(float) + 1j*g.reactance_ohm.to_numpy(float)))
if QUICK: SPECTRA = SPECTRA[:4]
used = qc[qc.source_file.isin([s['sf'] for s in SPECTRA])]
log('complete spectra:', len(SPECTRA), '| raw rows', int(used.raw_rows.sum()), '-> dedup', int(used.rows_after_exact_dedup.sum()))

## 4. Equivalent-circuit screening (11 models)

In [ ]:
%%run_if RUN_PART_A
def cpe(w,Q,a): return 1/(Q*(1j*w)**a)
def par(z1,z2): return 1/(1/z1+1/z2)
def ecm_predict(model, x, w):
    x = np.asarray(x, float); p = 10**x
    if model=='M1_SeriesRC': return p[0] + 1/(1j*w*p[1])
    if model=='M2_FrickeMorse': return p[0] + par(p[1], 1/(1j*w*p[2]))
    if model=='M3_SingleCole': return p[0] + p[1]/(1+(1j*w*p[2])**x[3])
    if model=='M4_RandlesCPE': return p[0] + par(p[1], cpe(w,p[2],x[3]))
    if model=='M5_DoubleRC': return p[0] + par(p[1],1/(1j*w*p[2])) + par(p[3],1/(1j*w*p[4]))
    if model=='M6_FrickeCPE': return p[0] + par(p[1], cpe(w,p[2],x[3]))
    if model=='M7_DoubleCPE': return p[0] + par(p[1],cpe(w,p[3],x[5])) + par(p[2],cpe(w,p[4],x[6]))
    if model=='M8_DoubleCole': return p[0] + p[1]/(1+(1j*w*p[2])**x[5]) + p[3]/(1+(1j*w*p[4])**x[6])
    if model=='M9_ColeElectrodeCPE': return p[0] + cpe(w,p[4],x[6]) + p[1] + p[2]/(1+(1j*w*p[3])**x[5])
    if model=='M10_ColeL': return predict('M10_ColeL', x, w)
    if model=='M11_RCPEL': return predict('M11_RCPEL', x, w)
    raise KeyError(model)
SPECS = {
 'M1_SeriesRC':([-3,-12],[7,-1]), 'M2_FrickeMorse':([-3,-3,-12],[7,8,-1]),
 'M3_SingleCole':([-3,-3,-7,.3],[7,8,1,1]), 'M4_RandlesCPE':([-3,-3,-12,.3],[7,8,0,1]),
 'M5_DoubleRC':([-3,-3,-12,-3,-12],[7,8,-1,8,-1]), 'M6_FrickeCPE':([-3,-3,-12,.3],[7,8,0,1]),
 'M7_DoubleCPE':([-3,-3,-3,-12,-12,.3,.3],[7,8,8,0,0,1,1]), 'M8_DoubleCole':([-3,-3,-7,-3,-7,.3,.3],[7,8,1,8,1,1,1]),
 'M9_ColeElectrodeCPE':([-3,-3,-3,-7,-12,.3,.3],[7,7,8,1,0,1,1]),
 'M10_ColeL':(list(MODELS['M10_ColeL'][1]), list(MODELS['M10_ColeL'][2])),
 'M11_RCPEL':(list(MODELS['M11_RCPEL'][1]), list(MODELS['M11_RCPEL'][2]))}
def ecm_res(x, m, w, z):
    e = (ecm_predict(m, x, w) - z)/np.maximum(np.abs(z), 1.0); return np.r_[e.real, e.imag]
def _ecm_one(s):
    rows = []
    rng = np.random.default_rng(SEED0 + s['si'])
    for m, (lo, hi) in SPECS.items():
        lo, hi = np.array(lo, float), np.array(hi, float); best = None
        for _ in range(ECM_STARTS):
            r = least_squares(ecm_res, rng.uniform(lo, hi), bounds=(lo, hi), args=(m, s['w'], s['z']), max_nfev=4000)
            if best is None or r.cost < best.cost: best = r
        n = 2*len(s['w']); k = len(lo); rss = 2*best.cost   # weighted RSS: same metric as the fitting objective
        aic = n*np.log(rss/n) + 2*k
        rows.append(dict(source_file=s['sf'], tissue=s['tissue'], model=m, k=k, NRMSE=np.sqrt(rss/len(s['w'])),
                         AIC=aic, AICc=aic + 2*k*(k+1)/(n-k-1), BIC=n*np.log(rss/n) + k*np.log(n)))
    return rows
rows = []
for s in SPECTRA: rows += cached('ecm_'+s['sf'], lambda s=s: _ecm_one(s))
ecm = pd.DataFrame(rows)
ecm['AICc_rank'] = ecm.groupby('source_file').AICc.rank(method='min'); ecm['BIC_rank'] = ecm.groupby('source_file').BIC.rank(method='min')
ecm['dAICc'] = ecm.AICc - ecm.groupby('source_file').AICc.transform('min')
ecm_sum = (ecm.groupby('model').agg(k=('k','first'), median_NRMSE=('NRMSE','median'), mean_NRMSE=('NRMSE','mean'),
           mean_AICc_rank=('AICc_rank','mean'), AICc_wins=('AICc_rank', lambda x: int((x==1).sum())),
           median_dAICc=('dAICc','median'), within2_AICc=('dAICc', lambda x: int((x<=2).sum())),
           mean_BIC_rank=('BIC_rank','mean'), BIC_wins=('BIC_rank', lambda x: int((x==1).sum())))
           .sort_values('mean_AICc_rank'))
ecm.to_csv(TAB/'ecm_all_models.csv', index=False); ecm_sum.to_csv(TAB/'Table_ECM_screening.csv')
log('ECM screening done\n' + ecm_sum.round(4).to_string())

## 5. Confirmatory optimizer benchmark and extended budget

In [ ]:
%%run_if RUN_PART_A
def bench(models, algs, runs, budget, cps, tag):
    rows, traces = [], []; t0 = time.time()
    for model in models:
        names = MODELS[model][0]
        for s in SPECTRA:
          def _one(s=s, model=model, names=names):
            rows, traces = [], []
            for r in range(runs):
                seed = SEED0 + s['si']*1000 + r
                for a in algs:
                    x, fb, tr, ex, sec = run_algorithm(a, model, s['w'], s['z'], budget, seed, cps)
                    rows.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], algorithm=a, run=r+1, seed=seed,
                                     NFE=budget, NRMSE=math.sqrt(max(fb,0)), objective=fb, runtime_sec=sec, **ex,
                                     **{n: v for n, v in zip(names, x)}))
                    for c, v in tr.items(): traces.append((model, s['sf'], a, r+1, c, math.sqrt(max(v,0))))
            return rows, traces
          r_, t_ = cached(f'{tag}_{model}_{s["sf"]}', _one); rows += r_; traces += t_
          log(f'[{tag}] {model} {s["sf"][:32]} done ({time.time()-t0:.0f}s)')
    return pd.DataFrame(rows), pd.DataFrame(traces, columns=['model','source_file','algorithm','run','NFE','best_NRMSE'])

runs_df, trace_df = bench(['M10_ColeL','M11_RCPEL'], ALGORITHMS, RUNS, BUDGET, CPS, 'main')
runs_df.to_csv(TAB/'optimizer_runs.csv', index=False); trace_df.to_csv(TAB/'convergence_traces.csv', index=False)
ext_df, ext_tr = bench(['M10_ColeL','M11_RCPEL'], ['MS-TRF','L-SHADE','ASHO-LSHADE'], EXT_RUNS, EXT_BUDGET,
                       (500,1000,2000,5000,10000), 'ext')
ext_df.to_csv(TAB/'optimizer_runs_extended_budget.csv', index=False); ext_tr.to_csv(TAB/'convergence_traces_extended.csv', index=False)

## 6a. Best-known reference solutions

In [ ]:
%%run_if RUN_PART_A
ref = {}
for model in ['M10_ColeL','M11_RCPEL']:
    names = MODELS[model][0]
    for s in SPECTRA:
        cand = pd.concat([runs_df, ext_df])
        cand = cand[(cand.model==model)&(cand.source_file==s['sf'])].nsmallest(5, 'objective')[names].to_numpy()
        x, rss = cached(f'ref_{model}_{s["sf"]}', lambda: best_fit(model, s['w'], s['z'], list(cand), np.random.default_rng(SEED0+7+s['si']), 30 if not QUICK else 5))
        ref[(model, s['sf'])] = (x, rss/len(s['w']))
ref_df = pd.DataFrame([dict(model=m, source_file=sf, ref_objective=v[1], ref_NRMSE=math.sqrt(v[1]),
                            **{n: val for n, val in zip(MODELS[m][0], v[0])}) for (m, sf), v in ref.items()])
ref_df.to_csv(TAB/'best_known_solutions.csv', index=False)

def add_gap(df):
    df = df.merge(ref_df[['model','source_file','ref_NRMSE']], on=['model','source_file'])
    df['rel_gap'] = (df.NRMSE - df.ref_NRMSE)/df.ref_NRMSE
    df['success'] = df.rel_gap <= 1e-3      # within 0.1% of best-known NRMSE
    return df
runs_df = add_gap(runs_df); ext_df = add_gap(ext_df)
runs_df.to_csv(TAB/'optimizer_runs.csv', index=False); ext_df.to_csv(TAB/'optimizer_runs_extended_budget.csv', index=False)

## 6b. Statistics (success rate, Friedman, Wilcoxon–Holm, bootstrap)

In [ ]:
%%run_if RUN_PART_A
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for rnk, j in enumerate(o):
        run = max(run, (len(p)-rnk)*p[j]); adj[j] = min(1, run)
    return adj
def boot_median(d, B=5000, seed=123):
    rng = np.random.default_rng(seed); d = np.asarray(d); n = len(d)
    v = np.array([np.median(d[rng.integers(0, n, n)]) for _ in range(B)])
    return np.median(d), np.quantile(v, .025), np.quantile(v, .975)
TIE = 1e-3   # relative NRMSE difference below 0.1% is treated as a practical tie
perf_rows, stat_rows, fried = [], [], {}
for model in ['M10_ColeL','M11_RCPEL']:
    d = runs_df[runs_df.model==model]
    med = d.groupby(['source_file','algorithm'], as_index=False).agg(NRMSE=('NRMSE','median'), success=('success','mean'),
          runtime=('runtime_sec','median'), rt=('rescue_trials','sum'), rs=('rescue_success','sum'),
          mean_pHO=('mean_pHO','mean'), final_pHO=('final_pHO','mean'))
    med['rank'] = med.groupby('source_file').NRMSE.rank(method='average')
    piv = med.pivot(index='source_file', columns='algorithm', values='NRMSE')[ALGORITHMS]
    fr = friedmanchisquare(*[piv[c] for c in ALGORITHMS]); fried[model] = (float(fr.statistic), float(fr.pvalue))
    for a in ALGORITHMS:
        m_ = med[med.algorithm==a]; da = d[d.algorithm==a]
        perf_rows.append(dict(model=model, algorithm=a, mean_spec_median_NRMSE=m_.NRMSE.mean(),
            median_spec_median_NRMSE=m_.NRMSE.median(), mean_rank=m_['rank'].mean(),
            success_rate=da.success.mean(), median_rel_gap=da.rel_gap.median(), p90_rel_gap=da.rel_gap.quantile(.9),
            worst_rel_gap=da.rel_gap.max(), median_runtime_ms=1000*da.runtime_sec.median(),
            rescue_trials=int(da.rescue_trials.sum()), rescue_success=int(da.rescue_success.sum()),
            mean_pHO=da.mean_pHO.mean(), final_pHO=da.final_pHO.mean()))
    tmp = []
    for c in [x for x in ALGORITHMS if x != 'ASHO-LSHADE']:
        delta = (piv[c] - piv['ASHO-LSHADE']).to_numpy(); rel = delta/piv['ASHO-LSHADE'].to_numpy()
        try: W, p = wilcoxon(piv['ASHO-LSHADE'], piv[c])
        except ValueError: W, p = np.nan, 1.0
        md, lo_, hi_ = boot_median(delta)
        nz = delta[delta != 0]; rk = rankdata(np.abs(nz)); rb = (rk[nz>0].sum()-rk[nz<0].sum())/rk.sum() if len(nz) else 0
        tmp.append(dict(model=model, comparator=c, W=W, p_raw=p, median_delta_NRMSE=md, boot95_low=lo_, boot95_high=hi_,
                        rank_biserial=rb, ASHO_wins=int((rel>TIE).sum()), ties=int((abs(rel)<=TIE).sum()), ASHO_losses=int((rel<-TIE).sum())))
    tmp = pd.DataFrame(tmp); tmp['p_holm'] = holm(tmp.p_raw); stat_rows.append(tmp)
perf = pd.DataFrame(perf_rows); stats = pd.concat(stat_rows, ignore_index=True)
perf['rescue_success_rate'] = perf.rescue_success/perf.rescue_trials.replace(0, np.nan)
perf.to_csv(TAB/'Table_optimizer_performance.csv', index=False); stats.to_csv(TAB/'Table_paired_statistics.csv', index=False)
log('performance\n' + perf.round(5).to_string()); log('stats\n' + stats.round(5).to_string()); log('friedman', fried)

ext_perf = ext_df.groupby(['model','algorithm']).agg(success_rate=('success','mean'), median_rel_gap=('rel_gap','median'),
           p90_rel_gap=('rel_gap', lambda x: x.quantile(.9)), median_NRMSE=('NRMSE','median')).reset_index()
ext_perf.to_csv(TAB/'Table_extended_budget.csv', index=False); log('extended\n' + ext_perf.round(5).to_string())

## 7a. Parameter spread among near-optimal runs

In [ ]:
%%run_if RUN_PART_A
spread = []
for model in ['M10_ColeL','M11_RCPEL']:
    names = MODELS[model][0]
    d = runs_df[(runs_df.model==model) & runs_df.success]
    for (sf, a), g in d.groupby(['source_file','algorithm']):
        if len(g) < 3: continue
        row = dict(model=model, source_file=sf, algorithm=a, n_success=len(g))
        for n in names: row[n+'_range'] = g[n].max() - g[n].min()
        spread.append(row)
spread = pd.DataFrame(spread); spread.to_csv(TAB/'Table_near_optimal_parameter_spread.csv', index=False)
spread_sum = spread.groupby(['model','algorithm']).median(numeric_only=True).reset_index()
spread_sum.to_csv(TAB/'Table_near_optimal_parameter_spread_summary.csv', index=False)
log('near-optimal spread\n' + spread_sum.round(4).to_string())

## 7b–8a. Jacobian conditioning and profile likelihood

In [ ]:
%%run_if RUN_PART_A
ident = []
for model in ['M10_ColeL','M11_RCPEL']:
    names = MODELS[model][0]
    for s in SPECTRA:
        x, J = ref[(model, s['sf'])]; cond, smin = jacobian_svd(model, x, s['w'], s['z'])
        ident.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], point='best-known', algorithm='-',
                          cond=cond, smin=smin))
        for a in ['L-SHADE','ASHO-LSHADE']:
            g = runs_df[(runs_df.model==model)&(runs_df.source_file==s['sf'])&(runs_df.algorithm==a)]
            row = g.iloc[(g.NRMSE - g.NRMSE.median()).abs().argmin()]
            c2, s2 = jacobian_svd(model, row[names].to_numpy(float), s['w'], s['z'])
            ident.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], point='median run', algorithm=a, cond=c2, smin=s2))
ident = pd.DataFrame(ident); ident.to_csv(TAB/'Table_jacobian_identifiability.csv', index=False)
cond_test = []
for model in ['M10_ColeL','M11_RCPEL']:
    q = ident[(ident.model==model)&(ident.point=='median run')].pivot(index='source_file', columns='algorithm', values='cond')
    W, p = wilcoxon(np.log10(q['ASHO-LSHADE']), np.log10(q['L-SHADE']))
    cond_test.append(dict(model=model, median_cond_LSHADE=q['L-SHADE'].median(), median_cond_ASHO=q['ASHO-LSHADE'].median(),
                          ASHO_lower_count=int((q['ASHO-LSHADE']<q['L-SHADE']).sum()), n=len(q), wilcoxon_p=p))
best_cond = ident[ident.point=='best-known'].groupby('model').cond.agg(['median','min','max']).reset_index()
cond_test = pd.DataFrame(cond_test); cond_test.to_csv(TAB/'Table_condition_number_tests.csv', index=False)
best_cond.to_csv(TAB/'Table_condition_number_best_known.csv', index=False)
log('cond tests\n' + cond_test.to_string()); log('best-known cond\n' + best_cond.to_string())

prof_rows, prof_curves = [], []
for model in ['M10_ColeL','M11_RCPEL']:
    names = MODELS[model][0]
    for s in SPECTRA:
        x, _ = ref[(model, s['sf'])]
        def _prof(x=x, s=s, model=model, names=names):
            rng = np.random.default_rng(SEED0+99+s['si']); pr, pc = [], []
            for pi, pn in enumerate(names):
                grid, lr, lo_ci, hi_ci, bounded = profile_likelihood(model, s['w'], s['z'], x, pi, 21 if QUICK else 41, rng)
                pr.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], parameter=pn, estimate=x[pi],
                               ci_low=lo_ci, ci_high=hi_ci, ci_width=hi_ci-lo_ci, identifiable=bounded))
                for g_, l_ in zip(grid, lr): pc.append((model, s['sf'], pn, g_, l_))
            return pr, pc
        pr, pc = cached(f'prof_{model}_{s["sf"]}', _prof); prof_rows += pr; prof_curves += pc
        log(f'profiles {model} {s["sf"][:32]}')
prof = pd.DataFrame(prof_rows); prof.to_csv(TAB/'Table_profile_likelihood.csv', index=False)
pd.DataFrame(prof_curves, columns=['model','source_file','parameter','value','LR']).to_csv(TAB/'profile_curves.csv', index=False)
prof_sum = prof.groupby(['model','parameter']).agg(identifiable_spectra=('identifiable','sum'), n=('identifiable','size'),
            median_ci_width=('ci_width','median')).reset_index()
prof_sum.to_csv(TAB/'Table_profile_summary.csv', index=False); log('profiles\n' + prof_sum.to_string())

## 8b. Characteristic-frequency band coverage

In [ ]:
%%run_if RUN_PART_A
band = []
for s in SPECTRA:
    x, _ = ref[('M10_ColeL', s['sf'])]
    p = prof[(prof.model=='M10_ColeL')&(prof.source_file==s['sf'])&(prof.parameter=='logtau')].iloc[0]
    band.append(dict(source_file=s['sf'], tissue=s['tissue'], fc_hat_hz=1/(2*np.pi*10**x[2]),
                     fc_ci_low_hz=1/(2*np.pi*10**p.ci_high), fc_ci_high_hz=1/(2*np.pi*10**p.ci_low),
                     fc_below_band=1/(2*np.pi*10**x[2]) < 10e3, Z10k_ohm=abs(s['z'][0]), Z100k_ohm=abs(s['z'][-1]),
                     R0_hat_ohm=10**x[0]+10**x[1]))
band = pd.DataFrame(band); band.to_csv(TAB/'Table_fc_band_coverage.csv', index=False)
log('band\n' + band.round(2).to_string())

## 9. Identifiable M11 physical descriptors

In [ ]:
%%run_if RUN_PART_A
phys = []
for s in SPECTRA:
    x, J = ref[('M11_RCPEL', s['sf'])]
    pr = prof[(prof.model=='M11_RCPEL')&(prof.source_file==s['sf'])].set_index('parameter')
    phys.append(dict(source_file=s['sf'], tissue=s['tissue'], NRMSE=math.sqrt(J), Rinf_ohm=10**x[0], Q=10**x[1], alpha=x[2],
                     L_H=10**x[3], ZCPE_50k_ohm=abs(1/(10**x[1]*(1j*2*np.pi*5e4)**x[2])),
                     Rinf_ci=f'{10**pr.loc["logRinf","ci_low"]:.0f}-{10**pr.loc["logRinf","ci_high"]:.0f}',
                     alpha_ci=f'{pr.loc["alpha","ci_low"]:.3f}-{pr.loc["alpha","ci_high"]:.3f}'))
phys = pd.DataFrame(phys); phys.to_csv(TAB/'Table_M11_physical_parameters.csv', index=False)
log('phys\n' + phys.round(5).to_string())

## 10. Synthetic parameter-recovery experiment

In [ ]:
%%run_if RUN_PART_A
def synth(regime, i):
    rng = np.random.default_rng(SEED0 + (0 if regime=='A' else 50000) + i)
    Rinf = 10**rng.uniform(np.log10(500), np.log10(5000)); a = rng.uniform(0.60, 0.95); L = 10**rng.uniform(-3.4, -2.6)
    fc = 10**rng.uniform(np.log10(15e3), np.log10(60e3)) if regime=='A' else 10**rng.uniform(np.log10(50), np.log10(2e3))
    tau = 1/(2*np.pi*fc)
    # choose dR so that |Z| at 10 kHz is ~2-4x Rinf (as in the real spectra)
    w = 2*np.pi*1000*TARGET_FREQ; target = rng.uniform(2, 4)*Rinf
    dR = 10**np.interp(0, [0,1], [0,0]) ; lo, hi = 0.0, 9.0
    for _ in range(60):
        mid = (lo+hi)/2; zt = predict('M10_ColeL', [np.log10(Rinf), mid, np.log10(tau), a, np.log10(L)], w)
        lo, hi = (mid, hi) if abs(zt[0]) < target else (lo, mid)
    x = np.array([np.log10(Rinf), (lo+hi)/2, np.log10(tau), a, np.log10(L)])
    z0 = predict('M10_ColeL', x, w)
    z = z0 + 0.01*np.abs(z0)*(rng.standard_normal(10) + 1j*rng.standard_normal(10))/np.sqrt(2)
    return x, w, z
syn = []
for regime in ['A','B']:
    for i in range(SYN_N):
        xt, w, z = synth(regime, i); logQt = xt[3]*xt[2] - xt[1]
        def _syn(regime=regime, i=i, xt=xt, w=w, z=z, logQt=logQt):
          syn = []
          for r in range(SYN_RUNS):
            seed = SEED0 + 777000 + (0 if regime=='A' else 1000) + i*100 + r
            for a in ALGORITHMS:
                xh, fb, tr, ex, sec = run_algorithm(a, 'M10_ColeL', w, z, BUDGET, seed)
                err = np.abs(xh - xt)
                syn.append(dict(regime=regime, spectrum=i, run=r+1, algorithm=a, NRMSE=math.sqrt(fb),
                                fc_true_hz=1/(2*np.pi*10**xt[2]), **{f'abs_err_{n}': e for n, e in zip(MODELS['M10_ColeL'][0], err)},
                                abs_err_logQ=abs((xh[3]*xh[2]-xh[1]) - logQt)))
          return syn
        syn += cached(f'syn_{regime}_{i}', _syn)
    log('synthetic regime', regime, 'done')
syn = pd.DataFrame(syn); syn.to_csv(TAB/'synthetic_recovery_runs.csv', index=False)
syn_sum = syn.groupby(['regime','algorithm']).median(numeric_only=True).drop(columns=['spectrum','run']).reset_index()
syn_sum.to_csv(TAB/'Table_synthetic_recovery.csv', index=False); log('synthetic\n' + syn_sum.round(4).to_string())

## Run manifest (main protocol)

In [ ]:
%%run_if RUN_PART_A
man = dict(protocol='ASHO-EIS-v2.0 (revised)', quick=QUICK, runs=RUNS, budget=BUDGET, ext_runs=EXT_RUNS, ext_budget=EXT_BUDGET,
           seed_formula='SEED0 + spectrum_index*1000 + run_index', SEED0=SEED0, success_definition='NRMSE within 0.1% of best-known',
           tie_definition='relative spectrum-median NRMSE difference <= 0.1%', n_spectra=len(SPECTRA),
           frequency_grid_khz=TARGET_FREQ.tolist(), raw_zip_sha256=hashlib.sha256(RAW_ZIP.read_bytes()).hexdigest(),
           python=sys.version, platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           friedman=fried)
json.dump(man, open(OUT/'reproducibility_manifest.json','w'), indent=2, default=str)
log('ALL DONE')

## 11. Higher-dimensional stress test: M12 double Cole + L (8 parameters)

In [ ]:
%%run_if RUN_PART_A
CPS_M12=(100,200,500,1000,1500,2000)
def bench(model, algs, runs, budget):
    rows=[]; names=MODELS[model][0]
    for s in SPECTRA:
        def _one(s=s):
            out=[]
            for r in range(runs):
                seed=SEED0+s['si']*1000+r
                for a in algs:
                    x,fb,tr,ex,sec=run_algorithm(a,model,s['w'],s['z'],budget,seed,CPS_M12)
                    out.append(dict(model=model,source_file=s['sf'],tissue=s['tissue'],algorithm=a,run=r+1,NRMSE=math.sqrt(fb),
                        runtime_sec=sec,**ex,**{f'trace_{k}':math.sqrt(v) for k,v in tr.items()},**dict(zip(names,x))))
            return out
        rows+=cached(f'm12_{s["sf"]}',_one); log('m12',s['sf'][:30])
    return pd.DataFrame(rows)
df=bench('M12_DoubleColeL',ALGORITHMS,RUNS,BUDGET)
best=df.groupby('source_file').NRMSE.min().rename('ref')
# polish reference with multistart least squares
refs={}
for s in SPECTRA:
    cand=df[df.source_file==s['sf']].nsmallest(5,'NRMSE')[MODELS['M12_DoubleColeL'][0]].to_numpy()
    x,rss=cached('m12ref_'+s['sf'],lambda: best_fit('M12_DoubleColeL',s['w'],s['z'],list(cand),np.random.default_rng(5+s['si']),30))
    refs[s['sf']]=min(math.sqrt(rss/len(s['w'])),best[s['sf']])
df['ref']=df.source_file.map(refs); df['rel_gap']=(df.NRMSE-df.ref)/df.ref; df['success']=df.rel_gap<=1e-3
df.to_csv(TAB/'optimizer_runs_M12.csv',index=False)
med=df.groupby(['source_file','algorithm']).NRMSE.median().unstack()[ALGORITHMS]
rows=[]
for a in ALGORITHMS:
    d=df[df.algorithm==a]
    rows.append(dict(algorithm=a,success_rate=d.success.mean(),median_rel_gap=d.rel_gap.median(),p90_rel_gap=d.rel_gap.quantile(.9),
        median_runtime_ms=1000*d.runtime_sec.median(),**{f'success_at_{c}':((d[f'trace_{c}']-d.ref)/d.ref<=1e-3).mean() for c in CPS}))
perf=pd.DataFrame(rows); perf.to_csv(TAB/'Table_M12_performance.csv',index=False)
st=[]
for c in ['L-SHADE','L-SHADE-Restart','SAHO-LSHADE','MS-TRF','HO']:
    W,p=wilcoxon(med['ASHO-LSHADE'],med[c]); st.append(dict(comparator=c,p_raw=p,ASHO_better=int((med[c]>med['ASHO-LSHADE']*(1+1e-3)).sum()),ASHO_worse=int((med[c]<med['ASHO-LSHADE']*(1-1e-3)).sum())))
for c in ['L-SHADE+TRF','MS-TRF']:
    W,p=wilcoxon(med['ASHO-LSHADE+TRF'],med[c]); st.append(dict(comparator=f'ASHO+TRF vs {c}',p_raw=p,ASHO_better=int((med[c]>med['ASHO-LSHADE+TRF']*(1+1e-3)).sum()),ASHO_worse=int((med[c]<med['ASHO-LSHADE+TRF']*(1-1e-3)).sum())))
st=pd.DataFrame(st); st.to_csv(TAB/'Table_M12_paired.csv',index=False)
# per-spectrum success (hardness)
ps=df.groupby(['source_file','algorithm']).success.mean().unstack()[ALGORITHMS]; ps.to_csv(TAB/'Table_M12_success_by_spectrum.csv')
print(perf.round(4).to_string()); print(st.to_string()); print(ps.round(2).to_string()); log('M12 DONE')

## 12. Kramers–Kronig consistency, measurement-level repeatability (ICC) and subset mapping

In [ ]:
%%run_if RUN_PART_A
ref = pd.read_csv(TAB/'best_known_solutions.csv')  # from Section 6a
# ---- reading-level data
raw = []
with zipfile.ZipFile(RAW_ZIP) as zf:
    for s in SPECTRA:
        name = [n for n in zf.namelist() if n.endswith(s['sf']) and '/byExperimentalTask/' in n][0]
        d = pd.read_csv(zf.open(name), sep=';', comment='%', header=None, names=COLS, engine='python').dropna(subset=['frequency_khz']).drop_duplicates()
        d['source_file'] = s['sf']; raw.append(d)
raw = pd.concat(raw)
nread = raw.groupby(['source_file','frequency_khz']).size()
log('readings per frequency: median', nread.median(), 'min', nread.min(), 'max', nread.max())
# ---- subset (acquisition group) mapping
sub = {}
with zipfile.ZipFile(RAW_ZIP) as zf:
    for n in zf.namelist():
        if '/byOriginalSubset/' in n and n.endswith('.txt'): sub.setdefault(Path(n).name, Path(n).parent.name)
mapping = pd.DataFrame([(s['sf'], s['tissue'], sub.get(s['sf'], 'n/a')) for s in SPECTRA], columns=['source_file','tissue','original_subset'])
mapping.to_csv(TAB/'Table_subset_mapping.csv', index=False); log('subset mapping\n' + mapping.original_subset.value_counts().to_string())
# ---- linear Kramers-Kronig (Voigt basis + R + L + C, |Z|-weighted linear least squares)
kk = []
for s in SPECTRA:
    w, z = s['w'], s['z']; M = 5
    taus = np.logspace(np.log10(1/w.max()), np.log10(1/w.min()), M)
    cols = [np.ones_like(w)+0j, 1j*w, 1/(1j*w)] + [1/(1+1j*w*t) for t in taus]
    A = np.array(cols).T; wt = 1/np.abs(z)
    Ar = np.vstack([(A*wt[:,None]).real, (A*wt[:,None]).imag]); br = np.r_[(z*wt).real, (z*wt).imag]
    c, *_ = np.linalg.lstsq(Ar, br, rcond=None); zk = A @ c
    rr = (zk.real - z.real)/np.abs(z); ri = (zk.imag - z.imag)/np.abs(z)
    kk.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_rms_pct=100*np.sqrt(np.mean(np.r_[rr, ri]**2)),
                   kk_max_pct=100*np.max(np.abs(np.r_[rr, ri]))))
kk = pd.DataFrame(kk); kk = kk.merge(ref[ref.model=='M11_RCPEL'][['source_file','ref_NRMSE']], on='source_file')
kk.to_csv(TAB/'Table_KK_check.csv', index=False)
log('KK\n' + kk.round(3).to_string()); log('Spearman KK rms vs M11 NRMSE', spearmanr(kk.kk_rms_pct, kk.ref_NRMSE))
# ---- reading-level bootstrap repeatability
B = 40
def _boot(s):
    d = raw[raw.source_file == s['sf']]; rng = np.random.default_rng(SEED0 + 4242 + s['si']); out = []
    groups = [g for _, g in d.groupby('frequency_khz')]
    for b in range(B):
        pick = [g.iloc[rng.integers(len(g))] for g in groups]
        zb = np.array([p.resistance_ohm + 1j*p.reactance_ohm for p in pick])
        for m in ['M10_ColeL', 'M11_RCPEL']:
            names = MODELS[m][0]; x0 = ref[(ref.model==m)&(ref.source_file==s['sf'])][names].to_numpy(float)[0]
            x, rss = best_fit(m, s['w'], zb, [x0], rng, 4)
            out.append(dict(source_file=s['sf'], tissue=s['tissue'], model=m, b=b, NRMSE=math.sqrt(rss/10), **dict(zip(names, x))))
    return out
rows = []
for s in SPECTRA:
    rows += cached('boot_' + s['sf'], lambda s=s: _boot(s)); log('boot', s['sf'][:30])
bt = pd.DataFrame(rows); bt['logQ_M10'] = np.where(bt.model=='M10_ColeL', bt.alpha*bt.logtau - bt.logdR, np.nan)
bt.to_csv(TAB/'repeatability_bootstrap_fits.csv', index=False)
rep = []
for m, params in [('M10_ColeL', ['logRinf','logdR','logtau','alpha','logL','logQ_M10']), ('M11_RCPEL', ['logRinf','logQ','alpha','logL'])]:
    d = bt[bt.model==m]
    for p in params:
        within_sd = d.groupby('source_file')[p].std(); means = d.groupby('source_file')[p].mean()
        vw = (within_sd**2).mean(); vb = means.var()
        rep.append(dict(model=m, parameter=p, median_within_SD=within_sd.median(), p90_within_SD=within_sd.quantile(.9),
                        between_SD=np.sqrt(vb), ICC=vb/(vb+vw)))
rep = pd.DataFrame(rep); rep.to_csv(TAB/'Table_repeatability.csv', index=False); log('repeatability\n' + rep.round(4).to_string())
log('REV3 DONE')

## 13. Development study of ASHO-LR (synthetic M12 spectra only)
The measured data are **not** used here. ASHO-LR did not beat MS-TRF on this development set and was therefore not carried into the confirmatory experiments (reported in one sentence in the manuscript).

In [ ]:
%%run_if RUN_PART_A
import numpy as np, math
DEV = {}
w=2*np.pi*1000*np.arange(10,101,10.)
specs=[]
for i in range(3 if QUICK else 8):
    r=np.random.default_rng(900+i)
    x=np.array([r.uniform(2.7,3.7),r.uniform(3,5),np.log10(1/(2*np.pi*10**r.uniform(4.2,4.9))),r.uniform(3,5),np.log10(1/(2*np.pi*10**r.uniform(2,3.5))),r.uniform(.6,.95),r.uniform(.6,.95),r.uniform(-3.4,-2.6)])
    z0=predict('M12_DoubleColeL',x,w); z=z0+0.01*abs(z0)*(r.standard_normal(10)+1j*r.standard_normal(10))/np.sqrt(2); specs.append(z)
for name,kw in [('MS-TRF',None),('LR c80',dict(cap_per_dim=80)),('LR full w0',dict(cap_per_dim=10**6,cap_max_per_dim=10**6,adapt_cap=False,warmup_gens=0)),('LR c200 w0',dict(cap_per_dim=200,cap_max_per_dim=400,warmup_gens=0))]:
    res=[]
    for z in specs:
        vals=[]
        for s in range(2 if QUICK else 8):
            if kw is None: vals.append(run_algorithm('MS-TRF','M12_DoubleColeL',w,z,2000,s)[1])
            else: vals.append(asho_lr('M12_DoubleColeL',w,z,*MODELS['M12_DoubleColeL'][1:],2000,s,**kw)[1])
        res.append(vals)
    res=np.sqrt(np.array(res)); best=res.min(1,keepdims=True)
    print(name, 'median NRMSE',np.median(res).round(5))
    DEV[name]=res
best=np.min([v.min(1) for v in DEV.values()],0)[:,None]
for k,v in DEV.items(): print(k,'success vs pooled best', ((v-best)/best<=1e-3).mean().round(3))

## 14. Figures
Main text: Fig04 (fits), Fig05 (convergence), Fig08 (profiles), Fig09 (f_c band), Fig10 (synthetic), Fig14 (repeatability). Supplementary: Fig02 (S1), Fig07 (S2), Fig11 (S3), Fig03 (S4), Fig06 (S5), Fig01 (S6), Fig13 (S7), Fig12 (S8).

In [ ]:
%%run_if RUN_PART_A
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
R = OUT; T = R/'tables'; F = R/'figures'; F.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
ALG = ['HO','MS-TRF','L-SHADE','L-SHADE-Restart','SAHO-LSHADE','ASHO-LSHADE','L-SHADE+TRF','ASHO-LSHADE+TRF']
MK = dict(zip(ALG, ['v','s','o','^','D','*','P','X']))
MNAME = {'M10_ColeL': 'M10 (Cole + L)', 'M11_RCPEL': 'M11 (R–CPE–L)'}
def save(name): plt.tight_layout(); plt.savefig(F/name, bbox_inches='tight'); plt.close()
short = lambda sf: sf.replace('pr_','').replace('_SP_yellow_bipolar_',' ').replace('.txt','')

# Fig 1 workflow
fig, ax = plt.subplots(figsize=(13, 2.9)); ax.set_xlim(0, 13.2); ax.set_ylim(0.6, 2.9); ax.axis('off')
lab = ['Raw EIS\n(10–100 kHz)', 'QC & complete-\nspectrum unit', 'ECM screening\n(weighted AICc/BIC)',
       'Optimizer benchmark\n(equal NFE, 25 runs)', 'Identifiability\n(profile likelihood,\nsynthetic recovery)',
       'Model reduction\nM10 → M11 (R–CPE–L)', 'Bounded physical\ninterpretation']
xs = np.linspace(0.1, 11.4, 7); wd = 1.65
for x, l in zip(xs, lab):
    ax.add_patch(FancyBboxPatch((x, 1.0), wd, 1.15, boxstyle='round,pad=0.04', fill=False, lw=1.2))
    ax.text(x+wd/2, 1.575, l, ha='center', va='center', fontsize=8.5)
for i in range(6):
    ax.add_patch(FancyArrowPatch((xs[i]+wd+0.02, 1.575), (xs[i+1]-0.04, 1.575), arrowstyle='->', mutation_scale=11))
ax.text(6.6, 2.6, 'Optimization–identifiability framework for tissue EIS inverse modelling', ha='center', weight='bold', fontsize=11)
save('Fig01_workflow.png')

comp = pd.read_csv(T/'ecm_ready_complete_scans.csv')
comp = comp[comp.source_file.isin(pd.read_csv(T/'best_known_solutions.csv').source_file)]
fig, ax = plt.subplots(figsize=(6.5, 3.4)); comp.groupby('tissue').source_file.nunique().sort_values(ascending=False).plot.bar(ax=ax, color='0.55')
ax.set_ylabel('Complete spectra'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=40); save('Fig02a_complete_spectra_by_tissue.png')
fig, ax = plt.subplots(figsize=(6.2, 4.6))
for sf, g in comp.groupby('source_file'):
    g = g.sort_values('frequency_khz'); ax.plot(g.resistance_ohm, -g.reactance_ohm, 'o-', ms=2.5, lw=0.9, label=g.tissue.iloc[0])
h, l = ax.get_legend_handles_labels(); u = dict(zip(l, h)); ax.legend(u.values(), u.keys(), fontsize=6.5, ncol=2)
ax.set_xlabel("Z′ (Ω)"); ax.set_ylabel("−Z″ (Ω)"); ax.set_title('All 20 complete spectra (10–100 kHz)'); save('Fig02b_nyquist_all.png')

ecm = pd.read_csv(T/'Table_ECM_screening.csv')
fig, ax = plt.subplots(figsize=(7.5, 3.4)); e = ecm.sort_values('mean_AICc_rank')
ax.bar(e.model.str.replace('_',' '), e.mean_AICc_rank, color=['0.2' if m in ('M10_ColeL','M11_RCPEL') else '0.65' for m in e.model])
ax.set_ylabel('Mean AICc rank (lower is better)'); ax.tick_params(axis='x', rotation=40); save('Fig03_ECM_AICc_rank.png')

# Fig 4 representative fit
ref = pd.read_csv(T/'best_known_solutions.csv'); sfs = sorted(comp.source_file.unique())
sfs = sorted(ref.source_file.unique()); rep = ref[ref.model=='M11_RCPEL'].sort_values('ref_NRMSE').source_file.iloc[len(sfs)//2]
g = comp[comp.source_file==rep].sort_values('frequency_khz'); w = 2*np.pi*1000*g.frequency_khz.to_numpy(); z = g.resistance_ohm.to_numpy()+1j*g.reactance_ohm.to_numpy()
wf = 2*np.pi*1000*np.linspace(10, 100, 200)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(z.real, -z.imag, 'ko', label='Measured')
for m, ls in [('M10_ColeL','-'), ('M11_RCPEL','--')]:
    x = ref[(ref.model==m)&(ref.source_file==rep)][MODELS[m][0]].to_numpy(float)[0]; zf = predict(m, x, wf)
    axs[0].plot(zf.real, -zf.imag, ls, label=MNAME[m])
axs[0].set_xlabel("Z′ (Ω)"); axs[0].set_ylabel("−Z″ (Ω)"); axs[0].legend(); axs[0].set_title(short(rep))
for m, mk in [('M10_ColeL','o'), ('M11_RCPEL','s')]:
    res = []
    for sf in sfs:
        gg = comp[comp.source_file==sf].sort_values('frequency_khz'); ww = 2*np.pi*1000*gg.frequency_khz.to_numpy()
        zz = gg.resistance_ohm.to_numpy()+1j*gg.reactance_ohm.to_numpy()
        x = ref[(ref.model==m)&(ref.source_file==sf)][MODELS[m][0]].to_numpy(float)[0]
        res.append(np.abs(predict(m, x, ww)-zz)/np.abs(zz))
    res = np.array(res); axs[1].plot(np.arange(10,101,10), np.median(res, 0), mk+'-', label=MNAME[m])
axs[1].set_xlabel('Frequency (kHz)'); axs[1].set_ylabel('Median |relative residual|'); axs[1].legend(); axs[1].set_title('Residuals across 20 spectra')
save('Fig04_representative_fit_residuals.png')

# Fig 5 convergence
tr = pd.read_csv(T/'convergence_traces.csv').merge(ref[['model','source_file','ref_NRMSE']], on=['model','source_file'])
tr['gap'] = (tr.best_NRMSE - tr.ref_NRMSE)/tr.ref_NRMSE
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4), sharey=True)
for ax, m in zip(axs, MNAME):
    d = tr[tr.model==m].groupby(['algorithm','NFE']).gap.median().reset_index()
    for a in ALG:
        q = d[d.algorithm==a]; ax.plot(q.NFE, np.maximum(q.gap, 1e-7), marker=MK[a], ms=4, label=a)
    ax.set_yscale('log'); ax.set_xlabel('NFE'); ax.set_title(MNAME[m])
axs[0].set_ylabel('Median relative NRMSE gap to best-known'); axs[1].legend(fontsize=7)
save('Fig05_convergence.png')

# Fig 6 gap distribution and success
runs = pd.read_csv(T/'optimizer_runs.csv')
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, m in zip(axs, MNAME):
    d = runs[runs.model==m]; data = [np.maximum(d[d.algorithm==a].rel_gap, 1e-7) for a in ALG]
    ax.boxplot(data, tick_labels=ALG, showfliers=False); ax.set_yscale('log'); ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.axhline(1e-3, ls=':', c='0.4'); ax.set_title(MNAME[m])
    for i, a in enumerate(ALG):
        ax.text(i+1, 2e-7, f"{100*d[d.algorithm==a].success.mean():.0f}%", ha='center', fontsize=7)
axs[0].set_ylabel('Relative NRMSE gap (per run)'); save('Fig06_gap_success.png')

# Fig 7 ASHO adaptation
a = runs[(runs.algorithm=='ASHO-LSHADE')]
fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.6))
for m in MNAME:
    q = a[a.model==m].groupby('source_file')[['rescue_trials','rescue_success']].sum()
    axs[0].hist(q.rescue_success/q.rescue_trials.replace(0, np.nan), bins=10, alpha=.6, label=MNAME[m])
    axs[1].hist(a[a.model==m].final_pHO.dropna(), bins=15, alpha=.6, label=MNAME[m])
axs[0].set_xlabel('HO rescue success rate per spectrum'); axs[0].set_ylabel('Spectra'); axs[0].legend(fontsize=7)
axs[1].axvline(0.25, ls='--', c='k', lw=1); axs[1].set_xlabel('Final adaptive $p_{HO}$ per run'); axs[1].set_ylabel('Runs')
save('Fig07_ASHO_adaptation.png')

# Fig 8 profiles
pc = pd.read_csv(T/'profile_curves.csv')
fig, axs = plt.subplots(2, 4, figsize=(13, 5.6))
panels = [('M10_ColeL','logtau','log$_{10}$ τ (M10)'), ('M10_ColeL','logdR','log$_{10}$ ΔR (M10)'),
          ('M10_ColeL','alpha','α (M10)'), ('M10_ColeL','logRinf','log$_{10}$ R$_∞$ (M10)'),
          ('M11_RCPEL','logQ','log$_{10}$ Q (M11)'), ('M11_RCPEL','alpha','α (M11)'),
          ('M11_RCPEL','logRinf','log$_{10}$ R$_∞$ (M11)'), ('M11_RCPEL','logL','log$_{10}$ L (M11)')]
for ax, (m, p, t) in zip(axs.ravel(), panels):
    for sf, g in pc[(pc.model==m)&(pc.parameter==p)].groupby('source_file'):
        ax.plot(g.value, np.minimum(g.LR, 60), lw=0.8, alpha=0.75)
    ax.axhline(3.84, ls='--', c='k', lw=0.9); ax.set_ylim(-1, 61); ax.set_title(t, fontsize=9)
for ax in axs[:, 0]: ax.set_ylabel('Profile LR (capped at 60)')
save('Fig08_profile_likelihood.png')

# Fig 9 fc band coverage
b = pd.read_csv(T/'Table_fc_band_coverage.csv').sort_values('fc_hat_hz').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 4.6)); y = np.arange(len(b))
ax.axvspan(10e3, 100e3, color='0.85', label='Measured band (10–100 kHz)')
lo = np.maximum(b.fc_ci_low_hz, 1e-2); hi = np.minimum(b.fc_ci_high_hz, 1e8)
ax.hlines(y, lo, hi, color='0.3'); ax.plot(b.fc_hat_hz, y, 'ko', ms=4, label='M10 estimate $f_c$')
ax.set_xscale('log'); ax.set_yticks(y); ax.set_yticklabels([f'{t} ({s[3:15]})' for t, s in zip(b.tissue, b.source_file)], fontsize=6.5)
ax.set_xlabel('Characteristic frequency $f_c$ (Hz) with 95% profile CI'); ax.legend(fontsize=7, loc='lower right')
save('Fig09_fc_band_coverage.png')

# Fig 10 synthetic recovery
s = pd.read_csv(T/'synthetic_recovery_runs.csv')
pars = ['abs_err_logRinf','abs_err_logdR','abs_err_logtau','abs_err_alpha','abs_err_logL','abs_err_logQ']
lbl = ['log R$_∞$','log ΔR','log τ','α','log L','log Q']
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
algs_show = ['HO','MS-TRF','L-SHADE','ASHO-LSHADE','ASHO-LSHADE+TRF']
for ax, rg, tt in zip(axs, ['A','B'], ['Regime A: $f_c$ inside band', 'Regime B: $f_c$ below band (as in real data)']):
    d = s[s.regime==rg]; wdt = 0.16
    for k, alg in enumerate(algs_show):
        med = [d[d.algorithm==alg][p].median() for p in pars]
        ax.bar(np.arange(len(pars)) + (k-2)*wdt, np.maximum(med, 1e-5), wdt, label=alg)
    ax.set_xticks(range(len(pars))); ax.set_xticklabels(lbl); ax.set_yscale('log'); ax.set_title(tt)
axs[0].set_ylabel('Median absolute recovery error'); axs[1].legend(fontsize=7); save('Fig10_synthetic_recovery.png')

# Fig 11 M11 descriptors by tissue
ph = pd.read_csv(T/'Table_M11_physical_parameters.csv')
fig, axs = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (c, t, lg) in zip(axs, [('Rinf_ohm','R$_∞$ (Ω)',True), ('Q','Q (S·s$^α$)',True), ('alpha','α',False), ('L_H','L (H)',True)]):
    o = ph.groupby('tissue')[c].median().sort_values().index
    for i, ti in enumerate(o):
        v = ph[ph.tissue==ti][c]; ax.plot([i]*len(v), v, 'o', c='0.3', ms=4)
    ax.set_xticks(range(len(o))); ax.set_xticklabels(o, rotation=60, fontsize=7); ax.set_title(t)
    if lg: ax.set_yscale('log')
save('Fig11_M11_descriptors_by_tissue.png')
print('figures:', sorted(p.name for p in F.glob('*.png')))

# Fig 12 higher-dimensional stress test (M12, 8 parameters)
p12 = T/'Table_M12_performance.csv'
if p12.exists():
    m = pd.read_csv(p12); cps = [100,200,500,1000,1500,2000]
    fig, ax = plt.subplots(figsize=(6.8, 4))
    for _, r in m.iterrows():
        ax.plot(cps, [100*r[f'success_at_{c}'] for c in cps], marker=MK[r.algorithm], ms=4, label=r.algorithm)
    ax.set_xlabel('NFE'); ax.set_ylabel('Runs within 0.1% of best-known (%)'); ax.set_title('M12 double Cole + L (8 parameters)')
    ax.legend(fontsize=7); save('Fig12_M12_stress_test.png')

In [ ]:
%%run_if RUN_PART_A
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
T=str(OUT/'tables')+'/'; F=str(OUT/'figures')+'/'
plt.rcParams.update({'savefig.dpi':300,'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
kk=pd.read_csv(T+'Table_KK_check.csv'); bt=pd.read_csv(T+'repeatability_bootstrap_fits.csv')
# Fig: KK vs NRMSE
fig,ax=plt.subplots(figsize=(5.5,4)); ax.scatter(kk.kk_rms_pct,100*kk.ref_NRMSE,c='0.25')
for _,r in kk[kk.kk_rms_pct>3].iterrows(): ax.annotate(r.tissue,(r.kk_rms_pct,100*r.ref_NRMSE),fontsize=7,xytext=(3,2),textcoords='offset points')
ax.axvline(3,ls='--',c='0.5'); ax.set_xlabel('Linear Kramers–Kronig residual, RMS (%)'); ax.set_ylabel('M11 NRMSE (%)')
plt.tight_layout(); plt.savefig(F+'Fig13_KK_vs_fit.png'); plt.close()
# repeatability, KK-consistent only
good=kk[kk.kk_rms_pct<=3].source_file
rows=[]
for m,params in [('M10_ColeL',['logRinf','logdR','logtau','alpha','logL']),('M11_RCPEL',['logRinf','logQ','alpha','logL'])]:
    d=bt[(bt.model==m)&bt.source_file.isin(good)]
    for p in params:
        sd=d.groupby('source_file')[p].std(); mu=d.groupby('source_file')[p].mean(); vw=(sd**2).mean(); vb=mu.var()
        rows.append(dict(model=m,parameter=p,median_within_SD=sd.median(),ICC=vb/(vb+vw),sds=sd.values))
r=pd.DataFrame(rows); r.drop(columns='sds').to_csv(T+'Table_repeatability_KKconsistent.csv',index=False); print(r.drop(columns='sds').round(4).to_string()); print('n good',len(good))
lab={('M10_ColeL','logRinf'):'M10 log R∞',('M10_ColeL','logdR'):'M10 log ΔR',('M10_ColeL','logtau'):'M10 log τ',('M10_ColeL','alpha'):'M10 α',('M10_ColeL','logL'):'M10 log L',
     ('M11_RCPEL','logRinf'):'M11 log R∞',('M11_RCPEL','logQ'):'M11 log Q',('M11_RCPEL','alpha'):'M11 α',('M11_RCPEL','logL'):'M11 log L'}
fig,axs=plt.subplots(1,2,figsize=(11,3.8))
axs[0].boxplot([np.maximum(x,1e-4) for x in r.sds],tick_labels=[lab[(a,b)] for a,b in zip(r.model,r.parameter)]); axs[0].set_yscale('log')
axs[0].set_ylabel('Within-spectrum SD (reading-level bootstrap)'); axs[0].tick_params(axis='x',rotation=45,labelsize=7)
axs[1].bar([lab[(a,b)] for a,b in zip(r.model,r.parameter)],r.ICC,color=['0.65']*5+['0.25']*4); axs[1].set_ylabel('ICC (between / total variance)')
axs[1].tick_params(axis='x',rotation=45,labelsize=7); axs[1].set_ylim(0,1)
plt.tight_layout(); plt.savefig(F+'Fig14_repeatability.png'); plt.close()
# Graphical abstract 200:500
fig,ax=plt.subplots(figsize=(10,4)); ax.set_xlim(0,10); ax.set_ylim(0,4); ax.axis('off')
boxes=[(0.2,'In vivo porcine EIS\n20 spectra, 12 tissues\n10–100 kHz'),(2.7,'Equal-budget benchmark\n8 optimizers\nASHO-LSHADE ≈ L-SHADE\nMS-TRF: 100% success'),
       (5.2,'Identifiability\nf$_c$ below band (19/20)\nτ, ΔR not recoverable'),(7.7,'Reduced R–CPE–L\nidentifiable, repeatable\n(ICC 0.7–0.8)')]
for x,t in boxes:
    ax.add_patch(FancyBboxPatch((x,1.0),2.1,2.0,boxstyle='round,pad=0.05',fill=False,lw=1.6)); ax.text(x+1.05,2.0,t,ha='center',va='center',fontsize=10.5)
for x in [2.3,4.8,7.3]: ax.add_patch(FancyArrowPatch((x,2.0),(x+0.4,2.0),arrowstyle='->',mutation_scale=16,lw=1.5))
ax.text(5,3.55,'When do metaheuristics help in tissue EIS inversion?',ha='center',fontsize=13,weight='bold')
ax.text(5,0.45,'Recommendation: multistart/memetic local refinement + profile-likelihood check + identifiable descriptors',ha='center',fontsize=9.5)
plt.savefig(F+'Graphical_abstract.png',dpi=300); plt.close()

## 15. Export
Collects tables, figures and the manifest into `EIS_ASHO_results_export.zip`.

In [ ]:
%%run_if RUN_PART_A
man = json.load(open(OUT/'reproducibility_manifest.json'))
man.update(notebook='EIS_ASHO_LSHADE_TiS_Complete.ipynb', data_citation='Falagario C, De Momi E, Mattos LS. In vivo tissue bioimpedance dataset. Sci Data (2026). doi:10.1038/s41597-026-07652-y')
json.dump(man, open(OUT/'reproducibility_manifest.json','w'), indent=2, default=str)
with zipfile.ZipFile(HERE/'EIS_ASHO_results_export.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for p in list((OUT/'tables').glob('*.csv'))+list((OUT/'figures').glob('*.png'))+[OUT/'reproducibility_manifest.json']:
        zf.write(p, p.relative_to(HERE))
print('exported', HERE/'EIS_ASHO_results_export.zip')

---
# Part B — Variable projection (VarPro)
The library files are written to `src/` and the experiments are run as scripts so that every step can be re-run independently.

### B.1 Library: core optimizers (`eis_core.py`), data loader, VarPro core

In [ ]:
%%writefile src/eis_core.py
"""
eis_core.py - Core library for the revised ASHO-LSHADE tissue-EIS study.

Pure Python/NumPy (no numba required). Algorithms are line-by-line ports of the
frozen numba implementation in the original notebook, generalised to any
dimension so that they can be applied to both the 5-parameter Cole+L model
(M10) and the identifiable 4-parameter R-CPE-L model (M11).

Hyperparameters (identical for every model and every spectrum):
  L-SHADE : N_init=20, N_min=4, H=6, p=0.11, |A|=NP, M_F=M_CR=0.5 initially,
            current-to-pbest/1/bin, linear population-size reduction.
  Trigger : stagnation >= max(4, round(12-7*progress)) generations AND
            normalised diversity < 0.20+0.20*progress; only the poorer half
            of the population is eligible.
  SAHO    : fixed rescue probability 0.25.
  ASHO    : p_HO(0)=0.25, p_min=0.05, p_max=0.45, lambda=0.25.
  Restart : same trigger, uniform re-sampling with probability 0.25.
"""
import math, random, time
import numpy as np
from scipy.optimize import least_squares

POP0, POPMIN, H, PBEST = 20, 4, 6, 0.11

# ------------------------------------------------------------------ models
MODELS = {
    # name: (parameter names, lower bounds, upper bounds)
    'M10_ColeL': (['logRinf', 'logdR', 'logtau', 'alpha', 'logL'],
                  np.array([-3., -3., -7., 0.30, -9.]), np.array([7., 8., 1., 1.0, 0.])),
    'M11_RCPEL': (['logRinf', 'logQ', 'alpha', 'logL'],
                  np.array([-3., -12., 0.30, -9.]), np.array([7., 0., 1.0, 0.])),
    # higher-dimensional stress test: double Cole + series inductance (8 parameters)
    'M12_DoubleColeL': (['logRinf', 'logdR1', 'logtau1', 'logdR2', 'logtau2', 'alpha1', 'alpha2', 'logL'],
                  np.array([-3., -3., -7., -3., -7., 0.30, 0.30, -9.]), np.array([7., 8., 1., 8., 1., 1.0, 1.0, 0.])),
}

def predict(model, x, w):
    x = np.asarray(x, float)
    if model == 'M10_ColeL':
        Rinf, dR, tau, L = 10**x[0], 10**x[1], 10**x[2], 10**x[4]; a = x[3]
        return Rinf + dR/(1 + (1j*w*tau)**a) + 1j*w*L
    if model == 'M11_RCPEL':
        Rinf, Q, L = 10**x[0], 10**x[1], 10**x[3]; a = x[2]
        return Rinf + 1/(Q*(1j*w)**a) + 1j*w*L
    if model == 'M12_DoubleColeL':
        Rinf, d1, t1, d2, t2, L = 10**x[0], 10**x[1], 10**x[2], 10**x[3], 10**x[4], 10**x[7]
        return Rinf + d1/(1 + (1j*w*t1)**x[5]) + d2/(1 + (1j*w*t2)**x[6]) + 1j*w*L
    raise KeyError(model)

def residual(model, x, w, z):
    e = (predict(model, x, w) - z)/np.maximum(np.abs(z), 1.0)
    return np.r_[e.real, e.imag]

def make_objective(model, w, z):
    """J(theta) = (1/N) sum_k |Zhat_k - Z_k|^2/|Z_k|^2 ; NRMSE = sqrt(J)."""
    sc = np.maximum(np.abs(z), 1.0); N = len(w)
    if model == 'M10_ColeL':
        jw = 1j*w
        def f(x):
            zh = 10**x[0] + 10**x[1]/(1 + (jw*10**x[2])**x[3]) + jw*10**x[4]
            e = np.abs(zh - z)/sc
            return float(np.dot(e, e)/N)
    elif model == 'M11_RCPEL':
        jw = 1j*w
        def f(x):
            zh = 10**x[0] + 1/(10**x[1]*jw**x[2]) + jw*10**x[3]
            e = np.abs(zh - z)/sc
            return float(np.dot(e, e)/N)
    else:
        def f(x):
            e = np.abs(predict(model, x, w) - z)/sc
            return float(np.dot(e, e)/N)
    return f

# ------------------------------------------------------------ utilities
class Tracker:
    """Counts NFE and records best-so-far objective at checkpoints."""
    def __init__(self, f, budget, checkpoints):
        self.f, self.budget, self.nfe = f, budget, 0
        self.best = math.inf; self.xbest = None
        self.cps = sorted(c for c in checkpoints if c <= budget); self.ci = 0
        self.trace = {}
    def __call__(self, x):
        v = self.f(x); self.nfe += 1
        if v < self.best: self.best = v; self.xbest = list(x)
        while self.ci < len(self.cps) and self.nfe >= self.cps[self.ci]:
            self.trace[self.cps[self.ci]] = self.best; self.ci += 1
        return v

def _clip(u, lo, hi):
    for d in range(len(u)):
        if u[d] < lo[d]: u[d] = lo[d]
        elif u[d] > hi[d]: u[d] = hi[d]
    return u

def _levy(rnd, beta=1.5):
    sigma = (math.gamma(1+beta)*math.sin(math.pi*beta/2) /
             (math.gamma((1+beta)/2)*beta*2**((beta-1)/2)))**(1/beta)
    return rnd.gauss(0, 1)*sigma/(abs(rnd.gauss(0, 1))**(1/beta) + 1e-12)

def _ho_rescue(rnd, x, best, progress, lo, hi):
    D = len(x); u = list(x)
    if rnd.random() < 0.5:   # HO river/pond position update (dominant-hippo guidance)
        I1 = 1 + rnd.randint(0, 1); y = rnd.random()
        for d in range(D): u[d] = x[d] + y*(best[d] - I1*x[d])
    else:                    # HO defence/escape-inspired heavy-tailed move
        lev = _levy(rnd); shrink = max(0.03, 0.35*(1.0 - progress))
        for d in range(D):
            u[d] = x[d] + rnd.random()*lev*shrink*(hi[d]-lo[d]) + 0.25*rnd.random()*(best[d]-x[d])
    return _clip(u, lo, hi)

def _init(rnd, lo, hi, n):
    D = len(lo)
    return [[lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)] for _ in range(n)]

# ------------------------------------------------------- L-SHADE family
def lshade_family(f, lo, hi, budget, seed, mode, checkpoints=()):
    """mode: 'lshade' | 'restart' | 'saho' | 'asho'"""
    rnd = random.Random(seed); D = len(lo); lo = list(lo); hi = list(hi)
    T = Tracker(f, budget, checkpoints)
    NP = POP0; X = _init(rnd, lo, hi, NP); fx = [T(x) for x in X]
    MF = [0.5]*H; MCR = [0.5]*H; mem = 0; archive = []
    best_prev = min(fx); stagn = 0; maxdiv = 1e-12
    rt = rs = 0; p_ho = 0.25; pmin, pmax, lam = 0.05, 0.45, 0.25; asum = 0.0; an = 0
    while T.nfe < budget and NP >= POPMIN:
        cen = [sum(X[i][d] for i in range(NP))/NP for d in range(D)]
        div = sum(math.sqrt(sum((X[i][d]-cen[d])**2 for d in range(D))) for i in range(NP))/NP
        maxdiv = max(maxdiv, div); dnorm = div/maxdiv; progress = T.nfe/budget
        stag_thr = max(4, int(round(12 - 7*progress)))
        active = mode != 'lshade' and stagn >= stag_thr and dnorm < (0.20 + 0.20*progress)
        prob = 0.25 if mode in ('saho', 'restart') else p_ho
        order = sorted(range(NP), key=fx.__getitem__); pnum = max(2, int(math.ceil(PBEST*NP)))
        Xold = [list(x) for x in X]; fold = list(fx); ib = order[0]
        sF = []; sCR = []; simp = []; arch_add = []; grt = grs = 0
        for i in range(NP):
            if T.nfe >= budget: break
            k = rnd.randrange(H)
            Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            while Fi <= 0: Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            Fi = min(1.0, Fi); CRi = min(1.0, max(0.0, rnd.gauss(MCR[k], 0.1)))
            special = active and i >= max(1, NP//2) and rnd.random() < prob
            if special and mode in ('saho', 'asho'):
                rt += 1; grt += 1; u = _ho_rescue(rnd, Xold[i], Xold[ib], progress, lo, hi)
            elif special and mode == 'restart':
                rt += 1; u = [lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)]
            else:
                special = False
                pb = order[rnd.randrange(pnum)]
                while True:
                    r1 = rnd.randrange(NP)
                    if r1 != i and r1 != pb: break
                tot = NP + len(archive)
                while True:
                    r2 = rnd.randrange(tot)
                    if r2 >= NP or (r2 != i and r2 != pb and r2 != r1): break
                xr2 = Xold[r2] if r2 < NP else archive[r2-NP]
                xi = Xold[i]; xp = Xold[pb]; x1 = Xold[r1]
                v = _clip([xi[d] + Fi*(xp[d]-xi[d]) + Fi*(x1[d]-xr2[d]) for d in range(D)], lo, hi)
                jr = rnd.randrange(D)
                u = [v[d] if (rnd.random() < CRi or d == jr) else xi[d] for d in range(D)]
            fu = T(u)
            if fu < fold[i]:
                X[i] = u; fx[i] = fu
                if special:
                    rs += 1; grs += 1
                else:
                    sF.append(Fi); sCR.append(CRi); simp.append(fold[i]-fu); arch_add.append(Xold[i])
        if mode == 'asho' and grt > 0:
            p_ho = (1-lam)*p_ho + lam*(pmin + (pmax-pmin)*grs/grt)
            p_ho = min(pmax, max(pmin, p_ho)); asum += p_ho; an += 1
        if arch_add:
            archive.extend(arch_add)
            if len(archive) > NP: archive = rnd.sample(archive, NP)
        if sF:
            tot = sum(simp); wts = [s/tot for s in simp]
            den = sum(wt*F for wt, F in zip(wts, sF))
            if den > 0: MF[mem] = sum(wt*F*F for wt, F in zip(wts, sF))/den
            MCR[mem] = sum(wt*c for wt, c in zip(wts, sCR)); mem = (mem+1) % H
        cur = min(fx)
        if cur < best_prev - 1e-14: best_prev = cur; stagn = 0
        else: stagn += 1
        if T.nfe < budget:
            tgt = int(round(POPMIN + (POP0-POPMIN)*(budget-T.nfe)/budget)); tgt = max(POPMIN, min(NP, tgt))
            if tgt < NP:
                ids = sorted(range(NP), key=fx.__getitem__)[:tgt]
                X = [X[j] for j in ids]; fx = [fx[j] for j in ids]; NP = tgt
                if len(archive) > NP: archive = rnd.sample(archive, NP)
    order = sorted(range(NP), key=fx.__getitem__)
    extra = {'final_pop': [X[j] for j in order], 'rescue_trials': rt, 'rescue_success': rs,
             'mean_pHO': (asum/an if an else p_ho) if mode == 'asho' else np.nan,
             'final_pHO': p_ho if mode == 'asho' else np.nan}
    return np.array(T.xbest), T.best, T.trace, extra

# ------------------------------------------------------------------- HO
def ho(f, lo, hi, budget, seed, checkpoints=()):
    """Compact NFE-fair HO (Amiri et al., 2024): position update, defence, escape."""
    rnd = random.Random(seed); D = len(lo); lo = list(lo); hi = list(hi)
    T = Tracker(f, budget, checkpoints)
    N = POP0; X = _init(rnd, lo, hi, N); fx = [T(x) for x in X]; it = 1
    maxit = max(1, (budget-N)//N)
    while T.nfe < budget:
        best = list(X[min(range(N), key=fx.__getitem__)]); half = max(1, N//2); Tt = math.exp(-it/maxit)
        for i in range(half):
            if T.nfe >= budget: break
            I1 = 1 + rnd.randint(0, 1); y = rnd.random()
            u = _clip([X[i][d] + y*(best[d]-I1*X[i][d]) for d in range(D)], lo, hi)
            fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        for i in range(half, N):
            if T.nfe >= budget: break
            pred = [lo[d] + (hi[d]-lo[d])*rnd.random() for d in range(D)]; fp = T(pred)
            if T.nfe >= budget: break
            lev = _levy(rnd); u = list(X[i])
            for d in range(D):
                dist = abs(pred[d]-X[i][d]) + 1e-9; step = 0.05*lev*(hi[d]-lo[d])/dist
                u[d] = pred[d] + step if fp < fx[i] else X[i][d] + rnd.random()*(best[d]-X[i][d]) + step*rnd.random()
            u = _clip(u, lo, hi); fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        for _ in range(max(1, N//4)):
            if T.nfe >= budget: break
            i = rnd.randrange(N); shrink = max(0.02, 0.25*Tt)
            u = _clip([X[i][d] + (2*rnd.random()-1)*shrink*(hi[d]-lo[d]) + 0.2*rnd.random()*(best[d]-X[i][d])
                       for d in range(D)], lo, hi)
            fu = T(u)
            if fu < fx[i]: X[i] = u; fx[i] = fu
        it += 1
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': 0, 'rescue_success': 0, 'mean_pHO': np.nan, 'final_pHO': np.nan}

# ----------------------------------------- multistart trust-region baseline
class _Budget(Exception): pass

def ms_trf(model, w, z, lo, hi, budget, seed, checkpoints=()):
    """Multistart bounded nonlinear least squares (trust-region reflective,
    2-point Jacobian). Every residual evaluation, including those used for the
    finite-difference Jacobian, is counted against the same NFE budget."""
    rng = np.random.default_rng(seed); f = make_objective(model, w, z)
    T = Tracker(f, budget, checkpoints)
    def res(x):
        if T.nfe >= budget: raise _Budget
        T(x); return residual(model, x, w, z)
    while T.nfe < budget:
        try:
            least_squares(res, rng.uniform(lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget:
            break
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': 0, 'rescue_success': 0, 'mean_pHO': np.nan, 'final_pHO': np.nan}

def hybrid_trf(model, w, z, lo, hi, budget, seed, mode, checkpoints=(), frac_global=0.5):
    """Memetic variant: L-SHADE-family global stage for frac_global*budget NFE,
    then bounded TRF refinement started from the best and the remaining final
    population members (then uniform random points) until the SAME total NFE
    budget is exhausted."""
    f = make_objective(model, w, z); b1 = int(frac_global*budget)
    x1, f1, tr1, ex = lshade_family(f, lo, hi, b1, seed, mode, checkpoints)
    T = Tracker(f, budget, checkpoints); T.nfe = b1; T.best = f1; T.xbest = list(x1)
    while T.ci < len(T.cps) and T.cps[T.ci] <= b1: T.ci += 1
    rng = np.random.default_rng(seed)
    starts = [np.asarray(p, float) for p in ex.pop('final_pop')]
    def res(x):
        if T.nfe >= budget: raise _Budget
        T(x); return residual(model, x, w, z)
    k = 0
    while T.nfe < budget:
        x0 = starts[k] if k < len(starts) else rng.uniform(lo, hi); k += 1
        try:
            least_squares(res, np.clip(x0, lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget:
            break
    trace = dict(tr1); trace.update(T.trace)
    return np.array(T.xbest), T.best, trace, ex

ALGORITHMS = ['HO', 'MS-TRF', 'L-SHADE', 'L-SHADE-Restart', 'SAHO-LSHADE', 'ASHO-LSHADE', 'L-SHADE+TRF', 'ASHO-LSHADE+TRF']

def run_algorithm(name, model, w, z, budget, seed, checkpoints=()):
    _, lo, hi = MODELS[model]; f = make_objective(model, w, z)
    t0 = time.perf_counter()
    if name == 'HO': out = ho(f, lo, hi, budget, seed, checkpoints)
    elif name == 'MS-TRF': out = ms_trf(model, w, z, lo, hi, budget, seed, checkpoints)
    elif name.endswith('+TRF'):
        out = hybrid_trf(model, w, z, lo, hi, budget, seed, 'asho' if name.startswith('ASHO') else 'lshade', checkpoints)
    else:
        mode = {'L-SHADE': 'lshade', 'L-SHADE-Restart': 'restart', 'SAHO-LSHADE': 'saho', 'ASHO-LSHADE': 'asho'}[name]
        out = lshade_family(f, lo, hi, budget, seed, mode, checkpoints)
    out[3].pop('final_pop', None)
    return (*out, time.perf_counter()-t0)

# --------------------------------------------------------- identifiability
def jacobian_svd(model, x, w, z):
    _, lo, hi = MODELS[model]; x = np.asarray(x, float); r0 = residual(model, x, w, z)
    J = np.empty((len(r0), len(x)))
    for d in range(len(x)):
        h = 1e-5*max(1, abs(x[d])); xp = x.copy(); xm = x.copy()
        xp[d] = min(hi[d], xp[d]+h); xm[d] = max(lo[d], xm[d]-h)
        J[:, d] = (residual(model, xp, w, z) - residual(model, xm, w, z))/max(xp[d]-xm[d], 1e-15)
    s = np.linalg.svd(J, compute_uv=False)
    return s[0]/max(s[-1], 1e-300), s[-1]

def best_fit(model, w, z, x0s, rng, extra_starts=20):
    _, lo, hi = MODELS[model]; best = None
    starts = [np.clip(np.asarray(x, float), lo, hi) for x in x0s] + [rng.uniform(lo, hi) for _ in range(extra_starts)]
    for x0 in starts:
        r = least_squares(lambda x: residual(model, x, w, z), x0, bounds=(lo, hi), max_nfev=5000,
                          xtol=1e-12, ftol=1e-12, gtol=1e-12)
        if best is None or r.cost < best.cost: best = r
    return best.x, 2*best.cost

def profile_likelihood(model, w, z, xhat, pidx, ngrid=41, rng=None):
    """Profile of parameter pidx (others re-optimised). Gaussian likelihood with
    unknown variance: LR(theta) = n*log(RSS(theta)/RSS_min); 95% CI where LR<=3.84."""
    names, lo, hi = MODELS[model]; n = 2*len(w)
    free = [d for d in range(len(lo)) if d != pidx]
    rss_min = float(np.sum(residual(model, xhat, w, z)**2))
    span = 0.05*(hi[pidx]-lo[pidx])
    grid = np.unique(np.clip(np.r_[np.linspace(lo[pidx], hi[pidx], ngrid),
                                   xhat[pidx] + np.linspace(-span, span, 21), xhat[pidx]], lo[pidx], hi[pidx]))
    prof = []
    for g in grid:
        best = None
        starts = [xhat[free]] + [rng.uniform(lo[free], hi[free]) for _ in range(3)]
        for x0 in starts:
            def rf(xf, g=g):
                x = np.array(xhat, float); x[free] = xf; x[pidx] = g; return residual(model, x, w, z)
            r = least_squares(rf, np.clip(x0, lo[free], hi[free]), bounds=(lo[free], hi[free]), max_nfev=3000)
            if best is None or r.cost < best: best = r.cost
        prof.append(2*best)
    prof = np.minimum(np.array(prof), np.inf); rss_min = min(rss_min, prof.min())
    lr = n*np.log(prof/rss_min); inside = grid[lr <= 3.84]
    i0 = int(np.argmin(lr)); thr = 3.84
    def cross(j_in, j_out):   # linear interpolation of the LR=3.84 crossing
        a, b = lr[j_in], lr[j_out]
        return grid[j_in] + (thr-a)*(grid[j_out]-grid[j_in])/(b-a) if b != a else grid[j_in]
    j = i0
    while j > 0 and lr[j-1] <= thr: j -= 1
    lo_ci = cross(j, j-1) if j > 0 else grid[0]
    j = i0
    while j < len(grid)-1 and lr[j+1] <= thr: j += 1
    hi_ci = cross(j, j+1) if j < len(grid)-1 else grid[-1]
    # identifiable (practically) if the 95% profile CI closes before both search bounds
    bounded = (lr[0] > 3.84) and (lr[-1] > 3.84)
    return grid, lr, lo_ci, hi_ci, bounded

# ------------------------------------------------------------------------------
# ASHO-LR: Adaptive Stagnation-triggered, HO-seeded Local Refinement (revision 3)
# ------------------------------------------------------------------------------
class _LocalCap(Exception): pass

def asho_lr(model, w, z, lo, hi, budget, seed, checkpoints=(), cap_per_dim=40, warmup_gens=2, cap_max_per_dim=160, adapt_cap=True):
    """L-SHADE backbone. When the best objective stagnates (same progress-dependent
    threshold as ASHO, no diversity condition), a bounded TRF refinement is launched.
    The first refinement starts from the population best; subsequent ones start from
    an HO-rescue move applied to a poorer-half individual, so HO proposes a new basin
    and TRF polishes it. Each refinement is capped at cap_per_dim*D evaluations
    (doubled after a refinement improves the global best, halved otherwise, within
    [10D, 160D]). The refined point replaces the worst individual if it is better.
    All evaluations (population, HO, TRF incl. Jacobians) share one NFE budget."""
    rnd = random.Random(seed); D = len(lo); lo_l = list(lo); hi_l = list(hi)
    f = make_objective(model, w, z); T = Tracker(f, budget, checkpoints)
    NP = POP0; X = _init(rnd, lo_l, hi_l, NP); fx = [T(x) for x in X]
    MF = [0.5]*H; MCR = [0.5]*H; mem = 0; archive = []
    best_prev = min(fx); stagn = 0; gen = 0; nref = 0; cap = cap_per_dim*D
    ref_trials = ref_success = 0; cap_max = cap_max_per_dim*D
    def refine(x0, cap_):
        start = T.nfe; best_seen = [math.inf, None]
        def res(x):
            if T.nfe >= budget: raise _Budget
            if T.nfe - start >= cap_: raise _LocalCap
            v = T(x)
            if v < best_seen[0]: best_seen[0] = v; best_seen[1] = list(x)
            return residual(model, x, w, z)
        try:
            least_squares(res, np.clip(np.asarray(x0, float), lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=100000)
        except (_Budget, _LocalCap):
            pass
        return best_seen
    while T.nfe < budget and NP >= POPMIN:
        progress = T.nfe/budget; stag_thr = max(4, int(round(12 - 7*progress)))
        if gen >= warmup_gens and stagn >= stag_thr:
            ib = min(range(NP), key=fx.__getitem__)
            if nref == 0: x0 = X[ib]
            else:
                i = rnd.randrange(max(1, NP//2), NP) if NP > 1 else 0
                order = sorted(range(NP), key=fx.__getitem__)
                x0 = _ho_rescue(rnd, X[order[i]], X[ib], progress, lo_l, hi_l)
            gb = T.best; v, xr = refine(x0, cap); nref += 1; ref_trials += 1
            if xr is not None:
                iw = max(range(NP), key=fx.__getitem__)
                if v < fx[iw]: X[iw] = xr; fx[iw] = v
            if T.best < gb*(1 - 1e-9): ref_success += 1; cap = min(cap_max, cap*2)
            else: cap = max(10*D, cap//2) if adapt_cap else cap
            stagn = 0; best_prev = min(fx)
            if T.nfe >= budget: break
        order = sorted(range(NP), key=fx.__getitem__); pnum = max(2, int(math.ceil(PBEST*NP)))
        Xold = [list(x) for x in X]; fold = list(fx)
        sF = []; sCR = []; simp = []; arch_add = []
        for i in range(NP):
            if T.nfe >= budget: break
            k = rnd.randrange(H)
            Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            while Fi <= 0: Fi = MF[k] + 0.1*math.tan(math.pi*(rnd.random()-0.5))
            Fi = min(1.0, Fi); CRi = min(1.0, max(0.0, rnd.gauss(MCR[k], 0.1)))
            pb = order[rnd.randrange(pnum)]
            while True:
                r1 = rnd.randrange(NP)
                if r1 != i and r1 != pb: break
            tot = NP + len(archive)
            while True:
                r2 = rnd.randrange(tot)
                if r2 >= NP or (r2 != i and r2 != pb and r2 != r1): break
            xr2 = Xold[r2] if r2 < NP else archive[r2-NP]; xi = Xold[i]
            v = _clip([xi[d] + Fi*(Xold[pb][d]-xi[d]) + Fi*(Xold[r1][d]-xr2[d]) for d in range(D)], lo_l, hi_l)
            jr = rnd.randrange(D); u = [v[d] if (rnd.random() < CRi or d == jr) else xi[d] for d in range(D)]
            fu = T(u)
            if fu < fold[i]:
                X[i] = u; fx[i] = fu; sF.append(Fi); sCR.append(CRi); simp.append(fold[i]-fu); arch_add.append(Xold[i])
        if arch_add:
            archive.extend(arch_add)
            if len(archive) > NP: archive = rnd.sample(archive, NP)
        if sF:
            tot = sum(simp); wts = [s/tot for s in simp]; den = sum(a*b for a, b in zip(wts, sF))
            if den > 0: MF[mem] = sum(a*b*b for a, b in zip(wts, sF))/den
            MCR[mem] = sum(a*b for a, b in zip(wts, sCR)); mem = (mem+1) % H
        cur = min(fx)
        if cur < best_prev - 1e-14: best_prev = cur; stagn = 0
        else: stagn += 1
        gen += 1
        if T.nfe < budget:
            tgt = int(round(POPMIN + (POP0-POPMIN)*(budget-T.nfe)/budget)); tgt = max(POPMIN, min(NP, tgt))
            if tgt < NP:
                ids = sorted(range(NP), key=fx.__getitem__)[:tgt]; X = [X[j] for j in ids]; fx = [fx[j] for j in ids]; NP = tgt
                if len(archive) > NP: archive = rnd.sample(archive, NP)
    return np.array(T.xbest), T.best, T.trace, {'rescue_trials': ref_trials, 'rescue_success': ref_success, 'mean_pHO': np.nan, 'final_pHO': np.nan}

_old_run_algorithm = run_algorithm
def run_algorithm(name, model, w, z, budget, seed, checkpoints=()):
    if name == 'ASHO-LR':
        _, lo, hi = MODELS[model]; t0 = time.perf_counter()
        out = asho_lr(model, w, z, lo, hi, budget, seed, checkpoints)
        return (*out, time.perf_counter()-t0)
    return _old_run_algorithm(name, model, w, z, budget, seed, checkpoints)

In [ ]:
%%writefile src/vp_common.py
import numpy as np, pandas as pd
from pathlib import Path
TAB = Path(__file__).parent.parent/'results'/'tables'
def load_spectra():
    d = pd.read_csv(TAB/'ecm_ready_complete_scans.csv'); S = []
    for si, (sf, g) in enumerate(d.groupby('source_file')):
        g = g.sort_values('frequency_hz')
        S.append(dict(si=si, sf=sf, tissue=g.tissue.iloc[0], w=2*np.pi*g.frequency_hz.to_numpy(float),
                      z=g.resistance_ohm.to_numpy(float)+1j*g.reactance_ohm.to_numpy(float)))
    return S

In [ ]:
%%writefile src/vp_core.py
"""
vp_core.py - Variable-projection (VarPro) reformulation of the ECM fitting problems.

Every model used in the study is *separable*: given its nonlinear parameters
(relaxation times and dispersion exponents) the impedance is a linear
combination of fixed basis functions,

    M10  Z = Rinf*1 + dR*C(w;tau,alpha)            + L*jw     nonlinear: (log tau, alpha)
    M11  Z = Rinf*1 + (1/Q)*(jw)^(-alpha)          + L*jw     nonlinear: (alpha)
    M12  Z = Rinf*1 + dR1*C(w;tau1,a1) + dR2*C(w;tau2,a2) + L*jw
                                                              nonlinear: (log tau1, log tau2, a1, a2)

with C(w;tau,a) = 1/(1+(jw tau)^a). For any value of the nonlinear parameters the
optimal linear coefficients follow from a weighted, box-bounded linear least-squares
problem (same weights 1/max(|Z|,1) as the original objective, same bounds as the
original log10 bounds). The reduced objective
        J_VP(theta_nl) = min_c J(theta_nl, c)
has the same global minimum as the full objective, but lives in 1 (M11), 2 (M10)
or 4 (M12) dimensions instead of 4, 5 and 8 (Golub & Pereyra 1973, 2003).

NFE accounting: one evaluation of J_VP builds the basis once (the cost of one
model evaluation) and solves a <=4-column linear least-squares problem; it is
counted as ONE function evaluation. Wall-clock time is reported separately so that
this convention can be checked.
"""
import itertools, math, random, time
import numpy as np
from scipy.optimize import least_squares, minimize_scalar, nnls
from eis_core import MODELS, Tracker, lshade_family, _Budget, POP0

# nonlinear parameter spaces and linear coefficient bounds (linear scale)
VP_SPECS = {
    'M10_ColeL': dict(nl=['logtau', 'alpha'], lo=np.array([-7., 0.30]), hi=np.array([1., 1.0]),
                      lin=['Rinf', 'dR', 'L'], llo=10.0**np.array([-3., -3., -9.]), lhi=10.0**np.array([7., 8., 0.])),
    'M11_RCPEL': dict(nl=['alpha'], lo=np.array([0.30]), hi=np.array([1.0]),
                      lin=['Rinf', 'invQ', 'L'], llo=10.0**np.array([-3., 0., -9.]), lhi=10.0**np.array([7., 12., 0.])),
    'M12_DoubleColeL': dict(nl=['logtau1', 'logtau2', 'alpha1', 'alpha2'],
                      lo=np.array([-7., -7., 0.30, 0.30]), hi=np.array([1., 1., 1.0, 1.0]),
                      lin=['Rinf', 'dR1', 'dR2', 'L'], llo=10.0**np.array([-3., -3., -3., -9.]),
                      lhi=10.0**np.array([7., 8., 8., 0.])),
}

def basis(model, t, w):
    jw = 1j*w; one = np.ones_like(w) + 0j
    if model == 'M10_ColeL':
        return np.column_stack([one, 1/(1 + (jw*10**t[0])**t[1]), jw])
    if model == 'M11_RCPEL':
        return np.column_stack([one, jw**(-t[0]), jw])
    if model == 'M12_DoubleColeL':
        return np.column_stack([one, 1/(1 + (jw*10**t[0])**t[2]), 1/(1 + (jw*10**t[1])**t[3]), jw])
    raise KeyError(model)

def _enum_lstsq(A, b, lb, ub):
    """Exact box-bounded least squares by active-set enumeration (k<=4 columns).
    Convex problem: the optimum is the best feasible candidate over all active sets."""
    k = A.shape[1]; s = np.linalg.norm(A, axis=0); s[s == 0] = 1.0; As = A/s
    best_c, best_v = None, math.inf
    for state in itertools.product((0, 1, 2), repeat=k):       # 0 free, 1 lower, 2 upper
        st = np.array(state); cc = np.where(st == 1, lb, np.where(st == 2, ub, 0.0))
        free = np.flatnonzero(st == 0)
        if len(free):
            cc[free] = np.linalg.lstsq(As[:, free], b - A @ cc, rcond=None)[0]/s[free]
            if np.any(cc[free] < lb[free]*(1-1e-12)) or np.any(cc[free] > ub[free]*(1+1e-12)): continue
        r = A @ cc - b; v = float(r @ r)
        if v < best_v: best_v, best_c = v, cc
    return best_c, best_v

def bounded_lstsq(A, b, lb, ub):
    """min ||A c - b||^2 s.t. lb <= c <= ub. Fast path: shift to c = lb + d, d >= 0, and
    solve the column-scaled NNLS problem (Lawson-Hanson). The upper bounds are far from
    any physically meaningful value and are almost never active; if the NNLS solution
    violates one, the exact active-set enumeration is used instead."""
    s = np.linalg.norm(A, axis=0); s[s == 0] = 1.0
    d, _ = nnls(A/s, b - A @ lb, maxiter=50*A.shape[1])
    c = lb + d/s
    if np.all(c <= ub):
        r = A @ c - b; return c, float(r @ r)
    # clamp the violated coefficients at their upper bound, re-solve NNLS for the rest and
    # verify the KKT conditions (exact certificate for this convex problem)
    up = c > ub
    for _ in range(A.shape[1]):
        fr = ~up; cc = np.where(up, ub, lb)
        rhs = b - A[:, up] @ ub[up] - A[:, fr] @ lb[fr]
        d, _ = nnls(A[:, fr]/s[fr], rhs, maxiter=50*A.shape[1]); cc[fr] = lb[fr] + d/s[fr]
        newup = up | (cc > ub)
        if np.array_equal(newup, up): break
        up = newup
    r = A @ cc - b; g = A.T @ r
    if np.all(cc <= ub*(1+1e-12)) and np.all(g[up] <= 1e-10*max(1.0, float(np.abs(g).max()))):
        return cc, float(r @ r)
    return _enum_lstsq(A, b, lb, ub)

class VarPro:
    def __init__(self, model, w, z):
        self.model, self.w, self.z = model, w, z
        sp = VP_SPECS[model]; self.sp = sp; self.N = len(w)
        self.wt = 1.0/np.maximum(np.abs(z), 1.0)
        zw = z*self.wt; self.b = np.r_[zw.real, zw.imag]
    def solve(self, t):
        A = basis(self.model, t, self.w)*self.wt[:, None]
        Ar = np.vstack([A.real, A.imag])
        c, rss = bounded_lstsq(Ar, self.b, self.sp['llo'], self.sp['lhi'])
        return c, rss, Ar
    def f(self, t):                       # reduced objective J = RSS/N (same scale as eis_core)
        return self.solve(np.asarray(t, float))[1]/self.N
    def res(self, t):                     # reduced residual vector for TRF
        c, _, Ar = self.solve(np.asarray(t, float)); return Ar @ c - self.b
    def full_params(self, t):
        """map (nonlinear, linear) back to the original log10 parameter vector of eis_core.MODELS."""
        t = np.asarray(t, float); c, _, _ = self.solve(t); lc = np.log10(np.maximum(c, self.sp['llo']))
        if self.model == 'M10_ColeL':  return np.array([lc[0], lc[1], t[0], t[1], lc[2]])
        if self.model == 'M11_RCPEL':  return np.array([lc[0], -lc[1], t[0], lc[2]])
        return np.array([lc[0], lc[1], t[0], lc[2], t[1], t[2], t[3], lc[3]])

# ------------------------------------------------------------- algorithms
def _vp_trf_loop(vp, T, starts, rng, lo, hi, budget):
    def res(t):
        if T.nfe >= budget: raise _Budget
        T(t); return vp.res(t)
    k = 0
    while T.nfe < budget:
        x0 = starts[k] if k < len(starts) else rng.uniform(lo, hi); k += 1
        try:
            least_squares(res, np.clip(x0, lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget:
            break

def vp_ms_trf(vp, budget, seed, cps):
    lo, hi = vp.sp['lo'], vp.sp['hi']; T = Tracker(vp.f, budget, cps); rng = np.random.default_rng(seed)
    _vp_trf_loop(vp, T, [], rng, lo, hi, budget); return T

def vp_screen_trf(vp, budget, seed, cps, n_screen=None, n_polish=3):
    """Space-filling screen of the (small) nonlinear space, TRF polish from the best
    screened points, then random multistart TRF until the budget is spent."""
    lo, hi = vp.sp['lo'], vp.sp['hi']; D = len(lo); rng = np.random.default_rng(seed)
    T = Tracker(vp.f, budget, cps)
    if n_screen is None: n_screen = {1: 21, 2: 60, 4: 150}[D]
    if D == 1:
        pts = np.linspace(lo, hi, n_screen)
        pts = np.clip(pts + rng.uniform(-0.5, 0.5, pts.shape)*(hi-lo)/(n_screen-1), lo, hi)
    else:   # Latin hypercube
        u = (np.argsort(rng.random((D, n_screen)), axis=1).T + rng.random((n_screen, D)))/n_screen
        pts = lo + u*(hi-lo)
    vals = [T(p) for p in pts]
    order = np.argsort(vals); starts = [pts[i] for i in order[:n_polish]]
    _vp_trf_loop(vp, T, starts, rng, lo, hi, budget); return T

def vp_lshade(vp, budget, seed, cps, mode):
    lo, hi = vp.sp['lo'], vp.sp['hi']
    x, fb, tr, ex = lshade_family(vp.f, lo, hi, budget, seed, mode, cps)
    T = Tracker(vp.f, budget, ()); T.best = fb; T.xbest = list(x); T.trace = tr; T.extra = ex; return T

def vp_lshade_trf(vp, budget, seed, cps, mode='lshade', frac=0.5):
    lo, hi = vp.sp['lo'], vp.sp['hi']; b1 = int(frac*budget)
    x1, f1, tr1, ex = lshade_family(vp.f, lo, hi, b1, seed, mode, cps)
    T = Tracker(vp.f, budget, cps); T.nfe = b1; T.best = f1; T.xbest = list(x1)
    while T.ci < len(T.cps) and T.cps[T.ci] <= b1: T.ci += 1
    starts = [np.asarray(p, float) for p in ex.pop('final_pop')]
    _vp_trf_loop(vp, T, starts, np.random.default_rng(seed), lo, hi, budget)
    T.trace = {**tr1, **T.trace}; return T

VP_ALGORITHMS = ['VP-L-SHADE', 'VP-ASHO-LSHADE', 'VP-MS-TRF', 'VP-L-SHADE+TRF', 'VP-Screen+TRF']

def run_vp(name, model, w, z, budget, seed, cps=()):
    vp = VarPro(model, w, z); t0 = time.perf_counter()
    if name == 'VP-L-SHADE':       T = vp_lshade(vp, budget, seed, cps, 'lshade')
    elif name == 'VP-ASHO-LSHADE': T = vp_lshade(vp, budget, seed, cps, 'asho')
    elif name == 'VP-MS-TRF':      T = vp_ms_trf(vp, budget, seed, cps)
    elif name == 'VP-L-SHADE+TRF': T = vp_lshade_trf(vp, budget, seed, cps)
    elif name == 'VP-Screen+TRF':  T = vp_screen_trf(vp, budget, seed, cps)
    else: raise KeyError(name)
    sec = time.perf_counter() - t0
    tr = dict(T.trace)
    for c in cps:                                  # carry best value forward if run ended early
        if c not in tr: tr[c] = T.best
    return vp.full_params(T.xbest), T.best, tr, sec

# ------------------------------------------------------ exact cheap profiles
def vp_profile(model, w, z, pname, grid, inner_grid=61):
    """Profile likelihood through VarPro: the profiled parameter is fixed, the remaining
    NONLINEAR parameters are optimised (grid + bounded refinement) and all linear ones
    are solved exactly. Returns RSS(grid) on the eis_core residual scale."""
    vp = VarPro(model, w, z); sp = vp.sp; j = sp['nl'].index(pname); others = [i for i in range(len(sp['nl'])) if i != j]
    out = []
    for g in grid:
        if not others:
            out.append(vp.f([g])*vp.N); continue
        i = others[0]                               # M10: one remaining nonlinear parameter
        def h(v, g=g):
            t = np.empty(2); t[j] = g; t[i] = v; return vp.f(t)
        vs = np.linspace(sp['lo'][i], sp['hi'][i], inner_grid); hv = [h(v) for v in vs]; k = int(np.argmin(hv))
        a, b = vs[max(0, k-1)], vs[min(len(vs)-1, k+1)]
        r = minimize_scalar(h, bounds=(a, b), method='bounded', options={'xatol': 1e-6})
        out.append(min(r.fun, hv[k])*vp.N)
    return np.array(out)

### B.2 VarPro benchmark scripts

In [ ]:
%%writefile src/run_vp.py
"""run_vp.py - benchmark of the variable-projection algorithms (same spectra, seeds, budget, checkpoints)."""
import sys, math, pickle, time, re
import numpy as np, pandas as pd
from pathlib import Path
from vp_common import *; from vp_core import *
SEED0 = 20261101; RUNS = int(sys.argv[2]) if len(sys.argv) > 2 else 25; BUDGET = 2000
CPS = (100, 200, 500, 1000, 1500, 2000)
CACHE = TAB.parent/'cache_vp'; CACHE.mkdir(exist_ok=True)
model = sys.argv[1]; names = MODELS[model][0]
S = load_spectra()
rows = []
for s in S:
    p = CACHE/(f'{model}_'+re.sub(r'[^A-Za-z0-9_.-]', '_', s['sf'])+f'_{RUNS}.pkl')
    if p.exists(): out = pickle.load(open(p, 'rb'))
    else:
        out = []
        for r in range(RUNS):
            seed = SEED0 + s['si']*1000 + r
            for a in VP_ALGORITHMS:
                x, fb, tr, sec = run_vp(a, model, s['w'], s['z'], BUDGET, seed, CPS)
                out.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], algorithm=a, run=r+1, seed=seed,
                                NRMSE=math.sqrt(fb), objective=fb, runtime_sec=sec,
                                **{f'trace_{c}': math.sqrt(v) for c, v in tr.items()}, **dict(zip(names, x))))
        pickle.dump(out, open(p, 'wb'))
    rows += out; print(time.strftime('%H:%M:%S'), model, s['si'], s['sf'][:40], flush=True)
pd.DataFrame(rows).to_csv(TAB/f'vp_runs_{model}.csv', index=False)
print('DONE', model)

In [ ]:
%%writefile src/analyze_vp.py
"""analyze_vp.py - merge the variable-projection benchmark with the original runs and
produce comparison tables and figures (same spectra, seeds, budget and success rule)."""
import math, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from vp_common import TAB
FIG = TAB.parent/'figures_vp'; FIG.mkdir(exist_ok=True)
CPS = [100, 200, 500, 1000, 1500, 2000]; TOL = 1e-3
OLD = ['HO', 'L-SHADE', 'ASHO-LSHADE', 'MS-TRF', 'L-SHADE+TRF', 'ASHO-LSHADE+TRF']
COL = dict(zip(['HO','L-SHADE','ASHO-LSHADE','MS-TRF','L-SHADE+TRF','ASHO-LSHADE+TRF','VP-L-SHADE','VP-ASHO-LSHADE','VP-MS-TRF','VP-L-SHADE+TRF','VP-Screen+TRF'],
 ['#bbbbbb','#f4a582','#d6604d','#b2182b','#c2a5cf','#9970ab','#4393c3','#92c5de','#1b7837','#2166ac','#5aae61']))
NEW = ['VP-L-SHADE', 'VP-ASHO-LSHADE', 'VP-MS-TRF', 'VP-L-SHADE+TRF', 'VP-Screen+TRF']

def load(model):
    if model == 'M12_DoubleColeL':
        o = pd.read_csv(TAB/'optimizer_runs_M12.csv').rename(columns={'ref': 'ref_old'})
    else:
        o = pd.read_csv(TAB/'optimizer_runs.csv'); o = o[o.model == model].rename(columns={'ref_NRMSE': 'ref_old'})
        tr = pd.read_csv(TAB/'convergence_traces.csv'); tr = tr[tr.model == model]
        tr = tr.pivot_table(index=['source_file', 'algorithm', 'run'], columns='NFE', values='best_NRMSE')
        tr.columns = [f'trace_{c}' for c in tr.columns]; o = o.merge(tr.reset_index(), on=['source_file', 'algorithm', 'run'], how='left')
    o = o[o.algorithm.isin(OLD)]
    n = pd.read_csv(TAB/f'vp_runs_{model}.csv')
    ref_old = o.groupby('source_file').ref_old.first()
    best_vp = n.groupby('source_file').NRMSE.min()
    ref = pd.concat([ref_old, best_vp], axis=1).min(axis=1)
    improved = ((ref_old - best_vp)/ref_old > TOL)
    d = pd.concat([o, n], ignore_index=True); d['ref'] = d.source_file.map(ref)
    d['rel_gap'] = (d.NRMSE - d.ref)/d.ref; d['success'] = d.rel_gap <= TOL
    for c in CPS: d[f'succ_{c}'] = (d[f'trace_{c}'] - d.ref)/d.ref <= TOL
    return d, ref_old, best_vp, improved

def first_success_nfe(row):
    for c in CPS:
        if row[f'succ_{c}']: return c
    return np.nan

summary, paired, refinfo = [], [], []
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, model in zip(axes, ['M11_RCPEL', 'M10_ColeL', 'M12_DoubleColeL']):
    d, ref_old, best_vp, improved = load(model)
    refinfo.append(dict(model=model, spectra=len(ref_old), ref_improved_by_VP=int(improved.sum()),
                        max_improvement_pct=100*float(((ref_old-best_vp)/ref_old).max()),
                        VP_worse_than_old_ref=int(((best_vp-ref_old)/ref_old > TOL).sum())))
    d['nfe_first_success'] = d.apply(first_success_nfe, axis=1)
    for a in OLD + NEW:
        x = d[d.algorithm == a]
        summary.append(dict(model=model, algorithm=a, success_rate=x.success.mean(),
                            median_rel_gap=x.rel_gap.median(), p90_rel_gap=x.rel_gap.quantile(.9), worst_rel_gap=x.rel_gap.max(),
                            spectra_all_runs_success=int(x.groupby('source_file').success.mean().eq(1).sum()),
                            **{f'success_at_{c}': x[f'succ_{c}'].mean() for c in CPS},
                            median_runtime_ms=1000*x.runtime_sec.median()))
        ax.plot(CPS, [100*d[d.algorithm == a][f'succ_{c}'].mean() for c in CPS],
                color=COL[a], ls='-' if a in NEW else '--', marker='o' if a in NEW else 's', ms=4, lw=1.6 if a in NEW else 1.0,
                alpha=1 if a in NEW else .7, label=a)
    ax.set_xscale('log'); ax.set_xlabel('NFE'); ax.set_title(model.replace('_', ' ')); ax.set_ylim(-3, 103); ax.grid(alpha=.3)
    if model == 'M11_RCPEL': ax.set_ylabel('runs within 0.1% of best-known (%)')
    med = d.groupby(['source_file', 'algorithm']).NRMSE.median().unstack()
    for a, b in [('VP-MS-TRF', 'MS-TRF'), ('VP-L-SHADE', 'L-SHADE'), ('VP-ASHO-LSHADE', 'ASHO-LSHADE'),
                 ('VP-ASHO-LSHADE', 'VP-L-SHADE'), ('VP-L-SHADE+TRF', 'L-SHADE+TRF'), ('VP-Screen+TRF', 'MS-TRF'),
                 ('VP-Screen+TRF', 'VP-MS-TRF')]:
        diff = med[a] - med[b]; rel = diff/med[b]
        try: p = wilcoxon(med[a], med[b]).pvalue if np.any(np.abs(rel) > 1e-12) else 1.0
        except ValueError: p = 1.0
        paired.append(dict(model=model, comparison=f'{a} vs {b}', better=int((rel < -TOL).sum()), worse=int((rel > TOL).sum()),
                           ties=int((rel.abs() <= TOL).sum()), median_rel_diff=float(rel.median()), p_raw=p))
    d.to_csv(TAB/f'vp_merged_{model}.csv', index=False)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=6, fontsize=8, frameon=False, bbox_to_anchor=(.5, -0.08))
plt.tight_layout(); plt.savefig(FIG/'FigVP1_success_vs_NFE.png', dpi=300, bbox_inches='tight'); plt.close()

S = pd.DataFrame(summary); P = pd.DataFrame(paired); R = pd.DataFrame(refinfo)
# Holm within model
P['p_holm'] = np.nan
for m, g in P.groupby('model'):
    o = g.p_raw.sort_values(); k = len(o); adj = np.maximum.accumulate([min(1, (k-i)*p) for i, p in enumerate(o)])
    P.loc[o.index, 'p_holm'] = adj
S.to_csv(TAB/'Table_VP_performance.csv', index=False); P.to_csv(TAB/'Table_VP_paired.csv', index=False)
R.to_csv(TAB/'Table_VP_reference_check.csv', index=False)

# M12 per-spectrum success: hardest spectra
d12 = pd.read_csv(TAB/'vp_merged_M12_DoubleColeL.csv')
ps = d12.groupby(['source_file', 'algorithm']).success.mean().unstack()[['MS-TRF', 'L-SHADE+TRF', 'ASHO-LSHADE+TRF'] + NEW]
ps.index = [i.split('bipolar_')[-1].replace('.txt', '') + '_' + i[3:15] for i in ps.index]
ps.to_csv(TAB/'Table_VP_M12_success_by_spectrum.csv')
fig, ax = plt.subplots(figsize=(9, 6)); im = ax.imshow(100*ps.values, cmap='viridis', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(ps.shape[1])); ax.set_xticklabels(ps.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(ps.shape[0])); ax.set_yticklabels(ps.index, fontsize=7)
for i in range(ps.shape[0]):
    for j in range(ps.shape[1]): ax.text(j, i, f'{100*ps.values[i, j]:.0f}', ha='center', va='center', fontsize=6,
                                         color='w' if ps.values[i, j] < .6 else 'k')
plt.colorbar(im, label='success (%)'); ax.set_title('M12: per-spectrum success at 2,000 NFE')
plt.tight_layout(); plt.savefig(FIG/'FigVP2_M12_success_heatmap.png', dpi=300); plt.close()

pd.set_option('display.width', 250)
print(R.to_string()); print(S.round(4).to_string()); print(P.round(4).to_string())

In [ ]:
%%writefile src/run_vp_profiles.py
"""run_vp_profiles.py - profile likelihood with a GLOBAL inner optimisation made cheap by VarPro.
For each profiled value, the linear coefficients are solved exactly and the remaining nonlinear
parameters (1-2 dims) are optimised by a dense grid followed by local polishing, so the inner
problem does not depend on random restarts. Same grid, likelihood-ratio statistic, 3.84 threshold
and identifiability rule as eis_core.profile_likelihood."""
import time, math, numpy as np, pandas as pd
from scipy.optimize import least_squares, minimize_scalar
from vp_common import *; from vp_core import VarPro, VP_SPECS, basis, bounded_lstsq
from eis_core import MODELS

class FixedVP(VarPro):
    """VarPro with one linear coefficient fixed at value cfix (column moved to the right-hand side)."""
    def __init__(self, model, w, z, jfix, cfix):
        super().__init__(model, w, z); self.j, self.cf = jfix, cfix
        k = len(self.sp['llo']); self.keep = [i for i in range(k) if i != jfix]
    def solve(self, t):
        A = basis(self.model, t, self.w)*self.wt[:, None]; Ar = np.vstack([A.real, A.imag])
        rhs = self.b - Ar[:, self.j]*self.cf
        c, rss = bounded_lstsq(Ar[:, self.keep], rhs, self.sp['llo'][self.keep], self.sp['lhi'][self.keep])
        return c, rss, None
    def res(self, t):
        A = basis(self.model, t, self.w)*self.wt[:, None]; Ar = np.vstack([A.real, A.imag])
        rhs = self.b - Ar[:, self.j]*self.cf
        c, _ = bounded_lstsq(Ar[:, self.keep], rhs, self.sp['llo'][self.keep], self.sp['lhi'][self.keep])
        return Ar[:, self.keep] @ c - rhs

def min_nonlinear(vp, lo, hi, fixed=None):
    """global minimum of the reduced RSS over the free nonlinear parameters (1 or 2 dims).
    fixed=(index, value) fixes one nonlinear parameter."""
    D = len(lo); free = [i for i in range(D) if not (fixed and i == fixed[0])]
    def full(u):
        t = np.empty(D); t[free] = u
        if fixed: t[fixed[0]] = fixed[1]
        return t
    if len(free) == 0: return vp.f(full([]))*vp.N
    if len(free) == 1:
        i = free[0]; vs = np.linspace(lo[i], hi[i], 81); hv = np.array([vp.f(full([v])) for v in vs]); k = int(hv.argmin())
        r = minimize_scalar(lambda v: vp.f(full([v])), bounds=(vs[max(0, k-1)], vs[min(80, k+1)]), method='bounded',
                            options={'xatol': 1e-7})
        return min(r.fun, hv[k])*vp.N
    g0 = np.linspace(lo[0], hi[0], 33); g1 = np.linspace(lo[1], hi[1], 15)
    P = np.array([[a, b] for a in g0 for b in g1]); hv = np.array([vp.f(p) for p in P]); best = hv.min()*vp.N
    for k in np.argsort(hv)[:3]:
        r = least_squares(vp.res, P[k], bounds=(lo, hi), xtol=1e-12, ftol=1e-12, gtol=1e-12, max_nfev=400)
        best = min(best, 2*r.cost)
    return best

def profile(model, s, xhat, pname):
    names, LO, HI = MODELS[model]; p = names.index(pname); n = 2*len(s['w'])
    span = 0.05*(HI[p]-LO[p])
    grid = np.unique(np.clip(np.r_[np.linspace(LO[p], HI[p], 41), xhat[p] + np.linspace(-span, span, 21), xhat[p]], LO[p], HI[p]))
    sp = VP_SPECS[model]; lo, hi = sp['lo'], sp['hi']; vp = VarPro(model, s['w'], s['z'])
    lin_map = {'M10_ColeL': {'logRinf': (0, 1), 'logdR': (1, 1), 'logL': (2, 1)},
               'M11_RCPEL': {'logRinf': (0, 1), 'logQ': (1, -1), 'logL': (2, 1)}}[model]
    prof = []
    for g in grid:
        if pname in sp['nl']:
            prof.append(min_nonlinear(vp, lo, hi, fixed=(sp['nl'].index(pname), g)))
        else:
            j, sgn = lin_map[pname]; fvp = FixedVP(model, s['w'], s['z'], j, 10.0**(sgn*g))
            prof.append(min_nonlinear(fvp, lo, hi))
    prof = np.array(prof); rss_min = min(prof.min(), min_nonlinear(vp, lo, hi))
    lr = n*np.log(prof/rss_min); thr = 3.84; i0 = int(np.argmin(lr))
    def cross(a, b): return grid[a] + (thr-lr[a])*(grid[b]-grid[a])/(lr[b]-lr[a]) if lr[b] != lr[a] else grid[a]
    j = i0
    while j > 0 and lr[j-1] <= thr: j -= 1
    lo_ci = cross(j, j-1) if j > 0 else grid[0]
    j = i0
    while j < len(grid)-1 and lr[j+1] <= thr: j += 1
    hi_ci = cross(j, j+1) if j < len(grid)-1 else grid[-1]
    return dict(ci_low=lo_ci, ci_high=hi_ci, ci_width=hi_ci-lo_ci, identifiable=bool(lr[0] > thr and lr[-1] > thr)), grid, lr

if __name__ == '__main__':
    S = load_spectra(); ref = pd.read_csv(TAB/'best_known_solutions.csv'); rows = []; curves = []
    t0 = time.perf_counter()
    for model in ['M10_ColeL', 'M11_RCPEL']:
        names = MODELS[model][0]
        for s in S:
            r = ref[(ref.model == model) & (ref.source_file == s['sf'])].iloc[0]; xhat = r[names].to_numpy(float)
            for pn in names:
                out, grid, lr = profile(model, s, xhat, pn)
                rows.append(dict(model=model, source_file=s['sf'], tissue=s['tissue'], parameter=pn, estimate=xhat[names.index(pn)], **out))
                curves += [dict(model=model, source_file=s['sf'], parameter=pn, value=g, LR=l) for g, l in zip(grid, lr)]
        print(time.strftime('%H:%M:%S'), model, 'done', flush=True)
    sec = time.perf_counter() - t0
    V = pd.DataFrame(rows); V.to_csv(TAB/'Table_VP_profile_likelihood.csv', index=False)
    pd.DataFrame(curves).to_csv(TAB/'vp_profile_curves.csv', index=False)
    old = pd.read_csv(TAB/'Table_profile_likelihood.csv')
    m = V.merge(old, on=['model', 'source_file', 'parameter'], suffixes=('_vp', '_old'))
    comp = m.groupby(['model', 'parameter']).agg(identifiable_old=('identifiable_old', 'sum'), identifiable_vp=('identifiable_vp', 'sum'),
              agree=('identifiable_vp', lambda x: int((x == m.loc[x.index, 'identifiable_old']).sum())),
              median_width_old=('ci_width_old', 'median'), median_width_vp=('ci_width_vp', 'median')).reset_index()
    comp.to_csv(TAB/'Table_VP_profile_comparison.csv', index=False)
    print(comp.round(3).to_string()); print('total VP profile time (s):', round(sec, 1))

In [ ]:
%%writefile src/time_fair.py
"""Same-machine timing: original MS-TRF vs VP-MS-TRF, wall-clock time to first reach the best-known value (0.1%)."""
import time, math, numpy as np, pandas as pd, sys
from vp_common import *; from vp_core import VarPro, vp_ms_trf
from eis_core import ms_trf, make_objective, MODELS, Tracker
import eis_core, vp_core
SEED0 = 20261101; RUNS = int(sys.argv[1]) if len(sys.argv) > 1 else 5
class TimedTracker(Tracker):
    target = None
    def __init__(self, *a, **k):
        super().__init__(*a, **k); self.t0 = time.perf_counter(); self.t_hit = None; self.n_hit = None
    def __call__(self, x):
        v = super().__call__(x)
        if self.t_hit is None and self.best <= TimedTracker.target:
            self.t_hit = time.perf_counter()-self.t0; self.n_hit = self.nfe
        return v
eis_core.Tracker = TimedTracker; vp_core.Tracker = TimedTracker
rows = []
for model in ['M11_RCPEL', 'M10_ColeL', 'M12_DoubleColeL']:
    d = pd.read_csv(TAB/f'vp_merged_{model}.csv'); ref = d.groupby('source_file').ref.first()
    _, lo, hi = MODELS[model]
    for s in load_spectra():
        TimedTracker.target = (ref[s['sf']]*(1+1e-3))**2
        for r in range(RUNS):
            seed = SEED0 + s['si']*1000 + r
            for name in ['MS-TRF', 'VP-MS-TRF']:
                cap = {}
                orig_init = TimedTracker.__init__
                def init(self, *a, **k): orig_init(self, *a, **k); cap['T'] = self
                TimedTracker.__init__ = init
                t0 = time.perf_counter()
                if name == 'MS-TRF': ms_trf(model, s['w'], s['z'], lo, hi, 2000, seed)
                else: vp_ms_trf(VarPro(model, s['w'], s['z']), 2000, seed, ())
                tot = time.perf_counter()-t0; TimedTracker.__init__ = orig_init; T = cap['T']
                rows.append(dict(model=model, source_file=s['sf'], run=r+1, algorithm=name, total_sec=tot,
                                 hit=T.t_hit is not None, sec_to_hit=T.t_hit, nfe_to_hit=T.n_hit))
    print(model, 'done', flush=True)
D = pd.DataFrame(rows); D.to_csv(TAB/f'vp_fair_timing_runs_{RUNS}.csv', index=False)
S = D.groupby(['model', 'algorithm']).agg(runs=('hit', 'size'), success=('hit', 'mean'), median_ms_to_hit=('sec_to_hit', lambda x: 1000*x.median()),
        p90_ms_to_hit=('sec_to_hit', lambda x: 1000*x.quantile(.9)), median_nfe_to_hit=('nfe_to_hit', 'median'),
        median_total_ms_2000NFE=('total_sec', lambda x: 1000*x.median())).reset_index()
S.to_csv(TAB/f'Table_VP_fair_timing_{RUNS}runs.csv', index=False); print(S.round(2).to_string())
# all-run statistics used in Table 4 of the manuscript (unsuccessful runs counted as infinite time)
D['t'] = np.where(D.hit, 1000*D.sec_to_hit, np.inf); rows = []
for m, g in D.groupby('model'):
    p = g.pivot_table(index=['source_file', 'run'], columns='algorithm', values='t', aggfunc='first'); a, b = p['MS-TRF'], p['VP-MS-TRF']
    rows.append(dict(model=m, ms_median_all=np.median(a), vp_median_all=np.median(b), ms_q1=np.quantile(a, .25), ms_q3=np.quantile(a, .75),
                     vp_q1=np.quantile(b, .25), vp_q3=np.quantile(b, .75), vp_faster_pct=100*np.mean(b < a),
                     ms_success=100*np.isfinite(a).mean(), vp_success=100*np.isfinite(b).mean()))
A = pd.DataFrame(rows); A.to_csv(TAB/'Table_VP_fair_timing_allruns.csv', index=False); print(A.round(2).to_string())

### B.3 Run the VarPro benchmark (M11 and M10 in parallel, then M12)

In [ ]:
%%time
!cd src && (python run_vp.py M11_RCPEL $RUNS_VP > ../results/log_vp_M11.txt 2>&1 & python run_vp.py M10_ColeL $RUNS_VP > ../results/log_vp_M10.txt 2>&1 & wait) && tail -n 1 ../results/log_vp_M11.txt ../results/log_vp_M10.txt

In [ ]:
%%time
!cd src && python run_vp.py M12_DoubleColeL $RUNS_VP 2>&1 | tail -n 1

### B.4 Merge with the original runs: success rates, paired tests (Table 3, Fig. 3, Fig. S8)

In [ ]:
!cd src && python analyze_vp.py > ../results/log_analyze_vp.txt 2>&1; tail -n 40 ../results/log_analyze_vp.txt

### B.5 Globally optimized VarPro profile likelihood (check of Table 6)

In [ ]:
%%time
!cd src && python run_vp_profiles.py 2>&1 | tail -n 12

### B.6 Same-machine time to solution (Table 4) — run this cell alone

In [ ]:
%%time
!cd src && python time_fair.py 25 2>&1 | tail -n 8

---
# Part C — Additional analyses

### C.1 Frequency-band design simulation (Fig. 8, Table S4)

In [ ]:
%%writefile src/band_design.py
"""band_design.py - how far below the current 10-100 kHz band must measurements extend for the
Cole relaxation time and resistive dispersion (M10) to become practically identifiable?

Synthetic Cole+L spectra with tissue-like parameters and f_c below 10 kHz (as in the measured
spectra) are generated on bands [f_min, 100 kHz] with 10 log-spaced points per decade and 1%
complex Gaussian noise. Each spectrum is fitted globally with VarPro and profiled with the
globally optimised VarPro profile likelihood (same 3.84 rule as the main analysis).
Usage: python3 band_design.py [n_spectra]
"""
import sys, time, pickle, numpy as np, pandas as pd
from pathlib import Path
from vp_core import VarPro, VP_SPECS
from run_vp_profiles import profile, min_nonlinear
from scipy.optimize import least_squares
from eis_core import MODELS

SEED0 = 20261101 + 990000
N = int(sys.argv[1]) if len(sys.argv) > 1 else 40
FMIN = [10e3, 3e3, 1e3, 300.0, 100.0]
NOISE = [0.01, 0.025]   # 2.5% ~ residual level of the KK-consistent measured spectra (M10 NRMSE 0.020-0.036)
TAB = Path(__file__).parent.parent/'results'/'tables'; CACHE = TAB.parent/'cache_band'; CACHE.mkdir(exist_ok=True)

def cole(x, w):
    Rinf, dR, tau, a, L = 10**x[0], 10**x[1], 10**x[2], x[3], 10**x[4]
    return Rinf + dR/(1 + (1j*w*tau)**a) + 1j*w*L

def truth(i):
    rng = np.random.default_rng(SEED0 + i)
    Rinf = 10**rng.uniform(np.log10(500), np.log10(5000)); a = rng.uniform(0.60, 0.95); L = 10**rng.uniform(-3.4, -2.6)
    fc = 10**rng.uniform(np.log10(100), np.log10(3000)); tau = 1/(2*np.pi*fc)
    target = rng.uniform(2, 4)*Rinf; w10 = np.array([2*np.pi*1e4]); lo, hi = 0.0, 9.0
    for _ in range(60):   # dR such that |Z(10 kHz)| is 2-4 x Rinf, as in the measured spectra
        mid = (lo+hi)/2; zt = cole([np.log10(Rinf), mid, np.log10(tau), a, np.log10(L)], w10)
        lo, hi = (mid, hi) if abs(zt[0]) < target else (lo, mid)
    return np.array([np.log10(Rinf), (lo+hi)/2, np.log10(tau), a, np.log10(L)]), fc, rng

def vp_fit(w, z):
    """global VarPro fit of M10 (dense 2-D grid + TRF polish); returns eis_core log10 parameter vector."""
    vp = VarPro('M10_ColeL', w, z); lo, hi = vp.sp['lo'], vp.sp['hi']
    g = np.array([[a, b] for a in np.linspace(lo[0], hi[0], 33) for b in np.linspace(lo[1], hi[1], 15)])
    v = np.array([vp.f(p) for p in g]); best, bx = np.inf, None
    for k in np.argsort(v)[:3]:
        r = least_squares(vp.res, g[k], bounds=(lo, hi), xtol=1e-12, ftol=1e-12, gtol=1e-12, max_nfev=400)
        if 2*r.cost < best: best, bx = 2*r.cost, r.x
    return vp.full_params(bx)

rows = []; t0 = time.time()
for i in range(N):
    p = CACHE/f'band_{i}.pkl'
    if p.exists(): rows += pickle.load(open(p, 'rb')); continue
    xt, fc, rng = truth(i); out = []
    noise_seed = rng.integers(1 << 30)
    for fmin, sig in [(a, b) for b in NOISE for a in FMIN]:
        f = np.logspace(np.log10(fmin), 5, int(round(10*np.log10(1e5/fmin))) + 1); w = 2*np.pi*f
        z0 = cole(xt, w); nr = np.random.default_rng(noise_seed)
        z = z0 + sig*np.abs(z0)*(nr.standard_normal(len(w)) + 1j*nr.standard_normal(len(w)))/np.sqrt(2)
        xh = vp_fit(w, z); s = dict(w=w, z=z)
        rec = dict(spectrum=i, noise=sig, fc_true_hz=fc, fmin_hz=fmin, n_freq=len(w), ratio_fmin_fc=fmin/fc)
        for pn in ['logdR', 'logtau']:
            o, _, _ = profile('M10_ColeL', s, xh, pn); j = MODELS['M10_ColeL'][0].index(pn)
            rec.update({f'{pn}_ident': o['identifiable'], f'{pn}_ciw': o['ci_width'], f'{pn}_abserr': abs(xh[j]-xt[j])})
        out.append(rec)
    pickle.dump(out, open(p, 'wb')); rows += out
    print(time.strftime('%H:%M:%S'), i, f'fc={fc:.0f}', f'{time.time()-t0:.0f}s', flush=True)

D = pd.DataFrame(rows); D.to_csv(TAB/'band_design_runs.csv', index=False)
S = D.groupby(['noise', 'fmin_hz']).agg(n=('spectrum', 'size'), n_freq=('n_freq', 'first'),
        tau_identifiable_pct=('logtau_ident', lambda x: 100*x.mean()), dR_identifiable_pct=('logdR_ident', lambda x: 100*x.mean()),
        median_tau_ciw=('logtau_ciw', 'median'), median_dR_ciw=('logdR_ciw', 'median'),
        median_tau_abserr=('logtau_abserr', 'median'), median_dR_abserr=('logdR_abserr', 'median')).reset_index().sort_values(['noise', 'fmin_hz'], ascending=[True, False])
S.to_csv(TAB/'Table_band_design.csv', index=False); print(S.round(3).to_string())
D['ratio_bin'] = pd.cut(D.ratio_fmin_fc, [0, 0.3, 1, 3, 10, 1000], labels=['<0.3', '0.3-1', '1-3', '3-10', '>10'])
R = D.groupby(['noise', 'ratio_bin'], observed=True).agg(n=('spectrum', 'size'), tau_identifiable_pct=('logtau_ident', lambda x: 100*x.mean()),
        dR_identifiable_pct=('logdR_ident', lambda x: 100*x.mean()), median_tau_ciw=('logtau_ciw', 'median')).reset_index()
R.to_csv(TAB/'Table_band_design_ratio.csv', index=False); print(R.round(3).to_string())

In [ ]:
%%writefile src/make_fig_band.py
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from pathlib import Path
T = Path(__file__).parent.parent/'results'; D = pd.read_csv(T/'tables'/'band_design_runs.csv')
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.5))
C = {0.01: '#2166ac', 0.025: '#b2182b'}
for sig, g in D.groupby('noise'):
    for pn, ls, lab in [('logtau', '-', r'$\log_{10}\tau$'), ('logdR', '--', r'$\log_{10}\Delta R$')]:
        q = g.groupby('fmin_hz')[f'{pn}_ciw'].quantile([.25, .5, .75]).unstack()
        ax[0].plot(q.index, q[0.5], ls=ls, marker='o', ms=3.5, color=C[sig], label=f'{lab}, noise {100*sig:g}%')
        ax[0].fill_between(q.index, q[0.25], q[0.75], color=C[sig], alpha=.10, lw=0)
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].invert_xaxis()
ax[0].set_xlabel('Lower band limit $f_{\\min}$ (Hz); upper limit 100 kHz'); ax[0].set_ylabel('95% profile CI width (decades)')
ax[0].axvline(1e4, color='0.5', lw=.8, ls=':'); ax[0].text(9e3, ax[0].get_ylim()[1]*0.6, 'measured\nband', fontsize=7.5, color='0.4', ha='left', va='top')
ax[0].legend(fontsize=7, frameon=False, loc='lower left'); ax[0].set_title('(a) Precision versus band extension', fontsize=9, loc='left')
for sig, g in D.groupby('noise'):
    ax[1].scatter(g.ratio_fmin_fc, g.logtau_ciw, s=9, alpha=.55, color=C[sig], label=f'noise {100*sig:g}%', edgecolors='none')
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].axvline(1, color='0.4', lw=.8, ls='--')
ax[1].set_xlabel('$f_{\\min}/f_c$'); ax[1].set_ylabel(r'$\log_{10}\tau$ CI width (decades)')
ax[1].legend(fontsize=7, frameon=False, loc='upper left'); ax[1].set_title(r'(b) Precision versus $f_{\min}/f_c$', fontsize=9, loc='left')
plt.tight_layout(); plt.savefig(T/'figures_vp'/'Fig16_band_design.png', dpi=300)

### C.2 Sensitivity to the small sample (Table S5)

In [ ]:
%%writefile src/sensitivity.py
"""sensitivity.py - robustness of the main conclusions to the small sample:
leave-one-animal(subset)-out, KK-consistent spectra only, and bootstrap over spectra."""
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path
T = Path(__file__).parent.parent/'results'/'tables'
sub = pd.read_csv(T/'Table_subset_mapping.csv').set_index('source_file').original_subset
kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct
prof = pd.read_csv(T/'Table_VP_profile_likelihood.csv')
SETS = {'All spectra': lambda s: True}
for g in sorted(sub.unique()): SETS[f'Without {g}'] = (lambda s, g=g: sub[s] != g)
SETS['KK-consistent only'] = lambda s: kk[s] < 3
rng = np.random.default_rng(20261017); rows = []
for name, keep in SETS.items():
    rec = dict(subset=name)
    for m, tag in [('M10_ColeL', 'M10'), ('M12_DoubleColeL', 'M12')]:
        d = pd.read_csv(T/f'vp_merged_{m}.csv'); d = d[d.source_file.map(keep)]
        rec['n_spectra'] = d.source_file.nunique()
        for a in ['L-SHADE', 'ASHO-LSHADE', 'MS-TRF', 'VP-L-SHADE', 'VP-ASHO-LSHADE', 'VP-MS-TRF']:
            rec[f'{tag} {a}'] = 100*d[d.algorithm == a].success.mean()
        med = d.groupby(['source_file', 'algorithm']).NRMSE.median().unstack()
        rel = (med['VP-MS-TRF']-med['MS-TRF'])/med['MS-TRF']
        rec[f'{tag} VP-MS better/worse'] = f"{int((rel < -1e-3).sum())}/{int((rel > 1e-3).sum())}"
        try: rec[f'{tag} p VP-ASHO vs VP-L-SHADE'] = wilcoxon(med['VP-ASHO-LSHADE'], med['VP-L-SHADE']).pvalue
        except ValueError: rec[f'{tag} p VP-ASHO vs VP-L-SHADE'] = 1.0
        # bootstrap over spectra: difference in success rate VP-MS-TRF - MS-TRF
        sr = d.groupby(['source_file', 'algorithm']).success.mean().unstack()
        diff = (sr['VP-MS-TRF']-sr['MS-TRF']).to_numpy(); B = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(10000)]
        rec[f'{tag} dSuccess VP-MS minus MS (pp)'] = 100*diff.mean(); rec[f'{tag} dSuccess CI95'] = f"{100*np.quantile(B, .025):.1f} to {100*np.quantile(B, .975):.1f}"
    p = prof[prof.source_file.map(keep)]
    for m, pn in [('M10_ColeL', 'logtau'), ('M10_ColeL', 'logdR'), ('M11_RCPEL', 'logQ'), ('M11_RCPEL', 'alpha')]:
        x = p[(p.model == m) & (p.parameter == pn)]
        rec[f'{m[:3]} {pn} identifiable'] = f"{int(x.identifiable.sum())}/{len(x)}"
    rows.append(rec)
R = pd.DataFrame(rows); R.to_csv(T/'Table_sensitivity.csv', index=False)
pd.set_option('display.width', 250); print(R.T.to_string())

### C.3 Weakly separable nested model M13 (Table S6)

In [ ]:
%%writefile src/nested_vp.py
"""nested_vp.py - limit of the variable-projection gain for a weakly separable (nested) circuit.

M13: Cole + L tissue model shunted by a parallel constant-phase element (stray/electrode capacitance),
    Z = 1 / (1/Z_C + Q_p (jw)^beta) + jwL,   Z_C = Rinf + dR/(1 + (jw tau)^alpha).
Because R_inf and dR are nested inside the inversion, only L enters linearly: partial VarPro
eliminates one of seven parameters (6-D nonlinear search), compared with 3 of 5 for M10.
Same 20 spectra, seeds, 25 runs, 2,000 NFE and 0.1% success rule as the main benchmark.
Usage: python3 nested_vp.py [runs]
"""
import sys, math, time, pickle, numpy as np, pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from scipy.stats import wilcoxon
from eis_core import lshade_family, Tracker, _Budget
from vp_common import load_spectra, TAB

NAMES = ['logRinf', 'logdR', 'logtau', 'alpha', 'logQp', 'beta', 'logL']
LO = np.array([-3., -3., -7., 0.30, -14., 0.50, -9.]); HI = np.array([7., 8., 1., 1.0, -6., 1.0, 0.])
RUNS = int(sys.argv[1]) if len(sys.argv) > 1 else 25; BUDGET = 2000; SEED0 = 20261101 + 330000
CPS = (100, 200, 500, 1000, 1500, 2000)
CACHE = TAB.parent/'cache_nested'; CACHE.mkdir(exist_ok=True)

def zpar(x, w):
    jw = 1j*w; zc = 10**x[0] + 10**x[1]/(1 + (jw*10**x[2])**x[3])
    return 1/(1/zc + 10**x[4]*jw**x[5])

class Problem:
    def __init__(self, w, z):
        self.w, self.z = w, z; self.wt = 1/np.maximum(np.abs(z), 1.0); self.N = len(w)
    # full parameterization
    def res(self, x):
        r = (zpar(x, self.w) + 1j*self.w*10**x[6] - self.z)*self.wt; return np.r_[r.real, r.imag]
    def f(self, x): r = self.res(x); return float(r @ r)/self.N
    # partial VarPro: L solved exactly (scalar bounded LS; clipping is exact for one variable)
    def res_vp(self, t):
        a = 1j*self.w*self.wt; b = (self.z - zpar(t, self.w))*self.wt
        A = np.r_[a.real, a.imag]; B = np.r_[b.real, b.imag]
        L = float(np.clip((A @ B)/(A @ A), 10**LO[6], 10**HI[6])); return A*L - B
    def f_vp(self, t): r = self.res_vp(t); return float(r @ r)/self.N

def ms(fun_res, fun_f, lo, hi, seed):
    rng = np.random.default_rng(seed); T = Tracker(fun_f, BUDGET, CPS)
    def r(x):
        if T.nfe >= BUDGET: raise _Budget
        T(x); return fun_res(x)
    while T.nfe < BUDGET:
        try: least_squares(r, rng.uniform(lo, hi), bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except _Budget: break
    return T.best, T.trace

def run(alg, P, seed):
    t0 = time.perf_counter()
    if alg == 'MS-TRF': fb, tr = ms(P.res, P.f, LO, HI, seed)
    elif alg == 'VP-MS-TRF': fb, tr = ms(P.res_vp, P.f_vp, LO[:6], HI[:6], seed)
    elif alg == 'L-SHADE': _, fb, tr, _ = lshade_family(P.f, LO, HI, BUDGET, seed, 'lshade', CPS)
    elif alg == 'VP-L-SHADE': _, fb, tr, _ = lshade_family(P.f_vp, LO[:6], HI[:6], BUDGET, seed, 'lshade', CPS)
    tr = {c: tr.get(c, fb) for c in CPS}
    return fb, tr, time.perf_counter()-t0

if __name__ == '__main__':
    ALGS = ['MS-TRF', 'VP-MS-TRF', 'L-SHADE', 'VP-L-SHADE']; rows = []
    for s in load_spectra():
        p = CACHE/f"m13_{s['si']}_{RUNS}.pkl"
        if p.exists(): rows += pickle.load(open(p, 'rb')); continue
        P = Problem(s['w'], s['z']); out = []
        for r in range(RUNS):
            seed = SEED0 + s['si']*1000 + r
            for a in ALGS:
                fb, tr, sec = run(a, P, seed)
                out.append(dict(source_file=s['sf'], tissue=s['tissue'], algorithm=a, run=r+1, NRMSE=math.sqrt(fb),
                                runtime_sec=sec, **{f'trace_{c}': math.sqrt(v) for c, v in tr.items()}))
        pickle.dump(out, open(p, 'wb')); rows += out
        print(time.strftime('%H:%M:%S'), s['si'], s['tissue'], flush=True)
    D = pd.DataFrame(rows); ref = D.groupby('source_file').NRMSE.min(); D['ref'] = D.source_file.map(ref)
    for c in CPS: D[f'succ_{c}'] = (D[f'trace_{c}'] - D.ref)/D.ref <= 1e-3
    D.to_csv(TAB/'nested_m13_runs.csv', index=False)
    S = D.groupby('algorithm').agg(**{f'success_at_{c}': (f'succ_{c}', 'mean') for c in CPS},
                                    spectra_all_success=('succ_2000', lambda x: 0)).reset_index()
    S['spectra_all_success'] = [int(D[D.algorithm == a].groupby('source_file').succ_2000.mean().eq(1).sum()) for a in S.algorithm]
    S.to_csv(TAB/'Table_nested_M13.csv', index=False)
    med = D.groupby(['source_file', 'algorithm']).NRMSE.median().unstack(); P_ = []
    for a, b in [('VP-MS-TRF', 'MS-TRF'), ('VP-L-SHADE', 'L-SHADE')]:
        rel = (med[a]-med[b])/med[b]
        try: p = wilcoxon(med[a], med[b]).pvalue
        except ValueError: p = 1.0
        P_.append(dict(comparison=f'{a} vs {b}', better=int((rel < -1e-3).sum()), worse=int((rel > 1e-3).sum()), p=p))
    pd.DataFrame(P_).to_csv(TAB/'Table_nested_M13_paired.csv', index=False)
    pd.set_option('display.width', 200); print(S.round(3).to_string()); print(pd.DataFrame(P_).to_string())

### C.4 Stability of fitted parameters as diagnostic features (Fig. 7, Table S7)

In [ ]:
%%writefile src/feature_stability.py
"""feature_stability.py - consequence of non-identifiability for diagnostic features.
Uses the 40 reading-level resamples per spectrum (repeatability_bootstrap_fits.csv). For each fitted
parameter, spectra are ranked separately in every resample; stability is summarized by
(i) the median Spearman correlation between rankings from different resamples, and
(ii) the share of spectrum pairs whose ordering is the same in at least 95% of resamples."""
import itertools, numpy as np, pandas as pd
from scipy.stats import spearmanr
from pathlib import Path
T = Path(__file__).parent.parent/'results'/'tables'
d = pd.read_csv(T/'repeatability_bootstrap_fits.csv'); kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct
FEATS = [('M10_ColeL', 'logtau', 'M10 log10 tau'), ('M10_ColeL', 'logdR', 'M10 log10 dR'), ('M10_ColeL', 'alpha', 'M10 alpha'),
         ('M10_ColeL', 'logRinf', 'M10 log10 Rinf'), ('M11_RCPEL', 'logQ', 'M11 log10 Q'), ('M11_RCPEL', 'alpha', 'M11 alpha'),
         ('M11_RCPEL', 'logRinf', 'M11 log10 Rinf')]
rows = []
for subset, keep in [('KK-consistent (12)', lambda s: kk[s] < 3), ('All spectra (20)', lambda s: True)]:
    for m, p, lab in FEATS:
        x = d[(d.model == m) & d.source_file.map(keep)].pivot_table(index='b', columns='source_file', values=p)
        X = x.to_numpy()                                   # resamples x spectra
        rho = [spearmanr(X[i], X[j]).statistic for i, j in itertools.combinations(range(len(X)), 2)]
        n = X.shape[1]; stable = 0
        for i, j in itertools.combinations(range(n), 2):
            agree = np.mean(X[:, i] > X[:, j]); stable += max(agree, 1-agree) >= 0.95
        npairs = n*(n-1)//2
        rows.append(dict(subset=subset, feature=lab, median_rank_rho=np.median(rho), q05_rank_rho=np.quantile(rho, .05),
                         stable_pairs_pct=100*stable/npairs, n_pairs=npairs))
R = pd.DataFrame(rows); R.to_csv(T/'Table_feature_stability.csv', index=False)
pd.set_option('display.width', 200); print(R.round(3).to_string())

In [ ]:
%%writefile src/make_fig_stability.py
import pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from pathlib import Path
T = Path(__file__).parent.parent/'results'
R = pd.read_csv(T/'tables'/'Table_feature_stability.csv'); R = R[R.subset.str.startswith('KK')]
order = ['M10 log10 tau', 'M10 log10 dR', 'M10 alpha', 'M11 log10 Q', 'M11 alpha', 'M10 log10 Rinf', 'M11 log10 Rinf']
lab = {'M10 log10 tau': r'M10 $\log_{10}\tau$', 'M10 log10 dR': r'M10 $\log_{10}\Delta R$', 'M10 alpha': r'M10 $\alpha$',
       'M11 log10 Q': r'M11 $\log_{10}Q$', 'M11 alpha': r'M11 $\alpha$', 'M10 log10 Rinf': r'M10 $\log_{10}R_\infty$', 'M11 log10 Rinf': r'M11 $\log_{10}R_\infty$'}
R = R.set_index('feature').loc[order[::-1]]
col = ['#b2182b' if f in order[:3] else '#2166ac' for f in R.index]
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
y = range(len(R))
ax[0].barh(y, R.median_rank_rho, color=col); ax[0].errorbar(R.q05_rank_rho, y, fmt='|', color='0.3', ms=8)
ax[0].set_yticks(list(y)); ax[0].set_yticklabels([lab[f] for f in R.index]); ax[0].set_xlim(-0.1, 1.02); ax[0].axvline(0, color='0.6', lw=.6)
ax[0].set_xlabel('Spearman correlation of spectrum rankings\nbetween resamples (bar: median; tick: 5th percentile)')
ax[0].set_title('(a) Ranking stability', fontsize=9, loc='left')
ax[1].barh(y, R.stable_pairs_pct, color=col); ax[1].set_xlim(0, 100)
for yi, v in zip(y, R.stable_pairs_pct): ax[1].text(v+1.5, yi, f'{v:.0f}%', va='center', fontsize=8)
ax[1].set_xlabel('Spectrum pairs ordered identically\nin at least 95% of resamples (%)')
ax[1].set_title('(b) Reliable pairwise discrimination', fontsize=9, loc='left')
plt.tight_layout(); plt.savefig(T/'figures_vp'/'Fig17_feature_stability.png', dpi=300)

### C.5 Reporting decision diagram (Fig. 9)

In [ ]:
%%writefile src/make_fig_decision.py
"""Reporting decision diagram for Cole parameters (manuscript Fig. 10)."""
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon
from pathlib import Path
plt.rcParams.update({'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold', 'font.size': 8.5})
fig = plt.figure(figsize=(8.6, 6.2)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 72); ax.axis('off')
def box(x, y, w, h, txt, fc='white', ec='black', bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.2', fc=fc, ec=ec, lw=1.1))
    ax.text(x+w/2, y+h/2, txt, ha='center', va='center', fontsize=8.5, fontweight='bold' if bold else 'normal', linespacing=1.35)
def diamond(xc, yc, w, h, txt):
    ax.add_patch(Polygon([(xc, yc+h/2), (xc+w/2, yc), (xc, yc-h/2), (xc-w/2, yc)], closed=True, fc='#f7f7f7', ec='black', lw=1.1))
    ax.text(xc, yc, txt, ha='center', va='center', fontsize=8.2, linespacing=1.3)
def arr(x1, y1, x2, y2, lab=None, lx=0, ly=0):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='-|>', lw=1.0, color='black', mutation_scale=11))
    if lab: ax.text((x1+x2)/2+lx, (y1+y2)/2+ly, lab, fontsize=8, fontstyle='italic')
box(30, 64, 40, 6, 'Measured spectrum $Z(f)$, band $[f_{\\min}, f_{\\max}]$', bold=True)
arr(50, 64, 50, 60.6)
diamond(50, 56, 34, 9, 'Linear Kramers–Kronig\nRMS residual < 3%?')
arr(67, 56, 74, 56, 'no', -3, 1)
box(74, 51.5, 24, 9, 'Flag spectrum:\nre-measure or interpret\nwith caution', fc='#fdecea', ec='#b2182b')
arr(50, 51.5, 50, 47.6, 'yes', 1, 0)
box(22, 40, 56, 7.6, 'Fit Cole+L and R–CPE–L by variable projection\nwith multistart local refinement; compare by AICc', bold=False)
arr(50, 40, 50, 36.6)
diamond(50, 30.5, 50, 12, 'Characteristic frequency inside the band\n($f_{\\min} \\leq f_c$) and calibrated 95% profile\ninterval of $\\log_{10}\\tau$ closed and < ~0.5 decade?')
ax.plot([25, 14], [30.5, 30.5], color='black', lw=1.0); arr(14, 30.5, 14, 27.6); ax.text(18.5, 31.6, 'yes', fontsize=8, fontstyle='italic')
box(1, 13, 26, 14.5, 'Report Cole parameters\n$R_0$, $R_\\infty$, $\\tau$ ($f_c$), $\\alpha$\nwith calibrated profile intervals', fc='#e8f1fb', ec='#2166ac', bold=False)
ax.plot([75, 84.5], [30.5, 30.5], color='black', lw=1.0); arr(84.5, 30.5, 84.5, 27.6); ax.text(78.5, 31.6, 'no', fontsize=8, fontstyle='italic')
box(70, 13, 29, 14.5, 'Report R–CPE–L descriptors\n$R_\\infty$, $Q$, $\\alpha$ (band-specific; state\nband and HF correction); do not\ninterpret $\\tau$, $\\Delta R$, $R_0$, $f_c$', fc='#e8f1fb', ec='#2166ac')
ax.plot([84.5, 84.5], [13, 7.8], color='black', lw=1.0, ls=(0, (3, 2))); arr(84.5, 7.8, 66.2, 7.8); ax.text(75.3, 5.4, 'if Cole parameters\nare needed', fontsize=7.6, fontstyle='italic', ha='center', va='top')
box(24, 3, 42, 9.6, 'To estimate Cole parameters, extend the band\nto $f_{\\min} \\leq f_c$ of the target tissue, control electrode\npolarization (e.g. tetrapolar), and repeat the assessment', fc='#fff7e6', ec='#b35806')
ax.text(50, 0.6, 'Thresholds are pragmatic suggestions from the band-design simulation; calibrate interval thresholds to the error model of the instrument.',
        ha='center', fontsize=7, color='0.35')
fig.savefig(Path(__file__).parent.parent/'results'/'figures'/'Fig18_reporting_decision_v2.png', dpi=300)


### C.6 Workflow diagram (Fig. 1)

In [ ]:
%%writefile src/make_fig1_workflow.py
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Rectangle
plt.rcParams.update({'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold'})
W,H=100,142
fig=plt.figure(figsize=(11,15.6)); ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,W); ax.set_ylim(0,H); ax.axis('off')
HL='#e8f1fb'; HE='#1f5f9e'
def box(x,y,w,h,title=None,lines=(),fs=10,tfs=11,fc='white',ec='black',lw=1.1,center=False,rounded=0.6):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle=f'round,pad=0,rounding_size={rounded}',fc=fc,ec=ec,lw=lw))
    if center:
        t=title if title else ''
        if lines: t=(r'$\bf{'+title.replace(' ',r'\ ')+'}$\n' if title else '')+'\n'.join(lines)
        ax.text(x+w/2,y+h/2,t if lines else r'$\bf{'+title.replace(' ',r'\ ')+'}$',ha='center',va='center',fontsize=fs,linespacing=1.35)
        return
    yy=y+h-1.3
    if title:
        ax.text(x+1.2,yy,title,ha='left',va='top',fontsize=tfs,fontweight='bold'); yy-=2.6
    for l in lines:
        ax.text(x+2.0,yy,'\u2022  '+l,ha='left',va='top',fontsize=fs); yy-=2.25
def para(xc,y,w,h,title,sub):
    s=2.2
    ax.add_patch(Polygon([(xc-w/2+s,y+h),(xc+w/2+s,y+h),(xc+w/2-s,y),(xc-w/2-s,y)],closed=True,fc='white',ec='black',lw=1.1))
    ax.text(xc,y+h*0.64,title,ha='center',va='center',fontsize=11,fontweight='bold')
    ax.text(xc,y+h*0.27,sub,ha='center',va='center',fontsize=9.5)
def arrow(x,y1,y2):
    ax.annotate('',xy=(x,y2),xytext=(x,y1),arrowprops=dict(arrowstyle='-|>',lw=1.1,color='black',mutation_scale=12))
def group(y,h,title,ec='black',fc='none'):
    ax.add_patch(Rectangle((2,y),96,h,fill=fc!='none',fc=fc,ec=ec,lw=1.0,ls=(0,(5,3))))
    ax.text(3,y+h-1.2,title,ha='left',va='top',fontsize=12,fontweight='bold',color=ec if ec!='black' else 'black')
cx=50
# start
box(42,138,16,3,'Start',center=True,rounded=1.4); arrow(cx,138,135.8)
para(cx,131,40,4.8,'Raw EIS spectra','In vivo porcine dataset: 12 tissues, 10\u2013100 kHz')
arrow(cx,131,129)
box(28,118.4,44,10.6,'Data preprocessing and quality control',['Median of readings; 51 complete repeated sweeps','Retain complete 10-frequency spectra (n = 20)','Linear Kramers\u2013Kronig check (3% threshold)'],fs=9.5)
arrow(cx,118.4,116.4)
box(28,105.8,44,10.6,'Equivalent-circuit model selection',['Fit 11 candidate ECMs; weighted AICc and BIC','Reduced R\u2013CPE\u2013L (M11) preferred','M10 (Cole + L) and M12 (double Cole + L) kept'],fs=9.5)
arrow(cx,105.8,103.4)
# optimization group
gy=67.2; gh=36.0
group(gy,gh,'Parameter optimization  (M10, M11, M12; 20 spectra \u00d7 25 runs \u00d7 2,000 NFE)')
ax.text(4,gy+gh-5.0,'(a) Original parameterization: all 4\u20138 parameters searched',fontsize=10.5,fontstyle='italic',va='top')
algs=['HO','L-SHADE','L-SHADE-\nRestart','SAHO-\nLSHADE','ASHO-\nLSHADE','MS-TRF','L-SHADE\n+TRF','ASHO-LSHADE\n+TRF']
bw=10.9; gap=(92-8*bw)/7
for i,a in enumerate(algs):
    box(4+i*(bw+gap),gy+gh-13.6,bw,6.4,a,center=True,fs=8.8) if False else None
    x=4+i*(bw+gap); ax.add_patch(FancyBboxPatch((x,gy+gh-13.6),bw,6.4,boxstyle='round,pad=0,rounding_size=0.5',fc='white',ec='black',lw=1))
    ax.text(x+bw/2,gy+gh-10.4,a,ha='center',va='center',fontsize=8.8,linespacing=1.15)
# VarPro box
vy=gy+1.4; vh=17.6
ax.add_patch(FancyBboxPatch((4,vy),92,vh,boxstyle='round,pad=0,rounding_size=0.8',fc=HL,ec=HE,lw=1.8))
ax.text(5.5,vy+vh-1.2,'(b) Variable-projection reformulation',fontsize=11,fontweight='bold',color=HE,va='top')
ax.text(5.5,vy+vh-4.3,r'$Z(\omega)=\boldsymbol{\varphi}(\omega;\boldsymbol{\eta})^{\top}\mathbf{c}$:  linear coefficients $\mathbf{c}=(R_\infty,\ \Delta R\ \mathrm{or}\ 1/Q,\ L)$ solved exactly by'
        '\nbound-constrained linear least squares;  search only the nonlinear $\\boldsymbol{\\eta}$ (1, 2 or 4 dimensions)',fontsize=9.6,va='top',linespacing=1.4)
vals=['VP-L-SHADE','VP-ASHO-\nLSHADE','VP-MS-TRF','VP-L-SHADE\n+TRF','VP-Screen\n+TRF']
bw2=16.4; gap2=(88-5*bw2)/4
for i,a in enumerate(vals):
    x=6+i*(bw2+gap2); ax.add_patch(FancyBboxPatch((x,vy+1.3),bw2,5.6,boxstyle='round,pad=0,rounding_size=0.5',fc='white',ec=HE,lw=1.1))
    ax.text(x+bw2/2,vy+4.1,a,ha='center',va='center',fontsize=8.8,linespacing=1.15)
arrow(cx,gy+gh-13.6,vy+vh+0.1)
ax.text(cx+1,gy+gh-15.3,'same spectra, seeds, budget and success criterion',fontsize=8.6,fontstyle='italic',va='center',ha='left')
arrow(cx,gy,gy-2.2)
# performance
py=46.4; ph=18.4
group(py,ph,'Performance evaluation')
box(4,py+1.2,29.5,13.4,'Benchmark comparison',['Success rate vs. NFE','Friedman, Wilcoxon, Holm','Bootstrap 95% CIs','10,000-NFE budget check'],fs=9,tfs=10)
box(35.2,py+1.2,29.5,13.4,'Mechanism and rescue',['VPinit control (projection vs. start)','Jacobian condition numbers','HO rescue: ±5 pp equivalence','Restart / fixed-rescue ablation'],fs=9,tfs=10)
box(66.5,py+1.2,29.5,13.4,'Computational cost',['NFE to first success','Wall-clock time to solution','Same computer for both','MS-TRF vs. VP-MS-TRF'],fs=9,tfs=10)
arrow(cx,py,py-2.2)
# identifiability
iy=25.6; ih=18.4
group(iy,ih,'Identifiability and calibrated uncertainty')
box(4,iy+1.2,29.5,13.4,'Profile likelihood',['Global inner fit via VP','Error model from 51 sweeps','Bootstrap-calibrated, GLS','Coverage validation'],fs=9,tfs=10)
box(35.2,iy+1.2,29.5,13.4,'Synthetic experiments',['$f_c$ inside vs. below band','Bias of R–CPE–L descriptors','Band design: range vs. $N$','Electrode polarization, mismatch'],fs=9,tfs=10)
box(66.5,iy+1.2,29.5,13.4,'Robustness',['Width thresholds, bounds','High-frequency correction','Model selection (GLS, LOFO)','Sweep-level repeatability'],fs=9,tfs=10)
arrow(cx,iy,iy-2.2)
# interpretation
qy=6.0; qh=17.4
group(qy,qh,'Physical and medical interpretation')
box(4,qy+1.2,45,12.4,'Identifiable parameters (M11)',['$R_\\infty$, $Q$ and $\\alpha$ interpreted','Effective $L$ interpreted conservatively','Extrapolated $R_0$ and $f_c$ not interpreted'],fs=9,tfs=10)
box(51,qy+1.2,45,12.4,'Descriptive tissue characterization',['Parameter distributions across tissues','Band-specific descriptors with intervals','Reporting recommendations for tissue EIS'],fs=9,tfs=10)
arrow(cx,qy,qy-2.2)
box(42,0.6,16,3,'End',center=True,rounded=1.4)
fig.savefig(str(__import__('pathlib').Path(__file__).parent.parent/'results'/'figures'/'Fig01_Methods_v2.png'),dpi=300)


In [ ]:
%%time
!cd src && python band_design.py 30 > ../results/log_band.txt 2>&1; tail -n 14 ../results/log_band.txt
!cd src && python make_fig_band.py
!cd src && python sensitivity.py 2>&1 | grep -v Warning
!cd src && python nested_vp.py 25 > ../results/log_nested.txt 2>&1; tail -n 8 ../results/log_nested.txt
!cd src && python feature_stability.py && python make_fig_stability.py && python make_fig_decision.py && python make_fig1_workflow.py

In [ ]:
from IPython.display import Image, display
for f in ['Fig01_Methods_EISTIS', 'FigVP1_success_vs_NFE', 'Fig17_feature_stability', 'Fig16_band_design', 'Fig18_reporting_decision']:
    display(Image(f'results/figures_vp/{f}.png', width=820))

---
# Part D — `eisvarpro` open-source package
Written to `eisvarpro/` (installable with `pip install ./eisvarpro`). The demo applies the complete workflow to every one of the 20 spectra.

In [ ]:
for d_ in ['eisvarpro/eisvarpro', 'eisvarpro/tests', 'eisvarpro/examples']: os.makedirs(d_, exist_ok=True)

In [ ]:
%%writefile eisvarpro/pyproject.toml
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "eisvarpro"
version = "0.3.3"
description = "Variable-projection fitting, Kramers-Kronig screening and identifiability checks for tissue impedance spectra"
readme = "README.md"
requires-python = ">=3.9"
license = {text = "MIT"}
authors = [{name = "Anonymous (withheld for double-blind review)"}]
dependencies = ["numpy>=1.22", "scipy>=1.8", "pandas>=1.4"]

[project.scripts]
eisvarpro = "eisvarpro.__main__:main"

[tool.setuptools]
packages = ["eisvarpro"]


In [ ]:
%%writefile eisvarpro/README.md
# eisvarpro

Reliable equivalent-circuit inversion of tissue electrical impedance spectra.

`eisvarpro` implements the workflow of the manuscript *Identifiability-Aware Parameter Estimation and
Frequency-Band Assessment of Equivalent-Circuit Models: Application to In Vivo Tissue Bioimpedance*
(authors withheld for double-blind review):

1. **Kramers–Kronig screening** (linear Voigt-element test; spectra above 3% RMS residual are flagged).
2. **Variable-projection fitting** of the Cole + inductance and resistance–CPE–inductance models: the
   resistive and inductive coefficients are solved exactly by bound-constrained linear least squares,
   so only one or two nonlinear parameters are searched (dense grid + local refinement = global fit).
3. **Globally optimised profile likelihood** for the Cole relaxation time (and any other parameter), with an
   optional **parametric-bootstrap calibration** of the likelihood-ratio threshold (`calibrated_threshold`,
   or `assess(..., calibrate=True, sigma=...)`). With few frequencies and correlated errors the chi-square
   value 3.84 can give intervals that cover the truth markedly less often than 95%.
4. **Reporting decision** (v0.3.3, Fig. 5 of the paper): every reported parameter must be practically identifiable,
   i.e. its (optionally calibrated) 95% profile interval is closed and narrower than 1 decade (log10 parameters) or
   0.2 (α). Cole parameters are reported only when the characteristic frequency lies in the band and both log10 τ and
   log10 ΔR meet the criterion; otherwise the R–CPE–L descriptors (R∞, Q, α) that meet it are reported and band
   extension is recommended. Calibration is conditional on the fitted model and the assumed error model.

## Install
```
pip install .            # from this folder; requires numpy, scipy, pandas
```

## Use
Command line (CSV columns `frequency_hz, resistance_ohm, reactance_ohm`):
```
eisvarpro examples/porcine_bladder_10-100kHz.csv
eisvarpro examples/synthetic_cole_100Hz-100kHz.csv --json
```
Python:
```python
import pandas as pd
from eisvarpro import assess, format_report, profile
d = pd.read_csv('examples/porcine_bladder_10-100kHz.csv')
Z = d.resistance_ohm + 1j*d.reactance_ohm
r = assess(d.frequency_hz, Z)          # dict with KK check, both fits, tau profile, decision
print(format_report(r))
p = profile('rcpe_l', d.frequency_hz, Z, 'log10_Q')   # profile of any parameter
```
Example output for the 10–100 kHz porcine bladder spectrum: Cole relaxation parameters are *not*
estimable (fc below the band, τ interval ≈ 4 decades); the R–CPE–L descriptors are reported.

Thresholds (`kk_threshold_pct=3.0`, `width_log=1.0`, `width_alpha=0.2`) are the paper's practical-identifiability
criterion; stricter values (e.g. 0.5 decade) can be passed when an application needs more precision.

Calibrated intervals:
```python
from eisvarpro import calibrated_threshold, profile
c = calibrated_threshold('rcpe_l', f, Z, 'log10_Q', n_boot=200, sigma=Sigma)  # Sigma: 2N x 2N covariance shape
P = profile('rcpe_l', f, Z, 'log10_Q', threshold=c)                           # of the stacked relative residual
```
`sigma=None` assumes independent errors. R–CPE–L descriptors are band-specific: report the band and the
high-frequency correction model with them.

Measurement design and reproducibility (`eisvarpro.design`, v0.3):
```python
from eisvarpro import band_criterion, rcpe_bias, cross_sweep_check, drift_variance, between_recording_sd, reproducibility_limits
c = band_criterion(f, Z)            # Gamma and detection limit f_c* = f_min/Gamma: tau identifiable only if f_c > f_c*
b = rcpe_bias(f, Z, fc_hz=300.0)    # predicted bias of log10 Q and alpha if the dispersion lies at 300 Hz
k = cross_sweep_check(f, sweeps, 'alpha')   # pairwise agreement of repeated sweeps within calibrated intervals
d = drift_variance([(est_rec1, se_rec1), (est_rec2, se_rec2)])   # REML drift SD between sweeps
b = between_recording_sd([logR_site_a, logR_site_b])              # pooled SD between replicate recordings (v0.3.2)
L = reproducibility_limits(s_fit, d['tau_drift'], b['sd'], differences=tissue_diffs)   # r = 2.77 s_w, R = 2.77 sqrt(s_w^2 + s_b^2)
```
Calibrated intervals describe one recording. When descriptors are compared across placements, sessions or probes,
use the reproducibility limit R, which includes the between-recording component.
The band criterion is a closed-form asymptotic result (valid for f_min/f_c >~ 1): the 95% width of the log10 tau
interval is about 2 z sigma (f_min/f_c)^alpha / (alpha ln10 D), with D computed from the spectrum.

## Tests
`python -m pytest tests` (nine tests: exact bounded solver, wide-band and narrow-band synthetic
spectra, real example, calibrated threshold, band criterion and bias formula, drift variance, refined width, reproducibility limits).

## Citation and data
Please cite the paper above. The example spectrum is taken from the in vivo tissue bioimpedance dataset
of Falagario et al., *Scientific Data* (2026), doi:10.1038/s41597-026-07652-y.

## License
MIT (see LICENSE).


In [ ]:
%%writefile eisvarpro/LICENSE
MIT License

Copyright (c) 2026 The authors (names withheld for double-blind review)

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and
associated documentation files (the "Software"), to deal in the Software without restriction, including
without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the
following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial
portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT
LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO
EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER
IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR
THE USE OR OTHER DEALINGS IN THE SOFTWARE.

In [ ]:
%%writefile eisvarpro/eisvarpro/__init__.py
"""eisvarpro - reliable equivalent-circuit inversion of tissue impedance spectra by variable projection,
with Kramers-Kronig screening, globally optimised profile likelihood and a band-coverage reporting rule."""
from .core import MODELS, VarPro, bounded_lstsq, global_min
from .analysis import kk_check, fit, profile, calibrated_threshold, assess, format_report
from .design import band_criterion, predicted_tau_width, rcpe_bias, cross_sweep_check, drift_variance, between_recording_sd, reproducibility_limits
__version__ = '0.3.3'
__all__ = ['MODELS', 'VarPro', 'bounded_lstsq', 'global_min', 'kk_check', 'fit', 'profile', 'calibrated_threshold', 'band_criterion', 'predicted_tau_width', 'rcpe_bias', 'cross_sweep_check', 'drift_variance', 'between_recording_sd', 'reproducibility_limits', 'assess', 'format_report']


In [ ]:
%%writefile eisvarpro/eisvarpro/core.py
"""Core numerics: models, exact bounded linear solve and variable projection (VarPro).

Impedance models (Z in ohm, w = 2*pi*f in rad/s):
  cole_l : Z = Rinf + dR / (1 + (j w tau)^alpha) + j w L      nonlinear (log10 tau, alpha), linear (Rinf, dR, L)
  rcpe_l : Z = Rinf + (1/Q) (j w)^(-alpha)       + j w L      nonlinear (alpha),            linear (Rinf, 1/Q, L)
All fits minimise J = mean_k |Z_model(w_k) - Z_k|^2 / max(|Z_k|, 1)^2 (relative complex residual).
"""
import itertools, math
import numpy as np
from scipy.optimize import nnls, least_squares, minimize_scalar

MODELS = {
    'cole_l': dict(nl=['log10_tau', 'alpha'], lo=np.array([-7.0, 0.30]), hi=np.array([1.0, 1.0]),
                   lin=['Rinf', 'dR', 'L'], llo=np.array([1e-3, 1e-3, 1e-9]), lhi=np.array([1e7, 1e8, 1.0])),
    'rcpe_l': dict(nl=['alpha'], lo=np.array([0.30]), hi=np.array([1.0]),
                   lin=['Rinf', 'invQ', 'L'], llo=np.array([1e-3, 1.0, 1e-9]), lhi=np.array([1e7, 1e12, 1.0])),
}

def basis(model, eta, w):
    jw = 1j*w; one = np.ones_like(w) + 0j
    if model == 'cole_l':
        return np.column_stack([one, 1/(1 + (jw*10**eta[0])**eta[1]), jw])
    if model == 'rcpe_l':
        return np.column_stack([one, jw**(-eta[0]), jw])
    raise KeyError(model)

def _enum_lstsq(A, b, lb, ub):
    k = A.shape[1]; s = np.linalg.norm(A, axis=0); s[s == 0] = 1.0; As = A/s
    best_c, best_v = None, math.inf
    for state in itertools.product((0, 1, 2), repeat=k):
        st = np.array(state); c = np.where(st == 1, lb, np.where(st == 2, ub, 0.0)); free = np.flatnonzero(st == 0)
        if len(free):
            c[free] = np.linalg.lstsq(As[:, free], b - A @ c, rcond=None)[0]/s[free]
            if np.any(c[free] < lb[free]*(1-1e-12)) or np.any(c[free] > ub[free]*(1+1e-12)): continue
        r = A @ c - b; v = float(r @ r)
        if v < best_v: best_v, best_c = v, c
    return best_c, best_v

def bounded_lstsq(A, b, lb, ub):
    """Exact solution of min ||A c - b||^2 subject to lb <= c <= ub (few columns):
    shifted NNLS, upper-bound clamping with KKT certificate, active-set enumeration fallback."""
    s = np.linalg.norm(A, axis=0); s[s == 0] = 1.0
    d, _ = nnls(A/s, b - A @ lb, maxiter=50*A.shape[1]); c = lb + d/s
    if np.all(c <= ub):
        r = A @ c - b; return c, float(r @ r)
    up = c > ub
    for _ in range(A.shape[1]):
        fr = ~up; cc = np.where(up, ub, lb)
        d, _ = nnls(A[:, fr]/s[fr], b - A[:, up] @ ub[up] - A[:, fr] @ lb[fr], maxiter=50*A.shape[1]); cc[fr] = lb[fr] + d/s[fr]
        new = up | (cc > ub)
        if np.array_equal(new, up): break
        up = new
    r = A @ cc - b; g = A.T @ r
    if np.all(cc <= ub*(1+1e-12)) and np.all(g[up] <= 1e-10*max(1.0, float(np.abs(g).max()))):
        return cc, float(r @ r)
    return _enum_lstsq(A, b, lb, ub)

class VarPro:
    """Reduced (variable-projection) problem for one spectrum and model. Optionally one linear
    coefficient can be fixed (used for profiling it)."""
    def __init__(self, model, f, Z, fix=None):
        self.model = model; self.sp = MODELS[model]; self.w = 2*np.pi*np.asarray(f, float); self.Z = np.asarray(Z, complex)
        self.wt = 1/np.maximum(np.abs(self.Z), 1.0); zw = self.Z*self.wt; self.b = np.r_[zw.real, zw.imag]; self.N = len(self.w)
        self.fix = fix                                    # (index, value) of a fixed linear coefficient
    def _system(self, eta):
        A = basis(self.model, eta, self.w)*self.wt[:, None]; A = np.vstack([A.real, A.imag]); b = self.b
        lb, ub = self.sp['llo'], self.sp['lhi']
        if self.fix is not None:
            j, v = self.fix; keep = [i for i in range(A.shape[1]) if i != j]
            return A[:, keep], b - A[:, j]*v, lb[keep], ub[keep], A, keep
        return A, b, lb, ub, A, list(range(A.shape[1]))
    def coef(self, eta):
        A, b, lb, ub, _, keep = self._system(np.asarray(eta, float)); c, rss = bounded_lstsq(A, b, lb, ub)
        if self.fix is not None:
            full = np.empty(len(keep)+1); full[keep] = c; full[self.fix[0]] = self.fix[1]; c = full
        return c, rss
    def rss(self, eta): return self.coef(eta)[1]
    def residual(self, eta):
        A, b, lb, ub, _, _ = self._system(np.asarray(eta, float)); c, _ = bounded_lstsq(A, b, lb, ub); return A @ c - b

def global_min(vp, fixed_nl=None, n1=81, grid2=(33, 15), polish=3):
    """Global minimum of the reduced RSS over the free nonlinear parameters (1-2 dims):
    dense grid followed by local refinement. fixed_nl=(index, value) fixes one nonlinear parameter."""
    lo, hi = vp.sp['lo'], vp.sp['hi']; D = len(lo); free = [i for i in range(D) if not (fixed_nl and i == fixed_nl[0])]
    def full(u):
        e = np.empty(D); e[free] = u
        if fixed_nl: e[fixed_nl[0]] = fixed_nl[1]
        return e
    if not free:
        e = full([]); return vp.rss(e), e
    if len(free) == 1:
        i = free[0]; g = np.linspace(lo[i], hi[i], n1); v = np.array([vp.rss(full([x])) for x in g]); k = int(v.argmin())
        r = minimize_scalar(lambda x: vp.rss(full([x])), bounds=(g[max(0, k-1)], g[min(n1-1, k+1)]), method='bounded', options={'xatol': 1e-8})
        return (r.fun, full([r.x])) if r.fun < v[k] else (v[k], full([g[k]]))
    g0 = np.linspace(lo[free[0]], hi[free[0]], grid2[0]); g1 = np.linspace(lo[free[1]], hi[free[1]], grid2[1])
    P = np.array([[a, b] for a in g0 for b in g1]); v = np.array([vp.rss(full(p)) for p in P])
    best, be = v.min(), full(P[int(v.argmin())])
    for k in np.argsort(v)[:polish]:
        r = least_squares(lambda u: vp.residual(full(u)), P[k], bounds=(lo[free], hi[free]), xtol=1e-12, ftol=1e-12, gtol=1e-12, max_nfev=400)
        if 2*r.cost < best: best, be = 2*r.cost, full(r.x)
    return best, be

In [ ]:
%%writefile eisvarpro/eisvarpro/analysis.py
"""Kramers-Kronig check, model fitting, profile likelihood and the reporting decision."""
import math
import numpy as np
from .core import MODELS, VarPro, global_min

def kk_check(f, Z, n_voigt=5):
    """Linear Kramers-Kronig test: |Z|-weighted fit of R + jwL + 1/(jwC) + n_voigt Voigt elements
    log-spaced over the band. Returns the RMS and maximum relative residual (%)."""
    w = 2*np.pi*np.asarray(f, float); Z = np.asarray(Z, complex)
    taus = np.logspace(np.log10(1/w.max()), np.log10(1/w.min()), n_voigt)
    A = np.array([np.ones_like(w)+0j, 1j*w, 1/(1j*w)] + [1/(1+1j*w*t) for t in taus]).T; wt = 1/np.abs(Z)
    Ar = np.vstack([(A*wt[:, None]).real, (A*wt[:, None]).imag]); br = np.r_[(Z*wt).real, (Z*wt).imag]
    c, *_ = np.linalg.lstsq(Ar, br, rcond=None); zk = A @ c
    r = np.r_[(zk.real-Z.real)/np.abs(Z), (zk.imag-Z.imag)/np.abs(Z)]
    return dict(kk_rms_pct=100*float(np.sqrt(np.mean(r**2))), kk_max_pct=100*float(np.max(np.abs(r))))

def fit(model, f, Z):
    """Global VarPro fit. Returns parameters, NRMSE and information criteria."""
    vp = VarPro(model, f, Z); rss, eta = global_min(vp); c, _ = vp.coef(eta); sp = MODELS[model]
    n = 2*vp.N; k = len(sp['nl']) + len(sp['lin']); aic = n*math.log(rss/n) + 2*k
    p = dict(zip(sp['nl'], eta)); p.update(dict(zip(sp['lin'], c)))
    if model == 'rcpe_l': p['Q'] = 1/p.pop('invQ')
    if model == 'cole_l':
        p['tau'] = 10**p['log10_tau']; p['fc_hz'] = 1/(2*math.pi*p['tau']); p['R0'] = p['Rinf'] + p['dR']
    return dict(model=model, params=p, eta=eta, rss=rss, NRMSE=math.sqrt(rss/vp.N), AICc=aic + 2*k*(k+1)/max(n-k-1, 1), n_obs=n)

def _ci(grid, lr, thr=3.84):
    i0 = int(np.argmin(lr))
    def cross(a, b): return grid[a] + (thr-lr[a])*(grid[b]-grid[a])/(lr[b]-lr[a]) if lr[b] != lr[a] else grid[a]
    j = i0
    while j > 0 and lr[j-1] <= thr: j -= 1
    lo = cross(j, j-1) if j > 0 else grid[0]
    j = i0
    while j < len(grid)-1 and lr[j+1] <= thr: j += 1
    hi = cross(j, j+1) if j < len(grid)-1 else grid[-1]
    return dict(ci_low=float(lo), ci_high=float(hi), ci_width=float(hi-lo), closed=bool(lr[0] > thr and lr[-1] > thr))

def _rss_fixed(model, f, Z, param, value):
    """Globally minimised RSS with one parameter fixed at `value` (profile scale)."""
    sp = MODELS[model]
    if param in sp['nl']:
        return global_min(VarPro(model, f, Z), fixed_nl=(sp['nl'].index(param), value))[0]
    name = param.replace('log10_', ''); sign = 1
    if name == 'Q': name, sign = 'invQ', -1
    j = sp['lin'].index(name); return global_min(VarPro(model, f, Z, fix=(j, 10**(sign*value))))[0]

def _profile_value(F, param):
    p = F['params']
    if param in ('log10_tau', 'alpha'): return p[param]
    return math.log10(p[param.replace('log10_', '')])

def calibrated_threshold(model, f, Z, param, n_boot=200, sigma=None, level=0.95, seed=0):
    """Parametric-bootstrap critical value of the profile likelihood-ratio statistic.
    Synthetic spectra are drawn from the fitted model with Gaussian noise on the stacked relative residual
    [Re; Im] / max(|Z|, 1); `sigma` (2N x 2N) gives its covariance shape (default: independent), scaled so that the
    expected RSS equals the observed RSS. Returns the `level` quantile of LR evaluated at the true value, to be used
    in place of 3.84. With correlated errors, the chi-square value can give substantially under-covering intervals."""
    f = np.asarray(f, float); Z = np.asarray(Z, complex); N = len(f); n = 2*N
    F = fit(model, f, Z); sp = MODELS[model]; k = len(sp['nl']) + len(sp['lin'])
    S = np.eye(n) if sigma is None else np.asarray(sigma, float)
    S = S*F['rss']/(np.trace(S)*(1 - k/n)); Lc = np.linalg.cholesky(S)
    w = 2*np.pi*f; vp = VarPro(model, f, Z); from .core import basis
    c, _ = vp.coef(F['eta']); zt = basis(model, F['eta'], w) @ c; truth = _profile_value(F, param)
    rng = np.random.default_rng(seed); sc = np.maximum(np.abs(zt), 1.0); lr = []
    for _ in range(n_boot):
        e = Lc @ rng.standard_normal(n); z = zt + sc*(e[:N] + 1j*e[N:])
        rmin = fit(model, f, z)['rss']; rf = _rss_fixed(model, f, z, param, truth)
        lr.append(n*math.log(max(rf, rmin)/rmin))
    return float(np.quantile(lr, level))

def profile(model, f, Z, param, n_grid=41, threshold=3.84):
    """Globally optimised profile likelihood (LR = n log(RSS/RSS_min), 95% threshold 3.84) for a
    nonlinear parameter ('log10_tau', 'alpha') or a log10 linear coefficient ('log10_dR', 'log10_Rinf',
    'log10_L', 'log10_Q')."""
    sp = MODELS[model]; base = fit(model, f, Z); n = base['n_obs']; rss_min = base['rss']
    if param in sp['nl']:
        i = sp['nl'].index(param); grid = np.linspace(sp['lo'][i], sp['hi'][i], n_grid)
        grid = np.unique(np.r_[grid, base['eta'][i] + np.linspace(-0.05, 0.05, 11)*(sp['hi'][i]-sp['lo'][i])].clip(sp['lo'][i], sp['hi'][i]))
        vp = VarPro(model, f, Z); prof = np.array([global_min(vp, fixed_nl=(i, g))[0] for g in grid])
    else:
        name = param.replace('log10_', ''); sign = 1
        if name == 'Q': name, sign = 'invQ', -1
        j = sp['lin'].index(name); lo, hi = np.log10(sp['llo'][j]), np.log10(sp['lhi'][j])
        if sign < 0: lo, hi = -hi, -lo
        est = math.log10(base['params']['Q'] if name == 'invQ' else base['params'][name])
        grid = np.unique(np.r_[np.linspace(lo, hi, n_grid), est + np.linspace(-0.05, 0.05, 11)*(hi-lo)].clip(lo, hi))
        prof = np.array([global_min(VarPro(model, f, Z, fix=(j, 10**(sign*g))))[0] for g in grid])
    rss_min = min(rss_min, prof.min()); lr = n*np.log(prof/rss_min)
    return dict(param=param, grid=grid, LR=lr, threshold=threshold, **_ci(grid, lr, threshold))

def _criterion(P, param, width_log, width_alpha):
    lim = width_alpha if param == 'alpha' else width_log
    return bool(P['closed'] and P['ci_width'] < lim)

def assess(f, Z, kk_threshold_pct=3.0, width_log=1.0, width_alpha=0.2, calibrate=False, sigma=None, n_boot=200, seed=0,
           tau_ci_max_decades=None):
    """Full workflow: KK screening, VarPro fits of Cole+L and R-CPE-L, profile likelihood of every parameter that
    might be reported, and the reporting decision (Fig. 5 of the paper). A parameter is practically identifiable when
    its 95% profile interval is closed and narrower than `width_log` decades (log10 parameters) or `width_alpha`
    (alpha); the same criterion applies to every reported parameter. Cole parameters are reported only when f_c lies
    in the band and both log10 tau and log10 dR meet the criterion; otherwise the R-CPE-L descriptors that meet it are
    reported. calibrate=True replaces the chi-square threshold by parametric-bootstrap critical values (one per
    parameter; `sigma` = covariance shape of the stacked relative residual): calibration is conditional on the fitted
    model and on that error model. `tau_ci_max_decades` is a deprecated alias of `width_log`."""
    if tau_ci_max_decades is not None: width_log = tau_ci_max_decades
    f = np.asarray(f, float); Z = np.asarray(Z, complex); o = np.argsort(f); f, Z = f[o], Z[o]
    kk = kk_check(f, Z); cole = fit('cole_l', f, Z); cpe = fit('rcpe_l', f, Z)
    def prof(model, param):
        thr = calibrated_threshold(model, f, Z, param, n_boot, sigma, seed=seed) if calibrate else 3.84
        P = profile(model, f, Z, param, threshold=thr); P = {k: v for k, v in P.items() if k not in ('grid', 'LR')}
        P['identifiable'] = _criterion(P, param, width_log, width_alpha); return P
    cole_prof = {p: prof('cole_l', p) for p in ['log10_tau', 'log10_dR', 'alpha', 'log10_Rinf']}
    cpe_prof = {p: prof('rcpe_l', p) for p in ['log10_Q', 'alpha', 'log10_Rinf']}
    fc = cole['params']['fc_hz']; in_band = f.min() <= fc
    core_ok = cole_prof['log10_tau']['identifiable'] and cole_prof['log10_dR']['identifiable']
    reportable = bool(in_band and core_ok)
    if reportable:
        ok = [p for p, P in cole_prof.items() if P['identifiable']]; bad = [p for p in cole_prof if p not in ok]
        decision = ('Cole parameters are estimable: report ' + ', '.join(ok) + ' with their profile intervals'
                    + (f"; do not report {', '.join(bad)} (interval not closed or too wide)." if bad else '.'))
    else:
        why = []
        if not in_band: why.append(f'the estimated fc ({fc:.3g} Hz) lies below the band (f_min = {f.min():.3g} Hz)')
        for p in ['log10_tau', 'log10_dR']:
            P = cole_prof[p]
            if not P['closed']: why.append(f'the 95% profile interval of {p} is not closed')
            elif not P['identifiable']: why.append(f"the {p} interval spans {P['ci_width']:.2f} decades")
        ok = [p for p, P in cpe_prof.items() if P['identifiable']]
        decision = ('Cole relaxation parameters are not estimable because ' + ' and '.join(why) + '. '
                    + (f"Report the R-CPE-L descriptors {', '.join(ok)} (band-specific; state band, high-frequency correction and probe)"
                       if ok else 'No R-CPE-L descriptor meets the identifiability criterion')
                    + '; do not interpret tau, dR, R0 or fc. To estimate them, extend the band down to the characteristic frequency.')
    warn = None if kk['kk_rms_pct'] < kk_threshold_pct else \
        f"Kramers-Kronig residual {kk['kk_rms_pct']:.2f}% exceeds {kk_threshold_pct}%: the spectrum may be inconsistent; interpret with caution."
    return dict(kk=kk, cole_l=cole, rcpe_l=cpe, tau_profile=cole_prof['log10_tau'], cole_profiles=cole_prof, rcpe_profiles=cpe_prof,
                fc_in_band=bool(in_band), cole_reportable=reportable, preferred_by_AICc='cole_l' if cole['AICc'] < cpe['AICc'] else 'rcpe_l',
                decision=decision, kk_warning=warn)

def format_report(r):
    c, q = r['cole_l']['params'], r['rcpe_l']['params']; t = r['tau_profile']
    lines = [f"Kramers-Kronig RMS residual: {r['kk']['kk_rms_pct']:.2f}%" + (f"  WARNING: {r['kk_warning']}" if r['kk_warning'] else ''),
             f"R-CPE-L : Rinf = {q['Rinf']:.4g} ohm, Q = {q['Q']:.4g} S s^alpha, alpha = {q['alpha']:.3f}, L = {q['L']:.3g} H, NRMSE = {r['rcpe_l']['NRMSE']:.4f}",
             f"Cole+L  : Rinf = {c['Rinf']:.4g} ohm, dR = {c['dR']:.4g} ohm, fc = {c['fc_hz']:.4g} Hz, alpha = {c['alpha']:.3f}, NRMSE = {r['cole_l']['NRMSE']:.4f}",
             f"log10 tau 95% profile interval: [{t['ci_low']:.2f}, {t['ci_high']:.2f}] (width {t['ci_width']:.3f} decades, closed: {t['closed']}, LR threshold {t.get('threshold', 3.84):.2f})",
             f"Preferred by AICc: {r['preferred_by_AICc']}", f"DECISION: {r['decision']}"]
    return '\n'.join(lines)


In [ ]:
%%writefile eisvarpro/eisvarpro/design.py
"""Measurement-design and reproducibility tools.

band_criterion   closed-form criterion for practical identifiability of the Cole relaxation time from a given band
rcpe_bias        predicted bias of the R-CPE-L descriptors when the true dispersion lies at f_c below the band
cross_sweep_check  assumption-light check of interval calibration from repeated sweeps of one recording
drift_variance   REML separation of fitting uncertainty and between-sweep drift
between_recording_sd  pooled between-recording SD (chi-square interval) from groups of replicate recordings
reproducibility_limits  ISO 5725-type repeatability and reproducibility limits and the share of differences exceeding them

Theory (see the paper): above f_c, with u = (j w tau)^-alpha and the CPE term w = (j w)^-alpha / Q, the Cole term is
w - w u + O(w u^2). Information on tau enters only through w u, giving
    W95(log10 tau) ~= 2 z sigma (f_min/f_c)^alpha / (alpha ln10 D),    Gamma = (alpha ln10 D W* / (2 z sigma))^(1/alpha),
and tau is practically identifiable (interval narrower than W* decades) when f_min / f_c < Gamma, i.e. when
f_c > f_min / Gamma (the detection limit). D depends on band, sampling and impedance composition, not on f_c.
The approximation is asymptotic (f_min/f_c >~ 1); below that, tau is identifiable in practice anyway.
"""
import itertools, math
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
from .analysis import fit, profile, calibrated_threshold, _profile_value

LN10 = math.log(10.0)

def _stack(c): return np.r_[c.real, c.imag]

def _rcpe_geometry(f, Z, p):
    w = 2*np.pi*np.asarray(f, float); jw = 1j*w; absZ = np.maximum(np.abs(np.asarray(Z, complex)), 1.0)
    cpe = jw**(-p['alpha'])/p['Q']
    J = np.column_stack([_stack(c) for c in (p['Rinf']/absZ, -cpe/absZ, -np.log(jw)*cpe/absZ, jw*p['L']/absZ)])
    ht = _stack(cpe*(jw/w.min())**(-p['alpha'])/absZ)
    return J, ht

def band_criterion(f, Z, sigma=None, width=1.0, z=1.96):
    """Closed-form band-coverage criterion for the Cole relaxation time.
    sigma: SD of each stacked real component of the relative residual (default: from the Cole+L fit, RSS/(2N-5)).
    width: required 95% interval width of log10 tau in decades; z: 1.96 or sqrt(calibrated critical value).
    Returns D, Gamma, the detection limit f_c* = f_min/Gamma (tau identifiable only if f_c > f_c*), and sigma."""
    f = np.asarray(f, float); Z = np.asarray(Z, complex); o = np.argsort(f); f, Z = f[o], Z[o]
    p = fit('rcpe_l', f, Z)['params']
    if sigma is None: sigma = math.sqrt(fit('cole_l', f, Z)['rss']/(2*len(f) - 5))
    J, ht = _rcpe_geometry(f, Z, p); P = np.eye(len(ht)) - J @ np.linalg.pinv(J); D = float(np.linalg.norm(P @ ht))
    G = (p['alpha']*LN10*D*width/(2*z*sigma))**(1/p['alpha'])
    return dict(D=D, Gamma=G, fc_detect_hz=float(f.min()/G), sigma=sigma, alpha=p['alpha'])

def predicted_tau_width(f, Z, fc_hz, sigma=None, z=1.96, refined=True):
    """95% interval width (decades) of log10 tau if the true characteristic frequency were fc_hz.
    refined=False: first-order closed form, W = 2 z sigma (f_min/f_c)^alpha / (alpha ln10 D), accurate when
    (f_min/f_c)^-alpha <~ 0.5 (the remainder of the expansion is bounded by |u|/(1-|u|)).
    refined=True (default): exact derivative of the Cole term with respect to 1/dR and the delta method for
    log tau = (log Q - log(1/dR))/alpha including the covariances with Q and alpha (equals the exact linearized width)."""
    f = np.asarray(f, float); Z = np.asarray(Z, complex); o = np.argsort(f); f, Z = f[o], Z[o]
    c = band_criterion(f, Z, sigma, 1.0, z); a = c['alpha']
    if not refined:
        return 2*z*c['sigma']*(f.min()/fc_hz)**a/(a*LN10*c['D'])
    p = fit('rcpe_l', f, Z)['params']; w = 2*np.pi*f; jw = 1j*w; absZ = np.maximum(np.abs(Z), 1.0)
    tau = 1/(2*np.pi*fc_hz); cpe = jw**(-a)/p['Q']; u = (jw*tau)**(-a); d = 1/(1 + u)**2
    cols = [p['Rinf']/absZ, d*(-cpe)/absZ, d*(-np.log(jw)*cpe)/absZ, jw*p['L']/absZ, -(cpe*u*d)/absZ]
    J = np.column_stack([_stack(x) for x in cols]); C = np.linalg.pinv(J.T @ J)*c['sigma']**2
    g = np.array([0, 1/a, -math.log(tau)/a, 0, -1/a])
    return float(2*z*math.sqrt(max(g @ C @ g, 0))/LN10)

def rcpe_bias(f, Z, fc_hz):
    """Predicted bias of log10 Q and alpha of the R-CPE-L fit when the data follow a Cole dispersion with
    characteristic frequency fc_hz below the band (linearized; accurate to a few percent for f_min/f_c >= 10)."""
    f = np.asarray(f, float); Z = np.asarray(Z, complex); p = fit('rcpe_l', f, Z)['params']
    J, ht = _rcpe_geometry(f, Z, p); b = -np.linalg.lstsq(J, ht, rcond=None)[0]; k = (f.min()/fc_hz)**(-p['alpha'])
    return dict(bias_log10_Q=float(b[1]*k/LN10), bias_alpha=float(b[2]*k), coef_log10_Q=float(b[1]/LN10), coef_alpha=float(b[2]))

def cross_sweep_check(f, sweeps, param='log10_Q', model='rcpe_l', calibrate=True, sigma=None, n_boot=100, seed=0):
    """Fit and profile every sweep of one recording; for each pair of sweeps test |theta_i - theta_j| <= sqrt(h_i^2 + h_j^2)
    (h: 95% half-width). With calibrated intervals and no drift, about 95% of pairs should pass."""
    est, half = [], []
    for k, Z in enumerate(sweeps):
        thr = calibrated_threshold(model, f, Z, param, n_boot, sigma, seed=seed + k) if calibrate else 3.84
        P = profile(model, f, Z, param, threshold=thr); est.append(_profile_value(fit(model, f, Z), param)); half.append(P['ci_width']/2)
    ok = [abs(est[i] - est[j]) <= math.hypot(half[i], half[j]) for i, j in itertools.combinations(range(len(est)), 2)]
    return dict(estimates=est, half_widths=half, pairs=len(ok), agreement=float(np.mean(ok)) if ok else float('nan'))

def drift_variance(groups):
    """REML estimate of the between-sweep drift SD tau_d from groups = [(estimates, standard_errors), ...] (one group per
    recording; standard error = 95% half-width / 1.96). Returns tau_d and the reproducibility half-width multiplier."""
    G = [(np.asarray(t, float), np.asarray(s, float)**2) for t, s in groups if len(t) >= 2]
    def reml(t2):
        ll = 0.0
        for th, s2 in G:
            v = s2 + t2; w = 1/v; mu = (w*th).sum()/w.sum(); ll += -0.5*(np.log(v).sum() + math.log(w.sum()) + (w*(th - mu)**2).sum())
        return ll
    top = 3*math.sqrt(4*max(np.var(th) for th, _ in G) + 1e-12)
    r = minimize_scalar(lambda t: -reml(t**2), bounds=(0, top), method='bounded', options={'xatol': 1e-10})
    t = r.x if reml(r.x**2) > reml(0.0) else 0.0
    return dict(tau_drift=float(t), loglik_ratio=float(2*(reml(t**2) - reml(0.0))))

def between_recording_sd(groups, level=0.95):
    """Pooled between-recording SD from groups of replicate values (e.g. log10 R of recordings of the same medium with the
    same probe, animal and frequency). Returns the degrees-of-freedom-weighted SD, its chi-square interval and df."""
    G = [np.asarray(g, float) for g in groups if len(g) > 1]
    nu = sum(len(g) - 1 for g in G)
    if nu == 0: raise ValueError('need at least one group with two or more recordings')
    s = math.sqrt(sum((len(g) - 1)*np.var(g, ddof=1) for g in G)/nu); a = (1 - level)/2
    return dict(sd=s, ci_low=s*math.sqrt(nu/chi2.ppf(1 - a, nu)), ci_high=s*math.sqrt(nu/chi2.ppf(a, nu)), df=nu)

def reproducibility_limits(s_fit, s_drift=0.0, s_between=0.0, differences=None):
    """Repeatability limit r = 2.77 s_w (s_w^2 = s_fit^2 + s_drift^2: fitting error and between-sweep drift) and
    reproducibility limit R = 2.77 sqrt(s_w^2 + s_between^2) (between-recording component), in the form of ISO 5725.
    Two values from the same recording (or from different recordings) differ by more than r (or R) with probability
    about 5% when nothing but measurement variability separates them. If differences are given (e.g. between tissues),
    the percentages exceeding r and R are returned as well."""
    s_w = math.hypot(s_fit, s_drift); out = dict(s_within=s_w, r=2.77*s_w, R=2.77*math.hypot(s_w, s_between))
    if differences is not None:
        d = np.abs(np.asarray(differences, float)); out.update(pct_gt_r=100*float(np.mean(d > out['r'])), pct_gt_R=100*float(np.mean(d > out['R'])))
    return out


In [ ]:
%%writefile eisvarpro/eisvarpro/__main__.py
"""Command line: python -m eisvarpro spectrum.csv
CSV columns: frequency_hz, resistance_ohm, reactance_ohm (one row per frequency)."""
import sys, json
import numpy as np, pandas as pd
from .analysis import assess, format_report

def main(argv=None):
    argv = sys.argv[1:] if argv is None else argv
    if not argv:
        print(__doc__); return 1
    d = pd.read_csv(argv[0]); r = assess(d.frequency_hz.to_numpy(float), d.resistance_ohm.to_numpy(float) + 1j*d.reactance_ohm.to_numpy(float))
    print(format_report(r))
    if '--json' in argv:
        conv = lambda o: o.tolist() if isinstance(o, np.ndarray) else float(o) if isinstance(o, np.floating) else str(o)
        print(json.dumps(r, default=conv, indent=2))
    return 0

if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile eisvarpro/tests/test_eisvarpro.py
import itertools
import numpy as np, pandas as pd
from pathlib import Path
from eisvarpro import bounded_lstsq, assess, fit, kk_check
from eisvarpro.core import _enum_lstsq
EX = Path(__file__).parent.parent/'examples'

def synth(fmin, fc=500.0, noise=0.01, seed=0):
    rng = np.random.default_rng(seed); f = np.logspace(np.log10(fmin), 5, int(10*np.log10(1e5/fmin))+1); w = 2*np.pi*f
    Z = 1000 + 3000/(1+(1j*w/(2*np.pi*fc))**0.75) + 1j*w*1e-3
    return f, Z + noise*np.abs(Z)*(rng.standard_normal(len(f)) + 1j*rng.standard_normal(len(f)))/np.sqrt(2)

def test_bounded_lstsq_matches_enumeration():
    rng = np.random.default_rng(0)
    for _ in range(200):
        A = rng.standard_normal((20, 3)); b = rng.standard_normal(20); lb = np.full(3, -0.2); ub = np.full(3, 0.3)
        c1, v1 = bounded_lstsq(A, b, lb, ub); c2, v2 = _enum_lstsq(A, b, lb, ub)
        assert v1 <= v2*(1+1e-10) + 1e-12

def test_wide_band_cole_is_reportable():
    f, Z = synth(100.0, fc=500.0); r = assess(f, Z)
    assert r['cole_reportable'] and abs(r['cole_l']['params']['fc_hz']/500 - 1) < 0.2

def test_narrow_band_cole_is_not_reportable():
    f, Z = synth(1e4, fc=500.0, noise=0.025); r = assess(f, Z)
    assert not r['cole_reportable'] and 'not estimable' in r['decision']

def test_real_example_runs_and_kk():
    d = pd.read_csv(EX/'porcine_bladder_10-100kHz.csv'); Z = d.resistance_ohm + 1j*d.reactance_ohm
    assert kk_check(d.frequency_hz, Z)['kk_rms_pct'] < 3
    r = fit('rcpe_l', d.frequency_hz, Z); assert 0.02 < r['NRMSE'] < 0.03

def test_calibrated_threshold_white_noise():
    """Under independent noise the calibrated critical value should be of the order of the chi-square value."""
    import numpy as np
    from eisvarpro import calibrated_threshold
    f = np.arange(1, 11)*1e4; w = 2*np.pi*f
    Z = 1000 + 1/(2e-8*(1j*w)**0.8) + 1j*w*1e-3
    rng = np.random.default_rng(3); Z = Z + 0.02*np.abs(Z)*(rng.standard_normal(10) + 1j*rng.standard_normal(10))
    c = calibrated_threshold('rcpe_l', f, Z, 'alpha', n_boot=60, seed=1)
    assert 1.5 < c < 15

def test_band_criterion_and_bias():
    """Wide-band synthetic Cole data: tau must be detectable (f_c above the detection limit); bias formula matches a refit."""
    import numpy as np
    from eisvarpro import band_criterion, rcpe_bias, fit
    f = np.arange(1, 11)*1e4; w = 2*np.pi*f
    Rinf, a, Q, L = 1000.0, 0.8, 2e-8, 1e-3
    Z = Rinf + 1/(Q*(1j*w)**a) + 1j*w*L
    c = band_criterion(f, Z, sigma=0.01)
    assert c['Gamma'] > 1 and c['fc_detect_hz'] < 1e4
    fc = 100.0; tau = 1/(2*np.pi*fc); dR = tau**a/Q
    Zc = Rinf + dR/(1 + (1j*w*tau)**a) + 1j*w*L
    b = rcpe_bias(f, Z, fc); p = fit('rcpe_l', f, Zc)['params']
    assert abs((np.log10(p['Q']) - np.log10(Q)) - b['bias_log10_Q']) < 0.02
    assert abs((p['alpha'] - a) - b['bias_alpha']) < 0.005

def test_drift_variance_recovers_drift():
    import numpy as np
    from eisvarpro import drift_variance
    rng = np.random.default_rng(0); groups = []
    for i in range(30):
        mu = rng.normal(); se = np.full(4, 0.05); groups.append((mu + rng.normal(0, 0.2, 4) + rng.normal(0, 0.05, 4), se))
    t = drift_variance(groups)['tau_drift']
    assert 0.12 < t < 0.3

def test_refined_width_is_wider_and_finite():
    import numpy as np
    from eisvarpro import predicted_tau_width
    f = np.arange(1, 11)*1e4; w = 2*np.pi*f
    Z = 1000 + 1/(2e-8*(1j*w)**0.8) + 1j*w*1e-3
    a = predicted_tau_width(f, Z, 2000.0, sigma=0.01, refined=False); b = predicted_tau_width(f, Z, 2000.0, sigma=0.01, refined=True)
    assert np.isfinite(a) and np.isfinite(b) and b > 0.5*a


def test_reproducibility_limits():
    from eisvarpro import between_recording_sd, reproducibility_limits
    rng = np.random.default_rng(1)
    groups = [rng.normal(0, 0.1, 6) for _ in range(40)]
    b = between_recording_sd(groups); assert 0.08 < b['sd'] < 0.12 and b['ci_low'] < b['sd'] < b['ci_high'] and b['df'] == 200
    L = reproducibility_limits(0.03, 0.04, 0.12, differences=[0.01, 0.2, 0.5])
    assert abs(L['s_within'] - 0.05) < 1e-12 and abs(L['r'] - 0.1385) < 1e-9 and abs(L['R'] - 2.77*0.13) < 1e-9
    assert L['pct_gt_r'] > L['pct_gt_R']


In [ ]:
# example spectra and demo
import sys, pandas as pd, numpy as np
os.makedirs('eisvarpro/examples', exist_ok=True)
d = pd.read_csv('results/tables/ecm_ready_complete_scans.csv')
g = d[d.source_file.str.contains('202309231536_SP_yellow_bipolar_bladder')].sort_values('frequency_hz')
g[['frequency_hz', 'resistance_ohm', 'reactance_ohm']].to_csv('eisvarpro/examples/porcine_bladder_10-100kHz.csv', index=False)
sys.path.insert(0, os.path.abspath('eisvarpro'))
from eisvarpro import assess, format_report
print(format_report(assess(g.frequency_hz.to_numpy(float), (g.resistance_ohm + 1j*g.reactance_ohm).to_numpy())))
rows = []
for sf, s in d.groupby('source_file'):
    s = s.sort_values('frequency_hz'); r = assess(s.frequency_hz.to_numpy(float), (s.resistance_ohm + 1j*s.reactance_ohm).to_numpy())
    rows.append(dict(tissue=s.tissue.iloc[0], kk_consistent=r['kk_warning'] is None, fc_in_band=r['fc_in_band'],
                     tau_interval_closed=r['tau_profile']['closed'], tau_ci_width=round(r['tau_profile']['ci_width'], 2),
                     cole_reportable=r['cole_reportable'], preferred_by_AICc=r['preferred_by_AICc']))
T = pd.DataFrame(rows); T.to_csv('results/tables/Table_tool_decisions.csv', index=False); print(T.to_string())
print('KK-inconsistent:', (~T.kk_consistent).sum(), '| R-CPE-L preferred:', (T.preferred_by_AICc == 'rcpe_l').sum(), '| Cole reportable:', T.cole_reportable.sum())

In [ ]:
# unit tests of the package (pytest optional)
import importlib.util, time
spec = importlib.util.spec_from_file_location('t', 'eisvarpro/tests/test_eisvarpro.py'); t = importlib.util.module_from_spec(spec); spec.loader.exec_module(t)
for n in [x for x in dir(t) if x.startswith('test_')]:
    t0 = time.time(); getattr(t, n)(); print('PASS', n, f'{time.time()-t0:.1f}s')

---
# Part E — Reanalyses requested in the critical review
KK sensitivity (Table S8), model-selection robustness and inductance influence (Table S9), equivalence of the HO rescue (Table S10), interval coverage and bound sensitivity (Table S11), tolerance, population size and L-SHADE time to target (Table S12), jackknife and polished M13 references. Requires Parts B–D (uses `vp_merged_*`, `nested_m13_runs.csv`, `eisvarpro`).

In [ ]:
os.makedirs('src/review', exist_ok=True)

In [ ]:
%%writefile src/review/review_fast.py
"""review_fast.py - reanalyses of existing data requested in the critical review (R3, R4, R6, R9, R10, R17, R5-tolerance).
All computations use the shipped spectra and run-level results; no new measurements."""
import sys, math, itertools, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon, binomtest
ROOT = Path(__file__).resolve().parents[2]; T = ROOT/'results'/'tables'
sys.path.insert(0, str(ROOT/'eisvarpro'))
from eisvarpro import kk_check, fit
from eisvarpro.core import VarPro, global_min, MODELS
rng = np.random.default_rng(20261017)
d = pd.read_csv(T/'ecm_ready_complete_scans.csv'); sub = pd.read_csv(T/'Table_subset_mapping.csv').set_index('source_file').original_subset
S = [(sf, g.sort_values('frequency_hz')) for sf, g in d.groupby('source_file')]
def spec(g): return g.frequency_hz.to_numpy(float), (g.resistance_ohm + 1j*g.reactance_ohm).to_numpy()
out = {}

# ---------- R9: Kramers-Kronig sensitivity (number of Voigt elements x threshold)
base = np.array([kk_check(*spec(g), n_voigt=5)['kk_rms_pct'] >= 3 for _, g in S]); rows = []
for M in [2, 3, 4, 5, 6, 7]:
    r = np.array([kk_check(*spec(g), n_voigt=M)['kk_rms_pct'] for _, g in S])
    rows.append(dict(n_voigt=M, median_rms_pct=np.median(r), **{f'inconsistent_at_{t}pct': int((r >= t).sum()) for t in (2, 3, 4, 5)},
                     agreement_with_5voigt_3pct=int(((r >= 3) == base).sum())))
K = pd.DataFrame(rows); K.to_csv(T/'Table_R9_KK_sensitivity.csv', index=False); out['R9'] = K

# ---------- R3: model selection robustness (M10 vs M11)
def wfit(model, f, Z, weight):
    if weight == 'relative': return fit(model, f, Z)
    s = np.mean(np.abs(Z)); r = fit(model, f, Z/s*1000.0)    # constant weight: residuals scaled by mean |Z|
    return r
def lofo(model, f, Z):
    err = []
    for k in range(len(f)):
        m = np.ones(len(f), bool); m[k] = False; F = fit(model, f[m], Z[m]); p = F['params']; w = 2*np.pi*f[k]
        if model == 'cole_l': zp = p['Rinf'] + p['dR']/(1 + (1j*w*p['tau'])**p['alpha']) + 1j*w*p['L']
        else: zp = p['Rinf'] + (1/p['Q'])*(1j*w)**(-p['alpha']) + 1j*w*p['L']
        err.append(abs(zp - Z[k])**2/abs(Z[k])**2)
    return math.sqrt(np.mean(err))
rows = []
for sf, g in S:
    f, Z = spec(g); rec = dict(source_file=sf, tissue=g.tissue.iloc[0], subset=sub[sf], kk_ok=kk_check(f, Z)['kk_rms_pct'] < 3)
    for wgt in ['relative', 'constant']:
        a, b = wfit('cole_l', f, Z, wgt), wfit('rcpe_l', f, Z, wgt); rec[f'dAICc_M10minusM11_{wgt}'] = a['AICc'] - b['AICc']
    rec['lofo_M10'] = lofo('cole_l', f, Z); rec['lofo_M11'] = lofo('rcpe_l', f, Z)
    rows.append(rec)
MS = pd.DataFrame(rows); MS.to_csv(T/'Table_R3_model_selection_runs.csv', index=False)
summ = dict(M11_pref_relative=int((MS.dAICc_M10minusM11_relative > 0).sum()), M11_pref_constant=int((MS.dAICc_M10minusM11_constant > 0).sum()),
            lofo_M11_better=int((MS.lofo_M11 < MS.lofo_M10).sum()), lofo_median_M10=MS.lofo_M10.median(), lofo_median_M11=MS.lofo_M11.median(),
            lofo_wilcoxon_p=wilcoxon(MS.lofo_M11, MS.lofo_M10).pvalue)
ref = pd.read_csv(T/'best_known_solutions.csv')
m10 = ref[ref.model == 'M10_ColeL']; m11 = ref[ref.model == 'M11_RCPEL']
summ.update(M10_alpha_at_lower=int((m10.alpha <= 0.3001).sum()), M10_alpha_at_upper=int((m10.alpha >= 0.9999).sum()), M10_logdR_at_upper=int((m10.logdR >= 7.999).sum()),
            M10_logtau_at_bound=int(((m10.logtau <= -6.999) | (m10.logtau >= 0.999)).sum()), M11_alpha_at_lower=int((m11.alpha <= 0.3001).sum()),
            M11_alpha_at_upper=int((m11.alpha >= 0.9999).sum()))
pd.DataFrame([summ]).to_csv(T/'Table_R3_model_selection_summary.csv', index=False); out['R3'] = pd.DataFrame([summ]).T

# ---------- R4 / R10: influence of the inductive term and of the two highest frequencies on M11 descriptors
rows = []
for sf, g in S:
    f, Z = spec(g); full = fit('rcpe_l', f, Z)['params']; m = f <= 80e3; trunc = fit('rcpe_l', f[m], Z[m])['params']
    vp = VarPro('rcpe_l', f, Z, fix=(2, 1e-9)); rss, eta = global_min(vp); c, _ = vp.coef(eta)   # L fixed at ~0
    rows.append(dict(source_file=sf, tissue=g.tissue.iloc[0], kk_ok=kk_check(f, Z)['kk_rms_pct'] < 3,
                     omegaL_rel_100k=2*math.pi*1e5*full['L']/abs(Z[-1]), L_mH=1e3*full['L'],
                     dlogQ_trunc=math.log10(trunc['Q']) - math.log10(full['Q']), dalpha_trunc=trunc['alpha'] - full['alpha'],
                     dlogRinf_trunc=math.log10(trunc['Rinf']) - math.log10(full['Rinf']),
                     NRMSE_full=math.sqrt(fit('rcpe_l', f, Z)['rss']/len(f)), NRMSE_noL=math.sqrt(rss/len(f))))
L = pd.DataFrame(rows); L.to_csv(T/'Table_R4_inductance_runs.csv', index=False)
q = lambda x: f"{np.median(x):.3f} [{np.quantile(x, .25):.3f}, {np.quantile(x, .75):.3f}]"
out['R4'] = pd.Series(dict(omegaL_rel_100k=q(L.omegaL_rel_100k), L_mH=q(L.L_mH), abs_dlogQ_drop90_100=q(L.dlogQ_trunc.abs()),
                           abs_dalpha_drop90_100=q(L.dalpha_trunc.abs()), abs_dlogRinf=q(L.dlogRinf_trunc.abs()),
                           NRMSE_with_L=q(L.NRMSE_full), NRMSE_without_L=q(L.NRMSE_noL),
                           kk_ok_abs_dlogQ=q(L[L.kk_ok].dlogQ_trunc.abs()), kk_ok_abs_dalpha=q(L[L.kk_ok].dalpha_trunc.abs())))
out['R4'].to_csv(T/'Table_R4_inductance_summary.csv')

# ---------- R6: equivalence analysis ASHO vs L-SHADE (per-spectrum success-rate and NRMSE differences)
def boot_ci(x, B=10000):
    x = np.asarray(x); bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]; return np.quantile(bs, [.025, .975])
rows = []
o = pd.read_csv(T/'optimizer_runs.csv'); m12 = pd.read_csv(T/'optimizer_runs_M12.csv')
vpm = {m: pd.read_csv(T/f'vp_merged_{m}.csv') for m in ['M10_ColeL', 'M11_RCPEL', 'M12_DoubleColeL']}
for model, data in [('M10_ColeL', vpm['M10_ColeL']), ('M11_RCPEL', vpm['M11_RCPEL']), ('M12_DoubleColeL', vpm['M12_DoubleColeL'])]:
    for a, b in [('ASHO-LSHADE', 'L-SHADE'), ('VP-ASHO-LSHADE', 'VP-L-SHADE'), ('ASHO-LSHADE+TRF', 'L-SHADE+TRF')]:
        x = data[data.algorithm.isin([a, b])]
        if x.algorithm.nunique() < 2: continue
        sr = x.groupby(['source_file', 'algorithm']).success.mean().unstack(); dsr = 100*(sr[a]-sr[b])
        med = x.groupby(['source_file', 'algorithm']).NRMSE.median().unstack(); drel = 100*(med[a]-med[b])/med[b]
        lo, hi = boot_ci(dsr); lo2, hi2 = boot_ci(drel)
        rows.append(dict(model=model, comparison=f'{a} - {b}', mean_dsuccess_pp=dsr.mean(), ci_low_pp=lo, ci_high_pp=hi,
                         equivalent_within_5pp=bool(lo > -5 and hi < 5), mean_dNRMSE_pct=drel.mean(), ci_low_pct=lo2, ci_high_pct=hi2,
                         equivalent_within_1pct=bool(lo2 > -1 and hi2 < 1)))
E = pd.DataFrame(rows); E.to_csv(T/'Table_R6_equivalence.csv', index=False); out['R6'] = E

# ---------- R5 (tolerance) and R17 (jackknife): VP-MS-TRF vs MS-TRF and VP-L-SHADE vs L-SHADE on M12 and M10
rows = []
for model in ['M10_ColeL', 'M12_DoubleColeL']:
    x = vpm[model]
    for tol in [1e-2, 1e-3, 1e-4]:
        s = (x.NRMSE - x.ref)/x.ref <= tol
        r = x.assign(s=s).groupby('algorithm').s.mean()
        rows.append(dict(model=model, tolerance=tol, **{k: 100*r[k] for k in ['L-SHADE', 'VP-L-SHADE', 'MS-TRF', 'VP-MS-TRF', 'VP-Screen+TRF']}))
TOL = pd.DataFrame(rows); TOL.to_csv(T/'Table_R5_tolerance.csv', index=False); out['R5tol'] = TOL
x = vpm['M12_DoubleColeL']; sr = x.groupby(['source_file', 'algorithm']).success.mean().unstack(); g = 100*(sr['VP-MS-TRF']-sr['MS-TRF'])
jk = [g.drop(i).mean() for i in g.index]
sub_strat = {s: dict(n=int((sub[g.index] == s).sum()), mean_gain=g[sub[g.index] == s].mean(), positive=int((g[sub[g.index] == s] > 0).sum()),
                     negative=int((g[sub[g.index] == s] < 0).sum())) for s in sorted(sub.unique())}
out['R17'] = pd.Series(dict(gain_mean=g.mean(), jackknife_min=min(jk), jackknife_max=max(jk), spectra_with_gain=int((g > 0).sum()),
                            spectra_with_loss=int((g < 0).sum()), sign_test_p=binomtest(int((g > 0).sum()), int((g != 0).sum()), .5).pvalue, **{f'{k}_{kk}': vv for k, v in sub_strat.items() for kk, vv in v.items()}))
out['R17'].to_csv(T/'Table_R17_jackknife.csv')

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
for k, v in out.items(): print('=====', k); print(v.round(4).to_string() if hasattr(v, 'round') else v)

In [ ]:
%%writefile src/review/review_slow.py
"""review_slow.py - simulation-based coverage of profile-likelihood intervals (R2), search-bound sensitivity (R2),
polished M13 references (R12), L-SHADE population-size sensitivity (R13) and same-machine time-to-target for
L-SHADE vs VP-L-SHADE (R5). Usage: python review_slow.py [R2|R12|R13|R5|all]"""
import sys, math, time, pickle, numpy as np, pandas as pd
from pathlib import Path
ROOT = Path(__file__).resolve().parents[2]; T = ROOT/'results'/'tables'; SRC = ROOT/'src'
sys.path.insert(0, str(ROOT/'eisvarpro')); sys.path.insert(0, str(SRC))
import eisvarpro.core as C
from eisvarpro import fit, profile, kk_check
what = sys.argv[1] if len(sys.argv) > 1 else 'all'
d = pd.read_csv(T/'ecm_ready_complete_scans.csv'); kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct
S = [(sf, g.sort_values('frequency_hz')) for sf, g in d.groupby('source_file')]
def spec(g): return g.frequency_hz.to_numpy(float), (g.resistance_ohm + 1j*g.reactance_ohm).to_numpy()

def noise(z, kind, rng, level=0.025):
    n = len(z)
    if kind == 'white': e = rng.standard_normal(n) + 1j*rng.standard_normal(n)
    elif kind == 'AR1':                                     # correlated across frequency, rho = 0.6
        def ar():
            x = np.empty(n); x[0] = rng.standard_normal()
            for k in range(1, n): x[k] = 0.6*x[k-1] + math.sqrt(1-0.36)*rng.standard_normal()
            return x
        e = ar() + 1j*ar()
    elif kind == 'hetero':                                  # relative noise rising from 1% to 4% with frequency
        s = np.linspace(0.4, 1.6, n); e = s*(rng.standard_normal(n) + 1j*rng.standard_normal(n))
    return z + level*np.abs(z)*e/math.sqrt(2)

if what in ('R2', 'all'):
    t0 = time.time(); rng = np.random.default_rng(1); rows = []
    good = [(sf, g) for sf, g in S if kk[sf] < 3]
    for sf, g in good:
        f, Z = spec(g); p = fit('rcpe_l', f, Z)['params']; w = 2*np.pi*f
        z11 = p['Rinf'] + (1/p['Q'])*(1j*w)**(-p['alpha']) + 1j*w*p['L']
        # Cole truth with f_c below the band, matched to the same high-frequency tail (dR from Q, tau, alpha)
        fc = 10**rng.uniform(np.log10(200), np.log10(3000)); tau = 1/(2*np.pi*fc); a10 = min(max(p['alpha'], 0.5), 0.95)
        dR = tau**a10/p['Q']; z10 = p['Rinf'] + dR/(1 + (1j*w*tau)**a10) + 1j*w*p['L']
        for kind in ['white', 'AR1', 'hetero']:
            for r in range(10):
                zn = noise(z11, kind, rng)
                for par, truth in [('log10_Q', math.log10(p['Q'])), ('alpha', p['alpha'])]:
                    P = profile('rcpe_l', f, zn, par); rows.append(dict(source_file=sf, model='M11', noise=kind, param=par, covered=P['ci_low'] <= truth <= P['ci_high'],
                                                                     closed=P['closed'], width=P['ci_width']))
                zn = noise(z10, kind, rng); P = profile('cole_l', f, zn, 'log10_tau')
                rows.append(dict(source_file=sf, model='M10', noise=kind, param='log10_tau', covered=P['ci_low'] <= math.log10(tau) <= P['ci_high'],
                                 closed=P['closed'], width=P['ci_width']))
        print(time.strftime('%H:%M:%S'), 'R2', sf[3:15], flush=True)
    R = pd.DataFrame(rows); R.to_csv(T/'review_R2_coverage_runs.csv', index=False)
    Sm = R.groupby(['model', 'param', 'noise']).agg(n=('covered', 'size'), coverage_pct=('covered', lambda x: 100*x.mean()),
            closed_pct=('closed', lambda x: 100*x.mean()), median_width=('width', 'median')).reset_index()
    Sm.to_csv(T/'Table_R2_coverage.csv', index=False); print(Sm.round(3).to_string())
    # search-bound sensitivity on the measured spectra
    rows = []
    for lo, hi in [(-7.0, 1.0), (-9.0, 3.0), (-6.0, 0.0)]:
        C.MODELS['cole_l']['lo'][0], C.MODELS['cole_l']['hi'][0] = lo, hi
        for sf, g in S:
            P = profile('cole_l', *spec(g), 'log10_tau'); rows.append(dict(bounds=f'[{lo:g},{hi:g}]', source_file=sf, closed=P['closed'], width=P['ci_width']))
    C.MODELS['cole_l']['lo'][0], C.MODELS['cole_l']['hi'][0] = -7.0, 1.0
    B = pd.DataFrame(rows).groupby('bounds').agg(closed=('closed', 'sum'), median_width=('width', 'median')).reset_index()
    B.to_csv(T/'Table_R2_bound_sensitivity.csv', index=False); print(B.to_string()); print('R2 sec', round(time.time()-t0))

if what in ('R12', 'all'):
    t0 = time.time(); _argv = sys.argv; sys.argv = [sys.argv[0]]; from nested_vp import Problem, LO, HI; sys.argv = _argv
    from scipy.optimize import least_squares
    runs = pd.read_csv(T/'nested_m13_runs.csv'); best = {}
    for si, (sf, g) in enumerate(S):
        f, Z = spec(g); P = Problem(2*np.pi*f, Z) if False else Problem(f*2*np.pi, Z); rng = np.random.default_rng(500+si); b = np.inf
        for k in range(60):                                   # 60 multistart partial-VarPro TRF fits from random starts
            r = least_squares(P.res_vp, rng.uniform(LO[:6], HI[:6]), bounds=(LO[:6], HI[:6]), max_nfev=3000, xtol=1e-12, ftol=1e-12)
            b = min(b, 2*r.cost/P.N)
        best[sf] = math.sqrt(b)
    runs['ref_old'] = runs.ref; runs['ref'] = [min(o, best[s]) for o, s in zip(runs.ref_old, runs.source_file)]
    for c in [200, 500, 1000, 2000]: runs[f'succ_{c}'] = (runs[f'trace_{c}'] - runs.ref)/runs.ref <= 1e-3
    improved = int(((runs.groupby('source_file').ref_old.first() - pd.Series(best))/runs.groupby('source_file').ref_old.first() > 1e-3).sum())
    Sm = runs.groupby('algorithm')[[f'succ_{c}' for c in [200, 500, 1000, 2000]]].mean().mul(100).reset_index()
    Sm['refs_improved_by_polishing'] = improved; Sm.to_csv(T/'Table_R12_M13_polished.csv', index=False); print(Sm.round(1).to_string()); print('R12 sec', round(time.time()-t0))

if what in ('R13', 'all'):
    t0 = time.time(); import eis_core as E
    from vp_common import load_spectra
    ref = pd.read_csv(T/'best_known_solutions.csv'); rows = []
    for pop in [10, 20, 40]:
        E.POP0 = pop
        for s in load_spectra():
            rr = ref[(ref.model == 'M10_ColeL') & (ref.source_file == s['sf'])].ref_NRMSE.iloc[0]; f = E.make_objective('M10_ColeL', s['w'], s['z'])
            _, lo, hi = E.MODELS['M10_ColeL']
            for r in range(10):
                _, fb, _, _ = E.lshade_family(f, lo, hi, 2000, 20261101 + s['si']*1000 + r, 'lshade')
                rows.append(dict(pop=pop, source_file=s['sf'], run=r, success=(math.sqrt(fb)-rr)/rr <= 1e-3))
    E.POP0 = 20
    P = pd.DataFrame(rows).groupby('pop').success.mean().mul(100).reset_index(); P.to_csv(T/'Table_R13_population.csv', index=False); print(P.to_string()); print('R13 sec', round(time.time()-t0))

if what in ('R5', 'all'):
    t0 = time.time(); import eis_core as E; import vp_core as V
    from vp_common import load_spectra
    rows = []
    for model in ['M10_ColeL', 'M12_DoubleColeL']:
        mrg = pd.read_csv(T/f'vp_merged_{model}.csv'); ref = mrg.groupby('source_file').ref.first(); _, lo, hi = E.MODELS[model]
        for s in load_spectra():
            target = (ref[s['sf']]*(1+1e-3))**2
            for r in range(5):
                seed = 20261101 + s['si']*1000 + r
                for name in ['L-SHADE', 'VP-L-SHADE']:
                    if name == 'L-SHADE': fobj = E.make_objective(model, s['w'], s['z']); a, b = lo, hi
                    else: vp = V.VarPro(model, s['w'], s['z']); fobj = vp.f; a, b = vp.sp['lo'], vp.sp['hi']
                    st = dict(t0=None, hit=None, n=0)
                    def g(x, fobj=fobj, st=st):
                        v = fobj(x); st['n'] += 1
                        if st['hit'] is None and v <= target: st['hit'] = (time.perf_counter()-st['t0'], st['n'])
                        return v
                    st['t0'] = time.perf_counter(); E.lshade_family(g, a, b, 2000, seed, 'lshade')
                    rows.append(dict(model=model, algorithm=name, source_file=s['sf'], run=r, hit=st['hit'] is not None,
                                     ms=1000*st['hit'][0] if st['hit'] else np.inf, nfe=st['hit'][1] if st['hit'] else np.nan))
    D = pd.DataFrame(rows); D.to_csv(T/'review_R5_timing_lshade_runs.csv', index=False)
    Sm = D.groupby(['model', 'algorithm']).agg(success_pct=('hit', lambda x: 100*x.mean()), median_ms_all=('ms', 'median'),
          median_nfe_success=('nfe', 'median')).reset_index()
    Sm.to_csv(T/'Table_R5_timing_lshade.csv', index=False); print(Sm.to_string()); print('R5 sec', round(time.time()-t0))

In [ ]:
%%time
!cd src/review && python review_fast.py 2>&1 | grep -v Warning
!cd src/review && python review_slow.py R13 && python review_slow.py R5 && python review_slow.py R12 && python review_slow.py R2 2>&1 | grep -v 'R2 pr'

---
# Part F — Second-round analyses (uncertainty calibration, model discrepancy, band design, mechanism)
F1 empirical error model from the 51 repeated sweeps (Table S14); F2 calibrated and GLS profile-likelihood intervals and coverage validation (Tables S15–S16, Fig. 3); F3 bias of the R–CPE–L descriptors under a Cole truth (Table S17); F4 high-frequency correction treatments (Table S18); F5 uncertainty-aware model selection (Table S19); F6 threshold and bound sensitivity (Table S16); F7 sweep-level repeatability (Table S20); F8 band-design factorial (Table S21, Fig. 5); F9 mechanism of the VarPro gain (Table S23); F10 error-model checks against repeated sweeps (Table S24); F11 confidence intervals of ICC and coverage; F12 closed-form band criterion and bias formula (Eqs. 7–10, Fig. 3, Tables S25–S27); F13 drift decomposition between sweeps with profile-likelihood and Bayesian bounds (Table S28); F14 refined closed-form width (Table S30); F15 optional same-environment rerun of the benchmark and timing (Tables 2, S29, S31). Approximate single-core run times: F2 M11 10 min, M10 15 min, double bootstrap 10 min; F5 5 min; F8 12 min; F9 2 min.

In [ ]:
%%writefile src/review/r2_core.py
"""r2_core.py - shared engine for the second-round reanalyses (Experiments E1-E6).

Generic bounded variable projection with an optional whitening matrix (generalised least squares).
Residual vector: r = W [Re(e); Im(e)], e_k = (Z_model(w_k) - Z_k) / max(|Z_k|, 1).
W = I gives the published (independent, relatively weighted) objective; W = chol(Sigma)^-1 gives GLS.
For fixed nonlinear parameters the inner problem stays a box-constrained linear LS, so VarPro is exact.
"""
import math, glob, sys
import numpy as np, pandas as pd
from pathlib import Path
from scipy.optimize import least_squares, minimize_scalar
ROOT = Path(__file__).resolve().parents[2]; T = ROOT/'results'/'tables'
sys.path.insert(0, str(ROOT/'eisvarpro'))
from eisvarpro.core import bounded_lstsq

def _jw(w): return 1j*w
MODELS = {
    # name: nl names, nl lo, nl hi, lin names, lin lo, lin hi, basis(eta, w)
    'cole_l': (['log10_tau', 'alpha'], [-7.0, 0.30], [1.0, 1.0], ['Rinf', 'dR', 'L'], [1e-3, 1e-3, 1e-9], [1e7, 1e8, 1.0],
               lambda e, w: np.column_stack([np.ones_like(w)+0j, 1/(1 + (_jw(w)*10**e[0])**e[1]), _jw(w)])),
    'rcpe_l': (['alpha'], [0.30], [1.0], ['Rinf', 'invQ', 'L'], [1e-3, 1.0, 1e-9], [1e7, 1e12, 1.0],
               lambda e, w: np.column_stack([np.ones_like(w)+0j, _jw(w)**(-e[0]), _jw(w)])),
    'rcpe': (['alpha'], [0.30], [1.0], ['Rinf', 'invQ'], [1e-3, 1.0], [1e7, 1e12],
             lambda e, w: np.column_stack([np.ones_like(w)+0j, _jw(w)**(-e[0])])),
    # pure delay (equivalently a phase error linear in frequency): Z = Z_RCPE exp(-j w t_d), t_d in microseconds (signed)
    'rcpe_delay': (['alpha', 'td_us'], [0.30, -5.0], [1.0, 5.0], ['Rinf', 'invQ'], [1e-3, 1.0], [1e7, 1e12],
                   lambda e, w: np.column_stack([np.ones_like(w)+0j, _jw(w)**(-e[0])])*np.exp(-1j*w*e[1]*1e-6)[:, None]),
    # frequency-independent phase offset: Z = Z_RCPE exp(j phi0)
    'rcpe_phase': (['alpha', 'phi0'], [0.30, -0.8], [1.0, 0.8], ['Rinf', 'invQ'], [1e-3, 1.0], [1e7, 1e12],
                   lambda e, w: np.column_stack([np.ones_like(w)+0j, _jw(w)**(-e[0])])*np.exp(1j*e[1])),
    # generalised high-frequency CPE-type term K (j w)^nu (nu = 1 recovers the series inductance)
    'rcpe_hfcpe': (['alpha', 'nu'], [0.30, 0.30], [1.0, 1.0], ['Rinf', 'invQ', 'K'], [1e-3, 1.0, 1e-12], [1e7, 1e12, 1e6],
                   lambda e, w: np.column_stack([np.ones_like(w)+0j, _jw(w)**(-e[0]), _jw(w)**e[1]])),
}
def spec_of(name):
    nl, lo, hi, lin, llo, lhi, B = MODELS[name]
    return dict(nl=nl, lo=np.array(lo, float), hi=np.array(hi, float), lin=lin, llo=np.array(llo, float), lhi=np.array(lhi, float), basis=B)

class VP:
    def __init__(self, model, f, Z, W=None, fix=None):
        self.model = model; self.sp = spec_of(model); self.w = 2*np.pi*np.asarray(f, float); self.Z = np.asarray(Z, complex)
        self.N = len(self.w); self.n = 2*self.N; self.wt = 1/np.maximum(np.abs(self.Z), 1.0)
        self.W = np.eye(self.n) if W is None else W; zw = self.Z*self.wt; self.b = self.W @ np.r_[zw.real, zw.imag]; self.fix = fix
    def _sys(self, eta):
        A = self.sp['basis'](eta, self.w)*self.wt[:, None]; A = self.W @ np.vstack([A.real, A.imag]); b = self.b
        lb, ub = self.sp['llo'], self.sp['lhi']
        if self.fix is not None:
            j, v = self.fix; keep = [i for i in range(A.shape[1]) if i != j]
            return A[:, keep], b - A[:, j]*v, lb[keep], ub[keep], keep
        return A, b, lb, ub, list(range(A.shape[1]))
    def coef(self, eta):
        A, b, lb, ub, keep = self._sys(np.asarray(eta, float)); c, rss = bounded_lstsq(A, b, lb, ub)
        if self.fix is not None:
            full = np.empty(len(keep)+1); full[keep] = c; full[self.fix[0]] = self.fix[1]; c = full
        return c, rss
    def rss(self, eta): return self.coef(eta)[1]
    def residual(self, eta):
        A, b, lb, ub, _ = self._sys(np.asarray(eta, float)); c, _ = bounded_lstsq(A, b, lb, ub); return A @ c - b

def gmin(vp, fixed_nl=None, n1=61, grid2=(25, 13), polish=3):
    lo, hi = vp.sp['lo'], vp.sp['hi']; D = len(lo); free = [i for i in range(D) if not (fixed_nl and i == fixed_nl[0])]
    def full(u):
        e = np.empty(D); e[free] = u
        if fixed_nl: e[fixed_nl[0]] = fixed_nl[1]
        return e
    if not free:
        e = full([]); return vp.rss(e), e
    if len(free) == 1:
        i = free[0]; g = np.linspace(lo[i], hi[i], n1); v = np.array([vp.rss(full([x])) for x in g]); k = int(v.argmin())
        r = minimize_scalar(lambda x: vp.rss(full([x])), bounds=(g[max(0, k-1)], g[min(n1-1, k+1)]), method='bounded', options={'xatol': 1e-7})
        return (r.fun, full([r.x])) if r.fun < v[k] else (v[k], full([g[k]]))
    g0 = np.linspace(lo[free[0]], hi[free[0]], grid2[0]); g1 = np.linspace(lo[free[1]], hi[free[1]], grid2[1])
    P = np.array([[a, b] for a in g0 for b in g1]); v = np.array([vp.rss(full(p)) for p in P]); best, be = v.min(), full(P[int(v.argmin())])
    for k in np.argsort(v)[:polish]:
        r = least_squares(lambda u: vp.residual(full(u)), P[k], bounds=(lo[free], hi[free]), xtol=1e-10, ftol=1e-12, gtol=1e-12, max_nfev=200)
        if 2*r.cost < best: best, be = 2*r.cost, full(r.x)
    return best, be

def fit(model, f, Z, W=None):
    vp = VP(model, f, Z, W); rss, eta = gmin(vp); c, _ = vp.coef(eta); sp = vp.sp
    p = dict(zip(sp['nl'], eta)); p.update(dict(zip(sp['lin'], c)))
    if 'invQ' in p: p['Q'] = 1/p['invQ']
    return dict(params=p, eta=eta, c=c, rss=rss, n=vp.n, k=len(sp['nl'])+len(sp['lin']))

def predict(model, p, f):
    sp = spec_of(model); w = 2*np.pi*np.asarray(f, float); eta = [p[k] for k in sp['nl']]; c = np.array([p[k] for k in sp['lin']])
    return sp['basis'](eta, w) @ c

# --- parameter handling for profiles: parameter -> (kind, index, transform)
def _locate(model, param):
    sp = spec_of(model)
    if param in sp['nl']: return 'nl', sp['nl'].index(param), lambda g: g
    name = param.replace('log10_', ''); sign = 1
    if name == 'Q': name, sign = 'invQ', -1
    j = sp['lin'].index(name); return 'lin', j, (lambda g, s=sign: 10**(s*g))

def rss_fixed(model, f, Z, param, value, W=None):
    """Globally minimised RSS with one parameter fixed (value on the profile scale)."""
    kind, j, tr = _locate(model, param)
    if kind == 'nl': return gmin(VP(model, f, Z, W), fixed_nl=(j, value))[0]
    return gmin(VP(model, f, Z, W, fix=(j, tr(value))))[0]

def lr_at(model, f, Z, param, value, W=None, rss_min=None):
    rmin = fit(model, f, Z, W)['rss'] if rss_min is None else rss_min; rf = rss_fixed(model, f, Z, param, value, W)
    n = 2*len(f); return n*math.log(max(rf, rmin)/rmin)

def profile_curve(model, f, Z, param, W=None, n_grid=41):
    sp = spec_of(model); base = fit(model, f, Z, W); kind, j, _ = _locate(model, param)
    if kind == 'nl': lo, hi = sp['lo'][j], sp['hi'][j]; est = base['eta'][j]
    else:
        lo, hi = np.log10(sp['llo'][j]), np.log10(sp['lhi'][j]); est = math.log10(base['c'][j])
        if param == 'log10_Q': lo, hi, est = -hi, -lo, -est
    grid = np.unique(np.r_[np.linspace(lo, hi, n_grid), est + np.linspace(-0.06, 0.06, 13)*(hi-lo)].clip(lo, hi))
    prof = np.array([rss_fixed(model, f, Z, param, g, W) for g in grid]); rmin = min(base['rss'], prof.min())
    return grid, 2*len(f)*np.log(prof/rmin), est

def ci_from_curve(grid, lr, thr):
    i0 = int(np.argmin(lr))
    def cross(a, b): return grid[a] + (thr-lr[a])*(grid[b]-grid[a])/(lr[b]-lr[a]) if lr[b] != lr[a] else grid[a]
    j = i0
    while j > 0 and lr[j-1] <= thr: j -= 1
    lo = cross(j, j-1) if j > 0 else grid[0]
    j = i0
    while j < len(grid)-1 and lr[j+1] <= thr: j += 1
    hi = cross(j, j+1) if j < len(grid)-1 else grid[-1]
    return float(lo), float(hi), float(hi-lo), bool(lr[0] > thr and lr[-1] > thr)

def true_value(model, p, param):
    if param == 'log10_Q': return math.log10(p['Q'])
    if param.startswith('log10_') and param != 'log10_tau': return math.log10(p[param[6:]])
    return p[param]

# --- data
def spectra():
    d = pd.read_csv(T/'ecm_ready_complete_scans.csv'); kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct
    out = []
    for sf, g in d.groupby('source_file'):
        g = g.sort_values('frequency_hz'); out.append(dict(sf=sf, tissue=g.tissue.iloc[0], f=g.frequency_hz.to_numpy(float),
                     Z=(g.resistance_ohm + 1j*g.reactance_ohm).to_numpy(), kk_ok=bool(kk[sf] < 3)))
    return out

def sweeps(raw_dir):
    """Complete frequency sweeps per recording: each block of consecutive readings at one frequency is reduced to its
    median, and ascending runs covering all ten frequencies are kept."""
    res = {}
    for s in spectra():
        p = glob.glob(f'{raw_dir}/**/{s["sf"]}', recursive=True)[0]
        d = pd.read_csv(p, sep=';', comment='%', header=None, names=['st', 'f', 'mag', 'ph', 'R', 'X', 't', 'tis'])
        d['blk'] = (d.f.diff() != 0).cumsum(); B = d.groupby('blk').agg(f=('f', 'first'), R=('R', 'median'), X=('X', 'median')).reset_index(drop=True)
        cur, sw = [], []
        for _, r in B.iterrows():
            if cur and r.f <= cur[-1][0]:
                if len(cur) == 10: sw.append(cur)
                cur = []
            cur.append((r.f, r.R + 1j*r.X))
        if len(cur) == 10: sw.append(cur)
        res[s['sf']] = np.array([[z for _, z in c] for c in sw])
    return res

# --- structured covariance of the stacked relative residual [Re_1..Re_N, Im_1..Im_N]
def sigma_struct(psi, N=10):
    s0, gam, rf, rri, ratio = psi; k = np.arange(N); fk = (k+1)/1.0
    sd = s0*fk**gam; sd = np.r_[sd, ratio*sd]
    Rf = rf**np.abs(k[:, None]-k[None, :]); Rri = np.array([[1, rri], [rri, 1]])
    return np.outer(sd, sd)*np.kron(Rri, Rf)

def whitener(S):
    L = np.linalg.cholesky(S); return np.linalg.inv(L)

def noise_draw(rng, S_chol, Z):
    e = S_chol @ rng.standard_normal(S_chol.shape[0]); N = len(Z); sc = np.maximum(np.abs(Z), 1.0)
    return Z + sc*(e[:N] + 1j*e[N:])


In [ ]:
%%writefile src/review/e1_error_model.py
"""E1 steps 1-2: empirical error model from the 51 complete sweeps.

(a) Sweep-to-sweep deviations within a recording (12 recordings with >= 2 sweeps) are decomposed into the part
    absorbed by refitting R-CPE-L (in-model: descriptor variability) and the part left as residual (out-of-model).
(b) The residual covariance used for calibration is estimated from per-sweep M11 residuals (all 51 sweeps, all 20
    recordings) with a 5-parameter structured model (scale, frequency trend, AR(1) frequency correlation, Re-Im
    correlation, Im/Re scale ratio). Because fitted residuals are projections, r_s = P_s e_s with P_s = I - H_s (H_s the
    hat matrix of the linearized fit), the likelihood is evaluated in the 16-dimensional residual space of each sweep:
    U_s^T r_s ~ N(0, U_s^T Sigma U_s), U_s an orthonormal basis of range(P_s) (exact for a locally linear model).
    The naive estimate that ignores the projection is reported for comparison. Ledoit-Wolf shrinkage as a check;
    bootstrap over recordings.
(c) Between-sweep variability of the M11 descriptors (log10 Rinf, log10 Q, alpha) is tabulated per recording.
Usage: python e1_error_model.py RAW_DIR"""
import sys, math, json, numpy as np, pandas as pd
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import r2_core as R
raw = sys.argv[1]; rng = np.random.default_rng(7)
SP = {s['sf']: s for s in R.spectra()}; SW = R.sweeps(raw); N = 10

rows, resid, rec_of = [], [], []
for k, s in SP.items():
    X = SW[k]; zbar = X.mean(0)
    for i, z in enumerate(X):
        F = R.fit('rcpe_l', s['f'], z); vp = R.VP('rcpe_l', s['f'], z); r = vp.residual(F['eta'])
        sc = np.maximum(np.abs(z), 1.0); dev = (z - zbar)/sc; dev = np.r_[dev.real, dev.imag]
        rows.append(dict(source_file=k, tissue=s['tissue'], kk_ok=s['kk_ok'], sweep=i, n_sweeps=len(X), rss=F['rss'],
                         dev_energy=float(dev @ dev) if len(X) > 1 else np.nan, log10_Rinf=math.log10(F['params']['Rinf']),
                         log10_Q=math.log10(F['params']['Q']), alpha=F['params']['alpha'], L_mH=1e3*F['params']['L']))
        resid.append(r); rec_of.append(k)
SWF = pd.DataFrame(rows); SWF.to_csv(R.T/'e1_sweep_fits.csv', index=False); resid = np.array(resid); rec_of = np.array(rec_of)

def fit_psi(S, df, starts=None):
    def nll(u):
        psi = [math.exp(u[0]), u[1], math.tanh(u[2]), math.tanh(u[3]), math.exp(u[4])]
        C = R.sigma_struct(psi); sgn, ld = np.linalg.slogdet(C)
        return 0.5*df*(ld + np.trace(np.linalg.solve(C, S))) if sgn > 0 else 1e12
    best = None
    for st in (starts or [[math.log(0.01), 0, 0.0, 0, 0], [math.log(0.02), 0.5, 0.8, 0.3, 0], [math.log(0.005), -0.5, -0.5, -0.3, 0.3]]):
        r = minimize(nll, st, method='Nelder-Mead', options=dict(maxiter=4000 if starts else 20000, xatol=1e-7, fatol=1e-10))
        if best is None or r.fun < best.fun: best = r
    u = best.x; return np.array([math.exp(u[0]), u[1], math.tanh(u[2]), math.tanh(u[3]), math.exp(u[4])])

P_LIN = 4   # fitted parameters of M11 (alpha + 3 linear)
def jac_basis(s, z, F):
    """orthonormal basis of the residual space (complement of the fitted-parameter Jacobian range)"""
    vp = R.VP('rcpe_l', s['f'], z); eta = F['eta']; c, _ = vp.coef(eta)
    A = vp._sys(eta)[0]; h = 1e-6; Ap = vp._sys(eta + h)[0]; da = (Ap - A) @ c/h
    sp = vp.sp; free = [j for j in range(3) if c[j] > sp['llo'][j]*(1+1e-9) and c[j] < sp['lhi'][j]*(1-1e-9)]
    J = np.column_stack([da] + [A[:, j] for j in free]) if 0.3+1e-9 < eta[0] < 1-1e-9 else A[:, free]
    Qf, _ = np.linalg.qr(J, mode='complete'); return Qf[:, J.shape[1]:]
Us = []
for k, s in SP.items():
    for z in SW[k]: Us.append(jac_basis(s, z, R.fit('rcpe_l', s['f'], z)))
Zs = [U.T @ r for U, r in zip(Us, resid)]
def unpack(u): return np.array([math.exp(u[0]), u[1], math.tanh(u[2]), math.tanh(u[3]), math.exp(u[4])])
def fit_proj(idx, starts=None, maxiter=20000):
    def nll(u):
        C = R.sigma_struct(unpack(u)); t = 0.0
        for i in idx:
            M = Us[i].T @ C @ Us[i]; sgn, ld = np.linalg.slogdet(M)
            if sgn <= 0: return 1e12
            t += ld + Zs[i] @ np.linalg.solve(M, Zs[i])
        return 0.5*t
    best = None
    for st in (starts or [[math.log(0.02), 0, 0.0, 0, 0], [math.log(0.05), 0.5, 0.8, 0.3, 0], [math.log(0.01), -0.5, -0.5, -0.3, 0.3]]):
        r = minimize(nll, st, method='Nelder-Mead', options=dict(maxiter=maxiter, xatol=1e-7, fatol=1e-10))
        if best is None or r.fun < best.fun: best = r
    return unpack(best.x)
def pooled(idx): Rm = resid[idx]; return Rm.T @ Rm/len(idx)     # sample covariance of residual vectors
allidx = np.arange(len(resid)); S = pooled(allidx)
psi_naive = fit_psi(S, len(resid)); psi = fit_proj(allidx)
print('psi naive', psi_naive, 'psi projected', psi, flush=True)
sd = np.sqrt(np.diag(S)); Cr = S/np.outer(sd, sd)
lw = LedoitWolf().fit(resid); Clw = lw.covariance_/np.outer(np.sqrt(np.diag(lw.covariance_)), np.sqrt(np.diag(lw.covariance_)))
recs = np.unique(rec_of); boots = []
u0 = [math.log(psi[0]), psi[1], math.atanh(psi[2]), math.atanh(psi[3]), math.log(psi[4])]
for b in range(100):
    kb = rng.choice(recs, len(recs), replace=True); idx = np.concatenate([np.flatnonzero(rec_of == k) for k in kb])
    boots.append(fit_proj(idx, starts=[u0], maxiter=2500))
    if b % 20 == 0: print('boot', b, flush=True)
boots = np.array(boots); ci = np.quantile(boots, [0.025, 0.975], axis=0)
nb = []; un = [math.log(psi_naive[0]), psi_naive[1], math.atanh(psi_naive[2]), math.atanh(psi_naive[3]), math.log(psi_naive[4])]
for b in range(200):
    kb = rng.choice(recs, len(recs), replace=True); idx = np.concatenate([np.flatnonzero(rec_of == k) for k in kb])
    nb.append(fit_psi(pooled(idx), len(idx), starts=[un]))
nci = np.quantile(np.array(nb), [0.025, 0.975], axis=0)
names = ['sigma0', 'gamma', 'rho_f', 'rho_ReIm', 'ratio_Im_Re']
EM = pd.DataFrame(dict(parameter=names, estimate=psi, ci_low=ci[0], ci_high=ci[1], naive_estimate=psi_naive, naive_ci_low=nci[0], naive_ci_high=nci[1]))
extra = dict(sweeps_used=len(resid), recordings=len(recs), residual_df=len(resid)*(2*N - P_LIN),
             sample_lag1_corr_Re=np.mean([Cr[k, k+1] for k in range(N-1)]), sample_lag1_corr_Im=np.mean([Cr[N+k, N+k+1] for k in range(N-1)]),
             sample_ReIm_corr=np.mean([Cr[k, N+k] for k in range(N)]), LW_shrinkage=lw.shrinkage_,
             LW_lag1_corr_Re=np.mean([Clw[k, k+1] for k in range(N-1)]), LW_lag1_corr_Im=np.mean([Clw[N+k, N+k+1] for k in range(N-1)]))
for k, v in extra.items(): EM.loc[len(EM)] = [k, v, np.nan, np.nan, np.nan, np.nan, np.nan]
EM.to_csv(R.T/'Table_E1_error_model.csv', index=False); np.save(R.T/'E1_psi_boot.npy', boots)
json.dump(dict(psi=list(map(float, psi_naive))), open(R.T/'E1_psi.json', 'w')); json.dump(dict(psi=list(map(float, psi))), open(R.T/'E1_psi_projected.json', 'w'))   # residual-based model is used for calibration
print(EM.round(4).to_string(), flush=True)

# (a) in-model vs out-of-model share of sweep-to-sweep deviation energy
m = SWF[SWF.n_sweeps > 1]
share = m.groupby('source_file').apply(lambda g: g.rss.mean()/g.dev_energy.mean(), include_groups=False)
# (c) between-sweep descriptor SD per recording
V = m.groupby(['source_file', 'tissue', 'kk_ok']).agg(n_sweeps=('sweep', 'size'), sd_log10_Rinf=('log10_Rinf', 'std'), sd_log10_Q=('log10_Q', 'std'),
        sd_alpha=('alpha', 'std'), sd_L_mH=('L_mH', 'std')).reset_index()
V['residual_share_of_deviation'] = V.source_file.map(share); V.to_csv(R.T/'Table_E1_between_sweep.csv', index=False)
print(V.drop(columns='source_file').round(4).to_string())
print('median residual share of sweep deviation energy:', share.median(), share.quantile([.25, .75]).tolist())


In [ ]:
%%writefile src/review/e1_calibrate.py
"""E1 steps 3-4: bootstrap-calibrated and GLS profile-likelihood intervals, measured intervals and coverage validation.
Usage: python e1_calibrate.py M11|M10|DB   (M11: calibration + validation of log10 Q, alpha, log10 Rinf;
       M10: calibration + measured intervals for log10 tau, log10 dR and tau coverage; DB: reduced double bootstrap, M11)
Noise models are all scaled per spectrum so that the expected RSS equals the observed RSS of the published fit."""
import sys, json, math, time, pickle, numpy as np, pandas as pd
import r2_core as R
MODE = sys.argv[1]; T = R.T; CACHE = T.parent/'cache_e1'; CACHE.mkdir(exist_ok=True)
psi = np.array(json.load(open(T/'E1_psi.json'))['psi']); S_emp = R.sigma_struct(psi); W_gls = R.whitener(S_emp)
SP = R.spectra(); THR = 3.84; B = 200; RV = 100

def scaled(S, rss, p):          # scale covariance so that E[RSS] = observed RSS (relative residual space)
    return S*(rss/(np.trace(S)*(1 - p/S.shape[0])))
def cond_cov(name):
    if name in ('emp', 'mis'): return S_emp
    if name == 'rho_lo': q = psi.copy(); q[2] = max(q[2]-0.2, -0.95); return R.sigma_struct(q)
    if name == 'rho_hi': q = psi.copy(); q[2] = min(q[2]+0.2, 0.95); return R.sigma_struct(q)
    if name == 'white': return np.eye(20)
    if name == 'ar1': return R.sigma_struct([1.0, 0.0, 0.6, 0.0, 1.0])

def lr_pair(model, f, z, params_truth, pars):
    """LR at the true values under the independent (nominal) and GLS objectives."""
    out = {}
    for tag, W in (('nom', None), ('gls', W_gls)):
        rmin = R.fit(model, f, z, W)['rss']
        for pn in pars: out[(tag, pn)] = R.lr_at(model, f, z, pn, params_truth[pn], W, rmin)
    return out

def truths(model, F):
    p = F['params']
    if model == 'rcpe_l': return {'log10_Q': math.log10(p['Q']), 'alpha': p['alpha'], 'log10_Rinf': math.log10(p['Rinf'])}
    return {'log10_tau': p['log10_tau'], 'log10_dR': math.log10(p['dR'])}

def calibrate(model, s, pars, zt, tv, rss, seed, nb=B, Sshape=None):
    Sc = scaled(S_emp if Sshape is None else Sshape, rss, 4 if model == 'rcpe_l' else 5); Lc = np.linalg.cholesky(Sc); rng = np.random.default_rng(seed); L = []
    for b in range(nb): L.append(lr_pair(model, s['f'], R.noise_draw(rng, Lc, zt), tv, pars))
    return {k: float(np.quantile([d[k] for d in L], 0.95)) for k in L[0]}, L

if MODE == 'M11':
    pars = ['log10_Q', 'alpha', 'log10_Rinf']; cal_rows, cov_rows, int_rows = [], [], []
    for si, s in enumerate(SP):
        cp = CACHE/f'm11_{si}.pkl'
        if cp.exists(): a, b_, c = pickle.load(open(cp, 'rb')); cal_rows += a; cov_rows += b_; int_rows += c; continue
        t0 = time.time(); F = R.fit('rcpe_l', s['f'], s['Z']); tv = truths('rcpe_l', F); zt = R.predict('rcpe_l', F['params'], s['f'])
        cstar, _ = calibrate('rcpe_l', s, pars, zt, tv, F['rss'], 1000+si)
        a = [dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], model='M11', objective=k[0], param=k[1], c_star=v) for k, v in cstar.items()]
        # measured intervals: nominal and GLS profile curves with chi2 and calibrated thresholds
        c = []
        for tag, W in (('nom', None), ('gls', W_gls)):
            for pn in pars:
                g, lr, est = R.profile_curve('rcpe_l', s['f'], s['Z'], pn, W)
                for thr_name, thr in (('chi2', THR), ('cal', cstar[(tag, pn)])):
                    lo, hi, wd, cl = R.ci_from_curve(g, lr, thr)
                    c.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], model='M11', objective=tag, param=pn, threshold=thr_name,
                                  thr=thr, estimate=est, ci_low=lo, ci_high=hi, width=wd, closed=cl))
        b_ = []
        if s['kk_ok']:   # coverage validation on independent draws
            rng = np.random.default_rng(5000+si)
            for cond in ['emp', 'rho_lo', 'rho_hi', 'white', 'ar1', 'mis']:
                Lc = np.linalg.cholesky(scaled(cond_cov(cond), F['rss'], 4)); p = F['params']
                for r in range(RV):
                    if cond == 'mis':
                        w = 2*np.pi*s['f']; fc = 10**rng.uniform(np.log10(200), np.log10(3000)); tau = 1/(2*np.pi*fc); a10 = min(max(p['alpha'], 0.5), 0.95)
                        dR = tau**a10/p['Q']; z0 = p['Rinf'] + dR/(1 + (1j*w*tau)**a10) + 1j*w*p['L']
                        tvv = {'log10_Q': math.log10(p['Q']), 'alpha': a10, 'log10_Rinf': math.log10(p['Rinf'])}
                    else: z0, tvv = zt, tv
                    lr = lr_pair('rcpe_l', s['f'], R.noise_draw(rng, Lc, z0), tvv, pars)
                    for pn in pars:
                        b_.append(dict(source_file=s['sf'], cond=cond, rep=r, param=pn, lr_nom=lr[('nom', pn)], lr_gls=lr[('gls', pn)],
                                       c_nom=cstar[('nom', pn)], c_gls=cstar[('gls', pn)]))
        pickle.dump((a, b_, c), open(cp, 'wb')); cal_rows += a; cov_rows += b_; int_rows += c
        print(time.strftime('%H:%M:%S'), 'M11', si, s['tissue'], f'{time.time()-t0:.0f}s', flush=True)
    pd.DataFrame(cal_rows).to_csv(T/'e1_cstar_M11.csv', index=False); pd.DataFrame(int_rows).to_csv(T/'e1_intervals_M11.csv', index=False)
    CV = pd.DataFrame(cov_rows); CV.to_csv(T/'e1_coverage_runs_M11.csv', index=False)

if MODE == 'M10':
    pars = ['log10_tau', 'log10_dR']; cal_rows, int_rows, cov_rows = [], [], []
    for si, s in enumerate(SP):
        cp = CACHE/f'm10_{si}.pkl'
        if cp.exists(): a, c, b_ = pickle.load(open(cp, 'rb')); cal_rows += a; int_rows += c; cov_rows += b_; continue
        t0 = time.time(); F = R.fit('cole_l', s['f'], s['Z']); tv = truths('cole_l', F); zt = R.predict('cole_l', F['params'], s['f'])
        cstar, _ = calibrate('cole_l', s, pars, zt, tv, F['rss'], 2000+si, nb=100)
        a = [dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], model='M10', objective=k[0], param=k[1], c_star=v) for k, v in cstar.items()]
        c = []
        for tag, W in (('nom', None), ('gls', W_gls)):
            for pn in pars:
                g, lr, est = R.profile_curve('cole_l', s['f'], s['Z'], pn, W)
                for thr_name, thr in (('chi2', THR), ('cal', cstar[(tag, pn)])):
                    lo, hi, wd, cl = R.ci_from_curve(g, lr, thr)
                    c.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], model='M10', objective=tag, param=pn, threshold=thr_name,
                                  thr=thr, estimate=est, ci_low=lo, ci_high=hi, width=wd, closed=cl))
        b_ = []
        if s['kk_ok']:   # tau coverage: Cole truth with f_c below the band, calibrated at that truth, emp noise
            rng = np.random.default_rng(6000+si); p = R.fit('rcpe_l', s['f'], s['Z'])['params']; w = 2*np.pi*s['f']
            fc = 10**rng.uniform(np.log10(200), np.log10(3000)); tau = 1/(2*np.pi*fc); a10 = min(max(p['alpha'], 0.5), 0.95); dR = tau**a10/p['Q']
            z0 = p['Rinf'] + dR/(1 + (1j*w*tau)**a10) + 1j*w*p['L']; tvv = {'log10_tau': math.log10(tau), 'log10_dR': math.log10(dR)}
            cst, _ = calibrate('cole_l', s, ['log10_tau'], z0, tvv, F['rss'], 7000+si, nb=100)
            Lc = np.linalg.cholesky(scaled(S_emp, F['rss'], 5))
            for r in range(50):
                lr = lr_pair('cole_l', s['f'], R.noise_draw(rng, Lc, z0), tvv, ['log10_tau'])
                b_.append(dict(source_file=s['sf'], rep=r, param='log10_tau', lr_nom=lr[('nom', 'log10_tau')], lr_gls=lr[('gls', 'log10_tau')],
                               c_nom=cst[('nom', 'log10_tau')], c_gls=cst[('gls', 'log10_tau')]))
        pickle.dump((a, c, b_), open(cp, 'wb')); cal_rows += a; int_rows += c; cov_rows += b_
        print(time.strftime('%H:%M:%S'), 'M10', si, s['tissue'], f'{time.time()-t0:.0f}s', flush=True)
    pd.DataFrame(cal_rows).to_csv(T/'e1_cstar_M10.csv', index=False); pd.DataFrame(int_rows).to_csv(T/'e1_intervals_M10.csv', index=False)
    pd.DataFrame(cov_rows).to_csv(T/'e1_coverage_runs_M10.csv', index=False)

if MODE == 'DB':   # reduced double bootstrap: recalibrate at each replicate's own estimate (plug-in effect), emp noise, M11
    pars = ['log10_Q', 'alpha']; rows = []
    for si, s in enumerate(SP):
        if not s['kk_ok']: continue
        cp = CACHE/f'db_{si}.pkl'
        if cp.exists(): rows += pickle.load(open(cp, 'rb')); continue
        t0 = time.time(); F = R.fit('rcpe_l', s['f'], s['Z']); tv = truths('rcpe_l', F); zt = R.predict('rcpe_l', F['params'], s['f'])
        Lc = np.linalg.cholesky(scaled(S_emp, F['rss'], 4)); rng = np.random.default_rng(8000+si); out = []
        for r in range(25):
            z = R.noise_draw(rng, Lc, zt); lr = lr_pair('rcpe_l', s['f'], z, tv, pars)
            Fs = R.fit('rcpe_l', s['f'], z); cs, _ = calibrate('rcpe_l', dict(f=s['f']), pars, R.predict('rcpe_l', Fs['params'], s['f']),
                                                                 truths('rcpe_l', Fs), Fs['rss'], 9000+100*si+r, nb=80)
            for pn in pars: out.append(dict(source_file=s['sf'], rep=r, param=pn, lr_nom=lr[('nom', pn)], lr_gls=lr[('gls', pn)], c_nom=cs[('nom', pn)], c_gls=cs[('gls', pn)]))
        pickle.dump(out, open(cp, 'wb')); rows += out; print(time.strftime('%H:%M:%S'), 'DB', si, f'{time.time()-t0:.0f}s', flush=True)
    pd.DataFrame(rows).to_csv(T/'e1_coverage_runs_DB.csv', index=False)


In [ ]:
%%writefile src/review/e1b_bias.py
"""E1b: deterministic (noise-free) bias of the R-CPE-L descriptors when the truth is a Cole dispersion with f_c below the band.
For each KK-consistent spectrum, the Cole truth shares Rinf, L and the high-frequency asymptote Q = tau^alpha/dR with the fitted
R-CPE-L; f_c is varied on a grid. Bias = fitted - asymptotic value."""
import math, numpy as np, pandas as pd
import r2_core as R
rows = []
for s in R.spectra():
    if not s['kk_ok']: continue
    p = R.fit('rcpe_l', s['f'], s['Z'])['params']; w = 2*np.pi*s['f']; a = min(max(p['alpha'], 0.5), 0.95)
    for fc in [10, 30, 100, 300, 1000, 3000]:
        tau = 1/(2*np.pi*fc); dR = tau**a/p['Q']; z = p['Rinf'] + dR/(1 + (1j*w*tau)**a) + 1j*w*p['L']
        q = R.fit('rcpe_l', s['f'], z)['params']
        rows.append(dict(source_file=s['sf'], fc_hz=fc, fmin_over_fc=1e4/fc, bias_log10_Q=math.log10(q['Q']) - math.log10(p['Q']),
                         bias_alpha=q['alpha'] - a, bias_log10_Rinf=math.log10(q['Rinf']) - math.log10(p['Rinf'])))
D = pd.DataFrame(rows); D.to_csv(R.T/'e1b_bias_runs.csv', index=False)
S = D.groupby(['fc_hz', 'fmin_over_fc'])[['bias_log10_Q', 'bias_alpha', 'bias_log10_Rinf']].median().reset_index()
S.to_csv(R.T/'Table_E1b_bias.csv', index=False); print(S.round(4).to_string())


In [ ]:
%%writefile src/review/e2_hf.py
"""E2: sensitivity of the R-CPE descriptors (Rinf, Q, alpha) to the treatment of the high-frequency correction.
T0 R-CPE + series L (published); T1 R-CPE without correction, 10-80 kHz; T1b R-CPE + L, 10-80 kHz;
T2 R-CPE x exp(-j w t_d) (pure delay = phase error linear in frequency); T3 R-CPE x exp(j phi0) (constant phase offset);
T4 R-CPE + K (j w)^nu (generalised high-frequency CPE term; nu = 1 is the inductance)."""
import math, numpy as np, pandas as pd
import r2_core as R
TR = [('T0', 'rcpe_l', None), ('T1', 'rcpe', 80e3), ('T1b', 'rcpe_l', 80e3), ('T2', 'rcpe_delay', None), ('T3', 'rcpe_phase', None), ('T4', 'rcpe_hfcpe', None)]
rows = []
for s in R.spectra():
    for name, model, fmax in TR:
        m = np.ones(len(s['f']), bool) if fmax is None else s['f'] <= fmax; f, Z = s['f'][m], s['Z'][m]
        F = R.fit(model, f, Z); p = F['params']; n = F['n']; k = F['k']
        aicc = n*math.log(F['rss']/n) + 2*k + 2*k*(k+1)/(n-k-1)
        sp = R.spec_of(model); at_b = [nm for nm, v, lo, hi in zip(sp['nl'], F['eta'], sp['lo'], sp['hi']) if abs(v-lo) < 1e-6*(hi-lo) or abs(v-hi) < 1e-6*(hi-lo)]
        rows.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], treatment=name, model=model, band_kHz=f'{f.min()/1e3:g}-{f.max()/1e3:g}',
                         NRMSE=math.sqrt(F['rss']/len(f)), AICc_fullband=aicc if fmax is None else np.nan, log10_Rinf=math.log10(p['Rinf']),
                         log10_Q=math.log10(p['Q']), alpha=p['alpha'], extra=p.get('td_us', p.get('phi0', p.get('nu', p.get('L', np.nan)))),
                         K=p.get('K', np.nan), nl_at_bound=';'.join(at_b)))
D = pd.DataFrame(rows); base = D[D.treatment == 'T0'].set_index('source_file')
for c in ['log10_Rinf', 'log10_Q', 'alpha']: D['d_'+c] = D[c] - D.source_file.map(base[c])
D.to_csv(R.T/'e2_hf_runs.csv', index=False)
q = lambda x: f"{np.median(x):.3f} [{np.quantile(x, .25):.3f}, {np.quantile(x, .75):.3f}]"
S = D.groupby('treatment').agg(NRMSE=('NRMSE', q), abs_d_log10_Rinf=('d_log10_Rinf', lambda x: q(x.abs())), abs_d_log10_Q=('d_log10_Q', lambda x: q(x.abs())),
        abs_d_alpha=('d_alpha', lambda x: q(x.abs())), extra=('extra', q), at_bound=('nl_at_bound', lambda x: int((x != '').sum()))).reset_index()
best = D[D.AICc_fullband.notna()].loc[lambda d: d.groupby('source_file').AICc_fullband.idxmin()].treatment.value_counts()
S['AICc_best_count'] = S.treatment.map(best).fillna(0).astype(int)
S.to_csv(R.T/'Table_E2_hf_treatments.csv', index=False); print(S.to_string())
for kk in [True, False]:
    print('KK ok' if kk else 'KK inconsistent'); print(D[D.kk_ok == kk].groupby('treatment')[['d_log10_Q', 'd_alpha', 'd_log10_Rinf']].apply(lambda g: g.abs().median()).round(4).to_string())


In [ ]:
%%writefile src/review/e3_model_selection.py
"""E3: uncertainty-aware comparison of R-CPE-L (M11) and Cole+L (M10).
AICc under the independent and GLS (empirical covariance) objectives; leave-one-frequency-out split into interior
(20-90 kHz, interpolation) and endpoint (10, 100 kHz, extrapolation) errors; parametric-bootstrap LR test with M11 as
the null (boundary of M10), B = 100 draws from the fitted M11 with empirical noise; frequency-range sensitivity."""
import json, math, time, pickle, numpy as np, pandas as pd
from scipy.stats import wilcoxon
import r2_core as R
T = R.T; psi = np.array(json.load(open(T/'E1_psi.json'))['psi']); S_emp = R.sigma_struct(psi)
CACHE = T.parent/'cache_e3'; CACHE.mkdir(exist_ok=True)

def sub_W(mask):
    m2 = np.r_[mask, mask]; return R.whitener(S_emp[np.ix_(m2, m2)])
def aicc(F):
    n, k = F['n'], F['k']; return n*math.log(F['rss']/n) + 2*k + 2*k*(k+1)/(n-k-1)
def lofo(model, f, Z):
    e = []
    for k in range(len(f)):
        m = np.ones(len(f), bool); m[k] = False; F = R.fit(model, f[m], Z[m]); zp = R.predict(model, F['params'], f[k:k+1])[0]
        e.append(abs(zp - Z[k])**2/abs(Z[k])**2)
    return np.array(e)

rows = []
for si, s in enumerate(R.spectra()):
    cp = CACHE/f'e3_{si}.pkl'
    if cp.exists(): rows.append(pickle.load(open(cp, 'rb'))); continue
    t0 = time.time(); f, Z = s['f'], s['Z']; rec = dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'])
    for band, m in [('10-100', f > 0), ('10-80', f <= 80e3), ('20-100', f >= 20e3)]:
        W = sub_W(m)
        for tag, Wt in (('nom', None), ('gls', W)):
            a, b = R.fit('cole_l', f[m], Z[m], Wt), R.fit('rcpe_l', f[m], Z[m], Wt); rec[f'dAICc_{tag}_{band}'] = aicc(a) - aicc(b)
    e10, e11 = lofo('cole_l', f, Z), lofo('rcpe_l', f, Z); inner = (f > 10e3) & (f < 100e3)
    rec.update(lofo_int_M10=math.sqrt(e10[inner].mean()), lofo_int_M11=math.sqrt(e11[inner].mean()),
               lofo_end_M10=math.sqrt(e10[~inner].mean()), lofo_end_M11=math.sqrt(e11[~inner].mean()))
    # parametric bootstrap LR test, null = M11 (GLS and nominal objectives)
    W = R.whitener(S_emp); rng = np.random.default_rng(3000+si)
    for tag, Wt in (('nom', None), ('gls', W)):
        F11, F10 = R.fit('rcpe_l', f, Z, Wt), R.fit('cole_l', f, Z, Wt); lam = 20*math.log(F11['rss']/min(F10['rss'], F11['rss']))
        zt = R.predict('rcpe_l', F11['params'], f); F11n = R.fit('rcpe_l', f, Z)
        Sc = S_emp*(F11n['rss']/(np.trace(S_emp)*(1-4/20))); Lc = np.linalg.cholesky(Sc); lb = []
        for b in range(100):
            z = R.noise_draw(rng, Lc, zt); r11, r10 = R.fit('rcpe_l', f, z, Wt)['rss'], R.fit('cole_l', f, z, Wt)['rss']
            lb.append(20*math.log(r11/min(r10, r11)))
        rec[f'LR_obs_{tag}'] = lam; rec[f'p_boot_{tag}'] = (1 + sum(x >= lam for x in lb))/(1 + len(lb)); rec[f'LR_boot95_{tag}'] = float(np.quantile(lb, .95))
    pickle.dump(rec, open(cp, 'wb')); rows.append(rec); print(time.strftime('%H:%M:%S'), si, s['tissue'], f'{time.time()-t0:.0f}s', flush=True)

D = pd.DataFrame(rows); D.to_csv(T/'e3_model_selection_runs.csv', index=False)
summ = {}
for tag in ['nom', 'gls']:
    for band in ['10-100', '10-80', '20-100']: summ[f'M11_pref_AICc_{tag}_{band}'] = int((D[f'dAICc_{tag}_{band}'] > 0).sum())
    summ[f'bootLR_reject_M11_{tag}'] = int((D[f'p_boot_{tag}'] < 0.05).sum()); summ[f'median_p_boot_{tag}'] = D[f'p_boot_{tag}'].median()
for part in ['int', 'end']:
    a, b = D[f'lofo_{part}_M10'], D[f'lofo_{part}_M11']
    summ[f'lofo_{part}_M11_better'] = int((b < a).sum()); summ[f'lofo_{part}_median_M10'] = a.median(); summ[f'lofo_{part}_median_M11'] = b.median()
    summ[f'lofo_{part}_wilcoxon_p'] = wilcoxon(b, a).pvalue
S = pd.Series(summ); S.to_csv(T/'Table_E3_model_selection.csv'); print(S.round(4).to_string())


In [ ]:
%%writefile src/review/e4_thresholds.py
"""E4: sensitivity of the practical-identifiability classification to the precision threshold and to the M11 search bounds.
Uses the measured intervals from E1 (nominal and calibrated thresholds, independent and GLS objectives).
Bound perturbation (independent objective): alpha lower bound 0.30 -> 0.20; 1/Q bounds [1, 1e12] -> [1e-2, 1e14]."""
import math, numpy as np, pandas as pd
import r2_core as R
T = R.T
I = pd.concat([pd.read_csv(T/'e1_intervals_M11.csv'), pd.read_csv(T/'e1_intervals_M10.csv')])
rows = []
for (lab_log, lab_a) in [(0.5, 0.1), (1.0, 0.2), (1.5, 0.3)]:
    lim = I.param.map(lambda p: lab_a if p == 'alpha' else lab_log); I['id'] = I.closed & (I.width < lim)
    for (m, p, o, t), g in I.groupby(['model', 'param', 'objective', 'threshold']):
        rows.append(dict(log_threshold_dec=lab_log, alpha_threshold=lab_a, model=m, param=p, objective=o, interval=t,
                         ident_all=int(g.id.sum()), ident_kk_ok=int(g[g.kk_ok].id.sum()), n_kk_ok=int(g.kk_ok.sum())))
S = pd.DataFrame(rows); S.to_csv(T/'Table_E4_threshold_sensitivity.csv', index=False)
print(S[(S.objective == 'nom')].pivot_table(index=['model', 'param', 'interval'], columns='log_threshold_dec', values=['ident_all', 'ident_kk_ok']).to_string())
print(S[(S.objective == 'gls')].pivot_table(index=['model', 'param', 'interval'], columns='log_threshold_dec', values=['ident_all', 'ident_kk_ok']).to_string())

# bound perturbation for the M11 descriptors
cst = pd.read_csv(T/'e1_cstar_M11.csv'); cst = cst[cst.objective == 'nom'].set_index(['source_file', 'param']).c_star
nl, lo, hi, lin, llo, lhi, B = R.MODELS['rcpe_l']; R.MODELS['rcpe_l'] = (nl, [0.20], hi, lin, [1e-3, 1e-2, 1e-9], [1e7, 1e14, 1.0], B)
rows = []
for s in R.spectra():
    for pn in ['log10_Q', 'alpha']:
        g, lr, est = R.profile_curve('rcpe_l', s['f'], s['Z'], pn)
        for tn, thr in (('chi2', 3.84), ('cal', cst[(s['sf'], pn)])):
            a, b, wd, cl = R.ci_from_curve(g, lr, thr)
            rows.append(dict(source_file=s['sf'], kk_ok=s['kk_ok'], param=pn, interval=tn, width=wd, closed=cl, ident=bool(cl and wd < (0.2 if pn == 'alpha' else 1.0)), estimate=est))
B2 = pd.DataFrame(rows); B2.to_csv(T/'e4_bounds_runs.csv', index=False)
BS = B2.groupby(['param', 'interval']).agg(ident_all=('ident', 'sum'), ident_kk_ok=('ident', lambda x: int(x[B2.loc[x.index, 'kk_ok']].sum())), median_width=('width', 'median')).reset_index()
BS.to_csv(T/'Table_E4_bounds.csv', index=False); print(BS.to_string())


In [ ]:
%%writefile src/review/e5_band_factorial.py
"""E5: frequency-grid design with range and number of frequencies separated.
Truth: Cole+L with tissue-like parameters (as band_design.py) and f_c drawn from three classes.
Fitted model: Cole+L (bounded VarPro, global). Practical identifiability of log10 tau: 95% profile interval (chi2 3.84)
closed and narrower than 1 decade; sensitivity with the median calibrated critical value of log10 tau (6.38, Table S15). Designs (f_max = 100 kHz):
  base   : equal-N log grids, N = 10 and 20, white 2.5% noise
  linear : N = 20 linear grid
  corr   : N = 20 log grid, AR(1)-correlated noise (rho = 0.5 between adjacent frequencies)
  ep10/ep50 : N = 20 log grid, series electrode-polarization CPE (nu = 0.6) with |Z_EP(1 kHz)| = 10% / 50% of |Z_tissue(1 kHz)|,
              not included in the fitted model
  mis    : N = 20 log grid, second (high-frequency) Cole dispersion in the truth, single Cole fitted
Usage: python e5_band_factorial.py [n_per_cell]"""
import sys, math, time, pickle, numpy as np, pandas as pd
import r2_core as R
T = R.T; NS = int(sys.argv[1]) if len(sys.argv) > 1 else 20; CACHE = T.parent/'cache_e5'; CACHE.mkdir(exist_ok=True)
FMIN = [10e3, 3e3, 1e3, 300.0]; FC = {'low': (100, 1000), 'mid': (1000, 10000), 'near': (10000, 30000)}
THR_CAL = 6.38
DESIGNS = [('base', 10), ('base', 20), ('linear', 20), ('corr', 20), ('ep10', 20), ('ep50', 20), ('mis', 20)]

def truth(i, fc_rng):
    rng = np.random.default_rng(424242 + i)
    Rinf = 10**rng.uniform(np.log10(500), np.log10(5000)); a = rng.uniform(0.60, 0.95); L = 10**rng.uniform(-3.4, -2.6)
    fc = 10**rng.uniform(*np.log10(fc_rng)); tau = 1/(2*np.pi*fc); dR = rng.uniform(1, 3)*Rinf
    return dict(Rinf=Rinf, dR=dR, tau=tau, alpha=a, L=L, fc=fc, nu_ep=0.6, fc2=10**rng.uniform(np.log10(3e4), np.log10(3e5))), rng

def zcole(p, w): return p['Rinf'] + p['dR']/(1 + (1j*w*p['tau'])**p['alpha']) + 1j*w*p['L']

rows = []
for di, (design, N) in enumerate(DESIGNS):
    for fcn, fcr in FC.items():
        cp = CACHE/f'v2_{design}_{N}_{fcn}_{NS}.pkl'
        if cp.exists(): rows += pickle.load(open(cp, 'rb')); continue
        t0 = time.time(); out = []
        for i in range(NS):
            p, rng = truth(i, fcr); nseed = rng.integers(1 << 30)
            for fmin in FMIN:
                f = np.linspace(fmin, 1e5, N) if design == 'linear' else np.logspace(np.log10(fmin), 5, N); w = 2*np.pi*f
                z0 = zcole(p, w)
                if design.startswith('ep'):
                    frac = 0.1 if design == 'ep10' else 0.5; w1 = 2*np.pi*1e3; K = frac*abs(zcole(p, np.array([w1]))[0])*w1**p['nu_ep']
                    z0 = z0 + K*(1j*w)**(-p['nu_ep'])
                if design == 'mis': z0 = z0 + 0.2*p['dR']/(1 + 1j*w/(2*np.pi*p['fc2']))**0.8
                nr = np.random.default_rng(nseed); e = nr.standard_normal((2, N))
                if design == 'corr':
                    for k in range(1, N): e[:, k] = 0.5*e[:, k-1] + math.sqrt(0.75)*e[:, k]
                z = z0 + 0.025*np.abs(z0)*(e[0] + 1j*e[1])/math.sqrt(2)
                g, lr, est = R.profile_curve('cole_l', f, z, 'log10_tau', n_grid=33); lo, hi, wd, cl = R.ci_from_curve(g, lr, 3.84)
                lo2, hi2, wd2, cl2 = R.ci_from_curve(g, lr, THR_CAL)
                F = R.fit('cole_l', f, z)
                out.append(dict(design=design, N=N, fc_class=fcn, spectrum=i, fmin_hz=fmin, fc_true=p['fc'], ratio_fmin_fc=fmin/p['fc'],
                                tau_width=wd, tau_closed=cl, tau_ident=bool(cl and wd < 1.0), tau_ident_cal=bool(cl2 and wd2 < 1.0), tau_abserr=abs(est - math.log10(p['tau'])),
                                dR_abserr=abs(math.log10(F['params']['dR']) - math.log10(p['dR']))))
        pickle.dump(out, open(cp, 'wb')); rows += out
        print(time.strftime('%H:%M:%S'), design, N, fcn, f'{time.time()-t0:.0f}s', flush=True)
D = pd.DataFrame(rows); D.to_csv(T/'e5_band_runs.csv', index=False)
S = D.groupby(['design', 'N', 'fc_class', 'fmin_hz']).agg(n=('spectrum', 'size'), tau_ident_pct=('tau_ident', lambda x: 100*x.mean()), tau_ident_cal_pct=('tau_ident_cal', lambda x: 100*x.mean()),
        median_tau_width=('tau_width', 'median'), median_tau_abserr=('tau_abserr', 'median'), median_dR_abserr=('dR_abserr', 'median')).reset_index()
S.to_csv(T/'Table_E5_band_factorial.csv', index=False)
P = S.pivot_table(index=['design', 'N', 'fc_class'], columns='fmin_hz', values='tau_ident_pct').iloc[:, ::-1]
print(P.round(0).to_string())
D['ratio_bin'] = pd.cut(D.ratio_fmin_fc, [0, 0.3, 1, 3, 10, 1e4], labels=['<0.3', '0.3-1', '1-3', '3-10', '>10'])
Q = D.groupby(['design', 'N', 'ratio_bin'], observed=True).tau_ident.mean().mul(100).unstack(); Q.to_csv(T/'Table_E5_ratio.csv'); print(Q.round(0).to_string())
Q2 = D.groupby(['design', 'N', 'ratio_bin'], observed=True).tau_ident_cal.mean().mul(100).unstack(); Q2.to_csv(T/'Table_E5_ratio_cal.csv'); print(Q2.round(0).to_string())


In [ ]:
%%writefile src/review/e6_mechanism.py
"""E6: mechanism of the VarPro gain.
VPinit-MS-TRF: full-parameter multistart TRF in which each start draws the nonlinear parameters exactly as VP-MS-TRF does
and sets the linear coefficients to their bounded-LS optimum c*(eta0); the search then proceeds in the full space.
Same spectra, seeds (first 10 runs of the main benchmark), 2,000 NFE (the start-up solve counts as one evaluation) and
0.1% success rule. Conditioning: singular-value condition number of the full and reduced residual Jacobians at the
best-known solutions (finite differences on the log10 parameterisation)."""
import sys, math, time, pickle, numpy as np, pandas as pd
from pathlib import Path
SRC = Path(__file__).resolve().parents[1]; sys.path.insert(0, str(SRC))
from scipy.optimize import least_squares
import eis_core as E, vp_core as V
from vp_common import load_spectra
T = SRC.parent/'results'/'tables'; CACHE = T.parent/'cache_e6'; CACHE.mkdir(exist_ok=True); RUNS = 10; BUD = 2000; CPS = (200, 500, 1000, 2000)

def vpinit_ms_trf(model, w, z, seed):
    vp = V.VarPro(model, w, z); _, lo, hi = E.MODELS[model]; f = E.make_objective(model, w, z); Tr = E.Tracker(f, BUD, CPS)
    rng = np.random.default_rng(seed)
    def res(x):
        if Tr.nfe >= BUD: raise E._Budget
        Tr(x); return E.residual(model, x, w, z)
    while Tr.nfe < BUD:
        eta0 = rng.uniform(vp.sp['lo'], vp.sp['hi']); Tr.nfe += 1; x0 = np.clip(vp.full_params(eta0), lo, hi)
        try: least_squares(res, x0, bounds=(lo, hi), jac='2-point', max_nfev=10000)
        except E._Budget: break
    return Tr.best

def jac_cond(fun, x, lo, hi, h=1e-6):
    x = np.asarray(x, float); r0 = fun(x); J = []
    for k in range(len(x)):
        d = h*max(1.0, abs(x[k])); xp = x.copy(); xp[k] = min(x[k]+d, hi[k]) if x[k]+d <= hi[k] else x[k]-d
        J.append((fun(xp) - r0)/(xp[k]-x[k]))
    s = np.linalg.svd(np.array(J).T, compute_uv=False); return float(s[0]/s[-1])

rows, crow = [], []
for model in ['M10_ColeL', 'M12_DoubleColeL']:
    M = pd.read_csv(T/f'vp_merged_{model}.csv'); ref = M.groupby('source_file').ref.first()
    for s in load_spectra():
        cp = CACHE/f'{model}_{s["si"]}.pkl'
        if cp.exists(): a, c = pickle.load(open(cp, 'rb')); rows += a; crow += c; continue
        t0 = time.time(); a = []
        seeds = [20261101 + s['si']*1000 + r for r in range(RUNS)]   # seed rule of the main benchmark (run index r)
        for r, sd in enumerate(seeds):
            fb = vpinit_ms_trf(model, s['w'], s['z'], int(sd)); nr = math.sqrt(fb)
            a.append(dict(model=model, source_file=s['sf'], run=r+1, seed=int(sd), algorithm='VPinit-MS-TRF', NRMSE=nr, success=(nr - ref[s['sf']])/ref[s['sf']] <= 1e-3))
        # conditioning at the best-known solution
        b = M[M.source_file == s['sf']].sort_values('NRMSE').iloc[0]; vp = V.VarPro(model, s['w'], s['z']); _, lo, hi = E.MODELS[model]
        names = {'M10_ColeL': ['logRinf', 'logdR', 'logtau', 'alpha', 'logL'],
                 'M12_DoubleColeL': ['logRinf', 'logdR1', 'logtau1', 'logdR2', 'logtau2', 'alpha1', 'alpha2', 'logL']}[model]
        x = np.array([b[n] for n in names]); x = np.clip(x, lo, hi)
        nlx = [x[2], x[3]] if model == 'M10_ColeL' else [x[2], x[4], x[5], x[6]]
        c = [dict(model=model, source_file=s['sf'], cond_full=jac_cond(lambda u: E.residual(model, u, s['w'], s['z']), x, lo, hi),
                  cond_reduced=jac_cond(vp.res, np.array(nlx), vp.sp['lo'], vp.sp['hi']))]
        pickle.dump((a, c), open(cp, 'wb')); rows += a; crow += c; print(time.strftime('%H:%M:%S'), model, s['si'], f'{time.time()-t0:.0f}s', flush=True)
D = pd.DataFrame(rows); D.to_csv(T/'e6_vpinit_runs.csv', index=False); Cn = pd.DataFrame(crow); Cn.to_csv(T/'e6_conditioning.csv', index=False)
out = []
for model in ['M10_ColeL', 'M12_DoubleColeL']:
    M = pd.read_csv(T/f'vp_merged_{model}.csv'); key = D[D.model == model][['source_file', 'run']]
    Mm = M.merge(key, on=['source_file', 'run'])
    r = dict(model=model, runs=len(key), **{a: 100*Mm[Mm.algorithm == a].success.mean() for a in ['MS-TRF', 'VP-MS-TRF']},
             **{'VPinit-MS-TRF': 100*D[D.model == model].success.mean()})
    c = Cn[Cn.model == model]; r.update(median_cond_full=c.cond_full.median(), median_cond_reduced=c.cond_reduced.median()); out.append(r)
O = pd.DataFrame(out); O.to_csv(T/'Table_E6_mechanism.csv', index=False); print(O.to_string())


In [ ]:
%%writefile src/review/e7_sweep_repeat.py
"""E7: sweep-level repeatability. M10 and M11 are refitted (global VarPro) to every complete sweep; the between-sweep SD
within a recording is compared with the reading-level resampling SD reported previously (Table_repeatability)."""
import sys, math, numpy as np, pandas as pd
import r2_core as R
SW = R.sweeps(sys.argv[1]); rows = []
for s in R.spectra():
    for i, z in enumerate(SW[s['sf']]):
        a = R.fit('cole_l', s['f'], z)['params']; b = R.fit('rcpe_l', s['f'], z)['params']
        rows.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], sweep=i, n_sweeps=len(SW[s['sf']]),
                         M10_log10_tau=a['log10_tau'], M10_log10_dR=math.log10(a['dR']), M10_log10_Rinf=math.log10(a['Rinf']), M10_alpha=a['alpha'],
                         M11_log10_Q=math.log10(b['Q']), M11_alpha=b['alpha'], M11_log10_Rinf=math.log10(b['Rinf'])))
D = pd.DataFrame(rows); D.to_csv(R.T/'e7_sweep_params.csv', index=False)
cols = [c for c in D.columns if c.startswith('M1')]
m = D[D.n_sweeps > 1]
sd = m.groupby(['source_file', 'kk_ok'])[cols].std().reset_index()
out = []
for c in cols:
    for kk in [True, False]:
        x = sd[sd.kk_ok == kk][c]; out.append(dict(param=c, kk_ok=kk, recordings=len(x), median_between_sweep_sd=x.median(), max_sd=x.max()))
# ICC (one-way, unbalanced) over KK-consistent multi-sweep recordings
for c in cols:
    g = m[m.kk_ok].groupby('source_file')[c]; k = g.size(); n = k.sum(); a = len(k); gm = m[m.kk_ok][c].mean()
    msb = (k*(g.mean()-gm)**2).sum()/(a-1); msw = ((m[m.kk_ok][c] - g.transform('mean'))**2).sum()/(n-a); k0 = (n - (k**2).sum()/n)/(a-1)
    out.append(dict(param=c, kk_ok='ICC', recordings=a, median_between_sweep_sd=(msb-msw)/(msb+(k0-1)*msw), max_sd=np.nan))
O = pd.DataFrame(out); O.to_csv(R.T/'Table_E7_sweep_repeatability.csv', index=False); print(O.round(4).to_string())


In [ ]:
%%writefile src/review/r2_figs.py
"""Figures for the second-round analyses: calibrated coverage (E1) and band design (E5)."""
import numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import r2_core as R
T = R.T; OUT = T.parent/'figures'
C1, C2, C3, C4, C5, C6, C7 = '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.size': 8.5, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold'})
def wilson(k, n, z=1.96):
    p = k/n; d = 1+z*z/n; c = (p+z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h, c+h

# ---- E1 coverage
Cv = pd.read_csv(T/'e1_coverage_runs_M11.csv')
meth = [('Nominal $\\chi^2$', lambda d: d.lr_nom <= 3.84, C1, 'o'), ('Calibrated', lambda d: d.lr_nom <= d.c_nom, C2, 's'),
        ('GLS $\\chi^2$', lambda d: d.lr_gls <= 3.84, C3, '^'), ('GLS calibrated', lambda d: d.lr_gls <= d.c_gls, C7, 'D')]
conds = [('emp', 'Empirical'), ('rho_lo', '$\\rho_f-0.2$'), ('rho_hi', '$\\rho_f+0.2$'), ('white', 'White'), ('ar1', 'AR(1) 0.6')]
pars = [('log10_Q', '$\\log_{10}Q$'), ('alpha', '$\\alpha$'), ('log10_Rinf', '$\\log_{10}R_\\infty$')]
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
rows = []
for a, (p, pl) in zip(ax, pars):
    a.axhline(95, color=INK2, lw=1, ls='--', zorder=0); a.grid(axis='y', color=GRID, lw=0.6); a.set_axisbelow(True)
    for mi, (mn, fn, col, mk) in enumerate(meth):
        xs, ys, lo, hi = [], [], [], []
        for ci, (c, cl) in enumerate(conds):
            d = Cv[(Cv.cond == c) & (Cv.param == p)]; k = int(fn(d).sum()); n = len(d); l, h = wilson(k, n)
            xs.append(ci + (mi-1.5)*0.17); ys.append(100*k/n); lo.append(100*(k/n-l)); hi.append(100*(h-k/n))
            rows.append(dict(param=p, cond=c, method=mn, n=n, coverage=100*k/n, ci_low=100*l, ci_high=100*h))
        a.errorbar(xs, ys, yerr=[lo, hi], fmt=mk, ms=4.5, color=col, mec='#fcfcfb', mew=0.6, elinewidth=1, capsize=0, label=mn)
    a.set_xticks(range(len(conds))); a.set_xticklabels([c[1] for c in conds], rotation=35, ha='right'); a.set_title(pl, color=INK, fontsize=9)
    a.set_ylim(60, 101)
ax[0].set_ylabel('Coverage of nominal 95% interval (%)')
fig.legend(*ax[0].get_legend_handles_labels(), loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout(rect=(0, 0, 1, 0.92)); fig.savefig(OUT/'Fig19_calibrated_coverage.png', dpi=300); plt.close(fig)
pd.DataFrame(rows).to_csv(T/'Table_E1_coverage.csv', index=False)

# ---- E5 band design
S = pd.read_csv(T/'Table_E5_band_factorial.csv'); D = pd.read_csv(T/'e5_band_runs.csv')
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
a = ax[0]; fmin = [10e3, 3e3, 1e3, 300.0]
for fc, col, lab in [('near', C3, '$f_c$ 10–30 kHz'), ('mid', C1, '$f_c$ 1–10 kHz'), ('low', C2, '$f_c$ 0.1–1 kHz')]:
    for N, ls, mk in [(20, '-', 'o'), (10, ':', 's')]:
        y = [S[(S.design == 'base') & (S.N == N) & (S.fc_class == fc) & (S.fmin_hz == f)].tau_ident_pct.iloc[0] for f in fmin]
        a.plot(fmin, y, ls=ls, marker=mk, ms=4, lw=1.6 if N == 20 else 1.2, color=col, mec='#fcfcfb', mew=0.6, label=f'{lab}, N={N}')
a.set_xscale('log'); a.invert_xaxis(); a.set_xticks(fmin); a.set_xticklabels(['10k', '3k', '1k', '300']); a.set_xlabel('Lower band limit $f_{\\min}$ (Hz)')
a.set_ylabel('$\\log_{10}\\tau$ practically identifiable (%)'); a.set_ylim(-3, 103); a.grid(color=GRID, lw=0.6); a.set_axisbelow(True)
a.legend(frameon=False, fontsize=6.8, loc='lower right'); a.set_title('(a) Range vs number of frequencies', fontsize=9, color=INK, loc='left')
a = ax[1]; bins = ['<0.3', '0.3-1', '1-3', '3-10', '>10']
D['ratio_bin'] = pd.cut(D.ratio_fmin_fc, [0, 0.3, 1, 3, 10, 1e4], labels=bins)
for (dz, N), col, mk, lab in [(('base', 20), C1, 'o', 'White noise'), (('corr', 20), C3, '^', 'Correlated noise'), (('linear', 20), C4, 'v', 'Linear grid'),
                              (('ep10', 20), C5, 's', 'Electrode pol. 10%'), (('ep50', 20), C2, 'D', 'Electrode pol. 50%'), (('mis', 20), C7, 'P', 'Model mismatch')]:
    q = D[(D.design == dz) & (D.N == N)].groupby('ratio_bin', observed=False).tau_ident.mean().mul(100).reindex(bins)
    a.plot(range(5), q.values, marker=mk, ms=4, lw=1.4, color=col, mec='#fcfcfb', mew=0.6, label=lab)
a.set_xticks(range(5)); a.set_xticklabels(bins); a.set_xlabel('$f_{\\min}/f_c$'); a.set_ylim(-3, 103); a.grid(color=GRID, lw=0.6); a.set_axisbelow(True)
a.legend(frameon=False, fontsize=6.8, loc='upper right'); a.set_title('(b) Robustness of the $f_{\\min}/f_c$ rule (N = 20)', fontsize=9, color=INK, loc='left')
fig.tight_layout(); fig.savefig(OUT/'Fig16_band_design_v2.png', dpi=300); plt.close(fig)
print('ok')


In [ ]:
%%writefile src/review/make_supp_round2.py
"""Writes supp_round2.tex (Figs S13-S14, Tables S14-S22) from the result tables."""
import sys, json, numpy as np, pandas as pd
import r2_core as R
T = R.T; out = []
def tab(cap, cols, rows, align=None, size='\\small'):
    a = align or 'l' + 'c'*(len(cols)-1)
    s = ['\\begin{table}[H]\\centering', f'\\caption{{{cap}}}', size, f'\\begin{{tabular}}{{{a}}}\\toprule', ' & '.join(cols) + ' \\\\\\midrule']
    s += [' & '.join(str(x) for x in r) + ' \\\\' for r in rows]; s.append('\\bottomrule\\end{tabular}\\end{table}\n'); out.append('\n'.join(s))
f2 = lambda x, d=2: f'{x:.{d}f}'
out.append('\\begin{figure}[H]\\centering\\includegraphics[width=\\textwidth]{figures/Fig17_feature_stability.png}\n\\caption{Stability of fitted parameters as features for ranking the 12 KK-consistent spectra across 40 reading-level resamples. (a) Spearman correlation between rankings from different resamples (bar: median; tick: 5th percentile). (b) Share of spectrum pairs ordered identically in at least 95\\% of resamples. Red: non-identifiable Cole parameters; blue: identifiable parameters. Reading-level resampling understates within-recording variability (Table~S20).}\\end{figure}\n')
out.append('\\begin{figure}[H]\\centering\\includegraphics[width=\\textwidth]{figures/Fig08_profile_likelihood.png}\n\\caption{Nominal profile likelihood-ratio curves for all 20 spectra (one line per spectrum; dashed line: $\\chi^2_1$ threshold 3.84). Calibrated thresholds were 1.8--2 times larger (Table~S15). Flat profiles for $\\log_{10}\\tau$ and $\\log_{10}\\Delta R$ in M10 indicate practical non-identifiability.}\\end{figure}\n')
# S14 error model
E = pd.read_csv(T/'Table_E1_error_model.csv').set_index('parameter')
B = pd.read_csv(T/'Table_E1_between_sweep.csv'); naive = json.load(open(T/'E1_psi.json'))['psi']
nm = {'sigma0': '$\\sigma_0$ (relative scale, real part)', 'gamma': '$\\gamma$ (frequency trend)', 'rho_f': '$\\rho_f$ (adjacent-frequency correlation)',
      'rho_ReIm': '$\\rho_{\\mathrm{RI}}$ (real--imaginary correlation)', 'ratio_Im_Re': '$\\kappa$ (imaginary/real scale)'}
rows = []
for i, k in enumerate(nm):
    rows.append([nm[k], f'{naive[i]:.3f}', f'{E.loc[k,"naive_ci_low"]:.3f} to {E.loc[k,"naive_ci_high"]:.3f}', f'{E.loc[k,"estimate"]:.3f}', f'{E.loc[k,"ci_low"]:.3f} to {E.loc[k,"ci_high"]:.3f}'])
sh = B.residual_share_of_deviation
rows += [['Sweeps / recordings / residual df', f"{int(E.loc['sweeps_used','estimate'])} / {int(E.loc['recordings','estimate'])} / {int(E.loc['residual_df','estimate'])}", '', '', ''],
         ['Sample lag-1 correlation, real / imaginary', f"{E.loc['sample_lag1_corr_Re','estimate']:.2f} / {E.loc['sample_lag1_corr_Im','estimate']:.2f}", '', '', ''],
         ['Ledoit--Wolf shrinkage intensity', f"{E.loc['LW_shrinkage','estimate']:.2f}", '', '', ''],
         ['Share of sweep-to-sweep deviation left as residual$^{a}$', f'{sh.median():.2f}', f'IQR {sh.quantile(.25):.2f}--{sh.quantile(.75):.2f}', '', '']]
tab('Empirical error model (Eq.~(5) of the main text) estimated from the R--CPE--L residuals of the 51 complete sweeps; 95\\% intervals from resampling recordings. Residual-based: residual vectors treated as errors (used for calibration). Projection-corrected: likelihood evaluated in the 16-dimensional residual space of each sweep, which accounts for the projection of errors by the fit; it is weakly identified, because a smooth error component that lies in the model tangent space is absorbed by the parameters and leaves no trace in the residuals (Table~S24). $^{a}$Mean RSS of the sweep fits divided by the mean squared deviation of sweeps from the recording mean (12 recordings with $\\geq2$ sweeps).',
    ['Quantity', '\\makecell{Residual-based\\\\estimate}', '95\\% interval', '\\makecell{Projection-\\\\corrected}', '95\\% interval'], rows, 'lcccc', '\\footnotesize')
# S15 coverage
C = pd.read_csv(T/'Table_E1_coverage.csv'); cn = {'emp': 'Empirical $\\Sigma$', 'rho_lo': '$\\rho_f-0.2$', 'rho_hi': '$\\rho_f+0.2$', 'white': 'White', 'ar1': 'AR(1), $\\rho=0.6$'}
Cv = pd.read_csv(T/'e1_coverage_runs_M11.csv'); mis = Cv[Cv.cond == 'mis']
rows = []
for c, cl in list(cn.items()) + [('mis', 'Misspecified (Cole truth)')]:
    for p, pl in [('log10_Q', '$\\log_{10}Q$'), ('alpha', '$\\alpha$'), ('log10_Rinf', '$\\log_{10}R_\\infty$')]:
        if c == 'mis':
            d = mis[mis.param == p]; v = [100*(d.lr_nom <= 3.84).mean(), 100*(d.lr_nom <= d.c_nom).mean(), 100*(d.lr_gls <= 3.84).mean(), 100*(d.lr_gls <= d.c_gls).mean()]
        else:
            d = C[(C.cond == c) & (C.param == p)].set_index('method').coverage; v = [d['Nominal $\\chi^2$'], d['Calibrated'], d['GLS $\\chi^2$'], d['GLS calibrated']]
        rows.append([cl if p == 'log10_Q' else '', pl] + [f'{x:.1f}' for x in v])
DB = pd.read_csv(T/'e1_coverage_runs_DB.csv'); M10 = pd.read_csv(T/'e1_coverage_runs_M10.csv')
for p, pl in [('log10_Q', '$\\log_{10}Q$'), ('alpha', '$\\alpha$')]:
    d = DB[DB.param == p]; rows.append(['Double bootstrap$^{a}$' if p == 'log10_Q' else '', pl, f'{100*(d.lr_nom<=3.84).mean():.1f}', f'{100*(d.lr_nom<=d.c_nom).mean():.1f}', f'{100*(d.lr_gls<=3.84).mean():.1f}', f'{100*(d.lr_gls<=d.c_gls).mean():.1f}'])
rows.append(['M10, Cole truth$^{b}$', '$\\log_{10}\\tau$', f'{100*(M10.lr_nom<=3.84).mean():.1f}', f'{100*(M10.lr_nom<=M10.c_nom).mean():.1f}', f'{100*(M10.lr_gls<=3.84).mean():.1f}', f'{100*(M10.lr_gls<=M10.c_gls).mean():.1f}'])
cs = pd.concat([pd.read_csv(T/'e1_cstar_M11.csv'), pd.read_csv(T/'e1_cstar_M10.csv')]).groupby(['objective', 'param']).c_star.median()
tab('Coverage (\\%) of 95\\% profile-likelihood intervals for the R--CPE--L descriptors (12 KK-consistent spectra $\\times$ 100 replicates per condition; cluster-bootstrap 95\\% intervals over spectra for the empirical-$\\Sigma$ rows: nominal 78.8--86.0, calibrated 90.9--95.5, GLS 89.3--93.7, GLS calibrated 90.5--96.4). Nominal: $\\chi^2_1$ threshold 3.84; calibrated: parametric-bootstrap critical value (200 replicates) under the empirical $\\Sigma$; GLS: generalized least squares with the empirical $\\Sigma$. Misspecified: Cole data with $f_c$ 200~Hz--3~kHz sharing the M11 high-frequency asymptote. $^{a}$Critical values recomputed at each replicate\'s own estimate (25 replicates $\\times$ 80 inner replicates per spectrum). $^{b}$Cole truth with $f_c$ below the band, empirical $\\Sigma$, 50 replicates per spectrum. ' +
    f"Median critical values: independent likelihood {cs[('nom','log10_Q')]:.1f} ($\\log_{{10}}Q$), {cs[('nom','alpha')]:.1f} ($\\alpha$), {cs[('nom','log10_Rinf')]:.1f} ($\\log_{{10}}R_\\infty$), {cs[('nom','log10_tau')]:.1f} ($\\log_{{10}}\\tau$), {cs[('nom','log10_dR')]:.1f} ($\\log_{{10}}\\Delta R$); GLS {cs[('gls','log10_Q')]:.1f}, {cs[('gls','alpha')]:.1f}, {cs[('gls','log10_Rinf')]:.1f}, {cs[('gls','log10_tau')]:.1f}, {cs[('gls','log10_dR')]:.1f}.",
    ['Condition', 'Parameter', 'Nominal', 'Calibrated', 'GLS', 'GLS calibrated'], rows, 'llcccc', '\\footnotesize')
# S16 measured intervals: procedures x thresholds + bounds
I = pd.concat([pd.read_csv(T/'e1_intervals_M11.csv'), pd.read_csv(T/'e1_intervals_M10.csv')]); rows = []
pl = {'log10_tau': 'M10 $\\log_{10}\\tau$', 'log10_dR': 'M10 $\\log_{10}\\Delta R$', 'log10_Rinf': 'M11 $\\log_{10}R_\\infty$', 'log10_Q': 'M11 $\\log_{10}Q$', 'alpha': 'M11 $\\alpha$'}
for p in pl:
    for o, t, lab in [('nom', 'chi2', 'Nominal'), ('nom', 'cal', 'Calibrated'), ('gls', 'chi2', 'GLS'), ('gls', 'cal', 'GLS calibrated')]:
        g = I[(I.param == p) & (I.objective == o) & (I.threshold == t)]; r = [pl[p] if lab == 'Nominal' else '', lab, int(g.closed.sum())]
        for lg, la in [(0.5, 0.1), (1.0, 0.2), (1.5, 0.3)]:
            lim = la if p == 'alpha' else lg; idd = g.closed & (g.width < lim); r.append(f'{int(idd.sum())} / {int(idd[g.kk_ok].sum())}')
        r.append(f'{g.width.median():.3f}'); rows.append(r)
Bd = pd.read_csv(T/'Table_E4_bounds.csv')
for p in ['log10_Q', 'alpha']:
    for t, lab in [('chi2', 'Nominal'), ('cal', 'Calibrated')]:
        b = Bd[(Bd.param == p) & (Bd.interval == t)].iloc[0]
        rows.append([pl[p] + ', wider bounds$^{a}$' if t == 'chi2' else '', lab, '', '', f'{int(b.ident_all)} / {int(b.ident_kk_ok)}', '', f'{b.median_width:.3f}'])
tab('Measured 95\\% profile intervals (20 spectra) by procedure: number closed within the bounds, number practically identifiable (all / 12 KK-consistent) at width thresholds of 0.5, 1.0 and 1.5 decades for $\\log_{10}$ parameters (0.1, 0.2 and 0.3 for $\\alpha$), and median width. $^{a}$$\\alpha\\in[0.2,1]$ and $1/Q\\in[10^{-2},10^{14}]$ (default $[0.3,1]$ and $[1,10^{12}]$), 1.0-decade threshold.',
    ['Parameter', 'Procedure', 'Closed', '0.5 (0.1)', '1.0 (0.2)', '1.5 (0.3)', 'Median width'], rows, 'llccccc', '\\footnotesize')
# S17 bias
Bi = pd.read_csv(T/'Table_E1b_bias.csv')
tab('Bias of the R--CPE--L descriptors when the data are a noise-free Cole dispersion sharing the same high-frequency asymptote (12 KK-consistent spectra; median over spectra). Bias = fitted value minus asymptotic value.',
    ['$f_c$ (Hz)', '$f_{\\min}/f_c$', '$\\log_{10}Q$', '$\\alpha$', '$\\log_{10}R_\\infty$'],
    [[int(r.fc_hz), f'{r.fmin_over_fc:.0f}', f'{r.bias_log10_Q:.3f}', f'{r.bias_alpha:.3f}', f'{r.bias_log10_Rinf:.3f}'] for r in Bi.itertuples()], 'ccccc')
# S18 HF treatments
H = pd.read_csv(T/'Table_E2_hf_treatments.csv').set_index('treatment')
lab = {'T0': 'Series $L$ (published), 10--100~kHz', 'T1': 'No correction, 10--80~kHz', 'T1b': 'Series $L$, 10--80~kHz', 'T2': 'Pure delay $e^{-j\\omega t_d}$',
       'T3': 'Constant phase offset $e^{j\\varphi_0}$', 'T4': 'Generalized term $K(j\\omega)^{\\nu}$'}
ex = {'T0': '$L$ (mH)', 'T1': '', 'T1b': '$L$ (mH)', 'T2': '$t_d$ ($\\mu$s)', 'T3': '$\\varphi_0$ (rad)', 'T4': '$\\nu$'}
D2 = pd.read_csv(T/'e2_hf_runs.csv'); rows = []
for k in lab:
    d = D2[D2.treatment == k]; q = lambda x, dd=3: f'{np.median(x):.{dd}f} [{np.quantile(x,.25):.{dd}f}, {np.quantile(x,.75):.{dd}f}]'
    e = d.extra*1e3 if k in ('T0', 'T1b') else d.extra
    rows.append([lab[k], q(d.NRMSE), q(d.d_log10_Q.abs()), q(d.d_alpha.abs()), q(d.d_log10_Rinf.abs()), (ex[k] + ' ' + q(e, 2)) if ex[k] else '--', int(H.loc[k, 'AICc_best_count']) if k not in ('T1', 'T1b') else '--'])
tab('Sensitivity of the R--CPE--L descriptors to the treatment of the high-frequency correction (20 spectra; median [interquartile range]). $\\Delta$: absolute difference from the published fit. AICc best: number of spectra in which the treatment had the lowest AICc among full-band treatments.',
    ['Treatment', 'NRMSE', '$|\\Delta\\log_{10}Q|$', '$|\\Delta\\alpha|$', '$|\\Delta\\log_{10}R_\\infty|$', 'Correction parameter', 'AICc best'], rows, 'p{2.6cm}cccccc', '\\scriptsize\\setlength{\\tabcolsep}{3pt}')
# S19 model selection
M = pd.read_csv(T/'Table_E3_model_selection.csv', index_col=0).iloc[:, 0]
rows = [['AICc, independent likelihood: 10--100 / 10--80 / 20--100~kHz', f"{int(M['M11_pref_AICc_nom_10-100'])} / {int(M['M11_pref_AICc_nom_10-80'])} / {int(M['M11_pref_AICc_nom_20-100'])}"],
        ['AICc, GLS likelihood: 10--100 / 10--80 / 20--100~kHz', f"{int(M['M11_pref_AICc_gls_10-100'])} / {int(M['M11_pref_AICc_gls_10-80'])} / {int(M['M11_pref_AICc_gls_20-100'])}"],
        ['Bootstrap LR test rejects M11 ($p<0.05$): independent / GLS', f"{int(M['bootLR_reject_M11_nom'])} / {int(M['bootLR_reject_M11_gls'])} (median $p$ {M['median_p_boot_nom']:.2f} / {M['median_p_boot_gls']:.2f})"],
        ['LOFO, interior 20--90~kHz: M11 better; medians M11 vs M10', f"{int(M['lofo_int_M11_better'])}; {M['lofo_int_median_M11']:.4f} vs {M['lofo_int_median_M10']:.4f} ($p={M['lofo_int_wilcoxon_p']:.2f}$)"],
        ['LOFO, band edges 10 and 100~kHz: M11 better; medians M11 vs M10', f"{int(M['lofo_end_M11_better'])}; {M['lofo_end_median_M11']:.4f} vs {M['lofo_end_median_M10']:.4f} ($p={M['lofo_end_wilcoxon_p']:.2f}$)"]]
tab('Uncertainty-aware comparison of M11 (R--CPE--L) and M10 (Cole+L) (number of spectra out of 20 preferring M11 unless stated). Bootstrap LR: 100 parametric-bootstrap replicates from the fitted M11 with the empirical $\\Sigma$. LOFO: leave-one-frequency-out relative prediction error; $p$: Wilcoxon signed-rank test.',
    ['Criterion', 'Result'], rows, 'p{8.2cm}l')
# S20 sweep repeatability
S7 = pd.read_csv(T/'Table_E7_sweep_repeatability.csv'); rr = pd.read_csv(T/'Table_repeatability_KKconsistent.csv')
rmap = {'M10_log10_tau': ('M10_ColeL', 'logtau'), 'M10_log10_dR': ('M10_ColeL', 'logdR'), 'M10_log10_Rinf': ('M10_ColeL', 'logRinf'), 'M10_alpha': ('M10_ColeL', 'alpha'),
        'M11_log10_Q': ('M11_RCPEL', 'logQ'), 'M11_alpha': ('M11_RCPEL', 'alpha'), 'M11_log10_Rinf': ('M11_RCPEL', 'logRinf')}
plab = {'M10_log10_tau': 'M10 $\\log_{10}\\tau$', 'M10_log10_dR': 'M10 $\\log_{10}\\Delta R$', 'M10_log10_Rinf': 'M10 $\\log_{10}R_\\infty$', 'M10_alpha': 'M10 $\\alpha$',
        'M11_log10_Q': 'M11 $\\log_{10}Q$', 'M11_alpha': 'M11 $\\alpha$', 'M11_log10_Rinf': 'M11 $\\log_{10}R_\\infty$'}
rows = []
for p in plab:
    a = S7[(S7.param == p) & (S7.kk_ok == 'True')].median_between_sweep_sd.iloc[0]; b = S7[(S7.param == p) & (S7.kk_ok == 'False')].median_between_sweep_sd.iloc[0]
    ic = pd.read_csv(T/'Table_E9_icc_ci.csv').set_index('param').loc[p]; m, n = rmap[p]; q = rr[(rr.model == m) & (rr.parameter == n)].iloc[0]
    rows.append([plab[p], f'{a:.3f}', f'{b:.3f}', f'{ic.ICC:.2f} [{ic.ci_low:.2f}, {ic.ci_high:.2f}]', f'{q.median_within_SD:.3f}', f'{q.ICC:.2f}'])
tab('Sweep-level repeatability (complete sweeps refitted by global VarPro; six KK-consistent and six KK-inconsistent recordings with $\\geq2$ sweeps) compared with the earlier reading-level resampling (12 KK-consistent spectra, 40 resamples). SD: median within-recording standard deviation; ICC: one-way intraclass correlation over the KK-consistent recordings with 95\\% interval from 2,000 resamples of recordings; with six recordings the sweep ICCs are imprecise.',
    ['Parameter', '\\makecell{Sweep SD\\\\KK-consistent}', '\\makecell{Sweep SD\\\\KK-inconsistent}', '\\makecell{Sweep ICC\\\\{[}95\\% CI{]}}', '\\makecell{Reading-level\\\\SD}', '\\makecell{Reading-level\\\\ICC}'], rows, 'lccccc', '\\footnotesize')
# S21 band factorial
F = pd.read_csv(T/'Table_E5_band_factorial.csv'); dl = {'base': 'Log grid, white', 'linear': 'Linear grid, white', 'corr': 'Log grid, correlated', 'ep10': 'Electrode pol. 10\\%', 'ep50': 'Electrode pol. 50\\%', 'mis': 'Model mismatch'}
rows = []
for (d, N), g in F.groupby(['design', 'N'], sort=False):
    for fc, fl in [('near', '10--30'), ('mid', '1--10'), ('low', '0.1--1')]:
        h = g[g.fc_class == fc].set_index('fmin_hz').tau_ident_pct
        rows.append([f'{dl[d]}, $N={N}$' if fc == 'near' else '', fl] + [f'{h[x]:.0f}' for x in [10000.0, 3000.0, 1000.0, 300.0]])
tab('Frequency-band design: percentage of synthetic spectra (20 per cell; 2.5\\% noise) in which $\\log_{10}\\tau$ was practically identifiable (nominal 95\\% profile interval closed and narrower than one decade), by design, $f_c$ class and lower band limit ($f_{\\max}=100$~kHz; $N$ frequencies at every $f_{\\min}$).',
    ['Design', '$f_c$ (kHz)', '10~kHz', '3~kHz', '1~kHz', '300~Hz'], rows, 'llcccc', '\\footnotesize')
Rn = pd.read_csv(T/'Table_E5_ratio.csv'); Rc = pd.read_csv(T/'Table_E5_ratio_cal.csv'); bins = ['<0.3', '0.3-1', '1-3', '3-10', '>10']; rows = []
for (rn, rc) in zip(Rn.itertuples(index=False), Rc.itertuples(index=False)):
    dn = rn._asdict(); dc = rc._asdict(); keys = list(dn.keys())
    rows.append([f"{dl[dn['design']]}, $N={dn['N']}$"] + [f"{dn[k]:.0f} / {dc[k]:.0f}" for k in keys[2:]])
tab('Practical identifiability of $\\log_{10}\\tau$ (\\%) by $f_{\\min}/f_c$, pooled over $f_c$ classes, with the nominal threshold (3.84) / the median calibrated critical value of $\\log_{10}\\tau$ (6.38; Table~S15).',
    ['Design'] + ['$<0.3$', '0.3--1', '1--3', '3--10', '$>10$'], rows, 'lccccc', '\\footnotesize')
# S22 mechanism
Mm = pd.read_csv(T/'Table_E6_mechanism.csv')
rows = [[('M10' if 'M10' in r.model else 'M12'), int(r.runs), f'{r._3:.0f}', f'{r._4:.0f}', f'{r._5:.0f}', f'{r.median_cond_full:.0f}', f'{r.median_cond_reduced:.0f}'] for r in Mm.itertuples()]
tab('Mechanism of the VarPro gain: success (\\%) at 2,000 NFE in the same runs (20 spectra $\\times$ 10 seeds) for full-parameter MS-TRF, VP-MS-TRF and VPinit-MS-TRF (full-parameter TRF started from VarPro-optimal linear coefficients), and median condition number of the full and reduced residual Jacobians at the best-known solutions.',
    ['Model', 'Runs', 'MS-TRF', 'VP-MS-TRF', 'VPinit-MS-TRF', '$\\kappa$ full', '$\\kappa$ reduced'], rows, 'lcccccc')
SA = pd.read_csv(T/'Table_E8_scenario.csv').set_index('scenario'); SB = pd.read_csv(T/'Table_E8_cross_sweep.csv')
obs = pd.read_csv(T/'e8_scenario_runs.csv').dropna(subset=['obs_sweep_sd_log10_Q']).groupby('scenario')[['obs_sweep_sd_log10_Q', 'obs_sweep_sd_alpha', 'obs_sweep_sd_log10_Rinf']].median().iloc[0]
rows = []
for sc, lab in [('residual-based', 'Residual-based (used)'), ('projection-corrected', 'Projection-corrected')]:
    r = SA.loc[sc]; rows.append([lab] + [f"{r['sd_'+p]:.3f} ({int(r['implied_gt_observed_'+p])}/6)" for p in ['log10_Q', 'alpha', 'log10_Rinf']] + [f"{r['cov_'+p]:.0f}" for p in ['log10_Q', 'alpha', 'log10_Rinf']])
rows.append(['Observed between-sweep SD$^{a}$'] + [f"{obs['obs_sweep_sd_'+p]:.3f}" for p in ['log10_Q', 'alpha', 'log10_Rinf']] + ['', '', ''])
for r in SB.itertuples():
    rows.append([f"Cross-sweep agreement$^{{b}}$, {'KK-consistent' if r.kk_ok else 'KK-inconsistent'} ({r.pairs} pairs)"] + [f"{getattr(r, 'nom_'+p+'_pct'):.0f} / {getattr(r, 'cal_'+p+'_pct'):.0f}" for p in ['log10_Q', 'alpha', 'log10_Rinf']] + ['', '', ''])
tab('Checks of the error model that do not rely on residuals alone (KK-consistent spectra unless stated). Upper rows: errors simulated under each covariance, scaled so that the expected residual sum of squares equals the observed one (100 replicates per spectrum): median implied SD of the fitted descriptor (number of the six recordings with repeated sweeps in which it exceeds the observed between-sweep SD) and coverage (\\%) of the calibrated intervals. $^{a}$Median over the six recordings; includes drift and is therefore an upper bound for noise-induced variability. $^{b}$Percentage of pairs of sweeps of the same recording whose estimates differ by less than $\\sqrt{h_i^2+h_j^2}$, $h$ the interval half-width, with nominal / calibrated intervals; about 95\\% is expected for calibrated intervals without drift.',
    ['', '$\\log_{10}Q$', '$\\alpha$', '$\\log_{10}R_\\infty$', '\\makecell{Cov.\\\\$\\log_{10}Q$}', '\\makecell{Cov.\\\\$\\alpha$}', '\\makecell{Cov.\\\\$\\log_{10}R_\\infty$}'], rows, 'p{4.6cm}cccccc', '\\scriptsize')

# S15 figure: f_c coverage (moved from main text)
out.append('\\begin{figure}[H]\\centering\\includegraphics[width=0.85\\textwidth]{figures/Fig09_fc_band_coverage.png}\n\\caption{M10 estimates of the characteristic frequency $f_c$ with nominal 95\\% profile-likelihood intervals relative to the measured band (shaded). Intervals extending to the left edge reach the search bound and are unbounded.}\\end{figure}\n')
# S25 asymptotic criterion: synthetic validation and bias
SA = pd.read_csv(T/'Table_E10_synthetic.csv'); dn = {'base': 'Log grid, white', 'linear': 'Linear grid, white', 'corr': 'Correlated noise', 'ep10': 'Electrode pol. 10\\%', 'ep50': 'Electrode pol. 50\\%', 'mis': 'Model mismatch'}
rows = [[dn[r.design], int(r.n), f'{r.agree_asym_pct:.1f}', f'{r.agree_wald_pct:.1f}', f'{r.false_ident_pct:.1f}', f'{r.missed_ident_pct:.1f}', f'{r.spearman_asym_prof:.2f}', f'{r.median_ratio_asym_prof:.2f}'] for r in SA.itertuples()]
A10 = pd.read_csv(T/'e10_synthetic.csv'); g = A10[A10.design.isin(['base', 'linear']) & A10.closed].copy()
g['bin'] = pd.cut(g.ratio, [0, 1, 3, 10, 1e4], labels=['$<1$', '1--3', '3--10', '$>10$'])
bb = g.groupby('bin', observed=True).apply(lambda h: (len(h), float(np.median(h.W_asym/h.W_prof)), float(np.median(h.W_wald/h.W_prof))), include_groups=False)
note = '; '.join(f'{k}: {v[1]:.2f} / {v[2]:.2f} (n = {v[0]})' for k, v in bb.items())
gm = A10[A10.design == 'base'].groupby('fmin_hz').Gamma.median()
tab('Validation of Eqs.~(8)--(9) on the band-design spectra (practical identifiability of $\\log_{10}\\tau$: closed profile interval narrower than one decade). Agreement: percentage of spectrum--band combinations classified identically by the closed-form width (Eq.~(8)) or the exact linearized (Wald) width and by profile likelihood; false/missed: predicted identifiable but not observed / the reverse. Spearman and ratio: closed profiles only. Median ratio of predicted (Eq.~(8) / Wald) to profile width by $f_{\\min}/f_c$, white-noise designs: ' + note + '. Median $\\Gamma$ (white log grid) for $f_{\\min}$ of 10, 3, 1 and 0.3~kHz: ' + ', '.join(f'{gm[x]:.1f}' for x in [10000.0, 3000.0, 1000.0, 300.0]) + '.',
    ['Design', 'n', '\\makecell{Agreement\\\\Eq.~(8) (\\%)}', '\\makecell{Agreement\\\\Wald (\\%)}', '\\makecell{False\\\\(\\%)}', '\\makecell{Missed\\\\(\\%)}', 'Spearman', '\\makecell{Median\\\\ratio}'], rows, 'lccccccc', '\\footnotesize')
Bz = pd.read_csv(T/'Table_E10_bias.csv')
rows = [[f'{r.fmin_over_fc:.0f}', f'{r.bias_logQ_num:.3f}', f'{r.bias_logQ_formula:.3f}', f'{r.bias_alpha_num:.3f}', f'{r.bias_alpha_formula:.3f}'] for r in Bz.itertuples()]
Bc = pd.read_csv(T/'e10_bias.csv')
tab(f'Bias of the R--CPE--L descriptors fitted to Cole spectra: numerical refits (Table~S17) and Eq.~(10) (median over the 12 KK-consistent spectra). Median coefficients: $b_Q={Bc.bQ.median():.2f}$ decades and $b_\\alpha={Bc.balpha.median():.2f}$ per unit $(f_{{\\min}}/f_c)^{{-\\alpha}}$.',
    ['$f_{\\min}/f_c$', '$\\log_{10}Q$ refit', '$\\log_{10}Q$ Eq.~(10)', '$\\alpha$ refit', '$\\alpha$ Eq.~(10)'], rows, 'ccccc')
# S26 measured criterion
Mm = pd.read_csv(T/'Table_E10_measured.csv')
fm = lambda x: f'{x:.0f}' if x < 1e5 else '$>10^5$'
rows = [[r.tissue + ('' if r.kk_ok else '$^{\\dagger}$'), f'{r.sigma:.3f}', f'{r.alpha:.2f}', f'{r.Gamma:.2f}', fm(r.fc_detect_hz), fm(r.fc_detect_cal_hz), fm(r.fc_hat_hz), fm(r.fc_ci_high_hz), f'{min(r.W_asym_at_fchat, 999):.2f}', f'{r.W_prof:.2f}', ('yes' if r.ident_pred else 'no') + ' / ' + ('yes' if r.ident_obs else 'no')] for r in Mm.itertuples()]
tab('Closed-form band criterion applied to the 20 measured spectra. $\\sigma$: residual SD per stacked relative component (Cole+L); $\\Gamma$: Eq.~(9) with $W^*=1$ decade; detection limit $f_{\\min}/\\Gamma$ (Hz) with $z=1.96$ and with the calibrated critical value; $\\hat f_c$ and upper 95\\% bound of $f_c$ (Hz) from the nominal profile; widths (decades) of the $\\log_{10}\\tau$ interval predicted by Eq.~(8) at $\\hat f_c$ and from the profile; identifiable predicted / observed. $^{\\dagger}$KK-inconsistent.',
    ['Tissue', '$\\sigma$', '$\\alpha$', '$\\Gamma$', '$f_c^*$', '$f_c^*$ cal.', '$\\hat f_c$', '$f_c$ upper', '$W$ Eq.~(8)', '$W$ profile', 'Ident.'], rows, 'lcccccccccc', '\\scriptsize\\setlength{\\tabcolsep}{3pt}')
# S27 drift decomposition
Dd = pd.read_csv(T/'Table_E11_drift.csv'); Pb = pd.read_csv(T/'Table_E11b_drift_ci.csv').set_index(['kk_ok', 'param']); pn = {'log10_Q': '$\\log_{10}Q$', 'alpha': '$\\alpha$', 'log10_Rinf': '$\\log_{10}R_\\infty$'}
rows = []
for r in Dd.itertuples():
    b = Pb.loc[(r.kk_ok, r.param)]; sc = (0.05, 0.2) if r.param != 'alpha' else (0.01, 0.05)
    prof = f'[{b.ci2_low:.3f}, {b.ci2_high:.3f}]' if b.tau_hat > 0 else f'$\\leq${b.upper95_onesided:.3f}'
    bay = f'{b[f"post_upper95_s{sc[0]}"]:.3f} / {b[f"post_upper95_s{sc[1]}"]:.3f}'
    rows.append([('KK-consistent' if r.kk_ok else 'KK-inconsistent') if r.param == 'log10_Q' else '', pn[r.param], f'{r.sweeps}', f'{r.median_fit_se:.4f}', f'{r.tau_drift:.4f}', prof, bay, f'{r.fit_halfwidth:.3f}', f'{r.reproducibility_halfwidth:.3f}'])
tab('Separation of fitting uncertainty and between-sweep drift (six recordings with $\\geq2$ sweeps in each group; REML). $s$: median fitting standard error (calibrated half-width/1.96); $\\tau_d$: drift SD with its 95\\% profile-likelihood interval (one-sided upper bound when $\\hat\\tau_d=0$, reference 2.71) and the 95\\% posterior upper bound under half-normal priors (scale 0.05 / 0.2 decade for $\\log_{10}$ parameters, 0.01 / 0.05 for $\\alpha$); half-widths: fitting $1.96s$ and reproducibility $1.96\\sqrt{s^2+\\tau_d^2}$.',
    ['Group', 'Parameter', 'Sweeps', '$s$', '$\\hat\\tau_d$', '\\makecell{Profile\\\\95\\% interval}', '\\makecell{Posterior\\\\upper 95\\%}', '\\makecell{Fitting\\\\half-width}', '\\makecell{Reprod.\\\\half-width}'], rows, 'llccccccc', '\\scriptsize\\setlength{\\tabcolsep}{3pt}')

open(sys.argv[1], 'w').write('\n'.join(out)); print('written', len(out))


In [ ]:
%%writefile src/review/e10_asymptotic.py
"""E10: closed-form band-coverage criterion and bias formula for the Cole model above its characteristic frequency.

Above f_c, with u = (j w tau)^-alpha and the CPE term w(w) = dR u = (j w)^-alpha / Q,
    Cole term = w/(1+u) = w - w u + O(w u^2).
Hence the data carry information on tau only through the second-order term w u, whose size relative to |Z| at f_min
is rho_min (f_min/f_c)^-alpha. Writing eta = 1/dR (eta -> 0 is the R-CPE-L limit), the Fisher information for eta with
R_inf, Q, alpha, L as nuisance parameters is ||P h||^2 / sigma^2 (h = w^2/|Z|, P the projector onto the complement of
the R-CPE-L tangent space, sigma the SD of each stacked real component of the relative residual). Since
alpha ln tau = ln Q - ln eta,
    W95(log10 tau) ~= 2 z sigma (f_min/f_c)^alpha / (alpha ln10 D),   D = ||P h~||,  h~_k = w_k (j w_k / w_min)^-alpha / |Z_k|,
where D depends on the band, the sampling and the impedance composition but not on f_c. Practical identifiability
(W95 < W*) therefore requires
    f_min / f_c < Gamma = (alpha ln10 D W* / (2 z sigma))^(1/alpha).
Fitting R-CPE-L to Cole data leaves the misfit -w u; its linearized least-squares projection gives the bias
    delta theta = -(J^T J)^-1 J^T h~ (f_min/f_c)^-alpha,   J the R-CPE-L Jacobian (relative, stacked).
Validation: (a) synthetic spectra of the band-design study (profile widths, Table S21) and the exact linearized
(Wald) width from the full Cole Jacobian; (b) bias against the numerical fits of Table S17; (c) the 20 measured spectra."""
import math, json, numpy as np, pandas as pd
from scipy.stats import spearmanr
import r2_core as R
T = R.T; LN10 = math.log(10)

def stack(c): return np.r_[c.real, c.imag]
def m11_tangent(w, Rinf, Q, alpha, L, absZ):
    jw = 1j*w; cpe = jw**(-alpha)/Q
    cols = [Rinf/absZ, -cpe/absZ, -np.log(jw)*cpe/absZ, jw*L/absZ]
    return np.column_stack([stack(c) for c in cols]), cpe
def D_factor(w, Rinf, Q, alpha, L, absZ):
    J, cpe = m11_tangent(w, Rinf, Q, alpha, L, absZ); wmin = w.min()
    ht = stack(cpe*(1j*w/wmin)**(-alpha)/absZ)
    P = np.eye(len(ht)) - J @ np.linalg.pinv(J); return float(np.linalg.norm(P @ ht)), J, ht
def W_asym(D, alpha, sigma, ratio, z=1.96): return 2*z*sigma*ratio**alpha/(alpha*LN10*D)
def Gamma(D, alpha, sigma, Wstar=1.0, z=1.96): return (alpha*LN10*D*Wstar/(2*z*sigma))**(1/alpha)
def cole(x, w): return x['Rinf'] + x['dR']/(1 + (1j*w*x['tau'])**x['alpha']) + 1j*w*x['L']
def W_wald(x, w, sigma, z=1.96):
    """exact linearized width from the full Cole+L Jacobian (log10 parameters, alpha)"""
    th = np.array([math.log10(x['Rinf']), math.log10(x['dR']), math.log10(x['tau']), x['alpha'], math.log10(x['L'])]); absZ = np.abs(cole(x, w))
    def m(t): return stack(cole(dict(Rinf=10**t[0], dR=10**t[1], tau=10**t[2], alpha=t[3], L=10**t[4]), w)/absZ)
    J = np.column_stack([(m(th + 1e-6*np.eye(5)[k]) - m(th - 1e-6*np.eye(5)[k]))/2e-6 for k in range(5)])
    C = np.linalg.pinv(J.T @ J)*sigma**2; return 2*z*math.sqrt(max(C[2, 2], 0))

# (a) synthetic band-design spectra
FC = {'low': (100, 1000), 'mid': (1000, 10000), 'near': (10000, 30000)}
def truth(i, fc_rng):
    rng = np.random.default_rng(424242 + i)
    Rinf = 10**rng.uniform(np.log10(500), np.log10(5000)); a = rng.uniform(0.60, 0.95); L = 10**rng.uniform(-3.4, -2.6)
    fc = 10**rng.uniform(*np.log10(fc_rng)); tau = 1/(2*np.pi*fc); dR = rng.uniform(1, 3)*Rinf
    return dict(Rinf=Rinf, dR=dR, tau=tau, alpha=a, L=L, fc=fc)
E = pd.read_csv(T/'e5_band_runs.csv'); rows = []
sig_white = 0.025/math.sqrt(2)
for r in E.itertuples():
    x = truth(r.spectrum, FC[r.fc_class]); N = r.N
    f = np.linspace(r.fmin_hz, 1e5, N) if r.design == 'linear' else np.logspace(np.log10(r.fmin_hz), 5, N); w = 2*np.pi*f
    absZ = np.abs(cole(x, w)); Q = x['tau']**x['alpha']/x['dR']
    D, _, _ = D_factor(w, x['Rinf'], Q, x['alpha'], x['L'], absZ); ratio = r.fmin_hz/x['fc']
    rows.append(dict(design=r.design, N=N, fc_class=r.fc_class, spectrum=r.spectrum, fmin_hz=r.fmin_hz, ratio=ratio, alpha=x['alpha'], D=D,
                     Gamma=Gamma(D, x['alpha'], sig_white), W_asym=W_asym(D, x['alpha'], sig_white, ratio), W_wald=W_wald(x, w, sig_white),
                     W_prof=r.tau_width, closed=r.tau_closed, ident_obs=r.tau_ident))
A = pd.DataFrame(rows); A['ident_pred'] = A.W_asym < 1.0; A['ident_pred_wald'] = A.W_wald < 1.0; A.to_csv(T/'e10_synthetic.csv', index=False)
out = []
for d, g in A.groupby('design'):
    c = g[g.closed]; rec = dict(design=d, n=len(g), closed=len(c))
    rec['spearman_asym_prof'] = spearmanr(np.log(c.W_asym), np.log(c.W_prof)).statistic if len(c) > 5 else np.nan
    rec['median_ratio_asym_prof'] = float(np.median(c.W_asym/c.W_prof)) if len(c) else np.nan
    rec['median_ratio_wald_prof'] = float(np.median(c.W_wald/c.W_prof)) if len(c) else np.nan
    rec['agree_asym_pct'] = 100*float((g.ident_pred == g.ident_obs).mean()); rec['agree_wald_pct'] = 100*float((g.ident_pred_wald == g.ident_obs).mean())
    rec['false_ident_pct'] = 100*float((g.ident_pred & ~g.ident_obs).mean()); rec['missed_ident_pct'] = 100*float((~g.ident_pred & g.ident_obs).mean())
    out.append(rec)
SA = pd.DataFrame(out); SA.to_csv(T/'Table_E10_synthetic.csv', index=False); print(SA.round(3).to_string())
# accuracy of the asymptotic width by f_min/f_c (white-noise designs, closed profiles)
g = A[A.design.isin(['base', 'linear']) & A.closed].copy(); g['bin'] = pd.cut(g.ratio, [0, 1, 3, 10, 1e4], labels=['<1', '1-3', '3-10', '>10'])
print(g.groupby('bin', observed=True).apply(lambda h: pd.Series(dict(n=len(h), med_asym_over_prof=np.median(h.W_asym/h.W_prof), med_wald_over_prof=np.median(h.W_wald/h.W_prof))), include_groups=False).round(2))
print('Gamma (white 2.5%, Wstar = 1 decade): median', A[A.design == 'base'].Gamma.median(), 'IQR', A[A.design == 'base'].Gamma.quantile([.25, .75]).round(2).tolist())

# (b) bias formula against numerical R-CPE-L fits of Cole data (Table S17)
Bn = pd.read_csv(T/'e1b_bias_runs.csv'); SP = {s['sf']: s for s in R.spectra()}; rows = []
for sf, g in Bn.groupby('source_file'):
    s = SP[sf]; p = R.fit('rcpe_l', s['f'], s['Z'])['params']; a = min(max(p['alpha'], 0.5), 0.95); w = 2*np.pi*s['f']
    Z11 = p['Rinf'] + (1j*w)**(-a)/p['Q'] + 1j*w*p['L']; absZ = np.maximum(np.abs(Z11), 1.0)
    D, J, ht = D_factor(w, p['Rinf'], p['Q'], a, p['L'], absZ); b = -np.linalg.lstsq(J, ht, rcond=None)[0]   # per unit (f_min/f_c)^-alpha
    for r in g.itertuples():
        k = r.fmin_over_fc**(-a)
        rows.append(dict(source_file=sf, fc_hz=r.fc_hz, fmin_over_fc=r.fmin_over_fc, alpha=a, bias_logQ_num=r.bias_log10_Q, bias_logQ_formula=b[1]*k/LN10,
                         bias_alpha_num=r.bias_alpha, bias_alpha_formula=b[2]*k, bQ=b[1]/LN10, balpha=b[2]))
Bb = pd.DataFrame(rows); Bb.to_csv(T/'e10_bias.csv', index=False)
SB = Bb.groupby('fmin_over_fc')[['bias_logQ_num', 'bias_logQ_formula', 'bias_alpha_num', 'bias_alpha_formula']].median().sort_index(ascending=False)
SB.to_csv(T/'Table_E10_bias.csv'); print(SB.round(3).to_string())
print('median coefficient b_Q (decades per unit (fmin/fc)^-alpha)', Bb.bQ.median(), 'b_alpha', Bb.balpha.median())

# (c) measured spectra: detection limit f_c* = f_min / Gamma and predicted vs profile width at the fitted f_c
I = pd.read_csv(T/'e1_intervals_M10.csv'); I = I[(I.param == 'log10_tau') & (I.objective == 'nom') & (I.threshold == 'chi2')].set_index('source_file')
Ic = pd.read_csv(T/'e1_intervals_M10.csv'); Ic = Ic[(Ic.param == 'log10_tau') & (Ic.objective == 'nom') & (Ic.threshold == 'cal')].set_index('source_file')
rows = []
for s in R.spectra():
    F11 = R.fit('rcpe_l', s['f'], s['Z']); p = F11['params']; F10 = R.fit('cole_l', s['f'], s['Z']); q = F10['params']
    w = 2*np.pi*s['f']; absZ = np.maximum(np.abs(s['Z']), 1.0); sig = math.sqrt(F10['rss']/(20 - 5))
    D, _, _ = D_factor(w, p['Rinf'], p['Q'], p['alpha'], p['L'], absZ)
    G = Gamma(D, p['alpha'], sig); Gc = Gamma(D, p['alpha'], sig, z=math.sqrt(Ic.loc[s['sf'], 'thr']))
    fc_hat = 1/(2*np.pi*10**q['log10_tau']); ratio = s['f'].min()/fc_hat
    rows.append(dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], sigma=sig, alpha=p['alpha'], D=D, Gamma=G, fc_detect_hz=s['f'].min()/G,
                     fc_detect_cal_hz=s['f'].min()/Gc, fc_hat_hz=fc_hat, fc_ci_high_hz=1/(2*np.pi*10**I.loc[s['sf'], 'ci_low']),
                     W_asym_at_fchat=W_asym(D, p['alpha'], sig, ratio), W_prof=I.loc[s['sf'], 'width'], closed=bool(I.loc[s['sf'], 'closed']),
                     ident_obs=bool(I.loc[s['sf'], 'closed'] and I.loc[s['sf'], 'width'] < 1)))
M = pd.DataFrame(rows); M['ident_pred'] = M.W_asym_at_fchat < 1; M.to_csv(T/'Table_E10_measured.csv', index=False)
print(M[['tissue', 'kk_ok', 'sigma', 'alpha', 'Gamma', 'fc_detect_hz', 'fc_hat_hz', 'fc_ci_high_hz', 'W_asym_at_fchat', 'W_prof', 'ident_obs', 'ident_pred']].round(3).to_string())
print('agreement measured', (M.ident_pred == M.ident_obs).mean(), 'spearman log fc_detect vs log fc_ci_high', spearmanr(np.log(M.fc_detect_hz), np.log(M.fc_ci_high_hz)).statistic)
print('median Gamma measured', M.Gamma.median(), M.Gamma.quantile([.25, .75]).tolist(), 'median f_c detect (Hz)', M.fc_detect_hz.median())


In [ ]:
%%writefile src/review/e10_fig.py
"""Figure for the asymptotic band criterion (E10)."""
import numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import r2_core as R
T = R.T; OUT = T.parent/'figures'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'; C1, C2 = '#2a78d6', '#eb6834'
plt.rcParams.update({'font.size': 8.3, 'axes.edgecolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False, 'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold'})
A = pd.read_csv(T/'e10_synthetic.csv'); B = pd.read_csv(T/'e10_bias.csv'); M = pd.read_csv(T/'Table_E10_measured.csv')
fig, ax = plt.subplots(1, 3, figsize=(7.4, 2.7))
a = ax[0]; g = A[A.design.isin(['base', 'linear', 'corr']) & A.closed]
bins = [(0, 1, '#cde2fb', '$<1$'), (1, 3, '#86b6ef', '1–3'), (3, 10, '#2a78d6', '3–10'), (10, 1e4, '#104281', '$>10$')]
for lo, hi, col, lab in bins:
    h = g[(g.ratio >= lo) & (g.ratio < hi)]; a.scatter(h.W_asym, h.W_prof, s=7, color=col, edgecolor='none', label=lab, alpha=0.9)
lim = [3e-4, 30]; a.plot(lim, lim, color=INK2, lw=0.8); a.axhline(1, color=INK2, lw=0.6, ls='--'); a.axvline(1, color=INK2, lw=0.6, ls='--')
a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(lim); a.set_ylim(lim); a.set_xlabel('Predicted width, Eq. (8) (decades)'); a.set_ylabel('Profile-likelihood width (decades)')
a.legend(title='$f_{\\min}/f_c$', frameon=False, fontsize=6.5, title_fontsize=6.5, loc='upper left', markerscale=1.5)
a.set_title('(a) Width of the $\\log_{10}\\tau$ interval', fontsize=8.5, loc='left')
a = ax[1]; m = B.groupby('fmin_over_fc')[['bias_logQ_num', 'bias_logQ_formula', 'bias_alpha_num', 'bias_alpha_formula']].median()
x = m.index.values
a.plot(x, m.bias_logQ_formula, color=C1, lw=1.6, label='$\\log_{10}Q$, Eq. (10)'); a.plot(x, m.bias_logQ_num, 'o', color=C1, ms=4.5, mec='#fcfcfb', mew=0.6, label='$\\log_{10}Q$, refit')
a.plot(x, -m.bias_alpha_formula, color=C2, lw=1.6, label='$-\\alpha$, Eq. (10)'); a.plot(x, -m.bias_alpha_num, 's', color=C2, ms=4.5, mec='#fcfcfb', mew=0.6, label='$-\\alpha$, refit')
a.set_xscale('log'); a.set_yscale('log'); a.set_xlabel('$f_{\\min}/f_c$'); a.set_ylabel('Bias of R–CPE–L descriptor'); a.grid(color=GRID, lw=0.5); a.set_axisbelow(True)
a.legend(frameon=False, fontsize=6.3); a.set_title('(b) Bias of $Q$ and $\\alpha$', fontsize=8.5, loc='left')
a = ax[2]
for kk, mk, col, lab in [(True, 'o', C1, 'KK-consistent'), (False, 's', C2, 'KK-inconsistent')]:
    h = M[(M.kk_ok == kk) & (M.fc_detect_hz < 1e5)]; a.plot(h.fc_detect_hz, h.fc_ci_high_hz, mk, color=col, ms=4.5, mec='#fcfcfb', mew=0.6, label=lab, ls='none')
lim = [3e2, 1e5]; a.plot(lim, lim, color=INK2, lw=0.8); a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(lim); a.set_ylim(lim)
a.set_xlabel('Detection limit $f_{\\min}/\\Gamma$ (Hz)'); a.set_ylabel('Upper 95% bound of $f_c$ (Hz)'); a.legend(frameon=False, fontsize=6.5, loc='upper left')
a.set_title('(c) Measured spectra', fontsize=8.5, loc='left')
fig.tight_layout(); fig.savefig(OUT/'Fig20_asymptotic_criterion.png', dpi=300); print('ok', (M.fc_detect_hz >= 1e5).sum(), 'off-scale')


In [ ]:
%%writefile src/review/e11_drift.py
"""E11: separation of fitting uncertainty and between-sweep drift.
For every sweep of the recordings with >= 2 sweeps, the R-CPE-L descriptors are estimated with calibrated 95% profile
intervals (B = 100, residual-based covariance); s = half-width/1.96 is the fitting standard error. Within recording i,
    theta_is = mu_i + d_is + e_is,   d_is ~ N(0, tau_d^2) (drift),   e_is ~ N(0, s_is^2) (fitting error, known),
and tau_d^2 is estimated by restricted maximum likelihood with mu_i profiled out, separately for KK-consistent and
KK-inconsistent recordings; 95% intervals from 1,000 resamples of recordings; test of tau_d = 0 by the likelihood
ratio with a 50:50 chi2_0/chi2_1 reference. Reproducibility half-width: 1.96 sqrt(s^2 + tau_d^2).
Usage: python e11_drift.py RAW_DIR"""
import sys, json, math, time, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import chi2
import r2_core as R
T = R.T; SW = R.sweeps(sys.argv[1]); S = R.sigma_struct(json.load(open(T/'E1_psi.json'))['psi']); PARS = ['log10_Q', 'alpha', 'log10_Rinf']
rows = []
for si, s in enumerate(R.spectra()):
    X = SW[s['sf']]
    if len(X) < 2: continue
    for i, z in enumerate(X):
        F = R.fit('rcpe_l', s['f'], z); p = F['params']; zt = R.predict('rcpe_l', p, s['f'])
        tv = {'log10_Q': math.log10(p['Q']), 'alpha': p['alpha'], 'log10_Rinf': math.log10(p['Rinf'])}
        Lc = np.linalg.cholesky(S*F['rss']/(np.trace(S)*(1 - 4/20))); rng = np.random.default_rng(9100 + 100*si + i); L = {q: [] for q in PARS}
        for b in range(100):
            zz = R.noise_draw(rng, Lc, zt); rm = R.fit('rcpe_l', s['f'], zz)['rss']
            for q in PARS: L[q].append(R.lr_at('rcpe_l', s['f'], zz, q, tv[q], None, rm))
        rec = dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], sweep=i)
        for q in PARS:
            g, lr, est = R.profile_curve('rcpe_l', s['f'], z, q); lo, hi, w, cl = R.ci_from_curve(g, lr, float(np.quantile(L[q], .95)))
            rec[q] = est; rec['s_'+q] = w/2/1.96; rec['closed_'+q] = cl
        rows.append(rec)
    print(time.strftime('%H:%M:%S'), si, len(X), flush=True)
D = pd.DataFrame(rows); D.to_csv(T/'e11_sweep_estimates.csv', index=False)

def reml(groups, t2):
    ll = 0.0
    for th, s2 in groups:
        v = s2 + t2; wgt = 1/v; mu = (wgt*th).sum()/wgt.sum()
        ll += -0.5*(np.log(v).sum() + math.log(wgt.sum()) + (wgt*(th - mu)**2).sum())
    return ll
def fit_tau(groups):
    top = max(1e-12, 4*max(np.var(th) for th, _ in groups) + 1e-6)
    r = minimize_scalar(lambda t: -reml(groups, t**2), bounds=(0, math.sqrt(top)*3), method='bounded', options={'xatol': 1e-9})
    t = r.x if reml(groups, r.x**2) > reml(groups, 0.0) else 0.0
    lr = 2*(reml(groups, t**2) - reml(groups, 0.0)); return t, max(lr, 0.0)
out = []; rng = np.random.default_rng(5)
for kk in [True, False]:
    d = D[D.kk_ok == kk]; recs = d.source_file.unique()
    for q in PARS:
        groups = {r: (g[q].to_numpy(), (g['s_'+q]**2).to_numpy()) for r, g in d.groupby('source_file')}
        t, lr = fit_tau(list(groups.values())); p = 0.5*chi2.sf(lr, 1) if lr > 0 else 1.0
        bs = [fit_tau([groups[r] for r in rng.choice(recs, len(recs), replace=True)])[0] for _ in range(1000)]
        s_med = float(np.sqrt(np.median(d['s_'+q]**2)))
        out.append(dict(kk_ok=kk, param=q, recordings=len(recs), sweeps=len(d), median_fit_se=s_med, tau_drift=t, tau_ci_low=np.quantile(bs, .025),
                        tau_ci_high=np.quantile(bs, .975), drift_to_fit_ratio=t/s_med, p_drift=p,
                        fit_halfwidth=1.96*s_med, reproducibility_halfwidth=1.96*math.sqrt(s_med**2 + t**2)))
O = pd.DataFrame(out); O.to_csv(T/'Table_E11_drift.csv', index=False); print(O.round(4).to_string())


In [ ]:
%%writefile src/review/e8_errcheck.py
"""E8: checks of the error model that do not rely on residuals alone.
(a) Scenario check. Residuals are projections of the errors, so a smooth error component lying in the model tangent
    space is invisible in them; the projection-corrected likelihood is weakly identified (rho_f 0.93, 95% CI 0.06-1.00).
    For each KK-consistent spectrum, errors are simulated under the residual-based covariance and under the
    projection-corrected covariance, both scaled so that the expected residual sum of squares, tr(P Sigma P), equals the
    observed RSS. Reported: implied SD of the fitted descriptors and coverage of the calibrated intervals. The implied SD
    is compared with the observed between-sweep SD of the same recording, which also contains drift and is therefore
    an upper bound for noise-induced variability.
(b) Cross-sweep predictive check. For every recording with >= 2 sweeps, each sweep is fitted and profiled (nominal and
    calibrated thresholds, residual-based covariance, B = 100). For every pair of sweeps, |theta_i - theta_j| is compared
    with sqrt(h_i^2 + h_j^2), h = interval half-width; with calibrated intervals and no drift this should hold in about
    95% of pairs. Usage: python e8_errcheck.py RAW_DIR"""
import sys, json, math, itertools, time, numpy as np, pandas as pd
import r2_core as R
T = R.T; SW = R.sweeps(sys.argv[1]); SP = R.spectra()
S_naive = R.sigma_struct(json.load(open(T/'E1_psi.json'))['psi']); S_proj = R.sigma_struct(json.load(open(T/'E1_psi_projected.json'))['psi'])
cst = pd.read_csv(T/'e1_cstar_M11.csv'); cst = cst[cst.objective == 'nom'].set_index(['source_file', 'param']).c_star
PARS = ['log10_Q', 'alpha', 'log10_Rinf']

def projector(f, Z, F):
    vp = R.VP('rcpe_l', f, Z); eta = F['eta']; c, _ = vp.coef(eta); A = vp._sys(eta)[0]; h = 1e-6
    da = (vp._sys(eta + h)[0] - A) @ c/h; sp = vp.sp
    free = [j for j in range(3) if sp['llo'][j]*(1+1e-9) < c[j] < sp['lhi'][j]*(1-1e-9)]
    J = np.column_stack(([da] if 0.3+1e-9 < eta[0] < 1-1e-9 else []) + [A[:, j] for j in free])
    return np.eye(20) - J @ np.linalg.pinv(J)
def truth(F):
    p = F['params']; return {'log10_Q': math.log10(p['Q']), 'alpha': p['alpha'], 'log10_Rinf': math.log10(p['Rinf'])}

# (a) scenario check
ev = pd.read_csv(T/'e1_sweep_fits.csv'); obs_sd = ev[ev.n_sweeps > 1].groupby('source_file')[['log10_Q', 'alpha', 'log10_Rinf']].std()
rows = []
for si, s in enumerate(SP):
    if not s['kk_ok']: continue
    F = R.fit('rcpe_l', s['f'], s['Z']); zt = R.predict('rcpe_l', F['params'], s['f']); tv = truth(F); P = projector(s['f'], s['Z'], F)
    for name, S in (('residual-based', S_naive), ('projection-corrected', S_proj)):
        Sc = S*F['rss']/np.trace(P @ S @ P); Lc = np.linalg.cholesky(Sc); rng = np.random.default_rng(100+si); est = {p: [] for p in PARS}; cov = {p: [] for p in PARS}
        for r in range(100):
            z = R.noise_draw(rng, Lc, zt); Fs = R.fit('rcpe_l', s['f'], z); e = truth(Fs)
            for p in PARS:
                est[p].append(e[p]); cov[p].append(R.lr_at('rcpe_l', s['f'], z, p, tv[p], None, Fs['rss']) <= cst[(s['sf'], p)])
        rec = dict(source_file=s['sf'], tissue=s['tissue'], scenario=name)
        for p in PARS:
            rec[f'sd_{p}'] = float(np.std(est[p], ddof=1)); rec[f'cov_{p}'] = 100*float(np.mean(cov[p]))
            rec[f'obs_sweep_sd_{p}'] = float(obs_sd.loc[s['sf'], p]) if s['sf'] in obs_sd.index else np.nan
        rows.append(rec)
    print(time.strftime('%H:%M:%S'), 'a', si, flush=True)
A = pd.DataFrame(rows); A.to_csv(T/'e8_scenario_runs.csv', index=False)
SA = A.groupby('scenario')[[c for c in A.columns if c.startswith(('sd_', 'cov_'))]].median()
m = A.dropna(subset=['obs_sweep_sd_log10_Q'])
SA['recordings_with_sweeps'] = m.groupby('scenario').size()
for p in PARS: SA[f'implied_gt_observed_{p}'] = m.groupby('scenario').apply(lambda g: int((g[f'sd_{p}'] > g[f'obs_sweep_sd_{p}']).sum()), include_groups=False)
SA.to_csv(T/'Table_E8_scenario.csv'); print(SA.round(3).T.to_string())

# (b) cross-sweep predictive check
rows = []
for si, s in enumerate(SP):
    X = SW[s['sf']]
    if len(X) < 2: continue
    per = []
    for i, z in enumerate(X):
        F = R.fit('rcpe_l', s['f'], z); zt = R.predict('rcpe_l', F['params'], s['f']); tv = truth(F)
        Lc = np.linalg.cholesky(S_naive*F['rss']/(np.trace(S_naive)*(1 - 4/20))); rng = np.random.default_rng(7000+100*si+i); L = {p: [] for p in PARS}
        for b in range(100):
            zz = R.noise_draw(rng, Lc, zt); rm = R.fit('rcpe_l', s['f'], zz)['rss']
            for p in PARS: L[p].append(R.lr_at('rcpe_l', s['f'], zz, p, tv[p], None, rm))
        rec = dict(sweep=i)
        for p in PARS:
            g, lr, est = R.profile_curve('rcpe_l', s['f'], z, p); rec[p] = est
            for tn, thr in (('nom', 3.84), ('cal', float(np.quantile(L[p], .95)))):
                lo, hi, w, cl = R.ci_from_curve(g, lr, thr); rec[f'h_{tn}_{p}'] = w/2
        per.append(rec)
    for a, b in itertools.combinations(per, 2):
        r = dict(source_file=s['sf'], tissue=s['tissue'], kk_ok=s['kk_ok'], i=a['sweep'], j=b['sweep'])
        for p in PARS:
            d = abs(a[p] - b[p]); r[f'diff_{p}'] = d
            for tn in ('nom', 'cal'): r[f'ok_{tn}_{p}'] = bool(d <= math.hypot(a[f'h_{tn}_{p}'], b[f'h_{tn}_{p}']))
        rows.append(r)
    print(time.strftime('%H:%M:%S'), 'b', si, len(X), flush=True)
Bx = pd.DataFrame(rows); Bx.to_csv(T/'e8_cross_sweep_pairs.csv', index=False)
out = []
for kk in [True, False]:
    g = Bx[Bx.kk_ok == kk]; r = dict(kk_ok=kk, recordings=g.source_file.nunique(), pairs=len(g))
    for p in PARS:
        for tn in ('nom', 'cal'): r[f'{tn}_{p}_pct'] = 100*g[f'ok_{tn}_{p}'].mean()
    out.append(r)
SB = pd.DataFrame(out); SB.to_csv(T/'Table_E8_cross_sweep.csv', index=False); print(SB.round(1).T.to_string())


In [ ]:
%%writefile src/review/e9_summary_ci.py
"""E9: confidence intervals for summary statistics.
(a) Sweep-level ICC (one-way, unbalanced) with 95% percentile intervals from 2,000 resamples of recordings.
(b) Coverage with cluster bootstrap intervals (resampling spectra, 2,000 replicates), which account for the
    dependence of replicates generated from the same spectrum."""
import numpy as np, pandas as pd
import r2_core as R
T = R.T; rng = np.random.default_rng(11)
D = pd.read_csv(T/'e7_sweep_params.csv'); m = D[(D.n_sweeps > 1) & D.kk_ok]
def icc(df, c):
    g = df.groupby('grp')[c]; k = g.size(); n = k.sum(); a = len(k)
    if a < 3: return np.nan
    gm = df[c].mean(); msb = (k*(g.mean()-gm)**2).sum()/(a-1); msw = ((df[c]-g.transform('mean'))**2).sum()/(n-a); k0 = (n-(k**2).sum()/n)/(a-1)
    return (msb-msw)/(msb+(k0-1)*msw)
recs = m.source_file.unique(); rows = []
for c in [x for x in D.columns if x.startswith('M1')]:
    base = icc(m.assign(grp=m.source_file), c); bs = []
    for b in range(2000):
        pick = rng.choice(recs, len(recs), replace=True)
        bb = pd.concat([m[m.source_file == r].assign(grp=f'{r}_{i}') for i, r in enumerate(pick)])
        v = icc(bb, c)
        if np.isfinite(v): bs.append(v)
    rows.append(dict(param=c, recordings=len(recs), ICC=base, ci_low=np.quantile(bs, .025), ci_high=np.quantile(bs, .975)))
I = pd.DataFrame(rows); I.to_csv(T/'Table_E9_icc_ci.csv', index=False); print(I.round(3).to_string())
C = pd.read_csv(T/'e1_coverage_runs_M11.csv')
C['nominal'] = C.lr_nom <= 3.84; C['calibrated'] = C.lr_nom <= C.c_nom; C['GLS'] = C.lr_gls <= 3.84; C['GLS_cal'] = C.lr_gls <= C.c_gls
rows = []
for (cond, p), g in C.groupby(['cond', 'param']):
    per = g.groupby('source_file')[['nominal', 'calibrated', 'GLS', 'GLS_cal']].mean(); sf = per.index.values; r = dict(cond=cond, param=p)
    for mth in per.columns:
        bs = [per.loc[rng.choice(sf, len(sf), replace=True), mth].mean() for _ in range(2000)]
        r[mth] = 100*per[mth].mean(); r[mth+'_lo'] = 100*np.quantile(bs, .025); r[mth+'_hi'] = 100*np.quantile(bs, .975)
    rows.append(r)
CV = pd.DataFrame(rows); CV.to_csv(T/'Table_E9_coverage_clusterCI.csv', index=False)
print(CV[CV.cond.isin(['emp'])].round(1).to_string())


In [ ]:
%%writefile src/review/e10b_refined.py
"""E10b: refinement of the closed-form criterion.
(1) Truncation bound: the expansion w/(1+u) = w - w u + w u^2 - ... has relative remainder |u|/(1-|u|) with
    |u| <= (f_min/f_c)^-alpha, so the first-order formula is accurate (remainder < 30%) when f_min/f_c > (0.3)^(-1/alpha)/... ;
    we report r_max = (f_min/f_c)^-alpha for every case and the accuracy of Eq. (8) as a function of r_max.
(2) Refined width: exact derivative of the Cole term with respect to eta = 1/dR at the true f_c, -w^2/(1+u)^2, and the
    full delta method for log10 tau = (log10 Q - log10 eta)/alpha including the alpha and Q covariances
    (Fisher matrix of (ln Rinf, ln Q, alpha, ln L, ln eta) at the truth).
Compared with the profile-likelihood widths of the band-design spectra (white and correlated noise, closed profiles)."""
import math, numpy as np, pandas as pd
import r2_core as R
LN10 = math.log(10); sig_white = 0.025/math.sqrt(2)
FC = {'low': (100, 1000), 'mid': (1000, 10000), 'near': (10000, 30000)}
def stack(c): return np.r_[c.real, c.imag]
def cole(x, w): return x['Rinf'] + x['dR']/(1 + (1j*w*x['tau'])**x['alpha']) + 1j*w*x['L']
def truth(i, fc_rng):     # identical to e5_band_factorial.py / e10_asymptotic.py
    rng = np.random.default_rng(424242 + i)
    Rinf = 10**rng.uniform(np.log10(500), np.log10(5000)); a = rng.uniform(0.60, 0.95); L = 10**rng.uniform(-3.4, -2.6)
    fc = 10**rng.uniform(*np.log10(fc_rng)); tau = 1/(2*np.pi*fc); dR = rng.uniform(1, 3)*Rinf
    return dict(Rinf=Rinf, dR=dR, tau=tau, alpha=a, L=L, fc=fc)
T = R.T; A = pd.read_csv(T/'e10_synthetic.csv'); rows = []
for r in A[A.design.isin(['base', 'linear', 'corr'])].itertuples():
    x = truth(r.spectrum, FC[r.fc_class]); f = np.linspace(r.fmin_hz, 1e5, r.N) if r.design == 'linear' else np.logspace(np.log10(r.fmin_hz), 5, r.N)
    w = 2*np.pi*f; jw = 1j*w; a = x['alpha']; Z = cole(x, w); absZ = np.abs(Z); Q = x['tau']**a/x['dR']; eta = 1/x['dR']
    cpe = jw**(-a)/Q; u = (jw*x['tau'])**(-a)
    # Cole term written as cpe/(1+eta*cpe); parameters p = (ln Rinf, ln Q, alpha, ln L, ln eta)
    d_cole = 1/(1 + u)**2                                          # d[cpe/(1+eta cpe)]/d cpe
    cols = [x['Rinf']/absZ, d_cole*(-cpe)/absZ, d_cole*(-np.log(jw)*cpe)/absZ, jw*x['L']/absZ, -(cpe*u/(1 + u)**2)/absZ]
    J = np.column_stack([stack(c) for c in cols]); C = np.linalg.pinv(J.T @ J)*sig_white**2
    g = np.array([0, 1/a, -math.log(x['tau'])/a, 0, -1/a])           # d ln tau / d p  (ln tau = (ln Q - ln eta)/alpha)
    W_ref = 2*1.96*math.sqrt(max(g @ C @ g, 0))/LN10
    g0 = np.array([0, 0, 0, 0, -1/a]); W_eta_only = 2*1.96*math.sqrt(max(g0 @ C @ g0, 0))/LN10   # without Q and alpha terms
    rows.append(dict(design=r.design, ratio=r.ratio, rmax=r.ratio**(-a), alpha=a, W_asym=r.W_asym, W_wald=r.W_wald, W_ref=W_ref, W_eta_only=W_eta_only,
                     W_prof=r.W_prof, closed=r.closed, ident_obs=r.ident_obs))
B = pd.DataFrame(rows); B.to_csv(T/'e10b_refined.csv', index=False)
c = B[B.closed].copy(); c['rbin'] = pd.cut(c.rmax, [0, 0.1, 0.3, 0.6, 1.0, 100], labels=['<0.1', '0.1-0.3', '0.3-0.6', '0.6-1', '>1'])
S = c.groupby('rbin', observed=True).apply(lambda h: pd.Series(dict(n=len(h), asym=np.median(h.W_asym/h.W_prof), refined=np.median(h.W_ref/h.W_prof),
        eta_only=np.median(h.W_eta_only/h.W_prof), wald=np.median(h.W_wald/h.W_prof))), include_groups=False)
S.to_csv(T/'Table_E10b_refined.csv'); print(S.round(2).to_string())
agree = lambda col: 100*((B[col] < 1) == B.ident_obs).mean()
print('classification agreement %: asym', round(agree('W_asym'), 1), 'refined', round(agree('W_ref'), 1), 'wald', round(agree('W_wald'), 1))
print('alpha/Q coupling inflates width by median factor', round(float(np.median(B.W_ref/B.W_eta_only)), 2), 'IQR', B.eval('W_ref/W_eta_only').quantile([.25, .75]).round(2).tolist())


In [ ]:
%%writefile src/review/e11b_drift_ci.py
"""E11b: inference for the drift SD tau_d that does not rely on the bootstrap (which is degenerate at tau_d = 0).
(a) Profile (REML) likelihood interval: one-sided 95% upper bound {tau: 2[l(tau_hat) - l(tau)] <= 2.71} (boundary at 0),
    and the two-sided 95% interval with 3.84 when tau_hat > 0.
(b) Bayesian sensitivity: posterior of tau_d under half-normal priors (scale 0.05 and 0.2 decade for log parameters,
    0.01 and 0.05 for alpha), with the recording means integrated out (REML likelihood), on a fine grid."""
import math, numpy as np, pandas as pd
import r2_core as R
T = R.T; D = pd.read_csv(T/'e11_sweep_estimates.csv'); PARS = ['log10_Q', 'alpha', 'log10_Rinf']
def reml(groups, t2):
    ll = 0.0
    for th, s2 in groups:
        v = s2 + t2; w = 1/v; mu = (w*th).sum()/w.sum(); ll += -0.5*(np.log(v).sum() + math.log(w.sum()) + (w*(th - mu)**2).sum())
    return ll
rows = []
for kk in [True, False]:
    d = D[D.kk_ok == kk]
    for q in PARS:
        G = [(g[q].to_numpy(), (g['s_'+q]**2).to_numpy()) for _, g in d.groupby('source_file')]
        top = 5*max(1e-6, max(np.std(th) for th, _ in G)) + 5*math.sqrt(np.median(d['s_'+q]**2))
        grid = np.linspace(0, top, 4001); ll = np.array([reml(G, t*t) for t in grid]); i = int(ll.argmax()); th = grid[i]; lmax = ll[i]
        up95 = grid[np.flatnonzero(2*(lmax - ll) <= 2.71)].max()
        two = grid[np.flatnonzero(2*(lmax - ll) <= 3.84)]
        rec = dict(kk_ok=kk, param=q, tau_hat=th, upper95_onesided=up95, ci2_low=two.min(), ci2_high=two.max(), fit_se=float(np.sqrt(np.median(d['s_'+q]**2))))
        for sc in ((0.05, 0.2) if q != 'alpha' else (0.01, 0.05)):
            post = np.exp(ll - lmax)*np.exp(-0.5*(grid/sc)**2); post /= np.trapezoid(post, grid); c = np.cumsum(post)*(grid[1]-grid[0])
            rec[f'post_median_s{sc}'] = grid[np.searchsorted(c, 0.5)]; rec[f'post_upper95_s{sc}'] = grid[np.searchsorted(c, 0.95)]
        rows.append(rec)
O = pd.DataFrame(rows); O.to_csv(T/'Table_E11b_drift_ci.csv', index=False); print(O.round(4).to_string())


In [ ]:
%%writefile src/review/make_supp_refined.py
"""Table S30: accuracy of the first-order and refined closed-form widths (E10b)."""
import sys, pandas as pd
import r2_core as R
S = pd.read_csv(R.T/'Table_E10b_refined.csv'); B = pd.read_csv(R.T/'e10b_refined.csv')
lab = {'<0.1': '$<0.1$', '0.1-0.3': '0.1--0.3', '0.3-0.6': '0.3--0.6', '0.6-1': '0.6--1', '>1': '$>1$ ($f_c$ in band)'}
rows = '\n'.join(f"{lab[r.rbin]} & {int(r.n)} & {r.asym:.2f} & {r.refined:.2f} & {r.eta_only:.2f} \\\\" for r in S.itertuples())
ag = lambda c: 100*((B[c] < 1) == B.ident_obs).mean()
cpl = (B.W_ref/B.W_eta_only).median()
open(sys.argv[1], 'w').write(r"""\begin{table}[H]\centering
\caption{Accuracy of the closed-form width of the $\log_{10}\tau$ interval by $r=(f_{\min}/f_c)^{-\alpha}$, the bound on the relative size of the neglected terms of Eq.~(7): median ratio of predicted to profile-likelihood width (band-design spectra with closed profiles, white or correlated noise). First order: Eq.~(8). Refined: exact derivative $-wu/(1+u)^2$ and delta method for $\log\tau=(\log Q-\log\eta)/\alpha$ including the covariances with $Q$ and $\alpha$ (identical to the exact linearized width). Without coupling: refined derivative, $\eta$ variance only. Coupling with $Q$ and $\alpha$ widened the interval by a median factor of """ + f"{cpl:.2f}" + r"""; classification agreement with profile likelihood: first order """ + f"{ag('W_asym'):.1f}" + r"""\%, refined """ + f"{ag('W_ref'):.1f}" + r"""\%.}
\small
\begin{tabular}{lcccc}\toprule
$r$ & n & First order & Refined & Without coupling \\\midrule
""" + rows + r"""
\bottomrule\end{tabular}\end{table}
""")
print('ok')


In [ ]:
%%writefile src/rerun_same_env.py
"""rerun_same_env.py - reruns the complete Table 2 benchmark (8 original + 5 VarPro algorithms, 20 spectra x 25 runs,
2,000 NFE, same seeds) in the environment used for the timing experiment, so that success rates and timings come from
one computer. Usage: python rerun_same_env.py MODEL   (M10_ColeL | M11_RCPEL | M12_DoubleColeL)"""
import sys, math, time, pickle, platform, numpy as np, pandas as pd
from pathlib import Path
from vp_common import load_spectra, TAB
import eis_core as E, vp_core as V
model = sys.argv[1]; SEED0 = 20261101; RUNS = 25; CPS = (500, 2000)
OUT = TAB.parent/'rerun_same_env'; OUT.mkdir(exist_ok=True)
rows = []
for s in load_spectra():
    cp = OUT/f'{model}_{s["si"]}.pkl'
    if cp.exists(): rows += pickle.load(open(cp, 'rb')); continue
    out = []
    for r in range(RUNS):
        seed = SEED0 + s['si']*1000 + r
        for a in E.ALGORITHMS:
            x, fb, tr, ex, sec = E.run_algorithm(a, model, s['w'], s['z'], 2000, seed, CPS)
            out.append(dict(source_file=s['sf'], run=r+1, algorithm=a, NRMSE=math.sqrt(fb), sec=sec, **{f't{c}': math.sqrt(tr.get(c, fb)) for c in CPS}))
        for a in V.VP_ALGORITHMS:
            x, fb, tr, sec = V.run_vp(a, model, s['w'], s['z'], 2000, seed, CPS)
            out.append(dict(source_file=s['sf'], run=r+1, algorithm=a, NRMSE=math.sqrt(fb), sec=sec, **{f't{c}': math.sqrt(tr.get(c, fb)) for c in CPS}))
    pickle.dump(out, open(cp, 'wb')); rows += out; print(time.strftime('%H:%M:%S'), model, s['si'], flush=True)
D = pd.DataFrame(rows); ref = pd.read_csv(TAB/f'vp_merged_{model}.csv').groupby('source_file').ref.first()
D['ref'] = D.source_file.map(ref)
for c in CPS: D[f's{c}'] = (D[f't{c}'] - D.ref)/D.ref <= 1e-3
D.to_csv(OUT/f'runs_{model}.csv', index=False)
S = D.groupby('algorithm')[['s500', 's2000']].mean().mul(100); S['new_minima'] = D.groupby('algorithm').apply(lambda g: int(((g.NRMSE - g.ref)/g.ref < -1e-6).sum()), include_groups=False)
S.to_csv(OUT/f'success_{model}.csv'); print(S.round(1).to_string()); print(platform.platform(), platform.processor())


In [ ]:
%%writefile src/review/e12_placement.py
"""E12: between-recording (placement) variability, reproducibility limits and per-animal analysis from existing data.
No new measurements. Uses (i) the single-frequency recordings of the dataset (tasks 2-4: static tissue recordings and
vessel punctures with the same bipolar probes and with a monopolar probe), (ii) the 100 kHz point of every sweep of the
20 frequency-scan recordings, and (iii) the R-CPE-L descriptor estimates of Sect. 3.4/3.7.

1. Every single-frequency recording is summarised by the median resistance in its most stable window (20% of readings,
   at least 15, minimum robust SD of log10 R); vessel recordings start in the surrounding tissue and end in blood, so
   each is a separate needle advance. Recordings of the same medium with the same probe, animal and frequency form a
   group; the pooled between-recording SD of log10 R (chi-square 95% CI) is compared with the within-recording SDs
   (short-term, between sweeps) and with the between-sweep drift of R_inf.
2. Transfer to R_inf: within recordings, between-sweep changes of log10 R_inf and log10 R(100 kHz) are regressed through
   the origin; the slope converts the between-placement SD of R(100 kHz) into an R_inf scale.
3. Descriptor-level between-recording SD from the recordings of the same tissue in the same animal (muscle x5, liver x2,
   vein x2), after subtracting the within-recording variance (fitting SE^2 + drift^2).
4. Repeatability and reproducibility limits r = 2.77 s_w, R = 2.77 sqrt(s_w^2 + s_b^2) (ISO 5725 form) and the fraction
   of tissue pairs (KK-consistent spectra) whose descriptor difference exceeds them.
5. Main results recomputed per animal (= subset A / subset D; animal, subset and date are confounded).
Usage: python e12_placement.py RAW_DIR"""
import sys, glob, os, math, itertools, json, numpy as np, pandas as pd
from scipy.stats import chi2
import r2_core as R
RAW = sys.argv[1]; T = R.T; LOG = math.log10

def read(p):
    txt = open(p).read().replace('\t', ';')
    rows = [l.split(';')[:8] for l in txt.splitlines() if l.count(';') >= 6]
    d = pd.DataFrame(rows, columns=['st', 'f', 'mag', 'ph', 'R', 'X', 't', 'tis'])
    for c in ['f', 'R', 'X']: d[c] = pd.to_numeric(d[c])
    return d

def stable(Rv):
    Rv = np.asarray(Rv, float); n = len(Rv); w = max(15, int(round(0.2*n)))
    if n < w: w = n
    best = None
    for i in range(0, n - w + 1):
        x = Rv[i:i+w]
        if (x <= 0).any(): continue
        lx = np.log10(x); s = 1.4826*np.median(np.abs(lx - np.median(lx)))
        if best is None or s < best[0]: best = (s, float(np.median(x)), i, w)
    return best   # (robust SD of log10 R, median R, start, window)

# ---------------- 1. single-frequency recordings
meta = []
for p in sorted(glob.glob(RAW + '/byExperimentalTask/_task[234]/**/*.txt', recursive=True)):
    fn = os.path.basename(p); m = fn.replace('.txt', '').split('_')
    stamp = m[1]; animal = 1 if stamp.startswith('2023') else 2
    probe = 'monopolar' if 'monop' in fn else m[3]
    medium = 'blood' if fn.endswith('_blood.txt') else ('surrounding' if fn.endswith('_surrounding.txt') else m[-1])
    vessel = 'vein' if '_vein_' in fn else ('artery' if '_artery_' in fn else '')
    if medium == 'surrounding': continue
    d = read(p); b = stable(d.R.to_numpy())
    if b is None: continue
    win = d.iloc[b[2]:b[2]+b[3]]; ph = float(np.degrees(np.arctan2(win.X, win.R)).median())
    meta.append(dict(source='single', file=fn, animal=animal, probe=probe, medium=medium, vessel=vessel, f_khz=int(d.f.iloc[0]),
                     n=len(d), R=b[1], sd_short=b[0], phase_deg=ph))
# 100 kHz point of the sweep recordings (yellow probe, animal 1/2)
SW = R.sweeps(RAW); sw100 = []
for s in R.spectra():
    X = SW[s['sf']]; k = int(np.argmin(np.abs(s['f'] - 1e5))); r100 = X[:, k].real; ph100 = np.degrees(np.angle(X[:, k]))
    animal = 1 if '_2023' in s['sf'] else 2
    for i, v in enumerate(r100): sw100.append(dict(file=s['sf'], sweep=i, logR100=LOG(v), phase=ph100[i], kk_ok=s['kk_ok']))
    meta.append(dict(source='sweep', file=s['sf'], animal=animal, probe='yellow', medium=s['tissue'], vessel='', f_khz=100, n=len(X),
                     R=float(np.median(r100)), sd_short=np.nan, kk_ok=s['kk_ok'], phase_deg=float(np.median(ph100))))
# 9-frequency skin recording (100 kHz block medians)
for p in glob.glob(RAW + '/byOriginalSubset/subset A/*skin*.txt'):
    d = read(p); d = d[d.f == 100].reset_index(drop=True); b = stable(d.R.to_numpy()); win = d.iloc[b[2]:b[2]+b[3]]
    meta.append(dict(source='sweep9', file=os.path.basename(p), animal=1, probe='yellow', medium='skin', vessel='', f_khz=100, n=len(d), R=b[1], sd_short=b[0],
                     phase_deg=float(np.degrees(np.arctan2(win.X, win.R)).median())))
M = pd.DataFrame(meta); M['logR'] = np.log10(M.R)
M['medium_g'] = np.where(M.medium == 'blood', 'blood_' + M.vessel, M.medium)
M.to_csv(T/'e12_recordings.csv', index=False)

def pooled(groups):
    num = sum((len(g) - 1)*np.var(g, ddof=1) for g in groups if len(g) > 1); nu = sum(len(g) - 1 for g in groups if len(g) > 1)
    if nu == 0: return np.nan, np.nan, np.nan, 0
    s = math.sqrt(num/nu); return s, s*math.sqrt(nu/chi2.ppf(.975, nu)), s*math.sqrt(nu/chi2.ppf(.025, nu)), nu
rows = []
for key, g in M.groupby(['animal', 'probe', 'medium_g', 'f_khz']):
    if len(g) < 2: continue
    s = np.std(g.logR, ddof=1)
    rows.append(dict(animal=key[0], probe=key[1], medium=key[2], f_khz=key[3], recordings=len(g), median_R_ohm=g.R.median(),
                     between_sd_log10=s, cv_pct=100*(10**s - 1), short_term_sd_log10=g.sd_short.median(), between_sd_phase_deg=np.std(g.phase_deg, ddof=1),
                     median_phase_deg=g.phase_deg.median()))
G = pd.DataFrame(rows)
sw = pd.DataFrame(sw100); within_sw = pooled([g.logR100.to_numpy() for _, g in sw[sw.kk_ok].groupby('file') if len(g) > 1])
within_sw_all = pooled([g.logR100.to_numpy() for _, g in sw.groupby('file') if len(g) > 1])
within_ph = pooled([g.phase.to_numpy() for _, g in sw[sw.kk_ok].groupby('file') if len(g) > 1])
summ = []
for lab, sel in [('blood, bipolar (same probe)', (G.medium.str.startswith('blood')) & (G.probe != 'monopolar')),
                 ('blood, monopolar', (G.medium.str.startswith('blood')) & (G.probe == 'monopolar')),
                 ('tissue, yellow bipolar 100 kHz', ~G.medium.str.startswith('blood'))]:
    keys = G[sel][['animal', 'probe', 'medium', 'f_khz']].itertuples(index=False)
    grp = [M[(M.animal == a) & (M.probe == p) & (M.medium_g == m) & (M.f_khz == f)].logR.to_numpy() for a, p, m, f in keys]
    s, lo, hi, nu = pooled(grp)
    grp_ph = [M[(M.animal == a) & (M.probe == p) & (M.medium_g == m) & (M.f_khz == f)].phase_deg.to_numpy() for a, p, m, f in G[sel][['animal', 'probe', 'medium', 'f_khz']].itertuples(index=False)]
    sph = pooled(grp_ph)
    summ.append(dict(set=lab, groups=int(sel.sum()), recordings=int(sum(len(x) for x in grp)), df=nu,
                     between_sd=s, ci_low=lo, ci_high=hi, cv_pct=100*(10**s - 1), phase_sd_deg=sph[0], phase_ci_high=sph[2]))
summ.append(dict(set='within recording, between sweeps, KK-consistent (R at 100 kHz)', groups=np.nan, recordings=int(sw[sw.kk_ok].groupby('file').size().gt(1).sum()),
                 df=within_sw[3], between_sd=within_sw[0], ci_low=within_sw[1], ci_high=within_sw[2], cv_pct=100*(10**within_sw[0] - 1), phase_sd_deg=within_ph[0], phase_ci_high=within_ph[2]))
summ.append(dict(set='within recording, between sweeps, all (R at 100 kHz)', groups=np.nan, recordings=int(sw.groupby('file').size().gt(1).sum()),
                 df=within_sw_all[3], between_sd=within_sw_all[0], ci_low=within_sw_all[1], ci_high=within_sw_all[2], cv_pct=100*(10**within_sw_all[0] - 1)))
summ.append(dict(set='within recording, short-term (stable window)', groups=np.nan, recordings=int(M.sd_short.notna().sum()), df=np.nan,
                 between_sd=float(M.sd_short.median()), ci_low=np.nan, ci_high=np.nan, cv_pct=100*(10**M.sd_short.median() - 1)))
S1 = pd.DataFrame(summ)
# probe effect: same tissue, same animal, yellow vs white bipolar (100 kHz)
pe = []
for tis in ['tendon', 'subcutaneous', 'bladder', 'intestine']:
    y = M[(M.probe == 'yellow') & (M.medium == tis) & (M.animal == 1)].logR; w = M[(M.probe == 'white') & (M.medium == tis)].logR
    if len(y) and len(w): pe.append(dict(tissue=tis, n_yellow=len(y), n_white=len(w), log10_ratio_white_yellow=float(w.mean() - y.mean())))
bl_y = M[(M.probe == 'yellow') & (M.medium == 'blood') & (M.animal == 1)].logR; bl_w = M[(M.probe == 'white') & (M.medium == 'blood')].logR
pe.append(dict(tissue='blood', n_yellow=len(bl_y), n_white=len(bl_w), log10_ratio_white_yellow=float(bl_w.mean() - bl_y.mean())))
PE = pd.DataFrame(pe)

# ---------------- 2. transfer from R(100 kHz) to R_inf (within-recording sweep changes)
E = pd.read_csv(T/'e11_sweep_estimates.csv').merge(sw.drop(columns='kk_ok').rename(columns={'file': 'source_file'}), on=['source_file', 'sweep'])
tr = {}
for kk in [True, False, None]:
    e = E if kk is None else E[E.kk_ok == kk]
    dx = e.logR100 - e.groupby('source_file').logR100.transform('mean'); dy = e.log10_Rinf - e.groupby('source_file').log10_Rinf.transform('mean')
    b = float((dx*dy).sum()/(dx*dx).sum()); r = float(np.corrcoef(dx, dy)[0, 1])
    tr['all' if kk is None else ('kk' if kk else 'nonkk')] = dict(slope=b, r=r, n=len(e))
sp = pd.DataFrame([dict(sf=s['sf'], logR100=LOG(s['Z'][np.argmin(np.abs(s['f'] - 1e5))].real)) for s in R.spectra()])
P = pd.read_csv(T/'Table_M11_physical_parameters.csv').merge(sp, left_on='source_file', right_on='sf')
tr['across_spectra'] = dict(slope=float(np.polyfit(P.logR100, np.log10(P.Rinf_ohm), 1)[0]), r=float(np.corrcoef(P.logR100, np.log10(P.Rinf_ohm))[0, 1]), n=len(P))

# ---------------- 3. descriptor-level between-recording SD (same tissue, same animal)
I = pd.read_csv(T/'e1_intervals_M11.csv'); I = I[(I.objective == 'nom') & (I.threshold == 'cal')]
I['se'] = I.width/2/1.96; I['animal'] = np.where(I.source_file.str.contains('_2023'), 1, 2)
DR = pd.read_csv(T/'Table_E11_drift.csv').set_index(['kk_ok', 'param'])
desc = []
for q in ['log10_Q', 'alpha', 'log10_Rinf']:
    d = I[I.param == q].copy(); d['w2'] = d.se**2 + [DR.loc[(k, q), 'tau_drift']**2 for k in d.kk_ok]
    for lab, sel in [('all', d.kk_ok | ~d.kk_ok), ('KK-consistent', d.kk_ok)]:
        dd = d[sel]; grp = [g for _, g in dd.groupby(['animal', 'tissue']) if len(g) > 1]
        if not grp: continue
        s, lo, hi, nu = pooled([g.estimate.to_numpy() for g in grp]); w2 = float(np.mean(np.concatenate([g.w2.to_numpy() for g in grp])))
        desc.append(dict(param=q, set=lab, groups='; '.join(f'{g.tissue.iloc[0]} x{len(g)} (animal {g.animal.iloc[0]})' for g in grp), df=nu,
                         between_recording_sd=s, ci_high=hi, within_sd=math.sqrt(w2), excess_sd=math.sqrt(max(s**2 - w2, 0)),
                         excess_sd_high=math.sqrt(max(hi**2 - w2, 0))))
D3 = pd.DataFrame(desc)

# ---------------- 4. repeatability / reproducibility limits and tissue discrimination (KK-consistent spectra)
# between-placement component for R_inf from the bipolar blood recordings transferred with the within-recording slope (KK);
# for Q and alpha from the KK-consistent replicate pair (upper 95% bound), stated as such
sb_R100 = S1.loc[0, 'between_sd']; sb_R100_hi = S1.loc[0, 'ci_high']; sw_R100 = within_sw[0]
exc_R100 = math.sqrt(max(sb_R100**2 - sw_R100**2, 0)); exc_R100_hi = math.sqrt(max(sb_R100_hi**2 - sw_R100**2, 0))
lim = []
for q in ['log10_Q', 'alpha', 'log10_Rinf']:
    s_fit = DR.loc[(True, q), 'median_fit_se']; s_dr = DR.loc[(True, q), 'tau_drift']; s_w = math.hypot(s_fit, s_dr)
    if q == 'log10_Rinf':
        k = abs(tr['kk']['slope']); s_b, s_b_hi, src = k*exc_R100, k*exc_R100_hi, 'bipolar blood recordings via R(100 kHz)'
    else:
        row = D3[(D3.param == q) & (D3.set == 'KK-consistent')]
        s_b, s_b_hi, src = (float(row.excess_sd.iloc[0]), float(row.excess_sd_high.iloc[0]), 'KK-consistent replicate pair') if len(row) else (np.nan, np.nan, 'none')
    lim.append(dict(param=q, s_fit=s_fit, s_drift=s_dr, s_within=s_w, r_limit=2.77*s_w, s_between=s_b, s_between_high=s_b_hi,
                    R_limit=2.77*math.hypot(s_w, s_b), R_limit_high=2.77*math.hypot(s_w, s_b_hi), source=src))
L = pd.DataFrame(lim).set_index('param')
K = I[I.kk_ok]; tis = K.groupby(['tissue', 'param']).estimate.mean().unstack()
disc = []
for q in ['log10_Q', 'alpha', 'log10_Rinf']:
    v = tis[q].dropna(); pairs = list(itertools.combinations(v.index, 2)); dif = np.array([abs(v[a] - v[b]) for a, b in pairs])
    disc.append(dict(param=q, tissues=len(v), pairs=len(pairs), median_abs_diff=float(np.median(dif)), pct_gt_r=100*np.mean(dif > L.loc[q, 'r_limit']),
                     pct_gt_R=100*np.mean(dif > L.loc[q, 'R_limit']), pct_gt_R_high=100*np.mean(dif > L.loc[q, 'R_limit_high'])))
DI = pd.DataFrame(disc)

# ---------------- 5. per-animal summary of the main results
an = lambda sf: 1 if '_2023' in sf else 2
E10 = pd.read_csv(T/'Table_E10_measured.csv'); E10['animal'] = E10.source_file.map(an)
VPs = pd.read_csv(T/'Table_VP_M12_success_by_spectrum.csv', index_col=0); VPs['animal'] = [1 if i.split('_')[-1].startswith('2023') else 2 for i in VPs.index]
pa = []
for a in [1, 2]:
    e = E10[E10.animal == a]; v = VPs[VPs.animal == a]; ii = I[I.animal == a]
    cl = ii[ii.kk_ok].groupby('param').closed.mean()
    pa.append(dict(animal=a, spectra=len(e), kk_consistent=int(e.kk_ok.sum()),
                   M12_success_MS_TRF=100*v['MS-TRF'].mean(), M12_success_VP_MS_TRF=100*v['VP-MS-TRF'].mean(),
                   VP_gain_pp=100*(v['VP-MS-TRF'] - v['MS-TRF']).mean(), VP_gain_positive=f"{int(((v['VP-MS-TRF'] - v['MS-TRF']) > 0).sum())}/{int(((v['VP-MS-TRF'] - v['MS-TRF']) != 0).sum())}",
                   tau_identifiable=f'{int(e.ident_obs.sum())}/{len(e)}', criterion_agreement=f'{int((e.ident_obs == e.ident_pred).sum())}/{len(e)}',
                   Gamma_median=float(e.Gamma.median()), fc_detect_median_hz=float(e.fc_detect_hz.median()), fc_detect_cal_median_hz=float(e.fc_detect_cal_hz.median()),
                   kk_closed_Q=float(cl.get('log10_Q', np.nan)), kk_closed_alpha=float(cl.get('alpha', np.nan)), kk_closed_Rinf=float(cl.get('log10_Rinf', np.nan)),
                   median_halfwidth_Q=float(ii[(ii.param == 'log10_Q') & ii.kk_ok].width.median()/2), median_halfwidth_alpha=float(ii[(ii.param == 'alpha') & ii.kk_ok].width.median()/2)))
PA = pd.DataFrame(pa)
# animal 1 vs 2 for tissues measured in both (KK-consistent): artery
both = []
for t, g in I[I.kk_ok].groupby('tissue'):
    if g.animal.nunique() < 2: continue
    for q, h in g.groupby('param'):
        a1 = h[h.animal == 1]; a2 = h[h.animal == 2]
        d = a2.estimate.mean() - a1.estimate.mean(); se = math.sqrt((a1.se**2).mean()/len(a1) + (a2.se**2).mean()/len(a2))
        both.append(dict(tissue=t, param=q, n1=len(a1), n2=len(a2), diff_2_minus_1=d, fit_se=se, R_limit=L.loc[q, 'R_limit'], R_limit_high=L.loc[q, 'R_limit_high']))
BO = pd.DataFrame(both)

# ---------------- 6. does the placement effect act as a scale factor?  Z -> k Z shifts log R_inf by +log k, log Q by -log k,
# leaves alpha and the phase unchanged; log10(R_inf Q) is invariant.
W = I.pivot_table(index=['source_file', 'tissue', 'animal', 'kk_ok'], columns='param', values='estimate').reset_index()
W['log10_RinfQ'] = W.log10_Rinf + W.log10_Q
sc = []
for lab, sel in [('all replicate groups', W.kk_ok | ~W.kk_ok), ('KK-consistent replicate groups', W.kk_ok)]:
    ww = W[sel]; grp = [g for _, g in ww.groupby(['animal', 'tissue']) if len(g) > 1]
    if not grp: continue
    out = dict(set=lab, groups='; '.join(f'{g.tissue.iloc[0]} x{len(g)}' for g in grp))
    for q in ['log10_Rinf', 'log10_Q', 'log10_RinfQ', 'alpha']: out['sd_' + q] = pooled([g[q].to_numpy() for g in grp])[0]
    dq = np.concatenate([g.log10_Q - g.log10_Q.mean() for g in grp]); dr = np.concatenate([g.log10_Rinf - g.log10_Rinf.mean() for g in grp])
    out['slope_dlogQ_on_dlogRinf'] = float((dq*dr).sum()/(dr*dr).sum()); sc.append(out)
e = E[E.kk_ok]; dq = e.log10_Q - e.groupby('source_file').log10_Q.transform('mean'); dr = e.log10_Rinf - e.groupby('source_file').log10_Rinf.transform('mean')
sc.append(dict(set='within recording, between sweeps (KK-consistent)', groups=f'{e.source_file.nunique()} recordings', slope_dlogQ_on_dlogRinf=float((dq*dr).sum()/(dr*dr).sum()),
               sd_log10_Rinf=float(dr.std()), sd_log10_Q=float(dq.std()), sd_log10_RinfQ=float((dq + dr).std()), sd_alpha=float((e.alpha - e.groupby('source_file').alpha.transform('mean')).std())))
SC = pd.DataFrame(sc)
for name, df in [('Table_E12_scale', SC), ('Table_E12_groups', G), ('Table_E12_summary', S1), ('Table_E12_probe', PE), ('Table_E12_descriptor_between', D3),
                 ('Table_E12_limits', L.reset_index()), ('Table_E12_discrimination', DI), ('Table_E12_per_animal', PA), ('Table_E12_between_animal', BO)]:
    df.to_csv(T/f'{name}.csv', index=False)
json.dump(tr, open(T/'E12_transfer.json', 'w'), indent=1)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
for name, df in [('scale', SC), ('groups', G), ('summary', S1), ('probe', PE), ('descriptor', D3), ('limits', L), ('discrimination', DI), ('per animal', PA), ('between animal', BO)]:
    print('\n==', name); print(df.round(4).to_string())
print('\ntransfer', json.dumps(tr, indent=1))


In [ ]:
%%writefile src/review/make_supp_e12.py
"""Tables S32-S36 (between-recording variability, probe effect, descriptor replicates and scale test, reproducibility
limits and tissue discrimination, per-animal results) from the outputs of e12_placement.py."""
import sys, json, numpy as np, pandas as pd
import r2_core as R
T = R.T; out = sys.argv[1] if len(sys.argv) > 1 else 'supp_e12.tex'
G = pd.read_csv(T/'Table_E12_groups.csv'); S = pd.read_csv(T/'Table_E12_summary.csv'); PE = pd.read_csv(T/'Table_E12_probe.csv')
D = pd.read_csv(T/'Table_E12_descriptor_between.csv'); SC = pd.read_csv(T/'Table_E12_scale.csv'); L = pd.read_csv(T/'Table_E12_limits.csv').set_index('param')
DI = pd.read_csv(T/'Table_E12_discrimination.csv').set_index('param'); PA = pd.read_csv(T/'Table_E12_per_animal.csv'); BO = pd.read_csv(T/'Table_E12_between_animal.csv')
tr = json.load(open(T/'E12_transfer.json'))
PN = {'log10_Q': r'$\log_{10}Q$', 'alpha': r'$\alpha$', 'log10_Rinf': r'$\log_{10}R_{\infty}$', 'log10_RinfQ': r'$\log_{10}(R_{\infty}Q)$'}
MED = lambda m: m.replace('blood_', 'blood (').replace('artery', 'artery)').replace('vein', 'vein)') if m.startswith('blood') else m
def tab(cap, spec, head, rows):
    return ("\\begin{table}[H]\n\\centering\n\\caption{" + cap + "}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n\\begin{tabular}{" + spec + "}\n\\toprule\n"
            + head + " \\\\\n\\midrule\n" + " \\\\\n".join(rows) + " \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
# S32 groups
r32 = []
for g in G.sort_values(['animal', 'probe', 'medium']).itertuples():
    med = MED(g.medium).replace('vein)', 'vein)') if g.medium.startswith('blood') else (g.medium + ' (scans)' if g.medium == 'vein' else g.medium)
    r32.append(f"{g.animal} & {g.probe} & {med} & {g.f_khz} & {g.recordings} & {g.median_R_ohm:.0f} & {g.between_sd_log10:.3f} & {g.cv_pct:.0f} & "
               + (f"{g.short_term_sd_log10:.4f}" if np.isfinite(g.short_term_sd_log10) else '--') + f" & {g.between_sd_phase_deg:.1f}")
r32.append('\\midrule\n\\multicolumn{10}{l}{\\emph{Pooled}}')
for s in S.itertuples():
    ci = f"[{s.ci_low:.3f}, {s.ci_high:.3f}]" if np.isfinite(s.ci_low) else ''
    ph = f"{s.phase_sd_deg:.1f}" if np.isfinite(s.phase_sd_deg) else '--'
    df = f"{int(s.df)}" if np.isfinite(s.df) else '--'
    r32.append(f"\\multicolumn{{4}}{{l}}{{{s.set}}} & {s.recordings} (df {df}) & & {s.between_sd:.3f} {ci} & {(f'{s.cv_pct:.1f}' if s.cv_pct < 1 else f'{s.cv_pct:.0f}')} & & {ph}")
t32 = tab(r"Between-recording variability of the resistance and phase angle in recordings of the same medium with the same probe, animal and frequency (most stable window of each single-frequency recording; median over sweeps of the 100~kHz point of the frequency scans). Vessel recordings begin in the surrounding tissue and end in blood (separate needle advances); whether static repeat recordings involved re-insertion at a new site is not documented. SD: standard deviation of the recording medians; CV $=10^{\mathrm{SD}}-1$; short-term SD: median robust SD of $\log_{10}R$ within the stable window. Pooled: degrees-of-freedom-weighted SD with $\chi^2$ 95\% interval.",
          'cllcccccccc'[:10], r"Animal & Probe & Medium & kHz & $n$ & \makecell{Median\\$R$ ($\Omega$)} & \makecell{SD\\$\log_{10}R$} & \makecell{CV\\(\%)} & \makecell{Short-term\\SD} & \makecell{SD phase\\($^{\circ}$)}", r32)
# S33 probe
r33 = [f"{p.tissue} & {p.n_yellow} & {p.n_white} & {p.log10_ratio_white_yellow:+.2f} & {100*(10**p.log10_ratio_white_yellow - 1):+.0f}" for p in PE.itertuples()]
t33 = tab(r"Probe effect: mean $\log_{10}R$ at 100~kHz with the second bipolar probe (white) minus that with the scan probe (yellow), first animal. Differences in blood, tendon and subcutaneous tissue exceed the between-recording SD (Table~S32), so absolute impedance descriptors are probe-specific.",
          'lcccc', r"Medium & $n$ yellow & $n$ white & $\Delta\log_{10}R$ & Difference (\%)", r33)
# S34 descriptors + scale test
r34 = []
for d in D.itertuples():
    r34.append(f"{PN[d.param]} & {d.set} & {d.df} & {d.between_recording_sd:.3f} & {d.within_sd:.3f} & {d.excess_sd:.3f} & {d.excess_sd_high:.2f}")
t34a = tab(r"Descriptor-level between-recording SD from recordings of the same tissue in the same animal (all: liver $\times$2 and muscle $\times$5, first animal; vein $\times$2, second animal; KK-consistent: the vein pair only). Within-recording SD: root mean of fitting variance (calibrated half-width/1.96) plus drift variance (Table~S28). Excess: $\sqrt{\max(\mathrm{SD}^2-\mathrm{within}^2,0)}$; upper bound from the $\chi^2$ 95\% upper limit of the SD. In the pooled set the within-recording variance of the KK-inconsistent recordings exceeds the between-recording variance, so no excess is estimable; with one degree of freedom the KK-consistent bound is uninformative.",
           'llccccc', r"Parameter & Set & df & \makecell{Between-recording\\SD} & \makecell{Within-recording\\SD} & Excess & \makecell{Excess, upper\\95\% bound}", r34)
r34b = []
for s in SC.itertuples():
    r34b.append(f"{s.set} & {s.sd_log10_Rinf:.3f} & {s.sd_log10_Q:.3f} & {s.sd_log10_RinfQ:.3f} & {s.sd_alpha:.3f} & {s.slope_dlogQ_on_dlogRinf:+.2f}")
t34b = tab(r"Scale test. A pure scale change of the impedance ($Z\to kZ$, e.g.\ a change of cell constant) shifts $\log_{10}R_{\infty}$ and $\log_{10}Q$ by equal and opposite amounts (slope $-1$), leaves $\alpha$ and the phase unchanged and cancels in $\log_{10}(R_{\infty}Q)$. Slope: regression of within-group deviations of $\log_{10}Q$ on those of $\log_{10}R_{\infty}$. Only the pooled set, dominated by KK-inconsistent muscle recordings, behaves partly like a scale change; the KK-consistent replicate pair and the between-sweep changes do not, consistent with the phase changes in Table~S32. Transfer slope from $\log_{10}R(100~\mathrm{kHz})$ to $\log_{10}R_{\infty}$ within KK-consistent recordings: " + f"{tr['kk']['slope']:.2f} ($r={tr['kk']['r']:.2f}$, {tr['kk']['n']} sweeps).",
           'lccccc', r"Set & \makecell{SD\\$\log_{10}R_{\infty}$} & \makecell{SD\\$\log_{10}Q$} & \makecell{SD\\$\log_{10}(R_{\infty}Q)$} & \makecell{SD\\$\alpha$} & Slope", r34b)
# S35 limits & discrimination
r35 = []
for q in ['log10_Rinf', 'log10_Q', 'alpha']:
    l = L.loc[q]; d = DI.loc[q]
    src = 'blood recordings' if q == 'log10_Rinf' else 'one replicate pair'
    r35.append(f"{PN[q]} & {l.s_within:.3f} & {l.r_limit:.3f} & {l.s_between:.3f} ({l.s_between_high:.2f}) & {l.R_limit:.2f} ({l.R_limit_high:.2f}) & {src} & {d.median_abs_diff:.3f} & {d.pct_gt_r:.0f} & {d.pct_gt_R:.0f} ({d.pct_gt_R_high:.0f})")
t35 = tab(r"Repeatability and reproducibility limits (ISO 5725 form, $r=2.77\,s_w$, $R=2.77\sqrt{s_w^2+s_b^2}$) and tissue discrimination. $s_w$: within-recording SD (fitting and drift, KK-consistent recordings); $s_b$: between-recording component (upper 95\% bound in parentheses), for $R_{\infty}$ from the bipolar blood recordings transferred with the slope of Table~S35, for $Q$ and $\alpha$ from the single KK-consistent replicate pair (point estimates only indicative; bounds uninformative). Discrimination: absolute differences between tissue means of the 10 tissues with KK-consistent spectra (45 pairs) and the percentage of pairs exceeding $r$ and $R$.",
          'lccccl cccc'.replace(' ', ''), r"Parameter & $s_w$ & $r$ & $s_b$ & $R$ & Source of $s_b$ & \makecell{Median\\$|\Delta|$} & \makecell{$>r$\\(\%)} & \makecell{$>R$\\(\%)}", r35)
# S36 per animal
pa = PA.set_index('animal')
items = [('Spectra (KK-consistent)', lambda a: f"{int(pa.loc[a, 'spectra'])} ({int(pa.loc[a, 'kk_consistent'])})"),
         ('M12 success, MS-TRF / VP-MS-TRF (\\%)', lambda a: f"{pa.loc[a, 'M12_success_MS_TRF']:.0f} / {pa.loc[a, 'M12_success_VP_MS_TRF']:.0f}"),
         ('Spectra with VP gain / spectra that differ', lambda a: pa.loc[a, 'VP_gain_positive']),
         ('$\\log_{10}\\tau$ practically identifiable', lambda a: pa.loc[a, 'tau_identifiable']),
         ('Band criterion agrees with profile likelihood', lambda a: pa.loc[a, 'criterion_agreement']),
         ('Median $\\Gamma$', lambda a: f"{pa.loc[a, 'Gamma_median']:.1f}"),
         ('Median detection limit, nominal / calibrated (kHz)', lambda a: f"{pa.loc[a, 'fc_detect_median_hz']/1e3:.1f} / {pa.loc[a, 'fc_detect_cal_median_hz']/1e3:.1f}"),
         ('Closed calibrated intervals $Q$, $\\alpha$, $R_{\\infty}$ (KK, \\%)', lambda a: f"{100*pa.loc[a, 'kk_closed_Q']:.0f}, {100*pa.loc[a, 'kk_closed_alpha']:.0f}, {100*pa.loc[a, 'kk_closed_Rinf']:.0f}"),
         ('Median calibrated half-width $\\log_{10}Q$ / $\\alpha$ (KK)', lambda a: f"{pa.loc[a, 'median_halfwidth_Q']:.2f} / {pa.loc[a, 'median_halfwidth_alpha']:.3f}")]
r36 = [f"{lab} & {fn(1)} & {fn(2)}" for lab, fn in items]
r36.append('\\midrule\n\\multicolumn{3}{l}{\\emph{Artery, one KK-consistent scan per animal: second minus first animal}}')
for b in BO.itertuples():
    r36.append(f"{PN[b.param]}: difference (fitting SE) & \\multicolumn{{2}}{{c}}{{{b.diff_2_minus_1:+.3f} ({b.fit_se:.3f}); $r$ = {L.loc[b.param, 'r_limit']:.3f}}}")
t36 = tab(r"Main results recomputed per animal (animal, recording subset and date are confounded: first animal = subset A, second = subset D). The detection limit is the per-spectrum median and differs from the pooled value in the main text.",
          'lcc', r"Quantity & First animal & Second animal", r36)
open(out, 'w').write('\n'.join([t32, t33, t34a, t34b, t35, t36]))
print('written', out)


In [ ]:
%%writefile src/review/e13_stats.py
"""E13: statistical robustness without new data.
A. Cluster-aware inference. Spectra are nested in tissue-within-animal clusters (14 clusters; muscle x5, liver x2 and
   vein x2 share a cluster). For the main benchmark contrasts (per-spectrum success difference at 2,000 NFE over 25 runs),
   95% intervals from (i) resampling spectra, (ii) resampling clusters, (iii) two-stage resampling (clusters, then spectra
   within clusters) and (iv) clusters defined by tissue only (12); p-values from the exact sign-flip permutation test
   of cluster-mean differences; and a run-level linear-probability model with cluster-robust (CR1) standard errors.
   Coverage of calibrated intervals and the Spearman correlation of the detection limit with the upper f_c bound are
   given cluster-bootstrap intervals in the same way.
B. Endpoint hierarchy: four confirmatory hypotheses with Holm control
   H1 VP-MS-TRF > MS-TRF on M12 (superiority; one-sided permutation p)
   H2 ASHO-LSHADE vs L-SHADE on M12 within +-5 pp (equivalence; TOST with cluster bootstrap, p = max of the two one-sided)
   H3 ASHO-LSHADE+TRF vs L-SHADE+TRF on M12 within +-5 pp (equivalence)
   H4 calibrated > nominal coverage (empirical Sigma, pooled over Q, alpha, R_inf; sign-flip over clusters)
   All other analyses are estimation with intervals.
C. Robustness of the between-recording SDs (E12): leave-one-group-out, a MAD-based pooled SD, and exclusion of the
   largest group contribution.
D. Precision and sample size for a replicate study, from the estimated variance components.
Usage: python e13_stats.py"""
import itertools, json, math, numpy as np, pandas as pd
from scipy.stats import chi2, spearmanr, norm
import r2_core as R
T = R.T; rng = np.random.default_rng(13); B = 4000
def animal(sf): return 'A1' if '_2023' in sf else 'A2'
def clusters(df, how):
    return (df.tissue + '|' + df.source_file.map(animal)) if how == 'tissue_animal' else df.tissue

def boot_ci(per_spec, how, stat=np.mean, two_stage=False):
    """per_spec: DataFrame with columns value, cluster. Returns 95% percentile interval."""
    if how == 'spectrum':
        v = per_spec.value.to_numpy(); bs = [stat(rng.choice(v, len(v))) for _ in range(B)]
    else:
        g = {c: x.value.to_numpy() for c, x in per_spec.groupby('cluster')}; keys = list(g)
        bs = []
        for _ in range(B):
            pick = rng.choice(keys, len(keys)); vals = []
            for k in pick:
                x = g[k]; vals.append(rng.choice(x, len(x)) if two_stage else x)
            bs.append(stat(np.concatenate(vals)))
    return np.quantile(bs, [.025, .975])

def signflip(per_spec, alternative='greater'):
    m = per_spec.groupby('cluster').value.mean().to_numpy(); obs = m.mean(); n = len(m)
    if n <= 16:
        signs = np.array(list(itertools.product([-1, 1], repeat=n)))
    else:
        signs = rng.choice([-1, 1], size=(200000, n))
    null = (signs*m).mean(1)
    if alternative == 'greater': return float(np.mean(null >= obs - 1e-12))
    return float(np.mean(np.abs(null) >= abs(obs) - 1e-12))

def cr1(runs, xcol, ycol, ccol):
    """Linear probability model y ~ 1 + x, cluster-robust CR1 SE for the x coefficient."""
    X = np.column_stack([np.ones(len(runs)), runs[xcol].to_numpy(float)]); y = runs[ycol].to_numpy(float)
    XtXi = np.linalg.inv(X.T @ X); b = XtXi @ X.T @ y; e = y - X @ b; G = runs[ccol].to_numpy()
    meat = np.zeros((2, 2)); cl = np.unique(G)
    for c in cl:
        idx = G == c; u = X[idx].T @ e[idx]; meat += np.outer(u, u)
    n, k, g = len(y), 2, len(cl); V = XtXi @ meat @ XtXi*(g/(g - 1))*((n - 1)/(n - k))
    return float(b[1]), float(math.sqrt(V[1, 1])), g

# ---------------- A1/B: benchmark contrasts
CONTR = [('M12_DoubleColeL', 'VP-MS-TRF', 'MS-TRF', 'sup'), ('M12_DoubleColeL', 'VP-L-SHADE+TRF', 'L-SHADE+TRF', 'sup'),
         ('M10_ColeL', 'VP-L-SHADE', 'L-SHADE', 'sup'), ('M12_DoubleColeL', 'VP-L-SHADE', 'L-SHADE', 'sup'),
         ('M12_DoubleColeL', 'ASHO-LSHADE', 'L-SHADE', 'eq'), ('M12_DoubleColeL', 'ASHO-LSHADE+TRF', 'L-SHADE+TRF', 'eq'),
         ('M11_RCPEL', 'ASHO-LSHADE', 'L-SHADE', 'eq'), ('M10_ColeL', 'ASHO-LSHADE', 'L-SHADE', 'eq')]
rows = []; P = {}
for m, a, b, kind in CONTR:
    d = pd.read_csv(T/f'vp_merged_{m}.csv'); d = d[d.algorithm.isin([a, b])].copy(); d['y'] = d.succ_2000.astype(float)
    s = d.pivot_table(index=['source_file', 'tissue'], columns='algorithm', values='y', aggfunc='mean').reset_index()
    s['value'] = 100*(s[a] - s[b])
    out = dict(model=m.split('_')[0], contrast=f'{a} - {b}', kind=kind, spectra=len(s), estimate_pp=s.value.mean())
    for how, lab, two in [('spectrum', 'ci_spectrum', False), ('tissue_animal', 'ci_cluster', False), ('tissue_animal', 'ci_two_stage', True), ('tissue', 'ci_tissue', False)]:
        ps = s.assign(cluster=clusters(s, how)) if how != 'spectrum' else s
        lo, hi = boot_ci(ps, how, two_stage=two); out[lab] = f'{lo:.1f} to {hi:.1f}'; out[lab + '_lo'] = lo; out[lab + '_hi'] = hi
    ps = s.assign(cluster=clusters(s, 'tissue_animal')); out['clusters'] = ps.cluster.nunique()
    if kind == 'sup':
        out['p_signflip'] = signflip(ps, 'greater')
    else:   # TOST at +-5 pp using the two-stage cluster bootstrap distribution
        g = {c: x.value.to_numpy() for c, x in ps.groupby('cluster')}; keys = list(g); bs = []
        for _ in range(B):
            pick = rng.choice(keys, len(keys)); bs.append(np.concatenate([rng.choice(g[k], len(g[k])) for k in pick]).mean())
        bs = np.array(bs); p_lo = float(np.mean(bs <= -5)); p_hi = float(np.mean(bs >= 5)); out['p_TOST'] = max(p_lo, p_hi)
        out['ci90_two_stage'] = f'{np.quantile(bs, .05):.1f} to {np.quantile(bs, .95):.1f}'
    d['cluster'] = d.tissue + '|' + d.source_file.map(animal); d['x'] = (d.algorithm == a).astype(float)
    bcoef, se, g = cr1(d, 'x', 'y', 'cluster'); out['lpm_pp'] = 100*bcoef; out['lpm_ci'] = f'{100*(bcoef - 1.96*se):.1f} to {100*(bcoef + 1.96*se):.1f}'
    rows.append(out)
A1 = pd.DataFrame(rows)

# ---------------- A2: coverage with cluster bootstrap (12 KK-consistent spectra)
cov_rows = []; cov_spec = {}
for m in ['M11']:
    d = pd.read_csv(T/f'e1_coverage_runs_{m}.csv'); kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct
    tis = pd.read_csv(T/'Table_subset_mapping.csv').set_index('source_file').tissue
    d['nom'] = d.lr_nom <= 3.84; d['cal'] = d.lr_nom <= d.c_nom
    for cond in ['emp', 'rho_hi', 'ar1']:
        for q in ['log10_Q', 'alpha', 'log10_Rinf']:
            x = d[(d.cond == cond) & (d.param == q)]
            s = x.groupby('source_file')[['nom', 'cal']].mean().reset_index(); s['tissue'] = s.source_file.map(tis)
            r = dict(condition=cond, param=q, spectra=len(s), nominal=100*s.nom.mean(), calibrated=100*s.cal.mean())
            for lab, col in [('nominal', 'nom'), ('calibrated', 'cal')]:
                ps = s.assign(value=100*s[col], cluster=clusters(s, 'tissue_animal'))
                lo, hi = boot_ci(ps, 'cluster', two_stage=True); r[lab + '_ci_two_stage'] = f'{lo:.1f} to {hi:.1f}'
            cov_rows.append(r)
            if cond == 'emp': cov_spec[q] = s
A2 = pd.DataFrame(cov_rows)
# H4: calibrated minus nominal coverage, averaged over the three descriptors per spectrum, sign-flip over clusters
cs = pd.concat([v.assign(param=k) for k, v in cov_spec.items()]).groupby(['source_file', 'tissue'])[['nom', 'cal']].mean().reset_index()
cs['value'] = 100*(cs.cal - cs.nom); cs['cluster'] = clusters(cs, 'tissue_animal')
p_H4 = signflip(cs, 'greater'); H4_est = cs.value.mean(); H4_ci = boot_ci(cs, 'cluster', two_stage=True)

# ---------------- A3: Spearman detection limit vs f_c upper bound; measured Gamma
E = pd.read_csv(T/'Table_E10_measured.csv'); E['cluster'] = clusters(E, 'tissue_animal')
def sp(df): return spearmanr(df.fc_detect_hz, df.fc_ci_high_hz).statistic
g = {c: x for c, x in E.groupby('cluster')}; keys = list(g); bs_s, bs_g = [], []
for _ in range(B):
    pick = rng.choice(keys, len(keys)); df = pd.concat([g[k] for k in pick])
    if df.fc_detect_hz.nunique() > 2: bs_s.append(sp(df))
    bs_g.append(np.median(df.Gamma))
A3 = pd.DataFrame([dict(quantity='Spearman(detection limit, upper f_c bound)', estimate=sp(E), ci=f'{np.nanquantile(bs_s, .025):.2f} to {np.nanquantile(bs_s, .975):.2f}'),
                   dict(quantity='median Gamma (all spectra)', estimate=float(np.median(E.Gamma)), ci=f'{np.quantile(bs_g, .025):.1f} to {np.quantile(bs_g, .975):.1f}')])

# ---------------- B: Holm over the four confirmatory hypotheses
def get(c, m='M12'): return A1[(A1.contrast == c) & (A1.model == m)].iloc[0]
H = pd.DataFrame([dict(H='H1', hypothesis='VP-MS-TRF > MS-TRF, M12 success', estimate=f"{get('VP-MS-TRF - MS-TRF').estimate_pp:+.1f} pp", interval=get('VP-MS-TRF - MS-TRF').ci_two_stage, p=get('VP-MS-TRF - MS-TRF').p_signflip, test='exact sign-flip over clusters (one-sided)'),
                  dict(H='H2', hypothesis='ASHO-LSHADE equivalent to L-SHADE (+-5 pp), M12', estimate=f"{get('ASHO-LSHADE - L-SHADE').estimate_pp:+.1f} pp", interval=get('ASHO-LSHADE - L-SHADE').ci90_two_stage + ' (90%)', p=get('ASHO-LSHADE - L-SHADE').p_TOST, test='TOST, two-stage cluster bootstrap'),
                  dict(H='H3', hypothesis='ASHO-LSHADE+TRF equivalent to L-SHADE+TRF (+-5 pp), M12', estimate=f"{get('ASHO-LSHADE+TRF - L-SHADE+TRF').estimate_pp:+.1f} pp", interval=get('ASHO-LSHADE+TRF - L-SHADE+TRF').ci90_two_stage + ' (90%)', p=get('ASHO-LSHADE+TRF - L-SHADE+TRF').p_TOST, test='TOST, two-stage cluster bootstrap'),
                  dict(H='H4', hypothesis='calibrated > nominal coverage (empirical Sigma)', estimate=f'{H4_est:+.1f} pp', interval=f'{H4_ci[0]:.1f} to {H4_ci[1]:.1f}', p=p_H4, test='exact sign-flip over clusters (one-sided)')])
o = np.argsort(H.p.to_numpy()); adj = np.empty(4); run = 0
for rank, i in enumerate(o): run = max(run, min(1, (4 - rank)*H.p.iloc[i])); adj[i] = run
H['p_holm'] = adj; H['rejected_at_0.05'] = H.p_holm < 0.05

# ---------------- C: robustness of the between-recording SDs
M = pd.read_csv(T/'e12_recordings.csv')
def groups_of(sel):
    out = {}
    for k, g in M[sel].groupby(['animal', 'probe', 'medium_g', 'f_khz']):
        if len(g) > 1: out[k] = g.logR.to_numpy()
    return out
def pooled(gs):
    nu = sum(len(x) - 1 for x in gs); return math.sqrt(sum((len(x) - 1)*np.var(x, ddof=1) for x in gs)/nu), nu
def mad_pooled(gs):   # pooled scaled MAD about group medians, with a small-sample consistency factor per group size
    dev = np.concatenate([np.abs(x - np.median(x)) for x in gs]); n = np.concatenate([[len(x)]*len(x) for x in gs])
    k = np.where(n == 2, 1/0.6745*1.0, 1.4826)   # for pairs |x - median| = |diff|/2; scaled so E = sigma approx
    return float(np.median(dev*k*np.sqrt(n/(n - 1))))
sw = pd.read_csv(T/'Table_E12_summary.csv'); s_within = float(sw[sw.set.str.contains('KK-consistent')].between_sd.iloc[0])
crow = []
for lab, sel in [('blood, bipolar', (M.medium == 'blood') & (M.probe != 'monopolar')), ('tissue, scan probe', (M.medium != 'blood') & (M.probe == 'yellow')),
                 ('all same-probe groups', M.medium.notna())]:
    G = groups_of(sel); s, nu = pooled(list(G.values())); loo = []
    for k in G:
        rest = [v for kk, v in G.items() if kk != k]
        if sum(len(v) - 1 for v in rest) > 0: loo.append((k, pooled(rest)[0]))
    smin = min(loo, key=lambda t: t[1]); smax = max(loo, key=lambda t: t[1])
    crow.append(dict(set=lab, groups=len(G), df=nu, pooled_sd=s, ratio_to_within=s/s_within, loo_min=smin[1], loo_min_dropped='/'.join(map(str, smin[0])),
                     loo_max=smax[1], loo_ratio_range=f'{smin[1]/s_within:.1f} to {smax[1]/s_within:.1f}', mad_pooled=mad_pooled(list(G.values())),
                     mad_ratio=mad_pooled(list(G.values()))/s_within))
C = pd.DataFrame(crow)

# ---------------- D: precision and sample size for a replicate study
L = pd.read_csv(T/'Table_E12_limits.csv').set_index('param'); DI = pd.read_csv(T/'Table_E12_discrimination.csv').set_index('param')
I = pd.read_csv(T/'e1_intervals_M11.csv'); I = I[(I.objective == 'nom') & (I.threshold == 'cal') & I.kk_ok]
tm = I.groupby(['tissue', 'param']).estimate.mean().unstack()
drow = []
for nu in [1, 2, 5, 10, 20, 30, 50, 100]:
    drow.append(dict(df=nu, sd_ci_low_factor=math.sqrt(nu/chi2.ppf(.975, nu)), sd_ci_high_factor=math.sqrt(nu/chi2.ppf(.025, nu))))
D1 = pd.DataFrame(drow)
need = next(r.df for r in D1.itertuples() if r.sd_ci_high_factor <= 1.5)
drow2 = []; sw_R, sb_R = L.loc['log10_Rinf', 's_within'], L.loc['log10_Rinf', 's_between']
dif = np.array([abs(tm.log10_Rinf[a] - tm.log10_Rinf[b]) for a, b in itertools.combinations(tm.log10_Rinf.dropna().index, 2)])
for n in [1, 2, 3, 4, 6, 8]:
    lim = 2.77*math.sqrt((sw_R**2 + sb_R**2)/n)
    drow2.append(dict(placements_per_tissue=n, R_limit_mean=lim, pct_pairs_exceeding=100*np.mean(dif > lim)))
D2 = pd.DataFrame(drow2)

for name, df in [('Table_E13_contrasts', A1), ('Table_E13_coverage', A2), ('Table_E13_corr', A3), ('Table_E13_hypotheses', H), ('Table_E13_robust_between', C),
                 ('Table_E13_precision_df', D1), ('Table_E13_placements', D2)]:
    df.to_csv(T/f'{name}.csv', index=False)
json.dump(dict(df_for_upper_factor_1p5=int(need), H4_estimate=H4_est, H4_ci=list(H4_ci), s_within_R100=s_within), open(T/'E13_misc.json', 'w'), indent=1)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
for name, df in [('contrasts', A1.drop(columns=[c for c in A1.columns if c.endswith('_lo') or c.endswith('_hi')])), ('coverage', A2), ('corr', A3), ('hypotheses', H),
                 ('robust between', C), ('precision', D1), ('placements', D2)]:
    print('\n==', name); print(df.round(4).to_string())
print('df needed for upper CI factor <= 1.5:', need)


In [ ]:
%%writefile src/review/make_supp_e13.py
"""Tables S38-S40 (cluster-aware inference and confirmatory hypotheses; robustness of the between-recording estimates;
precision and sample size) from the outputs of e13_stats.py."""
import sys, json, pandas as pd
import r2_core as R
T = R.T; out = sys.argv[1] if len(sys.argv) > 1 else 'supp_e13.tex'
A = pd.read_csv(T/'Table_E13_contrasts.csv'); H = pd.read_csv(T/'Table_E13_hypotheses.csv'); C = pd.read_csv(T/'Table_E13_coverage.csv')
K = pd.read_csv(T/'Table_E13_corr.csv'); RB = pd.read_csv(T/'Table_E13_robust_between.csv'); D1 = pd.read_csv(T/'Table_E13_precision_df.csv')
D2 = pd.read_csv(T/'Table_E13_placements.csv'); misc = json.load(open(T/'E13_misc.json'))
PN = {'log10_Q': r'$\log_{10}Q$', 'alpha': r'$\alpha$', 'log10_Rinf': r'$\log_{10}R_{\infty}$'}
CN = {'emp': 'empirical $\\Sigma$', 'rho_hi': '$\\rho_f+0.2$', 'ar1': 'AR(1)'}
def tab(cap, spec, head, rows):
    return ("\\begin{table}[H]\n\\centering\n\\caption{" + cap + "}\n\\footnotesize\n\\setlength{\\tabcolsep}{3.5pt}\n\\begin{tabular}{" + spec + "}\n\\toprule\n"
            + head + " \\\\\n\\midrule\n" + " \\\\\n".join(rows) + " \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
fp = lambda p: '$<10^{-4}$' if p < 1e-4 else (f'{p:.4f}' if p < 0.01 else f'{p:.3f}')
r = []
for h in H.itertuples():
    r.append(f"{h.H} & {h.hypothesis.replace('+-', '$\\pm$').replace('Sigma', '$\\Sigma$')} & {h.estimate} & {h.interval.replace('%', r'\%')} & {fp(h.p)} & {fp(h.p_holm)} & {'yes' if h._8 else 'no'}")
r.append('\\midrule\n\\multicolumn{7}{l}{\\emph{All contrasts: per-spectrum success difference (pp), 95\\% intervals by resampling unit}}')
r.append('Model & Contrast & Estimate & Spectra & Clusters (14) & Two-stage & Cluster-robust LPM')
for a in A.itertuples():
    r.append(f"{a.model} & {a.contrast} & {a.estimate_pp:+.1f} & {a.ci_spectrum} & {a.ci_cluster} & {a.ci_two_stage} & {a.lpm_ci}")
t38 = tab(r"Cluster-aware inference for the benchmark. Upper block: the four confirmatory hypotheses with Holm-adjusted p-values (H1, H4: exact one-sided sign-flip permutation of the means of the 14 tissue-within-animal clusters; H2, H3: two one-sided tests at $\pm5$ percentage points with the two-stage cluster bootstrap, 90\% interval shown; H4 averages $\log_{10}Q$, $\alpha$ and $\log_{10}R_{\infty}$ under the empirical covariance). Lower block: 95\% intervals of all benchmark contrasts at 2,000 NFE when resampling spectra, clusters, or clusters and then spectra within clusters, and from a run-level linear probability model with cluster-robust (CR1) standard errors. Clustering by tissue only (12 clusters) gave similar intervals (results files).",
          'lp{4.2cm}ccccc', r"H & Hypothesis & Estimate & Interval & $p$ & $p_{\mathrm{Holm}}$ & Supported", r)
t38 = t38.replace('{lp{4.2cm}ccccc}', '{llccccc}').replace('H & Hypothesis', 'H & Hypothesis')
# fix the column header of the lower block (7 columns)
r39 = []
for c in C.itertuples():
    r39.append(f"{CN[c.condition]} & {PN[c.param]} & {c.nominal:.1f} ({c.nominal_ci_two_stage}) & {c.calibrated:.1f} ({c.calibrated_ci_two_stage})")
r39.append('\\midrule\n' + f"\\multicolumn{{2}}{{l}}{{Spearman, detection limit vs upper $f_c$ bound}} & \\multicolumn{{2}}{{l}}{{{K.estimate[0]:.2f} ({K.ci[0]})}}")
r39.append(f"\\multicolumn{{2}}{{l}}{{Median $\\Gamma$, measured spectra}} & \\multicolumn{{2}}{{l}}{{{K.estimate[1]:.1f} ({K.ci[1]})}}")
t39a = tab(r"Coverage (\%) of nominal and calibrated 95\% profile intervals (12 KK-consistent spectra $\times$ 100 replicates) and measured-spectrum statistics, with two-stage cluster-bootstrap 95\% intervals (clusters: tissue within animal).",
           'llcc', r"Condition & Parameter & Nominal & Calibrated", r39)
r39b = []
for b in RB.itertuples():
    r39b.append(f"{b.set} & {b.groups} & {b.df} & {b.pooled_sd:.3f} & {b.ratio_to_within:.1f} & {b.loo_min:.3f}--{b.loo_max:.3f} & {b.loo_ratio_range.replace(' to ', '--')} & {b.mad_pooled:.3f} & {b.mad_ratio:.1f}")
t39b = tab(r"Robustness of the between-recording SD of $\log_{10}R$ (Table~S32): classical pooled SD, range when one group is left out, and a pooled median-absolute-deviation (MAD) estimate; ratios are relative to the between-sweep SD within KK-consistent recordings (" + f"{misc['s_within_R100']:.3f}" + r" decade). Between-recording variability exceeds within-recording variability in every analysis, but its size depends on a few discordant recordings; on the $R_{\infty}$ scale the reproducibility limit is 0.37 decade with the classical and 0.15 decade with the MAD estimate.",
           'lcccccccc', r"Set & Groups & df & SD & Ratio & \makecell{SD, leave one\\group out} & Ratio & \makecell{MAD\\SD} & Ratio", r39b)
r40 = [f"{d.df} & {d.sd_ci_low_factor:.2f} & {d.sd_ci_high_factor:.2f}" for d in D1.itertuples()]
t40a = tab(r"Precision of an estimated standard deviation: 95\% $\chi^2$ interval as a multiple of the estimate, by degrees of freedom. An upper limit within 1.5 times the estimate requires about " + f"{misc['df_for_upper_factor_1p5']}" + r" degrees of freedom, for example ten tissues with three placements each; the present KK-consistent replicate data for $Q$ and $\alpha$ have one.",
           'ccc', r"df & Lower factor & Upper factor", r40)
r40b = [f"{d.placements_per_tissue} & {d.R_limit_mean:.3f} & {d.pct_pairs_exceeding:.0f}" for d in D2.itertuples()]
t40b = tab(r"Planning a replicate study: reproducibility limit of tissue means of $\log_{10}R_{\infty}$ when $n$ placements per tissue are averaged, $2.77\sqrt{(s_w^2+s_b^2)/n}$ with the classical estimates of Table~S36, and the percentage of the 45 observed tissue pairs (KK-consistent spectra) whose difference exceeds it.",
           'ccc', r"Placements per tissue $n$ & Limit (decade) & Pairs exceeding (\%)", r40b)
open(out, 'w').write('\n'.join([t38, t39b, t40b, t39a, t40a])); print('written', out)


In [ ]:
%%writefile src/review/e14_kk_primary.py
"""E14: reviewer round 3.
(a) Physical/model-selection analyses with the 12 KK-consistent spectra as the primary set and all 20 as robustness:
    11-model screening (AICc/BIC wins, ranks), M10 vs M11 (AICc nominal/GLS on three bands, bootstrap LR, leave-one-
    frequency-out), high-frequency-correction sensitivity, M11 descriptors, sign of the fitted delay.
(b) Error model estimated from the sweeps of KK-consistent recordings only (residual-based, as in E1) and its effect on
    calibration: critical values c* recomputed under Sigma_KK and coverage under Sigma_KK of intervals calibrated with
    Sigma_KK and with the published Sigma (all 51 sweeps), 12 KK-consistent spectra x 100 replicates.
(c) Diagnosis of the closed-form width in the weak-identifiability regime (Table S30): share of closed profiles by r.
Usage: python e14_kk_primary.py RAW_DIR"""
import sys, json, math, time, numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.stats import wilcoxon
import r2_core as R
T = R.T; raw = sys.argv[1]
kk = pd.read_csv(T/'Table_KK_check.csv').set_index('source_file').kk_rms_pct < 3
# ---------------- (a) model selection and physical summaries
E = pd.read_csv(T/'ecm_all_models.csv'); E['kk'] = E.source_file.map(kk)
scr = []
for lab, d in [('KK-consistent (12)', E[E.kk]), ('all (20)', E)]:
    d = d.copy(); d['rA'] = d.groupby('source_file').AICc.rank(); d['rB'] = d.groupby('source_file').BIC.rank()
    for m, g in d.groupby('model'):
        scr.append(dict(set=lab, model=m, k=int(g.k.iloc[0]), median_NRMSE=g.NRMSE.median(), mean_AICc_rank=g.rA.mean(), AICc_wins=int((g.rA == 1).sum()),
                        BIC_wins=int((g.rB == 1).sum())))
SCR = pd.DataFrame(scr).sort_values(['set', 'mean_AICc_rank'])
M = pd.read_csv(T/'e3_model_selection_runs.csv')
ms = []
for lab, d in [('KK-consistent (12)', M[M.kk_ok]), ('all (20)', M)]:
    r = dict(set=lab, spectra=len(d))
    for obj in ['nom', 'gls']:
        for band in ['10-100', '10-80', '20-100']: r[f'M11_pref_AICc_{obj}_{band}'] = int((d[f'dAICc_{obj}_{band}'] > 0).sum())
        r[f'bootLR_reject_{obj}'] = int((d[f'p_boot_{obj}'] < 0.05).sum()); r[f'median_p_boot_{obj}'] = d[f'p_boot_{obj}'].median()
    for part in ['int', 'end']:
        a, b = d[f'lofo_{part}_M10'], d[f'lofo_{part}_M11']; r[f'lofo_{part}_M11_better'] = int((b < a).sum()); r[f'lofo_{part}_p'] = wilcoxon(b, a).pvalue
    r['median_dAICc_nom'] = d['dAICc_nom_10-100'].median(); r['median_dAICc_gls'] = d['dAICc_gls_10-100'].median()
    ms.append(r)
MS = pd.DataFrame(ms)
H = pd.read_csv(T/'e2_hf_runs.csv'); hf = []
for lab, d in [('KK-consistent (12)', H[H.kk_ok]), ('all (20)', H)]:
    for t, g in d.groupby('treatment'):
        hf.append(dict(set=lab, treatment=t, model=g.model.iloc[0], NRMSE=g.NRMSE.median(), abs_d_log10_Rinf=g.d_log10_Rinf.abs().median(),
                       abs_d_log10_Q=g.d_log10_Q.abs().median(), abs_d_alpha=g.d_alpha.abs().median(), extra_median=g.extra.median(),
                       extra_q25=g.extra.quantile(.25), extra_q75=g.extra.quantile(.75)))
HF = pd.DataFrame(hf)
P = pd.read_csv(T/'Table_M11_physical_parameters.csv'); P['kk'] = P.source_file.map(kk); de = []
for lab, d in [('KK-consistent (12)', P[P.kk]), ('all (20)', P)]:
    de.append(dict(set=lab, Rinf_ohm=d.Rinf_ohm.median(), Q=d.Q.median(), alpha=d.alpha.median(), L_mH=1e3*d.L_H.median(), NRMSE=d.NRMSE.median(),
                   Rinf_range=f'{d.Rinf_ohm.min():.0f}-{d.Rinf_ohm.max():.0f}', alpha_range=f'{d.alpha.min():.2f}-{d.alpha.max():.2f}'))
DE = pd.DataFrame(de)

# ---------------- (b) error model from KK-consistent recordings only
SP = {s['sf']: s for s in R.spectra()}; SW = R.sweeps(raw)
res_all, kk_of = [], []
for k, s in SP.items():
    for z in SW[k]:
        F = R.fit('rcpe_l', s['f'], z); res_all.append(R.VP('rcpe_l', s['f'], z).residual(F['eta'])); kk_of.append(s['kk_ok'])
res_all = np.array(res_all); kk_of = np.array(kk_of)
def fit_psi(S, df):
    def nll(u):
        C = R.sigma_struct([math.exp(u[0]), u[1], math.tanh(u[2]), math.tanh(u[3]), math.exp(u[4])]); sgn, ld = np.linalg.slogdet(C)
        return 0.5*df*(ld + np.trace(np.linalg.solve(C, S))) if sgn > 0 else 1e12
    best = None
    for st in [[math.log(0.01), 0, 0.0, 0, 0], [math.log(0.02), 0.5, 0.8, 0.3, 0], [math.log(0.005), -0.5, -0.5, -0.3, 0.3]]:
        r = minimize(nll, st, method='Nelder-Mead', options=dict(maxiter=20000, xatol=1e-7, fatol=1e-10))
        if best is None or r.fun < best.fun: best = r
    u = best.x; return np.array([math.exp(u[0]), u[1], math.tanh(u[2]), math.tanh(u[3]), math.exp(u[4])])
def S_of(idx): Rm = res_all[idx]; return Rm.T @ Rm/len(idx)
psi_all = fit_psi(S_of(np.arange(len(res_all))), len(res_all)); psi_kk = fit_psi(S_of(np.flatnonzero(kk_of)), int(kk_of.sum()))
psi_pub = np.array(json.load(open(T/'E1_psi.json'))['psi'])
EMK = pd.DataFrame(dict(parameter=['sigma0', 'gamma', 'rho_f', 'rho_ReIm', 'ratio_Im_Re'], published_all_51=psi_pub, refit_all_51=psi_all, kk_only=psi_kk))
EMK.loc[len(EMK)] = ['sweeps', len(res_all), len(res_all), int(kk_of.sum())]
S_kk = R.sigma_struct(psi_kk); S_pub = R.sigma_struct(psi_pub)
def scaled(S, rss, p=4): return S*(rss/(np.trace(S)*(1 - p/S.shape[0])))
pars = ['log10_Q', 'alpha', 'log10_Rinf']
def lr_nom(f, z, tv):
    rmin = R.fit('rcpe_l', f, z)['rss']; return {pn: R.lr_at('rcpe_l', f, z, pn, tv[pn], None, rmin) for pn in pars}
cst = pd.read_csv(T/'e1_cstar_M11.csv'); cst = cst[cst.objective == 'nom'].set_index(['source_file', 'param']).c_star
cov = []
for si, (k, s) in enumerate(SP.items()):
    if not s['kk_ok']: continue
    t0 = time.time(); F = R.fit('rcpe_l', s['f'], s['Z']); p = F['params']; zt = R.predict('rcpe_l', p, s['f'])
    tv = {'log10_Q': math.log10(p['Q']), 'alpha': p['alpha'], 'log10_Rinf': math.log10(p['Rinf'])}
    Lc = np.linalg.cholesky(scaled(S_kk, F['rss'])); rng = np.random.default_rng(14000 + si)
    boot = [lr_nom(s['f'], R.noise_draw(rng, Lc, zt), tv) for _ in range(200)]
    c_kk = {pn: float(np.quantile([b[pn] for b in boot], .95)) for pn in pars}
    for r in range(100):
        lr = lr_nom(s['f'], R.noise_draw(rng, Lc, zt), tv)
        for pn in pars:
            cov.append(dict(source_file=k, tissue=s['tissue'], rep=r, param=pn, lr=lr[pn], c_kk=c_kk[pn], c_pub=cst[(k, pn)]))
    print(time.strftime('%H:%M:%S'), si, s['tissue'], f'{time.time() - t0:.0f}s', flush=True)
CV = pd.DataFrame(cov); CV.to_csv(T/'e14_coverage_kkSigma.csv', index=False)
COV = CV.groupby('param').apply(lambda g: pd.Series(dict(nominal=100*(g.lr <= 3.84).mean(), calibrated_with_Sigma_KK=100*(g.lr <= g.c_kk).mean(),
        calibrated_with_published_Sigma=100*(g.lr <= g.c_pub).mean(), median_c_kk=g.groupby('source_file').c_kk.first().median(),
        median_c_pub=g.groupby('source_file').c_pub.first().median())), include_groups=False).reset_index()

# ---------------- (c) weak-identifiability diagnosis
B = pd.read_csv(T/'e10b_refined.csv'); B['rbin'] = pd.cut(B.rmax, [0, .1, .3, .6, 1, 100], labels=['<0.1', '0.1-0.3', '0.3-0.6', '0.6-1', '>1'])
WK = B.groupby('rbin', observed=True).apply(lambda h: pd.Series(dict(spectra=len(h), pct_closed=100*h.closed.mean(),
        median_W_asym_all=h.W_asym.median(), median_W_wald_all=h.W_wald.median(), pct_pred_nonident=100*(h.W_asym >= 1).mean(),
        pct_obs_nonident=100*(~h.ident_obs.astype(bool)).mean(), agreement=100*((h.W_asym < 1) == h.ident_obs.astype(bool)).mean())), include_groups=False).reset_index()

for name, df in [('Table_E14_screening', SCR), ('Table_E14_model_selection', MS), ('Table_E14_hf', HF), ('Table_E14_descriptors', DE),
                 ('Table_E14_error_model_kk', EMK), ('Table_E14_coverage_kk', COV), ('Table_E14_weak_ident', WK)]:
    df.to_csv(T/f'{name}.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
for name, df in [('screening', SCR[SCR.mean_AICc_rank < 6]), ('model selection', MS.T), ('hf', HF), ('descriptors', DE), ('error model', EMK), ('coverage', COV), ('weak', WK)]:
    print('\n==', name); print(df.round(4).to_string())


In [ ]:
%%writefile src/review/make_supp_e14.py
"""Tables S45-S47 (KK-consistent primary physical analysis; error model from KK-consistent recordings and calibration
sensitivity; closed-form criterion in the weak-identifiability regime) from the outputs of e14_kk_primary.py."""
import sys, pandas as pd
import r2_core as R
T = R.T; out = sys.argv[1] if len(sys.argv) > 1 else 'supp_e14.tex'
SCR = pd.read_csv(T/'Table_E14_screening.csv'); MS = pd.read_csv(T/'Table_E14_model_selection.csv').set_index('set'); HF = pd.read_csv(T/'Table_E14_hf.csv')
DE = pd.read_csv(T/'Table_E14_descriptors.csv').set_index('set'); EM = pd.read_csv(T/'Table_E14_error_model_kk.csv'); COV = pd.read_csv(T/'Table_E14_coverage_kk.csv')
WK = pd.read_csv(T/'Table_E14_weak_ident.csv')
K, A = 'KK-consistent (12)', 'all (20)'
def tab(cap, spec, head, rows):
    return ("\\begin{table}[H]\n\\centering\n\\caption{" + cap + "}\n\\footnotesize\n\\setlength{\\tabcolsep}{4pt}\n\\begin{tabular}{" + spec + "}\n\\toprule\n"
            + head + " \\\\\n\\midrule\n" + " \\\\\n".join(rows) + " \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
NM = {'M11_RCPEL': 'M11 R--CPE--L', 'M10_ColeL': 'M10 Cole+L', 'M2_FrickeMorse': 'M2 Fricke--Morse', 'M6_FrickeCPE': 'M6 Fricke--CPE', 'M4_RandlesCPE': 'M4 Randles--CPE'}
r = ['\\multicolumn{5}{l}{\\emph{(a) Screening, five best models}} & \\multicolumn{3}{l}{}']
k5 = SCR[SCR.set == K].sort_values('mean_AICc_rank').head(5)
for x in k5.itertuples():
    y = SCR[(SCR.set == A) & (SCR.model == x.model)].iloc[0]
    r.append(f"{NM.get(x.model, x.model)} & {x.k} & {x.median_NRMSE:.4f} & {x.AICc_wins} / {x.BIC_wins} & {x.mean_AICc_rank:.2f} & {y.median_NRMSE:.4f} & {y.AICc_wins} / {y.BIC_wins} & {y.mean_AICc_rank:.2f}")
r.append('\\midrule\n\\multicolumn{8}{l}{\\emph{(b) M10 versus M11 (number of spectra; KK-consistent / all)}}')
mk, ma = MS.loc[K], MS.loc[A]
for lab, col in [('M11 preferred, AICc, independent, 10--100 kHz', 'M11_pref_AICc_nom_10-100'), ('M11 preferred, AICc, independent, 10--80 / 20--100 kHz', None),
                 ('M11 preferred, AICc, GLS, 10--100 kHz', 'M11_pref_AICc_gls_10-100'), ('Bootstrap LR rejects M11 ($p<0.05$), independent / GLS', 'boot'),
                 ('Median $\\Delta$AICc (M10 $-$ M11), independent / GLS', 'dA'), ('LOFO interior: M11 better (Wilcoxon $p$)', 'lofo_int'), ('LOFO band edges: M11 better (Wilcoxon $p$)', 'lofo_end')]:
    if col is None: v = lambda m: f"{int(m['M11_pref_AICc_nom_10-80'])} / {int(m['M11_pref_AICc_nom_20-100'])}"
    elif col == 'boot': v = lambda m: f"{int(m['bootLR_reject_nom'])} / {int(m['bootLR_reject_gls'])}"
    elif col == 'dA': v = lambda m: f"{float(m['median_dAICc_nom']):.2f} / {float(m['median_dAICc_gls']):.2f}"
    elif col.startswith('lofo'): v = lambda m, c=col: f"{int(m[c + '_M11_better'])} ({float(m[c + '_p']):.2f})"
    else: v = lambda m, c=col: f"{int(m[c])}"
    r.append(f"\\multicolumn{{4}}{{l}}{{{lab}}} & \\multicolumn{{2}}{{c}}{{{v(mk)}}} & \\multicolumn{{2}}{{c}}{{{v(ma)}}}")
r.append('\\midrule\n\\multicolumn{8}{l}{\\emph{(c) M11 descriptors, median (KK-consistent / all)}}')
dk, da = DE.loc[K], DE.loc[A]
r.append(f"\\multicolumn{{4}}{{l}}{{$R_\\infty$ ($\\Omega$); $Q$ (S\\,s$^\\alpha$); $\\alpha$; $L$ (mH)}} & \\multicolumn{{4}}{{l}}{{{dk.Rinf_ohm:.0f} / {da.Rinf_ohm:.0f}; 7.4 / 7.1 $\\times10^{{-8}}$; {dk.alpha:.2f} / {da.alpha:.2f}; {dk.L_mH:.2f} / {da.L_mH:.2f}}}")
t45 = tab(r"Circuit representation with the 12 KK-consistent spectra as the primary set and all 20 spectra as a robustness analysis. (a) Weighted-AICc screening: median NRMSE, AICc/BIC wins and mean AICc rank. (b) M10 versus M11. (c) Descriptors. In the KK-consistent spectra, M10 predicted the band-edge frequencies better than M11 in 10 of 12 spectra (Wilcoxon $p=0.05$), a hint that the Cole model extrapolates slightly better although it does not fit the band better.",
          'lccccccc', r"Model & $k$ & \multicolumn{3}{c}{KK-consistent (12): NRMSE, AICc/BIC wins, rank} & \multicolumn{3}{c}{All (20)}", r)
TN = {'T0': 'T0 series $L$', 'T1': 'T1 none, 10--80 kHz', 'T1b': 'T1b $L$, 10--80 kHz', 'T2': 'T2 linear phase', 'T3': 'T3 phase offset', 'T4': 'T4 $K(j\\omega)^\\nu$'}
rh = []
for t in ['T0', 'T1', 'T1b', 'T2', 'T3', 'T4']:
    a = HF[(HF.set == K) & (HF.treatment == t)].iloc[0]; b = HF[(HF.set == A) & (HF.treatment == t)].iloc[0]
    ex = '--' if pd.isna(a.extra_median) else f"{a.extra_median:.3g}"
    rh.append(f"{TN[t]} & {a.NRMSE:.3f} & {a.abs_d_log10_Rinf:.3f} & {a.abs_d_log10_Q:.3f} & {a.abs_d_alpha:.3f} & {ex} & {b.abs_d_log10_Q:.3f} & {b.abs_d_alpha:.3f}")
t45b = tab(r"High-frequency-correction sensitivity in the 12 KK-consistent spectra: median NRMSE and median absolute change of the M11 descriptors relative to T0; extra parameter: $L$ (H), $t_d$ ($\mu$s), $\varphi_0$ (rad) or $\nu$. Last two columns: all 20 spectra.",
           'lccccccc', r"Treatment & NRMSE & $|\Delta\log_{10}R_\infty|$ & $|\Delta\log_{10}Q|$ & $|\Delta\alpha|$ & Extra & \multicolumn{2}{c}{All 20: $|\Delta\log_{10}Q|$, $|\Delta\alpha|$}", rh)
re_ = []
for x in EM.itertuples():
    if x.parameter == 'sweeps': re_.append(f"Sweeps (recordings) & 51 (20) & 27 (12)"); continue
    re_.append(f"{ {'sigma0': '$\\sigma_0$', 'gamma': '$\\gamma$', 'rho_f': '$\\rho_f$', 'rho_ReIm': '$\\rho_{\\mathrm{RI}}$', 'ratio_Im_Re': '$\\kappa$'}[x.parameter]} & {x.published_all_51:.3f} & {x.kk_only:.3f}")
t46a = tab(r"Residual-based error model estimated from all 51 sweeps (published; used for calibration in the main text) and from the 27 sweeps of the KK-consistent recordings only. The KK-only model is close to independent errors with a larger imaginary component; the correlated structure of the published model comes mainly from KK-inconsistent recordings. $\sigma_0$ is immaterial for calibration because the covariance is rescaled to the observed RSS of each spectrum.",
            'lcc', r"Parameter & All 51 sweeps & KK-consistent only", re_)
PN = {'log10_Q': r'$\log_{10}Q$', 'alpha': r'$\alpha$', 'log10_Rinf': r'$\log_{10}R_{\infty}$'}
rc = ['\\multicolumn{6}{l}{\\emph{(a) Errors drawn from the KK-only model}}'] + [f"{PN[x.param]} & {x.nominal:.1f} & {x.calibrated_with_Sigma_KK:.1f} & {x.calibrated_with_published_Sigma:.1f} & {x.median_c_kk:.2f} & {x.median_c_pub:.2f}" for x in COV.itertuples()]
cv = pd.read_csv(T/'e1_coverage_runs_M11.csv'); cv = cv[cv.cond == 'emp']; ck = pd.read_csv(T/'e14_coverage_kkSigma.csv').groupby(['source_file', 'param']).c_kk.first()
cv['c_kk'] = [ck[(a, b)] for a, b in zip(cv.source_file, cv.param)]
rc.append('\\midrule\n\\multicolumn{6}{l}{\\emph{(b) Errors drawn from the published model (all 51 sweeps)}}')
for q in ['alpha', 'log10_Q', 'log10_Rinf']:
    g = cv[cv.param == q]; rc.append(f"{PN[q]} & {100*(g.lr_nom <= 3.84).mean():.1f} & {100*(g.lr_nom <= g.c_kk).mean():.1f} & {100*(g.lr_nom <= g.c_nom).mean():.1f} & & ")
t46b = tab(r"Coverage (\%) of nominal intervals, of intervals calibrated with the KK-only error model and of the published calibrated intervals (critical values from the model estimated from all 51 sweeps); 12 KK-consistent spectra $\times$ 100 replicates. (a) Errors drawn from the KK-only model: nominal intervals are nearly adequate and the published calibration is conservative. (b) The reverse case: errors from the published model. The choice is asymmetric: intervals calibrated with the published model stay at or above the nominal level under either error structure (93--99\%), whereas intervals calibrated with the KK-only model under-cover when the errors are correlated (78--89\%). The size of the nominal under-coverage reported in the main text is conditional on the error model.",
            'lccccc', r"Parameter & Nominal & \makecell{Calibrated,\\KK-only $\Sigma$} & \makecell{Calibrated,\\published $\Sigma$} & \makecell{Median $c^*$\\KK-only} & \makecell{Median $c^*$\\published}", rc)
rw = [f"{x.rbin} & {int(x.spectra)} & {x.pct_closed:.0f} & {x.median_W_asym_all:.2f} & {x.median_W_wald_all:.2f} & {x.pct_pred_nonident:.0f} & {x.pct_obs_nonident:.0f} & {x.agreement:.0f}" for x in WK.itertuples()]
t47 = tab(r"Closed-form criterion by $r=(f_{\min}/f_c)^{-\alpha}$ for all band-design spectra (white noise, correlated noise and linear grid), complementing Table~S30, which is restricted to closed profiles. For $r<0.1$ only 7\% of profiles closed; in that regime the predicted first-order and exact linearized (Wald) widths are many decades and correctly classify $\log_{10}\tau$ as not practically identifiable (agreement 97\%), whereas the few profiles that happened to close are much narrower than any local (Fisher) approximation, because the likelihood is far from quadratic and the closed profiles are a selected subset. The width ratios of Table~S30 for $r<0.1$ therefore reflect non-local profiles, not a failure of the expansion (whose refined form equals the Wald width). Classification errors concentrate in the transition range $r=0.3$--0.6.",
          'lccccccc', r"$r$ & Spectra & \makecell{Closed\\(\%)} & \makecell{Median first-\\order width} & \makecell{Median Wald\\width} & \makecell{Predicted not\\ident. (\%)} & \makecell{Profile not\\ident. (\%)} & \makecell{Agreement\\(\%)}", rw)
open(out, 'w').write('\n'.join([t45, t45b, t46a, t46b, t47])); print('written', out)


In [ ]:
%%writefile src/make_fig1_workflow_v3.py
"""Simplified study workflow (manuscript Fig. 1, round 3): six stages, print-size fonts."""
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path
plt.rcParams.update({'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold'})
fig = plt.figure(figsize=(7.1, 3.3)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 46); ax.axis('off')
HL, HE = '#e8f1fb', '#1f5f9e'
def stage(x, y, n, title, lines, hl=False):
    w, h = 30, 18.5
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.2', fc=HL if hl else 'white', ec=HE if hl else 'black', lw=1.4 if hl else 1.0))
    ax.text(x + 1.6, y + h - 2.0, f'{n}', fontsize=11, fontweight='bold', color=HE, va='top')
    ax.text(x + 5.0, y + h - 2.0, title, fontsize=9.5, fontweight='bold', va='top')
    ax.text(x + 1.8, y + h - 6.2, '\n'.join(lines), fontsize=8.2, va='top', linespacing=1.4)
def arr(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='-|>', lw=1.1, color='black', mutation_scale=11))
xs = [2, 35, 68]; yt, yb = 25.5, 1.5
stage(xs[0], yt, 1, 'Data and quality control', ['20 in vivo spectra, 10–100 kHz', '51 repeated sweeps', 'Kramers–Kronig check:', '12 consistent, 8 not'])
stage(xs[1], yt, 2, 'Circuit representation', ['11 candidate circuits (AICc, BIC)', 'R–CPE–L (M11), Cole+L (M10),', 'double Cole+L (M12)', 'High-frequency correction variants'])
stage(xs[2], yt, 3, 'Inversion', ['Original vs variable projection', '13 algorithms, equal budget', '(2,000 evaluations, 25 runs)', 'Mechanism control, timing'], hl=True)
stage(xs[2], yb, 4, 'Uncertainty, identifiability', ['Error model from sweeps', 'Calibrated profile likelihood', 'Closed-form band criterion', 'Band-design simulation'], hl=True)
stage(xs[1], yb, 5, 'Reproducibility', ['Between sweeps: drift', 'Between recordings: 35', 'single-frequency recordings', 'Per animal'])
stage(xs[0], yb, 6, 'Reporting', ['Decision rule (Fig. 5)', 'Calibrated intervals,', 'reproducibility limits', 'Open-source eisvarpro'])
arr(32, 34.7, 35, 34.7); arr(65, 34.7, 68, 34.7); arr(83, 25.5, 83, 20); arr(68, 10.7, 65, 10.7); arr(35, 10.7, 32, 10.7)
out = Path(__file__).parent.parent/'results'/'figures'/'Fig01_Methods_v3.png'; fig.savefig(out, dpi=400); print(out)


In [ ]:
%%writefile src/make_fig_decision_v3.py
"""Reporting decision rule (manuscript Fig. 5, round 3): thresholds identical to the Methods; every reported
parameter must itself be practically identifiable with a calibrated interval."""
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon
from pathlib import Path
plt.rcParams.update({'font.family': 'Liberation Sans', 'mathtext.fontset': 'custom', 'mathtext.rm': 'Liberation Sans', 'mathtext.it': 'Liberation Sans:italic', 'mathtext.bf': 'Liberation Sans:bold', 'font.size': 8.5})
fig = plt.figure(figsize=(7.1, 5.6)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 79); ax.axis('off')
def box(x, y, w, h, txt, fc='white', ec='black', bold=False, fs=8.4):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.2', fc=fc, ec=ec, lw=1.1))
    ax.text(x+w/2, y+h/2, txt, ha='center', va='center', fontsize=fs, fontweight='bold' if bold else 'normal', linespacing=1.35)
def diamond(xc, yc, w, h, txt, fs=8.0):
    ax.add_patch(Polygon([(xc, yc+h/2), (xc+w/2, yc), (xc, yc-h/2), (xc-w/2, yc)], closed=True, fc='#f7f7f7', ec='black', lw=1.1))
    ax.text(xc, yc, txt, ha='center', va='center', fontsize=fs, linespacing=1.3)
def arr(x1, y1, x2, y2, lab=None, lx=0, ly=0):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='-|>', lw=1.0, color='black', mutation_scale=10))
    if lab: ax.text((x1+x2)/2+lx, (y1+y2)/2+ly, lab, fontsize=8, fontstyle='italic')
B, BE = '#e8f1fb', '#2166ac'; Rf, Re = '#fdecea', '#b2182b'
box(28, 72, 44, 6, 'Measured spectrum $Z(f)$ on band $[f_{\\min}, f_{\\max}]$', bold=True)
arr(50, 72, 50, 69)
diamond(50, 64.5, 34, 9, 'Linear Kramers–Kronig\nRMS residual < 3%?')
arr(67, 64.5, 73, 64.5, 'no', -3.2, 1)
box(73, 60, 26, 9, 'Flag: re-measure, or report\nas KK-inconsistent with\nreproducibility limits', fc=Rf, ec=Re)
arr(50, 60, 50, 56.6, 'yes', 1, 0)
box(16, 49, 68, 7.6, 'Fit Cole+L and R–CPE–L by variable projection with multistart refinement;\ncalibrate profile-likelihood thresholds to the error model of the instrument')
arr(50, 49, 50, 45.6)
diamond(50, 38.5, 72, 15, '$f_{\\min} \\leq f_c$ and, for every Cole parameter\nto be reported, a calibrated 95% interval\nthat is closed and < 1 decade (< 0.2 for $\\alpha$)?', fs=8.0)
ax.plot([14, 12], [38.5, 38.5], color='black', lw=1.0); arr(12, 38.5, 12, 30.6); ax.text(9, 39.8, 'yes', fontsize=8, fontstyle='italic')
box(1, 17, 25, 13.6, 'Report those Cole\nparameters with calibrated\nintervals; leave out any\nthat fail the criterion', fc=B, ec=BE)
ax.plot([86, 88], [38.5, 38.5], color='black', lw=1.0); arr(88, 38.5, 88, 30.6); ax.text(88.5, 39.8, 'no', fontsize=8, fontstyle='italic')
box(71, 17, 28, 13.6, 'Report $R_\\infty$, $Q$, $\\alpha$ that meet\nthe same criterion (band-\nspecific; state band, HF\ncorrection and probe); do not\ninterpret $\\tau$, $\\Delta R$, $R_0$, $f_c$', fc=B, ec=BE)
box(29, 17, 39, 13.6, 'With repeated sweeps or recordings,\nadd the reproducibility half-width\nor limit; detection limit $f_{\\min}/\\Gamma$ and\nEq. (10) state which dispersions\nthe band could resolve', fc='white', ec='0.4')
arr(26, 23.8, 29, 23.8); arr(71, 23.8, 68, 23.8)
box(24, 3, 52, 9.6, 'To estimate Cole parameters: extend the band to $f_{\\min} \\leq f_c$ of the\ntarget tissue, control electrode polarization (e.g. tetrapolar)\nand repeat the assessment', fc='#fff7e6', ec='#b35806')
ax.plot([85, 85], [17, 7.8], color='black', lw=1.0, ls=(0, (3, 2))); arr(85, 7.8, 76, 7.8)
out = Path(__file__).parent.parent/'results'/'figures'/'Fig18_reporting_decision_v3.png'; fig.savefig(out, dpi=400); print(out)


In [ ]:
%%time
# raw sweeps are needed for F1 and F7
import zipfile, os
if not os.path.exists('raw/PreparedData'): zipfile.ZipFile('PreparedData.zip').extractall('raw')
!pip -q install scikit-learn 2>/dev/null
!cd src/review && python e1_error_model.py ../../raw/PreparedData && python e1b_bias.py && python e2_hf.py
!cd src/review && python e1_calibrate.py M11 && python e1_calibrate.py M10 && python e1_calibrate.py DB
!cd src/review && python e3_model_selection.py && python e4_thresholds.py && python e7_sweep_repeat.py ../../raw/PreparedData
!cd src/review && python e5_band_factorial.py 20 && python e6_mechanism.py && python r2_figs.py
!cd src/review && python e8_errcheck.py ../../raw/PreparedData && python e9_summary_ci.py
!cd src/review && python e10_asymptotic.py && python e10b_refined.py && python e10_fig.py && python e11_drift.py ../../raw/PreparedData && python e11b_drift_ci.py
!cd src/review && python e12_placement.py ../../raw/PreparedData && python make_supp_e12.py ../../results/supp_e12.tex
!cd src/review && python e13_stats.py && python make_supp_e13.py ../../results/supp_e13.tex
!cd src/review && python e14_kk_primary.py ../../raw/PreparedData && python make_supp_e14.py ../../results/supp_e14.tex
!cd src && python make_fig1_workflow_v3.py && python make_fig_decision_v3.py
!cd src && python make_fig_decision.py && python make_fig1_workflow.py
# optional (about 1 h per model on one core): full Table 2 benchmark rerun in this environment, then same-machine timing
# !cd src && python rerun_same_env.py M10_ColeL && python rerun_same_env.py M11_RCPEL && python rerun_same_env.py M12_DoubleColeL && python time_fair.py 25

---
# Export
Collects all tables, figures and the environment description into `EIS_VarPro_results_<date>.zip`.

In [ ]:
import shutil, datetime, json, platform, numpy, scipy, pandas
json.dump(dict(python=sys.version.split()[0], numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__,
               platform=platform.platform(), processor=platform.processor()), open('results/env_info.json', 'w'), indent=2)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M'); out = f'EIS_VarPro_results_{stamp}'
os.makedirs(out, exist_ok=True)
for sub in ['tables', 'figures', 'figures_vp']:
    if os.path.exists(f'results/{sub}'): shutil.copytree(f'results/{sub}', f'{out}/{sub}', dirs_exist_ok=True)
for f in ['results/env_info.json', 'results/reproducibility_manifest.json']:
    if os.path.exists(f): shutil.copy(f, out)
zp = shutil.make_archive(out, 'zip', out); print(zp)
if IN_COLAB:
    from google.colab import files; files.download(zp)